# Week 1: pricing in the developed market

I am Manuel Rodríguez, an actuarial student and individual participant in the IAA AI Hackathon 2026. In this notebook I explain and reconstruct my Week 1 risk-premium model, developed with AI assistance.

I estimate the expected ultimate claim cost for each 2025 policy period in motor, buildings and contents. The target is net of excess and recoveries and excludes declined claims. It is a period risk premium, not a commercial premium. Claims valued at the end of 2024 do not reveal the full ultimate cost, so the development assumptions and their sensitivity matter.


## Notebook guide

| Purpose | What I reconstruct |
| --- | --- |
| Selected recipe | S41, my Week 1 submission |
| Required inputs | 11 developed-market Parquet files from Part 1 |
| Final exports | `submission_week1_S41.csv` |
| External model or repository download | None; source and frozen configuration are embedded |

For a first reading, I follow the [method](#approach-and-limitations), [execution plan](#inspect-the-dependency-graph) and [output checks](#apply-the-s41-level-factors-and-check-the-output). The [embedded source package](#embedded-source-package) preserves the individual programs for inspection. Its source cells are marked as hidden in JupyterLab's presentation metadata; I can expand them. GitHub may display them in full.

For execution, I attach the data, check [dependencies and storage](#dependencies-and-workspace) and run cells in order. The later [recipe execution](#materialize-and-run-the-frozen-recipe) trains models and can take hours. Reading the notebook does not require starting that execution.

My resumed local reconstruction recovered the historical S41 CSV. It also reported 160 intermediate differences, including three unexplained feature-table fingerprints. I have not established a hosted Kaggle execution or an uninterrupted Run All of this adapted notebook. AI-assisted reconstruction is distinct from independent professional review.


## Scope and reproducibility

I rebuild the historical S30 models and apply the S41 public-probe level factors from the official original Parquet files. All source code, parameters and aggregate verification hashes used below are embedded in this notebook; no team files, models or predictions are required. The complete adapted notebook has not yet been verified end to end on Kaggle.

The original F0 construction used restricted row-level decision ledgers for two audit-only capital flags. Both are excluded from every model. I reconstruct CAP01 directly from the original data and its frozen rule. For the other flag, I use a rule inferred after inspecting the historical matrix: a normal period in the same policy and, for buildings, the same address. The rule exactly matched the historical flag on this data; it is a retrospective reconstruction, not the recovered decision process or proof that the rule generalizes. I verify the complete F0 content hashes before training and check the final CSV separately.

Attach the Part 1 competition files and run in order. I keep intermediate data, models and logs in a fresh temporary directory outside `/kaggle/working`; only the final CSV goes into `/kaggle/working`. The temporary directory does not persist with the notebook output. Input files remain read-only, and the row-level final CSV must remain private under the competition rules.


## Attach the competition data

In Kaggle, add the IAA AI-Assisted Hackathon 2026 Part 1 competition as an input. Its archived file listing places the files in `public/`. The search below also supports Kaggle's competition mount prefix. If more than one matching folder is attached, set `DEVELOPED_DIR` to its exact path. Local testing may set `IAA_W1_DATA` to the directory containing the Parquet files.


## Optional dependency installation

Kaggle usually includes these libraries. If the environment does not match the recorded reference, set the switch to `True` and run this cell with Internet enabled. Restart the kernel after installation, set the switch back to `False`, and run from the top. Keep the same Kaggle session so the installed packages remain available. The reference versions may not support every Kaggle Python image.


In [ ]:
INSTALL_REFERENCE_PACKAGES = False
if INSTALL_REFERENCE_PACKAGES:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "numpy==2.5.3", "pandas==3.0.5", "scipy==1.18.1", "scikit-learn==1.9.1", "lightgbm==4.7.0", "catboost==1.2.10", "pyarrow==26.0.0"])
    raise RuntimeError("Packages installed. Restart the kernel, turn installation off, and run all cells again.")


In [ ]:
import os, sys, hashlib, importlib.metadata
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

for package in ("numpy", "pandas", "pyarrow"):
    print(f"{package}: {importlib.metadata.version(package)}")
print("Python:", sys.version)

DEVELOPED_DIR = os.environ.get("IAA_W1_DATA", "")  # e.g. /kaggle/input/competitions/iaa-ai-assisted-hackathon-2026-part-1/public


In [ ]:
LINES = ("motor", "buildings", "contents")
BASE_FILES = ["addresses.parquet", "claims.parquet", "customers.parquet",
              "weather.parquet", "sample_submission.parquet"] + [
    f"{prefix}_{line}.parquet" for prefix in ("policies", "test") for line in LINES]

def find_market(market, required, override):
    if override:
        candidates = [Path(override)]
    else:
        candidates = [p for p in Path("/kaggle/input").rglob(market) if p.is_dir()]
    candidates = [p for p in candidates if all((p / name).is_file() for name in required)]
    if len(candidates) != 1:
        raise RuntimeError(f"Expected exactly one {market} folder with the required files; "
                           f"found {candidates}. Set the override above explicitly.")
    return candidates[0]

def inventory(folder, filenames):
    rows = []
    for name in filenames:
        path = folder / name
        pf = pq.ParquetFile(path)
        digest = hashlib.sha256()
        with path.open("rb") as stream:
            for chunk in iter(lambda: stream.read(1024 * 1024), b""):
                digest.update(chunk)
        rows.append({"file": name, "rows": pf.metadata.num_rows,
                     "columns": len(pf.schema_arrow), "bytes": path.stat().st_size,
                     "sha256": digest.hexdigest()})
    return pd.DataFrame(rows)

def check_template(folder):
    sample = pd.read_parquet(folder / "sample_submission.parquet")
    test = pd.concat([pd.read_parquet(folder / f"test_{line}.parquet", columns=["row_id"])
                      for line in LINES], ignore_index=True)
    assert "row_id" in sample.columns and sample.row_id.notna().all()
    assert test.row_id.notna().all() and test.row_id.is_unique and sample.row_id.is_unique
    assert len(sample) == len(test) and sample.row_id.equals(test.row_id), "Template and test IDs/order differ"
    print(f"Template matches the concatenated test IDs in order: {len(test):,} rows")
    return sample


In [ ]:
developed = find_market("public", BASE_FILES, DEVELOPED_DIR)
print("Developed data:", developed)
display(inventory(developed, BASE_FILES))
sample = check_template(developed)


## Approach and limitations

I assign claims using policy, line and occurrence date, then build observed net costs and reporting-development adjustments. S30 combines total-cost Tweedie models with peril frequency times severity; its level uses historical temporal cuts. S41 applies three frozen credibility-weighted factors obtained from public leaderboard probes. Since the public leaderboard informed those factors, it is not an independent test of their level.

Case reserves and unreported claims remain uncertain. The historical temporal cuts informed model decisions; feature availability at quote time still needs confirmation. Claude developed the main modeling line, while Codex contributed data rules, reconstruction and checks under my direction. This is self-reproduction, not independent actuarial review.

The embedded source retains the historical programs and their stage boundaries. Each source is written into a fresh working directory at runtime; the notebook does not fetch a repository or execute a team file.


## From claims to a risk premium

Each prediction is the expected ultimate cost for one policy period, in the competition's fictional currency. It is already a period amount. The target is not the insurer's written premium and it should not be multiplied by exposure a second time when interpreting the output.

I first join each claim to a period using its line, policy identifier and occurrence date. A policy can have more than one period, so an identifier-only join can duplicate or misassign cost. I exclude declined claims and use incurred cost less recoveries for accepted, uniquely assigned claims. The valuation is a snapshot at the end of 2024: open claims may develop further, and some claims have not yet been reported.

I estimate reporting completeness by peril from a count development model. For training, the recipe reduces a period's effective exposure according to that completeness. It does not also inflate the observed label for the same unreported claims. This offset is a modeling assumption, not proof that case reserves are sufficient or that the development pattern will remain stable.

## Validation and information timing

The 2023 cut trains on earlier years and separates customers for calibration and selection. The 2024 cut trains on 2021–2023 and evaluates the next year. The 2025 fit uses the permitted 2021–2024 history. These historical cuts helped choose the model, so I do not describe them as untouched final tests.

The feature contract distinguishes policy and property attributes, prior claims history, prior-year weather, conditional customer attributes and existing pricing information. It excludes the current-period service-contact and renewal outcome fields and the audit flags from predictors. History features are cut at the start of the policy year. Whether every retained field is actually available at a real quote remains an open question.

## Models, aggregation and calibration

I combine total-cost Tweedie models with peril-specific frequency multiplied by severity. In motor the final weights are 40% total cost and 60% frequency–severity; the same split applies to buildings. In contents the split is 30% and 70%. Component predictions are normalized to a line-level reference before blending. The recipe applies the historical shape and exposure adjustments and then calibrates the 2025 level from the two historical cuts.

For Tweedie power 1.5, a constant multiplier that minimizes deviance over a fixed set of positive predictions is `sum(y / sqrt(mu)) / sum(sqrt(mu))`. S41 adds one fixed multiplier per line. Those factors were derived from six public leaderboard probes and a credibility rule specified before their scores were observed. They use information from the public part of 2025, so I do not treat that score as a blind evaluation. The exact factors, rather than their rounded presentation values, are embedded in the final cell.

## How to read the run

The source cells below preserve the original program structure, including the data producer, actuarial bases, feature contract, model stages and blend. The execution cell materializes them inside a fresh run folder and saves logs and a receipt for each node. If a stage fails, inspect its log before rerunning. Code 3 reports a matching final S30 CSV with intermediate content differences; it is not a clean match for every artifact. A different final CSV hash stops the run. The final check also enforces the template's identifiers, their order and positive finite predictions.


## Interpreting the metric

For nonnegative observed cost `y` and positive prediction `mu`, Tweedie deviance at power 1.5 is `4 * (y / sqrt(mu) + sqrt(mu) - 2 * sqrt(y))`. The local D² is one minus the total model deviance divided by the deviance of the constant mean prediction. Higher is better; a negative value means worse than that constant on the evaluated rows. The observed target is still a valuation proxy for ultimate cost. I do not claim this local calculation proves every detail of Kaggle's unpublished evaluator.

A score measures one aspect of the exercise. I also examine portfolio slices, large claims, reserve assumptions and whether the input fields exist at quote time. None of those checks turns an adaptive development cut into an independent professional review.


In [ ]:
from sklearn.metrics import mean_tweedie_deviance, d2_tweedie_score
# A small synthetic calculation, including a zero-cost observation.
y_demo = np.array([0.0, 0.25, 2.0, 100.0])
mu_demo = np.array([0.1, 0.3, 1.5, 25.0])
d_demo = 4 * (y_demo / np.sqrt(mu_demo) + np.sqrt(mu_demo) - 2 * np.sqrt(y_demo))
null_demo = 4 * (y_demo / np.sqrt(y_demo.mean()) + np.sqrt(y_demo.mean()) - 2 * np.sqrt(y_demo))
assert np.isclose(d_demo.mean(), mean_tweedie_deviance(y_demo, mu_demo, power=1.5), rtol=1e-13)
assert np.isclose(1 - d_demo.sum() / null_demo.sum(), d2_tweedie_score(y_demo, mu_demo, power=1.5), rtol=1e-13)
print("The formula agrees with the local scikit-learn calculation.")


## Dependencies and workspace

The historical reference used Python 3.13.16, NumPy 2.5.3, pandas 3.0.5, SciPy 1.18.1, scikit-learn 1.9.1, LightGBM 4.7.0, CatBoost 1.2.10 and PyArrow 26.0.0. Check the installed versions below; if needed install compatible versions separately and restart the kernel. Installation may require Internet.

The historical 36-node reconstruction, which also included the S22 backup, recorded about 10.6 hours on two cores. This notebook runs the 30-node S30 dependency graph; its time on hosted Kaggle has not been measured. [Kaggle's notebook documentation](https://www.kaggle.com/docs/notebooks), consulted on 28 October 2026, describes a 12-hour CPU session and 20 GB of saved working storage. Execution speed and disk use vary, so the available margin is not guaranteed. A hosted end-to-end run is still needed to establish whether this recipe finishes there.


In [ ]:
import json, shutil, subprocess, time, uuid
from datetime import datetime, timezone
from importlib import metadata
REFERENCE = {"numpy": "2.5.3", "pandas": "3.0.5", "scipy": "1.18.1", "scikit-learn": "1.9.1", "lightgbm": "4.7.0", "catboost": "1.2.10", "pyarrow": "26.0.0"}
installed = {name: metadata.version(name) for name in REFERENCE}
print("Installed:", installed, "Reference:", REFERENCE)
if installed != REFERENCE: print("Environment differs from the reference; verify output identity.")
import tempfile
RUN_ROOT = Path(os.environ.get("IAA_NOTEBOOK_WORK", tempfile.gettempdir()))
RUN = RUN_ROOT / ("iaa_week1_" + uuid.uuid4().hex[:12])
RUN.mkdir(parents=True, exist_ok=False)
EXPORT_ROOT = Path(os.environ.get("IAA_NOTEBOOK_EXPORT", "/kaggle/working" if Path("/kaggle/working").is_dir() else str(RUN)))
EXPORT = EXPORT_ROOT / (RUN.name + "_exports")
EXPORT.mkdir(parents=True, exist_ok=False)
print("Final outputs:", EXPORT)
PACKAGE = RUN / "embedded_code"
PACKAGE.mkdir()
SOURCES = {}
print("New run:", RUN)


## Embedded source package

I keep the source files below as text assignments. Running these cells registers the code in memory; the later execution section writes it into the new workspace and starts the recipe. I retain the original program boundaries, authorship and frozen configuration. No past row-level predictions or fitted models are embedded.


### Embedded source: `reproducir.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['reproducir.py'] = '''"""Punto de entrada único: reconstruye S30_claude_b (candidato principal) y S22_claude_propio (respaldo).

Versión 1.1 (19 de octubre de 2026): controles de conformidad del hallazgo GOV-007. El paquete solo termina con
código 0 cuando todo lo que se le pidió coincide con la receta; reanudar, verificar y las etapas de datos fallan
ante cualquier artefacto requerido que falte, no se pueda leer, no tenga huella esperada o difiera de ella.

Modos
  plan         Muestra el orden de los nodos de la receta y la orden exacta de cada uno, sin ejecutar nada.
  mezcla       Repite las mezclas finales a partir de los componentes guardados por el equipo (minutos). Antes de
               mezclar comprueba la huella de contenido de cada componente enlazado; al terminar, el SHA-256 de cada
               CSV con el entregado a Kaggle.
  componentes  Reconstruye desde las matrices F0 y la asignación de siniestros: variables, selecciones V23,
               componentes de producción, partes por peligro, severidades, línea B y mezclas (unas diez horas con dos
               núcleos). Compara cada artefacto con su huella esperada al terminar el nodo y se detiene si no coincide.
  verificar    Compara lo que haya en --trabajo con receta/huellas_esperadas.json, sin ejecutar nada; termina con
               error si algo falta, no se puede leer, no tiene huella esperada o difiere.
  datos        Etapas 00 a 02 desde los once Parquet originales: capa preparada (productor N62 de Codex), bases
               actuariales y matriz F0, en una raíz de proyecto mínima dentro de --trabajo (minutos). Cada paso se
               compara con su huella esperada y el modo falla si alguno no coincide.
  todo         datos y después componentes sobre lo que datos acaba de producir: de los originales a los dos CSV.

Opciones
  --semana1=<carpeta>   Carpeta «Hackaton/Semana 1» con Datos/, Matriz_F0_claude/runs/<F0>/ y
                        Bases_actuariales_claude/runs/<bases>/ (las entradas de la receta). En modo mezcla también
                        Modelos_claude/ y Modelos_claude_b/ con las corridas guardadas. Por defecto, IAA_SEMANA1.
  --trabajo=<carpeta>   Carpeta nueva donde se escriben work/, registros/ y registro.jsonl (por defecto ./trabajo).
  --objetivo=S30,S22    Objetivos a construir (por defecto los dos).
  --artefactos=<c>[,c]  Solo en modo mezcla: carpetas adicionales donde buscar las corridas guardadas.
  --solo=<nodo>[,...]   Ejecuta solo esos nodos (con sus salidas ya presentes para los que requieren). El resultado
                        final solo cubre los objetivos cuyo nodo esté entre los ejecutados.
  --originales=<c>      Modos datos y todo: carpeta con los once Parquet originales (por defecto <semana1>/Datos).
  --decisiones=<c>      Modos datos y todo: carpeta «Semana 1» del proyecto de donde se leen, con su SHA-256, las tres
                        listas congeladas que alimentan las marcas audit_* de F0 (por defecto --semana1).
  --continuar-si-difiere
                        No detenerse en el primer artefacto cuyo contenido difiere de la huella esperada: la
                        reconstrucción sigue, anota cada diferencia y termina con código 3 aunque los CSV finales
                        coincidan. Nunca convierte una diferencia en éxito. Un artefacto que falta, no se puede leer o
                        no tiene huella esperada detiene la reconstrucción también con esta opción.
  --relanzar-parciales  Si un nodo tiene solo parte de sus salidas (por ejemplo, porque el proceso murió a mitad),
                        aparta las existentes con el sufijo .parcial_<utc> y vuelve a ejecutar el nodo. Sin esta
                        opción el nodo se detiene y hay que revisar la carpeta de trabajo.

Reanudación. Si todas las salidas de un nodo existen ya en work/, el nodo se salta solo después de comprobar que
cada artefacto que la receta manda verificar coincide con su huella esperada (contenido; bytes para los CSV
finales). Cada nodo deja su salida estándar en registros/<nodo>.log y una línea en registro.jsonl con la orden, el
tiempo y las huellas obtenidas. Los scripts se ejecutan sin modificar, con las variables de entorno que esperan
(HAKATON_SEMANA1, MODELOS_WORK y MODELOS_CLAUDE).

Códigos de salida
  0  todo lo pedido coincide con la receta (contenido igual en los artefactos requeridos; SHA-256 exacto en los CSV
     finales construidos).
  1  detenido por un artefacto requerido que falta, no se puede leer, no tiene huella esperada o difiere; por un
     nodo que terminó con error; por entradas, originales, capa, bases, F0 o decisiones que no coinciden; o por un
     CSV final con SHA-256 distinto del entregado.
  2  uso incorrecto de la orden.
  3  terminó con --continuar-si-difiere y hubo diferencias de contenido en artefactos intermedios, aunque los CSV
     finales coincidan: reconstrucción documentada, no acreditada.
"""
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

VERSION = "1.2"
VERSION_FECHA = "2026-10-19"

REPO = Path(__file__).resolve().parent
sys.path.insert(0, str(REPO / "herramientas"))
from huella import huellas  # noqa: E402

RECETA = json.loads((REPO / "receta" / "receta.json").read_text(encoding="utf-8"))
NODOS = {n["id"]: n for n in RECETA["nodos"]}
ORDEN = [n["id"] for n in RECETA["nodos"]]
CSV_FINALES = {v["csv"] for v in RECETA["objetivos"].values()}


class NoConforme(SystemExit):
    """Detención por un artefacto o una entrada que no cumple el contrato de la receta (código 1)."""

    def __init__(self, mensaje: str):
        super().__init__(f"NO CONFORME: {mensaje}")


def opciones(argv):
    o = {}
    for a in argv:
        if a.startswith("--"):
            k, _, v = a[2:].partition("=")
            o[k] = v if v else True
    return o


def cierre(objetivos, solo=None):
    """Nodos necesarios para los objetivos, en el orden de la receta."""
    if solo:
        desconocidos = sorted(set(solo) - set(ORDEN))
        if desconocidos:
            raise SystemExit(f"--solo nombra nodos que no están en la receta: {desconocidos}")
        return [i for i in ORDEN if i in solo]
    raices = [RECETA["objetivos"][o]["nodo"] for o in objetivos]
    necesarios, pila = set(), list(raices)
    while pila:
        i = pila.pop()
        if i in necesarios:
            continue
        necesarios.add(i)
        pila += NODOS[i]["requiere"]
    return [i for i in ORDEN if i in necesarios]


def orden_de(nodo):
    args = [str(REPO / a) if a.startswith("receta/") else a for a in nodo["args"]]
    # Los dos escritores finales se ejecutan con serialización LF, como en la nube.
    # No normalizamos una salida después de verificarla ni relajamos su SHA exigido.
    if nodo["script"] in {"src/modelos/mezcla3.py", "src/linea_b/mezcla3_b.py"}:
        return [sys.executable, "-W", "ignore", str(REPO / "herramientas" / "ejecutar_csv_lf.py"), str(REPO / nodo["script"])] + args
    return [sys.executable, "-W", "ignore", str(REPO / nodo["script"])] + args


def entorno(semana1: Path, work: Path):
    env = dict(os.environ)
    env.update({"HAKATON_SEMANA1": str(semana1), "MODELOS_WORK": str(work), "MODELOS_CLAUDE": str(REPO / "src" / "modelos"),
                "PYTHONHASHSEED": "0"})
    return env


def esperadas():
    """Huellas esperadas de la receta. Sin este archivo no hay contrato, así que el paquete no puede continuar."""
    p = REPO / "receta" / "huellas_esperadas.json"
    if not p.is_file():
        raise NoConforme(f"falta {p.relative_to(REPO)}; sin huellas esperadas no se puede verificar nada")
    esp = json.loads(p.read_text(encoding="utf-8"))
    for clave in ("entradas", "salidas"):
        if not isinstance(esp.get(clave), dict) or not esp[clave]:
            raise NoConforme(f"receta/huellas_esperadas.json no tiene la sección «{clave}» o está vacía")
    return esp


def comparar(rel, work: Path, esp: dict, seccion: str = "salidas", clave: str | None = None):
    """Compara work/<rel> con la huella esperada esp[seccion][clave] (clave = rel salvo que se indique otra).
    Nunca lanza: deja el resultado en el diccionario."""
    p = work / rel
    e = esp.get(seccion, {}).get(clave or rel)
    r = {"ruta": rel, "existe": p.is_file(), "huella_esperada": isinstance(e, dict) and bool(e.get("contenido"))}
    if not p.is_file():
        return r
    try:
        h = huellas(p)
    except Exception as exc:  # archivo truncado, corrupto o de un formato que no se puede leer
        r.update({"legible": False, "error": f"{type(exc).__name__}: {exc}"})
        return r
    r.update({"legible": True, "sha256": h["sha256"], "contenido": h["contenido"]})
    if r["huella_esperada"]:
        r.update({"igual_bytes": h["sha256"] == e.get("sha256"), "igual_contenido": h["contenido"] == e["contenido"]})
    return r


def no_conformidades(filas, tolerar_diferencias=False):
    """Devuelve (bloqueantes, diferencias). Las bloqueantes detienen siempre; las diferencias de contenido solo
    pasan a la segunda lista cuando se pidió continuar. Los CSV finales exigen además SHA-256 exacto."""
    bloqueantes, diferencias = [], []
    for f in filas:
        if not f.get("existe"):
            bloqueantes.append((f["ruta"], "falta"))
        elif not f.get("legible", True):
            bloqueantes.append((f["ruta"], f"no se puede leer: {f.get('error')}"))
        elif not f.get("huella_esperada"):
            bloqueantes.append((f["ruta"], "sin huella esperada en la receta"))
        elif f.get("igual_contenido") is not True or (f["ruta"] in CSV_FINALES and f.get("igual_bytes") is not True):
            motivo = "contenido distinto" if f.get("igual_contenido") is not True else "CSV final con SHA-256 distinto"
            (diferencias if tolerar_diferencias else bloqueantes).append((f["ruta"], motivo))
    return bloqueantes, diferencias


def exigir_artefactos(filas, contexto: str, tolerar_diferencias=False):
    """Detiene el paquete si un artefacto requerido no cumple; devuelve las diferencias toleradas (lista vacía
    cuando todo coincide)."""
    if not filas:
        raise NoConforme(f"{contexto}: no hay artefactos requeridos que verificar")
    bloqueantes, diferencias = no_conformidades(filas, tolerar_diferencias)
    if bloqueantes:
        raise NoConforme(f"{contexto}: artefactos requeridos no conformes: {resumir(bloqueantes)}")
    return diferencias


def resumir(lista, maximo=8):
    """Los primeros elementos de una lista larga; el detalle completo queda en la salida JSON y en registro.jsonl."""
    return str(lista) if len(lista) <= maximo else f"{lista[:maximo]} y {len(lista) - maximo} más (detalle en la salida JSON y en registro.jsonl)"


def exigir_datos(informe):
    """Segunda barrera, en el punto de entrada, sobre el informe de las etapas de datos: cada control requerido del
    contrato «datos» de la receta tiene que estar presente y en verde."""
    contrato = esperadas().get("datos")
    if not contrato or not isinstance(informe, dict):
        raise NoConforme("datos: falta el contrato esperado de datos o el informe no es válido")
    fallos = []
    for nombre in contrato["capa"]:
        if informe.get("capa", {}).get(nombre) is not True:
            fallos.append("capa/" + nombre)
    for grupo in ("bases", "f0"):
        for nombre in contrato[grupo]:
            item = informe.get(grupo, {}).get(nombre)
            if not isinstance(item, dict) or item.get("igual_contenido") is not True:
                fallos.append(grupo + "/" + nombre)
    if informe.get("f0", {}).get("contrato_bloques_iguales") is not True:
        fallos.append("f0/contrato_bloques_iguales")
    if fallos:
        raise NoConforme(f"datos: controles requeridos no conformes: {fallos}")


def bloques_contrato(p: Path) -> str:
    """Huella de la asignación de bloques del contrato de F0, lo único que leen los modelos (comun.contract)."""
    import hashlib
    cols = json.loads(p.read_text(encoding="utf-8"))["columnas"]
    return hashlib.sha256(json.dumps({l: {c: v["bloque"] for c, v in cc.items()} for l, cc in cols.items()}, sort_keys=True).encode()).hexdigest()


def comprobar_entradas(semana1: Path, esp: dict):
    malas = []
    bloques_esperados = esp.get("datos", {}).get("f0_contrato_bloques")
    for rel, e in esp["entradas"].items():
        p = semana1 / rel
        if not p.is_file():
            malas.append((rel, "no existe"))
            continue
        try:
            if rel.endswith("contrato_disponibilidad.json"):
                # el contrato también anota el tipo Arrow de cada columna, que cambia con la versión de pandas
                if not bloques_esperados:
                    malas.append((rel, "la receta no tiene la huella esperada de los bloques (datos/f0_contrato_bloques)"))
                elif bloques_contrato(p) != bloques_esperados:
                    malas.append((rel, "bloques distintos"))
            elif not e.get("contenido"):
                malas.append((rel, "sin huella de contenido esperada"))
            elif huellas(p)["contenido"] != e["contenido"]:
                malas.append((rel, "contenido distinto"))
        except Exception as exc:
            malas.append((rel, f"no se puede leer: {type(exc).__name__}: {exc}"))
    return malas


def registrar(trabajo: Path, fila: dict):
    with open(trabajo / "registro.jsonl", "a", encoding="utf-8") as f:
        f.write(json.dumps(fila, ensure_ascii=False) + "\\n")


def ahora():
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


def apartar_parciales(nodo: dict, work: Path, trabajo: Path):
    """Aparta las salidas presentes de un nodo incompleto con el sufijo .parcial_<utc>; no borra nada."""
    marca = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    apartadas = []
    for s in nodo["salidas"]:
        p = work / s
        if p.exists() or p.is_symlink():
            destino = p.with_name(p.name + f".parcial_{marca}")
            p.rename(destino)
            apartadas.append({"de": s, "a": str(destino.relative_to(work))})
    registrar(trabajo, {"nodo": nodo["id"], "estado": "parciales_apartadas", "apartadas": apartadas, "utc": ahora()})
    print(f"[{nodo['id']}] salidas parciales apartadas: {[a['a'] for a in apartadas]}", flush=True)


def ejecutar(ids, semana1: Path, trabajo: Path, continuar=False, relanzar_parciales=False):
    """Ejecuta los nodos en orden. Devuelve la lista de diferencias toleradas (vacía si todo coincide)."""
    work = trabajo / "work"
    (trabajo / "registros").mkdir(parents=True, exist_ok=True)
    work.mkdir(parents=True, exist_ok=True)
    esp = esperadas()
    env = entorno(semana1, work)
    toleradas = []
    for i in ids:
        n = NODOS[i]
        presentes = [s for s in n["salidas"] if (work / s).is_file()]
        if len(presentes) == len(n["salidas"]):
            comp = [comparar(s, work, esp) for s in n["verificar"]]
            dif = exigir_artefactos(comp, f"reanudar {i}", continuar)
            estado = "reanudado_con_diferencias" if dif else "reanudado_verificado"
            registrar(trabajo, {"nodo": i, "estado": estado, "verificacion": comp, "utc": ahora()})
            print(f"[{i}] ya hecho, huellas requeridas verificadas" + (f"; difieren: {[d[0] for d in dif]}" if dif else ""), flush=True)
            toleradas += [(i, *d) for d in dif]
            continue
        if presentes or any((work / s).exists() or (work / s).is_symlink() for s in n["salidas"]):
            if not relanzar_parciales:
                raise NoConforme(f"{i}: salidas parciales en work/ ({presentes or 'rutas presentes sin archivo'}); "
                                 "revisar la carpeta de trabajo o relanzar con --relanzar-parciales")
            apartar_parciales(n, work, trabajo)
        cmd = orden_de(n)
        print(f"[{i}] {' '.join(cmd[3:])}", flush=True)
        t0 = time.time()
        with open(trabajo / "registros" / f"{i}.log", "a", encoding="utf-8") as log:
            log.write(f"# {ahora()} {' '.join(cmd)}\\n")
            log.flush()
            rc = subprocess.run(cmd, cwd=str(trabajo), env=env, stdout=log, stderr=subprocess.STDOUT).returncode  # cwd fuera del repositorio: CatBoost escribe catboost_info/ en el directorio actual
        seg = round(time.time() - t0, 1)
        comp = [comparar(s, work, esp) for s in n["verificar"]]
        bloq, dif = no_conformidades(comp, continuar)
        estado = "error" if rc else ("no_conforme" if bloq else ("difiere" if dif else "igual"))
        fila = {"nodo": i, "orden": cmd[2:], "rc": rc, "segundos": seg, "utc_fin": ahora(), "estado": estado, "verificacion": comp}
        registrar(trabajo, fila)
        print(f"[{i}] {estado} en {seg:.0f} s" + (f"; no conformes: {bloq}" if bloq else "") + (f"; difieren: {[d[0] for d in dif]}" if dif else ""), flush=True)
        if rc:
            raise SystemExit(f"El nodo {i} terminó con código {rc}; ver registros/{i}.log")
        exigir_artefactos(comp, f"salida {i}", continuar)
        toleradas += [(i, *d) for d in dif]
    return toleradas


def enlazar_archivo(src: Path, dst: Path):
    """Windows usa una copia privada: no requiere privilegios ni cambia el origen.
    Las comprobaciones posteriores exigen las mismas huellas que para un enlace.
    """
    if os.name == "nt":
        shutil.copy2(src, dst)
    else:
        dst.symlink_to(src)


def preparar_mezcla(ids, semana1: Path, trabajo: Path, extra: list[Path], esp: dict):
    """Enlaza en work/ las salidas guardadas de los nodos previos a las mezclas y las exposiciones, y comprueba la
    huella de contenido de cada archivo enlazado. Devuelve (faltan, no_conformes)."""
    work = trabajo / "work"
    work.mkdir(parents=True, exist_ok=True)
    origenes = [semana1 / "Modelos_claude", semana1 / "Modelos_claude_b"] + extra
    sys.path.insert(0, str(REPO / "herramientas"))
    from inventario import localizar
    faltan, comp = [], []
    directos = sorted({r for i in ids if NODOS[i]["etapa"] == "mezcla" for r in NODOS[i]["requiere"]})
    for i in directos:
        n = NODOS[i]
        for rel in n["verificar"]:
            if not rel.endswith(".parquet"):
                continue  # la mezcla solo lee v24.parquet, prediccion_2025.parquet y sel_<nombre>.parquet
            dst = work / rel
            if not dst.exists():
                src = localizar(rel, origenes)
                if src is None:
                    faltan.append(rel)
                    continue
                dst.parent.mkdir(parents=True, exist_ok=True)
                enlazar_archivo(src, dst)
            comp.append(comparar(rel, work, esp))
    for line in ("motor", "buildings", "contents"):
        rel = f"feat_{line}.parquet"
        dst = work / rel
        if not dst.exists():
            src = next((o / "insumos" / f"exposicion_{line}.parquet" for o in origenes if (o / "insumos" / f"exposicion_{line}.parquet").is_file()), None)
            src = src or next((o / rel for o in origenes if (o / rel).is_file()), None)
            if src is None:
                faltan.append(f"{rel} (o insumos/exposicion_{line}.parquet)")
                continue
            enlazar_archivo(src, dst)
        # la exposición puede venir de la matriz de variables completa (huella en «salidas») o del extracto
        # insumos/exposicion_<ramo>.parquet que guarda el equipo (huella en «mezcla»)
        # Acepta exactamente uno de los dos insumos permitidos por el contrato.
        # Una copia Windows no conserva el nombre del origen al resolver realpath.
        completo = comparar(rel, work, esp)
        extracto = comparar(rel, work, esp, "mezcla", clave=f"insumos/exposicion_{line}.parquet")
        if completo.get("igual_contenido") is True:
            completo["contrato"] = "salidas"
            comp.append(completo)
        else:
            extracto["contrato"] = "mezcla"
            comp.append(extracto)
    bloq, _ = no_conformidades(comp, tolerar_diferencias=False)
    registrar(trabajo, {"modo": "mezcla", "estado": "componentes_enlazados", "faltan": faltan, "no_conformes": bloq, "verificacion": comp, "utc": ahora()})
    return faltan, bloq


def main():
    if len(sys.argv) < 2 or sys.argv[1] not in ("plan", "mezcla", "componentes", "verificar", "datos", "todo"):
        print(__doc__)
        raise SystemExit(2)
    modo, o = sys.argv[1], opciones(sys.argv[2:])
    print(f"reproducir.py versión {VERSION} ({VERSION_FECHA}), modo {modo}", flush=True)
    if o.get("parar-si-difiere"):
        print("--parar-si-difiere ya es el comportamiento por defecto desde la versión 1.1; se ignora", flush=True)
    continuar = bool(o.get("continuar-si-difiere"))
    objetivos = [x for x in str(o.get("objetivo", "S30,S22")).split(",") if x]
    desconocidos = [x for x in objetivos if x not in RECETA["objetivos"]]
    if desconocidos:
        raise SystemExit(f"Objetivos desconocidos: {desconocidos}; la receta define {sorted(RECETA['objetivos'])}")
    semana1 = Path(o.get("semana1") or os.environ.get("IAA_SEMANA1", "")).expanduser().resolve()
    trabajo = Path(o.get("trabajo", "trabajo")).expanduser().resolve()
    solo = set(str(o["solo"]).split(",")) if o.get("solo") else None
    ids = cierre(objetivos, solo)
    if modo == "plan":
        for i in ids:
            n = NODOS[i]
            print(f"{i:30s} [{n['etapa']}, {n['linea']}, ~{n['minutos_ref']} min] {' '.join(orden_de(n)[3:])}")
        print(f"{len(ids)} nodos; tiempo de referencia ~{sum(NODOS[i]['minutos_ref'] or 0 for i in ids) / 60:.1f} h con dos núcleos")
        return
    if modo == "verificar":
        esp = esperadas()
        work = trabajo / "work"
        filas = [comparar(rel, work, esp) for i in ids for rel in NODOS[i]["verificar"]]
        bloq, _ = no_conformidades(filas, tolerar_diferencias=False)
        resumen = {"version": VERSION, "comparados": len(filas), "iguales_contenido": sum(f.get("igual_contenido") is True for f in filas),
                   "iguales_bytes": sum(f.get("igual_bytes") is True for f in filas),
                   "difieren": [f["ruta"] for f in filas if f.get("existe") and f.get("legible", True) and f.get("igual_contenido") is False],
                   "faltan": [f["ruta"] for f in filas if not f.get("existe")],
                   "ilegibles": [f["ruta"] for f in filas if f.get("existe") and not f.get("legible", True)],
                   "sin_huella_esperada": [f["ruta"] for f in filas if f.get("existe") and not f.get("huella_esperada")],
                   "conforme": not bloq}
        print(json.dumps(resumen, ensure_ascii=False, indent=1))
        exigir_artefactos(filas, "verificar")
        return
    if not semana1.is_dir():
        raise SystemExit("Falta --semana1=<carpeta Hackaton/Semana 1> (o la variable IAA_SEMANA1)")
    trabajo.mkdir(parents=True, exist_ok=True)
    registrar(trabajo, {"modo": modo, "estado": "inicio", "version": VERSION, "version_fecha": VERSION_FECHA, "orden": sys.argv[1:], "utc": ahora()})
    if modo in ("datos", "todo"):
        from etapas_datos import preparar_datos
        originales = Path(o.get("originales") or (semana1 / "Datos")).expanduser()
        decisiones = Path(o.get("decisiones") or semana1).expanduser()
        semana1, informe = preparar_datos(originales, decisiones, trabajo)
        exigir_datos(informe)
        registrar(trabajo, {"modo": "datos", "semana1": str(semana1), "informe": informe, "conforme": True, "utc": ahora()})
        print(json.dumps(informe, ensure_ascii=False, indent=1))
        if modo == "datos":
            print("datos: conforme", flush=True)
            return
        ids = cierre(objetivos, solo)
    esp = esperadas()
    if modo == "mezcla":
        extra = [Path(x) for x in str(o.get("artefactos", "")).split(",") if x]
        faltan, bloq = preparar_mezcla(ids, semana1, trabajo, extra, esp)
        if faltan:
            raise NoConforme(f"no encuentro estas salidas guardadas: {faltan}")
        if bloq:
            raise NoConforme(f"componentes guardados que no coinciden con la receta: {bloq}")
        ids = [i for i in ids if NODOS[i]["etapa"] == "mezcla"]
    else:
        malas = comprobar_entradas(semana1, esp)
        if malas:
            raise NoConforme(f"las entradas no coinciden con las de la receta: {malas}")
    toleradas = ejecutar(ids, semana1, trabajo, continuar=continuar, relanzar_parciales=bool(o.get("relanzar-parciales")))
    work = trabajo / "work"
    final, no_construidos = {}, []
    for ob in objetivos:
        t = RECETA["objetivos"][ob]
        if t["nodo"] not in ids:
            no_construidos.append(ob)
            continue
        p = work / t["csv"]
        sha = huellas(p)["sha256"] if p.is_file() else None
        final[ob] = {"csv": str(p), "sha256": sha, "esperado": t["sha256"], "coincide": sha == t["sha256"]}
    csv_ok = all(v["coincide"] for v in final.values())
    estado = "igual" if csv_ok and not toleradas else ("csv_iguales_con_diferencias_intermedias" if csv_ok else "csv_distinto")
    resultado = {"modo": modo, "version": VERSION, "objetivos": final, "objetivos_no_construidos": no_construidos,
                 "diferencias_toleradas": [{"nodo": n, "ruta": r, "motivo": m} for n, r, m in toleradas], "estado": estado, "utc": ahora()}
    registrar(trabajo, resultado)
    print(json.dumps(resultado, ensure_ascii=False, indent=1))
    if not csv_ok:
        raise SystemExit("NO CONFORME: algún CSV final no coincide con el entregado")
    if toleradas:
        print(f"Reconstrucción con {len(toleradas)} diferencias de contenido toleradas (--continuar-si-difiere): no acreditada", flush=True)
        raise SystemExit(3)


if __name__ == "__main__":
    main()
'''


### Embedded source: `herramientas/huella.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['herramientas/huella.py'] = '''"""Huellas de archivos para comparar artefactos reconstruidos con los originales.

Dos huellas por archivo:
- `sha256`: SHA-256 de los bytes. Es la prueba fuerte y la que se exige al CSV de envío.
- `contenido`: SHA-256 de los valores, independiente del escritor. Parquet se lee con pyarrow columna a columna
  (diccionarios decodificados, cadenas largas como cadenas, marcas de tiempo en nanosegundos), .npy por su tipo,
  forma y bytes, .npz miembro a miembro (el zip guarda la hora de escritura, así que sus bytes nunca coinciden),
  CSV con finales de línea normalizados y JSON en forma canónica. Sirve cuando el archivo se reescribe con otra
  versión de pyarrow o pandas, o en otra máquina, y los valores son los mismos.

Uso: python huella.py <archivo> [...]
"""
from __future__ import annotations

import hashlib
import json
import sys
from pathlib import Path

import numpy as np


def sha256_bytes(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


def _array_digest(h, arr: np.ndarray) -> None:
    arr = np.ascontiguousarray(arr)
    h.update(f"{arr.dtype.str}|{arr.shape}|".encode())
    if arr.dtype.kind == "O":
        h.update("\\x1f".join("\\x00" if v is None else str(v) for v in arr.ravel()).encode("utf-8"))
    else:
        h.update(arr.tobytes())


def _parquet_digest(path: Path) -> str:
    import pyarrow as pa
    import pyarrow.compute as pc
    import pyarrow.parquet as pq

    t = pq.read_table(path)
    h = hashlib.sha256()
    h.update(f"filas={t.num_rows}|".encode())
    for name in t.column_names:
        if name.startswith("__index_level_"):
            continue
        col = t.column(name).combine_chunks()
        if pa.types.is_dictionary(col.type):
            col = col.dictionary_decode()
        typ = col.type
        es_vista = getattr(pa.types, "is_string_view", lambda _t: False)
        if pa.types.is_large_string(typ) or es_vista(typ):
            col = col.cast(pa.string()); typ = col.type
        if pa.types.is_timestamp(typ):
            col = pc.cast(col.cast(pa.timestamp("ns", tz=typ.tz)), pa.int64()); tag = "ts_ns"
        elif pa.types.is_date(typ):
            col = pc.cast(col.cast(pa.date32()), pa.int32()); tag = "date32"
        elif pa.types.is_string(typ):
            tag = "str"
        elif pa.types.is_boolean(typ):
            tag = "bool"
        elif pa.types.is_integer(typ):
            tag = "int"
        elif pa.types.is_floating(typ):
            tag = "float"
        else:
            tag = str(typ)
        h.update(f"{name}|{tag}|nulos={col.null_count}|".encode())
        if tag == "str":
            vals = col.to_pylist()
            h.update("\\x1f".join("\\x00" if v is None else v for v in vals).encode("utf-8"))
        elif tag == "bool":
            h.update(np.asarray(col.fill_null(False).to_numpy(zero_copy_only=False), dtype=np.uint8).tobytes())
            if col.null_count:
                h.update(np.asarray(col.is_null().to_numpy(zero_copy_only=False), dtype=np.uint8).tobytes())
        elif tag in ("int", "ts_ns", "date32"):
            v = col.fill_null(0).to_numpy(zero_copy_only=False).astype(np.int64)
            h.update(v.tobytes())
            if col.null_count:
                h.update(np.asarray(col.is_null().to_numpy(zero_copy_only=False), dtype=np.uint8).tobytes())
        elif tag == "float":
            v = col.to_numpy(zero_copy_only=False).astype(np.float64)
            h.update(v.tobytes())
        else:
            h.update(json.dumps(col.to_pylist(), default=str).encode("utf-8"))
    return h.hexdigest()


def huella_contenido(path: Path) -> str:
    path = Path(path)
    suf = path.suffix.lower()
    if suf == ".parquet":
        return _parquet_digest(path)
    h = hashlib.sha256()
    if suf == ".npy":
        _array_digest(h, np.load(path, allow_pickle=False))
    elif suf == ".npz":
        with np.load(path, allow_pickle=False) as z:
            for k in sorted(z.files):
                h.update(f"[{k}]".encode())
                _array_digest(h, z[k])
    elif suf == ".csv":
        h.update(path.read_bytes().replace(b"\\r\\n", b"\\n"))
    elif suf == ".json":
        h.update(json.dumps(json.loads(path.read_text(encoding="utf-8-sig")), sort_keys=True, ensure_ascii=False).encode("utf-8"))
    else:
        h.update(path.read_bytes())
    return h.hexdigest()


def huellas(path: Path) -> dict:
    path = Path(path)
    return {"bytes": path.stat().st_size, "sha256": sha256_bytes(path), "contenido": huella_contenido(path)}


if __name__ == "__main__":
    for a in sys.argv[1:]:
        print(json.dumps({"archivo": a, **huellas(Path(a))}, ensure_ascii=False))
'''


### Embedded source: `herramientas/etapas_datos.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['herramientas/etapas_datos.py'] = '''"""Etapas 00 a 02: de los once Parquet originales a la capa preparada, las bases actuariales y la matriz F0.

bases_actuariales.py (D67) y construir_f0.py (D68) localizan sus entradas subiendo desde su propia carpeta hasta
RUTAS_PROYECTO.json y escriben la corrida nueva junto al script. Para ejecutarlos sin modificarlos se arma dentro de
<trabajo>/proyecto una raíz de proyecto mínima con la misma estructura de rutas que el espacio de trabajo original:

  proyecto/RUTAS_PROYECTO.json
  proyecto/Hackaton/Semana 1/Limpieza/runs/preaudit_20261016T211816Z_817285a5/   salida del productor N62 de Codex
  proyecto/Hackaton/Semana 1/Limpieza/current_dataset.json, dataset_catalog.json y verification.json
  proyecto/Hackaton/Semana 1/<tres listas de decisiones congeladas para las marcas audit_* de F0>
  proyecto/Hackaton/Semana 1/Bases_actuariales_claude/   copia de bases_actuariales.py y su corrida
  proyecto/Hackaton/Semana 1/Matriz_F0_claude/           copia de construir_f0.py y su corrida
  proyecto/Hackaton/Semana 1/Datos -> tablas de la capa (weather y sample_submission son los originales)

Las corridas nuevas se enlazan con los nombres que espera comun.py. Cada paso se compara con
receta/huellas_esperadas.json (sección «datos»). Devuelve la carpeta «Semana 1» de esa raíz, que es la que recibe
después el modo componentes.
"""
from __future__ import annotations

import hashlib
import json
import shutil
import subprocess
import sys
from pathlib import Path

REPO = Path(__file__).resolve().parent.parent
sys.path.insert(0, str(REPO / "herramientas"))
from huella import huellas  # noqa: E402

PREAUDIT = "preaudit_20261016T211816Z_817285a5"
F0_NOMBRE = "f0_20261017T141732Z_65df7fca"
BASES_NOMBRE = "bases_20261017T120251Z_45443ff0"


def sha(p: Path) -> str:
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()


def correr(cmd, cwd, log: Path):
    with open(log, "a", encoding="utf-8") as f:
        f.write("# " + " ".join(map(str, cmd)) + "\\n")
        f.flush()
        rc = subprocess.run([str(c) for c in cmd], cwd=str(cwd), stdout=f, stderr=subprocess.STDOUT).returncode
    if rc:
        raise SystemExit(f"Falló {' '.join(map(str, cmd[1:3]))}; ver {log}")


def copiar_script(src: Path, dst_dir: Path) -> Path:
    dst_dir.mkdir(parents=True, exist_ok=True)
    dst = dst_dir / src.name
    if not dst.exists():
        shutil.copy2(src, dst)
    if sha(dst) != sha(src):
        raise SystemExit(f"La copia de {src.name} en la raíz mínima no coincide con la del repositorio")
    return dst


def preparar_datos(originales: Path, decisiones: Path, trabajo: Path) -> tuple[Path, dict]:
    esp = json.loads((REPO / "receta" / "huellas_esperadas.json").read_text(encoding="utf-8"))["datos"]
    root = trabajo / "proyecto"
    s1 = root / "Hackaton" / "Semana 1"
    logs = trabajo / "registros"
    logs.mkdir(parents=True, exist_ok=True)
    s1.mkdir(parents=True, exist_ok=True)
    (root / "RUTAS_PROYECTO.json").write_text("{}\\n", encoding="utf-8")
    informe = {}

    # 00. originales congelados y capa preparada con el productor N62 de Codex
    malos = [n for n, h in esp["originales"].items() if sha(originales / (n if n.endswith(".parquet") else f"{n}.parquet")) != h]
    if malos:
        raise SystemExit(f"Los originales no tienen el SHA-256 congelado: {malos}")
    capa = s1 / "Limpieza" / "runs" / PREAUDIT
    if not (capa / "COMPLETE.json").exists():
        correr([sys.executable, REPO / "src/datos/n62_codex/rebuild_base.py", "--raw", originales,
                "--rules", REPO / "src/datos/n62_codex/rules.json", "--out", capa], trabajo, logs / "datos_capa.log")
    rel = f"Hackaton/Semana 1/Limpieza/runs/{PREAUDIT}"
    catalogo = {f.stem: {"workspace_relative_path": f"{rel}/data/{f.name}", "sha256": sha(f)} for f in sorted((capa / "data").glob("*.parquet"))}
    informe["capa"] = {n: catalogo.get(n, {}).get("sha256") == h for n, h in esp["capa"].items()}
    if not all(informe["capa"].values()):
        raise SystemExit(f"La capa reconstruida no coincide con preaudit: {informe['capa']}")
    (capa / "dataset_catalog.json").write_text(json.dumps(catalogo, indent=1), encoding="utf-8")
    (capa / "verification.json").write_text(json.dumps({"status": "capa reconstruida por N62 desde los once originales; SHA-256 iguales a preaudit",
                                                        "passed": len(catalogo), "failed": 0}, indent=1), encoding="utf-8")
    (s1 / "Limpieza" / "current_dataset.json").write_text(json.dumps({"workspace_relative_run": rel, "status": "rebuilt_N62_from_originals",
                                                                       "catalog_workspace_relative": f"{rel}/dataset_catalog.json", "approved_for_modeling": False}, indent=1), encoding="utf-8")
    datos = s1 / "Datos"
    if not datos.exists():
        shutil.copytree(capa / "data", datos) if sys.platform == "win32" else datos.symlink_to(capa / "data", target_is_directory=True)

    # Restricted audit ledgers are deliberately not copied into this run.

    # 01. bases actuariales
    bdir = s1 / "Bases_actuariales_claude"
    script = copiar_script(REPO / "src/datos/bases_actuariales.py", bdir)
    corridas = sorted(p for p in (bdir / "runs").glob("bases_*") if not p.is_symlink() and (p / "manifest.json").exists()) if (bdir / "runs").exists() else []
    if not corridas:
        correr([sys.executable, script], bdir, logs / "datos_bases.log")
        corridas = sorted(p for p in (bdir / "runs").glob("bases_*") if not p.is_symlink() and (p / "manifest.json").exists())
    bases = corridas[-1]
    (bdir / "current_run.json").write_text(json.dumps({"workspace_relative_run": f"Hackaton/Semana 1/Bases_actuariales_claude/runs/{bases.name}"}, indent=1), encoding="utf-8")
    alias = bdir / "runs" / BASES_NOMBRE
    if not alias.exists():
        shutil.copytree(bases, alias) if sys.platform == "win32" else alias.symlink_to(bases.name, target_is_directory=True)
    informe["bases"] = {}
    for relp, e in esp["bases"].items():
        h = huellas(bases / relp)
        informe["bases"][relp] = {"igual_bytes": h["sha256"] == e["sha256"], "igual_contenido": h["contenido"] == e["contenido"]}

    if not all(v["igual_contenido"] for v in informe["bases"].values()):
        raise SystemExit(f"Las bases requeridas difieren en contenido: {informe['bases']}")

    # 02. matriz F0 por partes
    fdir = s1 / "Matriz_F0_claude"
    script = copiar_script(REPO / "src/datos/construir_f0.py", fdir)
    corridas = sorted(p for p in (fdir / "runs").glob("f0_*") if not p.is_symlink() and (p / "manifest.json").exists()) if (fdir / "runs").exists() else []
    if not corridas:
        log = logs / "datos_f0.log"
        correr([sys.executable, script, "--ramo", "motor"], fdir, log)
        run_id = next(l.split(":", 1)[1].strip() for l in log.read_text(encoding="utf-8").splitlines() if l.startswith("Corrida creada:"))
        for parte in ("buildings", "contents"):
            correr([sys.executable, script, "--run", run_id, "--ramo", parte], fdir, log)
        correr([sys.executable, script, "--run", run_id, "--finalizar"], fdir, log)
        corridas = [fdir / "runs" / run_id]
    f0 = corridas[-1]
    alias = fdir / "runs" / F0_NOMBRE
    if not alias.exists():
        shutil.copytree(f0, alias) if sys.platform == "win32" else alias.symlink_to(f0.name, target_is_directory=True)
    informe["f0"] = {}
    for relp, e in esp["f0"].items():
        h = huellas(f0 / relp)
        informe["f0"][relp] = {"igual_bytes": h["sha256"] == e["sha256"], "igual_contenido": h["contenido"] == e["contenido"]}
    cols = json.loads((f0 / "contrato_disponibilidad.json").read_text(encoding="utf-8"))["columnas"]
    bloques = hashlib.sha256(json.dumps({l: {c: v["bloque"] for c, v in cc.items()} for l, cc in cols.items()}, sort_keys=True).encode()).hexdigest()
    informe["f0"]["contrato_bloques_iguales"] = bloques == esp["f0_contrato_bloques"]
    if not all(v["igual_contenido"] for k, v in informe["f0"].items() if k != "contrato_bloques_iguales"):
        raise SystemExit(f"Las tablas F0 requeridas difieren en contenido: {informe['f0']}")
    if not informe["f0"]["contrato_bloques_iguales"]:
        raise SystemExit("Los bloques del contrato F0 difieren")
    return s1, informe
'''


### Embedded source: `herramientas/ejecutar_csv_lf.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['herramientas/ejecutar_csv_lf.py'] = '''"""Arranca los escritores finales sin cambiar sus fuentes, con LF en pandas.

Pandas toma os.linesep como terminador por defecto; el CSV de referencia es LF.
El cambio vive solo en este proceso hijo. No se modifica un CSV ya producido.
"""
import os
import runpy
import sys
from pathlib import Path

REPO = Path(__file__).resolve().parents[1]
permitidos = {REPO / "src/modelos/mezcla3.py", REPO / "src/linea_b/mezcla3_b.py"}
script = Path(sys.argv[1]).resolve()
if script not in permitidos:
    raise SystemExit("El escritor no pertenece a los dos puntos de salida finales de la receta")
os.linesep = "\\n"
sys.argv = [str(script)] + sys.argv[2:]
sys.path.insert(0, str(script.parent))
runpy.run_path(str(script), run_name="__main__")
'''


### Embedded source: `herramientas/inventario.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['herramientas/inventario.py'] = '''"""Calcula las huellas esperadas de todos los artefactos de la receta a partir de las corridas originales.

Para cada salida de cada nodo de receta/receta.json (rutas relativas a la carpeta de trabajo de los modelos) busca el
archivo original en las carpetas indicadas, en este orden: <origen>/<ruta>, <origen>/runs/<ruta> y, para las
selecciones sel_<nombre>.parquet, <origen>/runs/<nombre>/sel_<nombre>.parquet y <origen>/selecciones_v23/. Hace lo
mismo con las entradas (relativas a «Hackaton/Semana 1»). Escribe receta/huellas_esperadas.json con bytes, SHA-256 y
huella de contenido de cada archivo y la ruta de origen, y avisa de lo que no encuentra.

Uso: python herramientas/inventario.py --semana1=<carpeta Semana 1> --origen=<carpeta> [--origen=<carpeta> ...]
"""
from __future__ import annotations

import json
import sys
from datetime import datetime, timezone
from pathlib import Path

AQUI = Path(__file__).resolve().parent
sys.path.insert(0, str(AQUI))
from huella import huellas  # noqa: E402

REPO = AQUI.parent


def localizar(rel: str, origenes: list[Path]) -> Path | None:
    for o in origenes:
        cands = [o / rel, o / "runs" / rel]
        name = Path(rel).name
        if "/" not in rel and name.startswith("sel_"):
            corrida = name[4:-8]
            cands += [o / "runs" / corrida / name, o / "selecciones_v23" / name, o / corrida / name]
        for c in cands:
            if c.is_file():
                return c
    # último recurso para las selecciones: guardadas dentro de la carpeta de otra corrida (p. ej. runs/S08h_hogar_F3s2/sel_hS2.parquet)
    if "/" not in rel and Path(rel).name.startswith("sel_"):
        for o in origenes:
            hall = sorted((o / "runs").glob(f"*/{Path(rel).name}")) if (o / "runs").is_dir() else []
            if hall:
                return hall[0]
    return None


def main():
    opts = [a for a in sys.argv[1:] if a.startswith("--")]
    semana1 = Path(next(a.split("=", 1)[1] for a in opts if a.startswith("--semana1=")))
    origenes = [Path(a.split("=", 1)[1]) for a in opts if a.startswith("--origen=")]
    receta = json.loads((REPO / "receta" / "receta.json").read_text(encoding="utf-8"))
    out = {"creado_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"), "origenes": [str(o) for o in origenes],
           "semana1": str(semana1), "entradas": {}, "salidas": {}, "faltan": []}
    for rel in receta["entradas"]["archivos"]:
        p = semana1 / rel
        if p.is_file():
            out["entradas"][rel] = huellas(p) | {"origen": str(p)}
        else:
            out["faltan"].append(rel)
    for n in receta["nodos"]:
        for rel in n["salidas"]:
            p = localizar(rel, origenes)
            if p is None:
                out["faltan"].append(rel)
                continue
            out["salidas"][rel] = huellas(p) | {"origen": str(p), "nodo": n["id"]}
            print(n["id"], rel, out["salidas"][rel]["sha256"][:12], flush=True)
    (REPO / "receta" / "huellas_esperadas.json").write_text(json.dumps(out, indent=1, ensure_ascii=False) + "\\n", encoding="utf-8")
    print("salidas:", len(out["salidas"]), "entradas:", len(out["entradas"]), "faltan:", out["faltan"])


if __name__ == "__main__":
    main()
'''


### Embedded source: `src/modelos/anexar_entrega.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/modelos/anexar_entrega.py'] = '''"""Completa la nota de entrega de una estimación propia de Claude con la batería de validación de 2024.

Lee, de la carpeta de trabajo, bootstrap_codex_<clave>.json (IC frente a A_G y H), publico_simulado_<clave>.json,
influencia_reservas_<clave>.json (reserva de caso ±25 %, influencia y cambio por peligro), robustez_<clave>_frente_<prev>.json
(frente a la estimación propia anterior), semestres_<clave>.json y deriva_nivel_<clave>.json, y añade esos campos a
Envios/<CLAVE>_claude_entrega_codex.json conservando CRLF. Si existen, añade también medias_por_ramo_<clave>.json,
la referencia (ruta y SHA-256) de predicciones_v23_v24_<clave>.parquet y criterio_<clave>.json (lo fijado antes de ver
resultados). No recalcula nada.
Uso: python3 anexar_entrega.py <clave> <nombre_candidato> <nombre_anterior> <sha256> <seleccion_de_pesos> [<forma>]
"""
from __future__ import annotations

import hashlib
import json
import sys
from pathlib import Path

from comun import WORK

O = Path("/mnt/user-data/outputs/Envios")


def c(v, k=5):
    return f"{v:+.{k}f}".replace(".", ",")


def main():
    key, cand, prev, sha, seleccion = sys.argv[1:6]
    forma = sys.argv[6] if len(sys.argv) > 6 else "motor:1.09:0.95,buildings:1.08:1.14,contents:1.07:1"
    P = O / f"{key.upper()}_claude_entrega_codex.json"
    j = json.loads(P.read_text(encoding="utf-8"))
    b = json.loads((WORK / f"bootstrap_codex_{key}.json").read_text())
    ps = json.loads((WORK / f"publico_simulado_{key}.json").read_text())
    inf = json.loads((WORK / f"influencia_reservas_{key}.json").read_text())
    rob = json.loads((WORK / f"robustez_{key}_frente_{prev.lower()}.json").read_text())["candidatos"][cand]
    sem = json.loads((WORK / f"semestres_{key}.json").read_text())
    der = json.loads((WORK / f"deriva_nivel_{key}.json").read_text())

    def ic_txt(d):
        t = f"{c(d['dif'], 6)}; IC95 cliente [{c(d['ic95_cliente'][0], 6)}; {c(d['ic95_cliente'][1], 6)}]"
        if "ic95_direccion" in d:
            t += f", dirección [{c(d['ic95_direccion'][0], 6)}; {c(d['ic95_direccion'][1], 6)}]"
        return t

    def fmt(d):
        return (f"{c(d['dif'])}; IC95 cliente [{c(d['ic95_cliente'][0])}; {c(d['ic95_cliente'][1])}], "
                f"dirección [{c(d['ic95_direccion'][0])}; {c(d['ic95_direccion'][1])}]")
    j["validacion"]["intervalos_2024_frente_a_A_G_y_H"] = {
        "valores": {k: {"frente_a_AG": fmt(b[k]["CL_menos_AG"]), "frente_a_H": fmt(b[k]["CL_menos_H"])} for k in ("motor", "buildings", "contents", "agregado")},
        "metodo": "Modelos_claude/bootstrap_codex.py: 1.000 réplicas Poisson por grupo, denominador fijo; mismas filas y T0",
        "nota": "A_G y H ajustan su meta dentro de 2024, así que la comparación local los favorece"}
    j["validacion"][f"frente_a_{prev}"] = {f: {k: f"{c(v['dif'])} (P>0 {str(v['P_mejora']).replace('.', ',')})" for k, v in rob[f].items()} for f in ("V23", "V24")}
    j["validacion"][f"frente_a_{prev}"]["sin_el_0_1_por_ciento_superior_V23_V24"] = {k: f"{c(rob['V23'][k]['dif_sin_top01'])} / {c(rob['V24'][k]['dif_sin_top01'])}" for k in rob["V24"]}
    tab = {}
    cn = inf["candidato"]; other = inf["comparadores"][0]
    for a in ("0.75", "1.0", "1.25"):
        tab[a] = {}
        for scope in ("motor", "buildings", "contents", "agregado"):
            row = inf["escenarios"][a][scope]
            tab[a][scope] = {f"d2_{cn}_{other}_H_A_G": [row["d2"][k] for k in (cn, other, "H", "A_G")],
                             f"{cn}_menos_H": ic_txt(row[f"{cn}_menos_H"]),
                             f"{cn}_menos_{other}": ic_txt(row[f"{cn}_menos_{other}"]),
                             f"{cn}_menos_A_G": ic_txt(row[f"{cn}_menos_A_G"]),
                             "diez_clientes_mas_favorables_frente_a_H": c(row[f"{cn}_menos_H"]["diez_clientes_mas_favorables"], 6),
                             "frente_a_H_sin_esos_diez": c(row[f"{cn}_menos_H"]["dif_sin_esos_diez"], 6)}
            if a == "1.25" and "cambio_numerador_por_peligro_a_mas25" in row[f"{cn}_menos_H"]:
                tab[a][scope]["cambio_numerador_frente_a_H_por_peligro"] = row[f"{cn}_menos_H"]["cambio_numerador_por_peligro_a_mas25"]
    j["validacion"]["reservas_de_caso_2024"] = {"tabla": tab, "metodo": "Modelos_claude/influencia_reservas.py: etiqueta T0 + (a - 1) x reserva de caso de los abiertos aceptados (a = 0,75; 1; 1,25), predicciones sin cambio; IC95 por cliente; los diez clientes más favorables son un diagnóstico de influencia, no un criterio de descarte"}
    st = {}
    for a, res in sem["resultados"].items():
        st[a] = {k: {"filas": v["filas"], "peso_exposicion": v["peso_exposicion"], "madurez": v["madurez"],
                     f"{cn}_menos_H": ic_txt(v[f"{cn}_menos_H"]), f"{cn}_menos_{other}": ic_txt(v[f"{cn}_menos_{other}"]),
                     f"{cn}_menos_A_G": ic_txt(v[f"{cn}_menos_A_G"])} for k, v in res.items()}
    j["validacion"]["semestres_2024"] = {"tabla": st, "metodo": "Modelos_claude/semestres.py: semestre de inicio del periodo (H1 enero-junio, H2 julio-diciembre); madurez = exposición efectiva / nominal; IC95 por cliente y, si se calculó, por dirección"}
    if (WORK / f"medias_por_ramo_{key}.json").exists():
        med = json.loads((WORK / f"medias_por_ramo_{key}.json").read_text())
        j["validacion"]["medias_por_ramo"] = {"tabla": med["medias"], "definiciones": med["definiciones"],
                                              "metodo": f"Modelos_claude/exportar_predicciones.py (experimentos/medias_por_ramo_{key}.json)"}
    pq = WORK / f"predicciones_v23_v24_{key}.parquet"
    if pq.exists():
        h = hashlib.sha256(pq.read_bytes()).hexdigest()
        j["predicciones_por_fila_v23_v24"] = {"archivo": f"Hackaton/Semana 1/Modelos_claude/experimentos/predicciones_v23_v24_{key}.parquet", "sha256": h,
                                              "columnas": ["row_id", "line", "corte", "y_T0", "reserva_caso", "prediction"],
                                              "nota": "las mismas predicciones que miden d2_sel (V23, sin factor) y d2_v24_f23 (V24, con el factor de V23); reserva_caso permite rehacer T0 + (a - 1) x reserva"}
    if (WORK / f"criterio_{key}.json").exists():
        j["criterio_fijado_antes_de_resultados"] = json.loads((WORK / f"criterio_{key}.json").read_text(encoding="utf-8"))
    p30 = ps["V24"]["30%"]; p30v23 = ps["V23"]["30%"]
    k_sp = f"{cand}-S09b"; k_prev = f"{prev}-{cand}"
    f1 = lambda x: f"{100 * x:.1f}".replace(".", ",")
    j["publico_simulado"] = (f"En 1.000 subconjuntos aleatorios del 30 % de los clientes, {cand} supera a S09b en el {f1(p30[k_sp]['P_mayor_que_0'])} % de los casos en V24 y el {f1(p30v23[k_sp]['P_mayor_que_0'])} % en V23, "
                             f"y a {prev} en el {f1(1 - p30[k_prev]['P_mayor_que_0'])} % (V24) y el {f1(1 - p30v23[k_prev]['P_mayor_que_0'])} % (V23) (Modelos_claude/experimentos/publico_simulado_{key}.json)")
    j["validacion"]["seleccion_de_pesos"] = seleccion
    j["validacion"]["regla_de_nivel"] = {"metodo": "mezcla3.py --nivel=s17: fuera de la referencia de nivel los componentes cuyo total 2025/V24 se aparta más de un 1,5 % del ancla del ramo",
                                         "deriva_por_componente": {line: {r: v["deriva"] for r, v in comps.items()} for line, comps in der.items()}}
    j["reproduccion"] = (f"En el equipo: python3 reproducir_mezcla.py experimentos/spec_{key}_propio.json {forma} s17 {cand}_claude_propio {sha} "
                         "(desde Modelos_claude; repite la mezcla a partir de las predicciones guardadas de cada componente)")
    txt = json.dumps(j, indent=1, ensure_ascii=False).replace("\\n", "\\r\\n") + "\\r\\n"
    P.write_text(txt, encoding="utf-8", newline="")
    print(j["publico_simulado"])
    print(json.dumps(j["validacion"][f"frente_a_{prev}"], ensure_ascii=False))


if __name__ == "__main__":
    main()
'''


### Embedded source: `src/modelos/bootstrap_codex.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/modelos/bootstrap_codex.py'] = '''"""Intervalos pareados de una mezcla de Claude frente a A_G y H de Codex en 2024, por cliente y por dirección.

Reproduce el criterio que usa Codex para decidir envíos: diferencia de D² por ramo y agregada, con IC del 95 % por
remuestreo de grupos (Poisson(1) por cliente o por dirección, 1.000 réplicas) y denominador fijo. Lee las predicciones de
Codex sin modificarlas. Su D² local es optimista (meta ajustada con validación cruzada por cliente dentro de 2024).
Uso: python3 bootstrap_codex.py <salida.json> <spec>@<forma>
"""
from __future__ import annotations

import json
import sys

import numpy as np
import pandas as pd

from comun import LINES, WORK, deviance
from mezcla3 import labels
from robustez import blend, parse

C = "/mnt/user-data/uploads/Hakaton/Hackaton/Semana 1/Modelado_codex/nocturna_20261017_N01"
B = 1000


def ci(dif, groups, null, rng):
    codes, inv = np.unique(groups, return_inverse=True)
    d = np.bincount(inv, weights=dif)
    bs = np.concatenate([rng.poisson(1.0, size=(50, len(d))) @ d for _ in range(B // 50)]) / null
    return [round(float(np.quantile(bs, 0.025)), 6), round(float(np.quantile(bs, 0.975)), 6)], round(float((bs > 0).mean()), 3)


def main():
    out, arg = sys.argv[1], sys.argv[2]
    _, spec, forma = parse("CL=" + arg)
    ag = pd.read_parquet(f"{C}/crossfit_meta_gam_guard/predictions_V24_crossfit.parquet", columns=["row_id", "meta_gam"]).set_index("row_id").meta_gam
    hh = pd.read_parquet(f"{C}/geometric_meta_nested/V24.parquet", columns=["row_id", "candidate"]).set_index("row_id").candidate
    rng = np.random.default_rng(20261018)
    rep, allv = {}, {"y": [], "CL": [], "AG": [], "H": [], "cu": [], "ad": []}
    for line in LINES:
        ysel, y24, idx25 = labels(line)
        fe = pd.read_parquet(WORK / f"feat_{line}.parquet", columns=["row_id", "customer_id", "address_id", "e_ef_coste", "exposure"])
        fe["row_id"] = fe.row_id.astype(str); fe = fe.set_index("row_id")
        expos = (fe.e_ef_coste.reindex(ysel.index).values, fe.e_ef_coste.reindex(y24.index).values, fe.exposure.reindex(idx25).values)
        _, _, b24, _ = blend(spec, forma, line, ysel, y24, idx25, expos)
        y = y24.values
        P = {"CL": b24, "AG": ag.reindex(y24.index).values, "H": hh.reindex(y24.index).values}
        cu = fe.customer_id.reindex(y24.index).astype(str).values; ad = fe.address_id.reindex(y24.index).astype(str).values
        null = deviance(y, np.full_like(y, y.mean())).sum()
        r = {}
        for ref in ("AG", "H"):
            dif = deviance(y, P[ref]) - deviance(y, P["CL"])
            r[f"CL_menos_{ref}"] = {"dif": round(float(dif.sum() / null), 6), "ic95_cliente": ci(dif, cu, null, rng)[0], "ic95_direccion": ci(dif, ad, null, rng)[0]}
        rep[line] = r
        for k, v in (("y", y), ("cu", cu), ("ad", ad)):
            allv[k].append(v)
        for k in ("CL", "AG", "H"):
            allv[k].append(P[k])
        print(line, r, flush=True)
    y = np.concatenate(allv["y"]); cu = np.concatenate(allv["cu"]); ad = np.concatenate(allv["ad"])
    null = deviance(y, np.full_like(y, y.mean())).sum()
    g = {}
    for ref in ("AG", "H"):
        dif = deviance(y, np.concatenate(allv[ref])) - deviance(y, np.concatenate(allv["CL"]))
        g[f"CL_menos_{ref}"] = {"dif": round(float(dif.sum() / null), 6), "ic95_cliente": ci(dif, cu, null, rng)[0], "ic95_direccion": ci(dif, ad, null, rng)[0]}
    rep["agregado"] = g
    print("agregado", g, flush=True)
    (WORK / out).write_text(json.dumps(rep, indent=1, ensure_ascii=False))


if __name__ == "__main__":
    main()
'''


### Embedded source: `src/modelos/calibracion_deciles.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/modelos/calibracion_deciles.py'] = '''"""Calibración por decil de predicción de una mezcla de Claude en V23 selección y V24: cociente real/esperado por decil.

Usa las predicciones de la mezcla tal como se evalúan (V24 con el factor de V23) y agrupa por deciles de la tasa predicha
(predicción / exposición efectiva). Informa también el cociente sin el 0,1 % superior de la pérdida, porque en hogar el
decil alto concentra la cola (sección 4.5 del informe).
Uso: python3 calibracion_deciles.py <spec>@<forma>@<nivel> <salida.json>
"""
import json
import sys

import numpy as np
import pandas as pd

from comun import LINES, WORK
from mezcla3 import labels
from robustez import blend, parse


def main():
    _, spec, forma = parse("C=" + sys.argv[1])
    rep = {}
    for line in LINES:
        ysel, y24, idx25 = labels(line)
        fe = pd.read_parquet(WORK / f"feat_{line}.parquet", columns=["row_id", "e_ef_coste", "exposure"])
        fe["row_id"] = fe.row_id.astype(str); fe = fe.set_index("row_id")
        expos = (fe.e_ef_coste.reindex(ysel.index).values, fe.e_ef_coste.reindex(y24.index).values, fe.exposure.reindex(idx25).values)
        _, bsel, b24, _ = blend(spec, forma, line, ysel, y24, idx25, expos)
        rep[line] = {}
        for fold, y, mu, e in (("V23", ysel.values, bsel, expos[0]), ("V24", y24.values, b24, expos[1])):
            rate = mu / e
            q = np.quantile(rate, np.linspace(0, 1, 11)); dec = np.clip(np.searchsorted(q, rate, side="right") - 1, 0, 9)
            keep = y <= np.quantile(y, 0.999)
            ae = [round(float(y[dec == k].sum() / mu[dec == k].sum()), 3) for k in range(10)]
            ae_sin = [round(float(y[(dec == k) & keep].sum() / mu[(dec == k) & keep].sum()), 3) for k in range(10)]
            rep[line][fold] = {"real_sobre_esperado_por_decil": ae, "sin_0_1_por_ciento_superior": ae_sin, "total": round(float(y.sum() / mu.sum()), 4)}
            print(line, fold, "total", rep[line][fold]["total"], "deciles", ae, flush=True)
            print("   sin top 0,1 %", ae_sin, flush=True)
    (WORK / sys.argv[2]).write_text(json.dumps(rep, indent=1, ensure_ascii=False))


if __name__ == "__main__":
    main()
'''


### Embedded source: `src/modelos/calibrar_siguiente_anio.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/modelos/calibrar_siguiente_anio.py'] = '''"""Calibración al año siguiente: factor por ramo que minimiza la desviación Tweedie (p=1,5) de las predicciones del modelo
sobre el año posterior a su entrenamiento, estimado fuera de tiempo en dos pliegues y aplicado a la predicción de 2025.

Pliegue V23: modelo entrenado 2021-2022, factor sobre la mitad de selección de 2023 (sel_<nombre_v23>.parquet).
Pliegue V24: modelo entrenado 2021-2023, factor sobre 2024 (v24.parquet de la corrida de producción).
Factor de cada pliegue: f = sum(y/sqrt(mu)) / sum(sqrt(mu)); factor aplicado: media de los dos pliegues.
Las predicciones ya están a nivel desarrollado (offset de exposición efectiva), así que el factor recoge tendencia y la
diferencia entre el balance agregado y el óptimo Tweedie, no el IBNR (que no se aplica dos veces, D70).

Comprobación de transferencia: el factor de V23 aplicado a las predicciones de V24 debe mejorar el D² de V24.
Uso: python3 calibrar_siguiente_anio.py <corrida_base> <nombre_salida> <ramo>:<sel_v23>[,<ramo>:<sel_v23>...]
"""
from __future__ import annotations

import json
import sys

import numpy as np
import pandas as pd

from comun import BASE, LINES, WORK, d2, digest


def ftw(y, mu):
    return float((y / np.sqrt(mu)).sum() / np.sqrt(mu).sum())


def main():
    base_run, out_name = sys.argv[1], sys.argv[2]
    sel_map = dict(kv.split(":") for kv in sys.argv[3].split(","))
    base = WORK / base_run
    pred = pd.read_parquet(base / "prediccion_2025.parquet")
    v24 = pd.read_parquet(base / "v24.parquet")
    out = WORK / out_name
    out.mkdir(parents=True, exist_ok=True)
    rep = {"base_run": base_run, "lines": {}}
    v24c = v24.copy()
    for line in LINES:
        s = pd.read_parquet(WORK / f"sel_{sel_map[line]}.parquet")
        s = s[s.line == line]
        b = v24[v24.line == line]
        f23, f24 = ftw(s.y.values, s.mu.values), ftw(b.y.values, b.mu.values)
        f = (f23 + f24) / 2
        rep["lines"][line] = {"f_v23": round(f23, 4), "f_v24": round(f24, 4), "f_aplicado": round(f, 4),
                              "d2_v24_sin": round(d2(b.y, b.mu), 5), "d2_v24_con_f_v23": round(d2(b.y, b.mu * f23), 5), "sel_v23": sel_map[line]}
        v24c.loc[v24c.line == line, "mu"] = b.mu.values * f23
        pred.loc[pred.line == line, "prediction"] = pred.loc[pred.line == line, "prediction"].values * f
    rep["d2_v24_global_sin"] = round(d2(v24.y, v24.mu), 5)
    rep["d2_v24_global_con_f_v23"] = round(d2(v24c.y, v24c.mu), 5)
    sample = pd.read_parquet(BASE / "Datos" / "sample_submission.parquet", columns=["row_id"])
    sub = sample.merge(pred[["row_id", "prediction"]], on="row_id", how="left", validate="one_to_one")
    assert len(sub) == 626639 and np.isfinite(sub.prediction).all() and (sub.prediction > 0).all()
    csv = out / f"{out_name}.csv"
    sub.to_csv(csv, index=False, float_format="%.6f")
    back = pd.read_csv(csv, dtype={"row_id": str})
    assert (back.row_id.values == sample.row_id.values).all() and len(back) == 626639
    rep["file"] = {"path": str(csv), "sha256": digest(csv), "rows": 626639, "total_CU": round(float(back.prediction.sum()), 2),
                   "by_line_CU": {l: round(float(pred[pred.line == l].prediction.sum()), 2) for l in LINES}}
    pred.to_parquet(out / "prediccion_2025.parquet", index=False)
    (out / "resumen.json").write_text(json.dumps(rep, indent=1))
    print(json.dumps(rep, indent=1))


if __name__ == "__main__":
    main()
'''


### Embedded source: `src/modelos/cat_experimento.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/modelos/cat_experimento.py'] = '''"""CatBoost Tweedie con el mismo contrato que el LightGBM de Claude (offset de exposición efectiva, tope de autos con
recargo uniforme, F3 solo en autos) en V23: parada temprana en la mitad de calibración, comparación en la de selección.
Guarda las predicciones de selección (sel_<nombre>.parquet) para mezclar con LightGBM.
Uso: python3 cat_experimento.py <nombre> [ramos] [--depth=6] [--lr=0.05] [--f3s2]
"""
from __future__ import annotations

import json
import sys
import time

import numpy as np
import pandas as pd
from catboost import CatBoostRegressor, Pool

from comun import LINES, WORK, d2
from experimentos import features, load

CAPS = {"motor": 15000.0}
# las siete variables de cliente F3 que el participante autorizó en hogar (D80); solo con --f3s2
F3S2 = ["prior_claims_5y", "credit_band", "missed_payments_12m", "household_size", "years_at_address", "marital_status", "licence_points"]


def frame(df, feats):
    X = df[feats].copy()
    cats = []
    for c in X.columns:
        s = X[c]
        if pd.api.types.is_bool_dtype(s.dtype):
            X[c] = s.astype("float32")
        elif pd.api.types.is_numeric_dtype(s.dtype):
            X[c] = s.astype("float64")
        else:
            if pd.api.types.infer_dtype(s, skipna=True) == "boolean":
                X[c] = s.map({True: 1.0, False: 0.0}).astype("float32")
            else:
                X[c] = s.astype(str).fillna("nan")
                cats.append(c)
    return X, cats


def main():
    name = sys.argv[1]
    lines = sys.argv[2].split(",") if len(sys.argv) > 2 and not sys.argv[2].startswith("--") else LINES
    opts = dict((a[2:].split("=") + ["1"])[:2] for a in sys.argv[2:] if a.startswith("--"))
    depth, lr = int(opts.get("depth", 6)), float(opts.get("lr", 0.05))
    power = float(opts.get("p", 1.5))
    if "cap" in opts:
        CAPS["motor"] = float(opts["cap"])
    rep, frames = {}, []
    for line in lines:
        t = time.time()
        df = load(line)
        feats = features(line, df, "F01234" if line == "motor" else "F0124")
        if line != "motor" and "f3s2" in opts:
            feats += [c for c in F3S2 if c in df.columns and c not in feats]
        X, cats = frame(df, feats)
        tr = df.year.isin([2021, 2022]).values
        cal = ((df.year == 2023) & (df.cal_half == 0)).values
        sel = ((df.year == 2023) & (df.cal_half == 1)).values
        y = df["T0"].fillna(0).values
        off = df["e_ef_coste"].values
        cap = CAPS.get(line)
        yt = np.minimum(y, cap) if cap else y
        loading = y[tr].sum() / yt[tr].sum() if cap else 1.0
        base = float(np.log(yt[tr].sum() / off[tr].sum()))
        ptr = Pool(X[tr], yt[tr], cat_features=cats, baseline=np.log(off[tr]) + base)
        pcal = Pool(X[cal], yt[cal], cat_features=cats, baseline=np.log(off[cal]) + base)
        m = CatBoostRegressor(loss_function=f"Tweedie:variance_power={power}", iterations=4000, learning_rate=lr, depth=depth, l2_leaf_reg=10,
                              od_type="Iter", od_wait=200, thread_count=2, random_seed=20261017, verbose=0, boost_from_average=False,
                              one_hot_max_size=8, use_best_model=True)
        m.fit(ptr, eval_set=pcal)
        raw = m.predict(Pool(X[sel], cat_features=cats), prediction_type="RawFormulaVal")
        mu = np.exp(raw + np.log(off[sel]) + base) * loading
        rep[line] = {"d2_sel": round(d2(y[sel], mu), 5), "best": int(m.get_best_iteration()), "balance_sel": round(float(mu.sum() / y[sel].sum()), 4), "secs": round(time.time() - t)}
        print(name, line, rep[line], flush=True)
        frames.append(pd.DataFrame({"row_id": df.row_id[sel].astype(str).values, "line": line, "y": y[sel], "mu": mu}))
    out = pd.concat(frames, ignore_index=True)
    out.to_parquet(WORK / f"sel_{name}.parquet", index=False)
    if len(lines) == 3:
        rep["global"] = round(d2(out.y, out.mu), 5)
        print("== ", name, "global", rep["global"], flush=True)
    (WORK / f"eval_{name}.json").write_text(json.dumps(rep, indent=1))


if __name__ == "__main__":
    main()
'''


### Embedded source: `src/modelos/cat_frecsev.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/modelos/cat_frecsev.py'] = '''"""Frecuencia por peligro con CatBoost Poisson por severidad media del peligro (versión CatBoost de FS_frecsev).

mu = sum_p lambda_p(x) x s_p, con lambda_p de CatBoost Poisson (baseline log e_ef_p + base) y s_p = coste del peligro /
número de siniestros del peligro en el entrenamiento de cada corte (sin tope). Mismo contrato que cat_experimento.py:
F3 solo en autos (y las siete variables de cliente D80 en hogar con --f3s2). V23: parada temprana por peligro en la mitad
de calibración y comparación en la de selección (guarda sel_<nombre>.parquet). Con --prod: V24 (iteraciones V23 x 1,15)
y 2025 (x 1,3, exposición nominal), con las semillas indicadas. Escribe WORK/<nombre>/ con el formato de produccion.py.
Reanudable: cada peligro guarda su resultado al terminar (v23_<p>.npz, v24_<p>.npy, prod_<p>.npy) y al relanzar la misma
orden se salta lo ya hecho. Las columnas numéricas pasan a float32 para ahorrar memoria (desde CFS_buildings; CFS_motor se
corrió con float64 salvo en su 2025).
Con --desde=<corrida> --semilla_inicial=k toma las mejores iteraciones de V23 de otra corrida y entrena solo V24 y 2025
con semillas nuevas, para promediarlas después con combinar_semillas.py.
Uso: python3 cat_frecsev.py <nombre> <ramo> [--prod] [--solo_prod] [--semillas=2] [--depth=6] [--lr=0.05] [--f3s2] [--desde=<corrida>] [--semilla_inicial=0]
"""
from __future__ import annotations

import gc
import json
import sys
import time

import numpy as np
import pandas as pd
from catboost import CatBoostRegressor, Pool

from cat_experimento import frame
from comun import PERILS, WORK, d2
from experimentos import features, load

F3S2 = ["prior_claims_5y", "credit_band", "missed_payments_12m", "household_size", "years_at_address", "marital_status", "licence_points"]


def main():
    name, line = sys.argv[1], sys.argv[2]
    opts = dict((a[2:].split("=") + ["1"])[:2] for a in sys.argv[3:] if a.startswith("--"))
    seeds, depth, lr = int(opts.get("semillas", 2)), int(opts.get("depth", 6)), float(opts.get("lr", 0.05))
    s0 = int(opts.get("semilla_inicial", 0))  # para añadir semillas nuevas a una corrida previa sin repetir las usadas
    t0 = time.time()
    df = load(line)
    feats = features(line, df, "F01234" if line == "motor" else "F0124")
    if line != "motor" and "f3s2" in opts:
        feats += [c for c in F3S2 if c in df.columns and c not in feats]
    X, cats = frame(df, feats)
    for c in X.columns:
        if c in cats:
            X[c] = X[c].astype("category")
        elif X[c].dtype == "float64":
            X[c] = X[c].astype("float32")
    y = df["T0"].fillna(0).values
    yr = df.year.values
    expo = df["exposure"].values
    row_ids = df.row_id.astype(str).values
    cal_half = df.cal_half.values
    per = {p: (df[f"n_{p}"].fillna(0).values.astype(float), df[f"y_{p}"].fillna(0).values, df[f"e_ef_{p}"].values) for p in PERILS[line]}
    del df
    gc.collect()
    tr = yr <= 2022; cal = (yr == 2023) & (cal_half == 0); sel = (yr == 2023) & (cal_half == 1)
    out = WORK / name
    out.mkdir(parents=True, exist_ok=True)
    if "solo_prod" in opts:
        # solo 2025, con las mejores iteraciones guardadas de V23; guarda cada peligro al terminar para poder reanudar
        rep = json.loads((out / "resumen.json").read_text())
        te, trm = yr == 2025, yr <= 2024
        # memoria: copias compactas de entrenamiento y test, sin el marco completo
        Xtr, Xte = X.loc[trm].copy(), X.loc[te].copy()
        del X
        for Z in (Xtr, Xte):
            for c in Z.columns:
                if Z[c].dtype == "float64":
                    Z[c] = Z[c].astype("float32")
        gc.collect()
        mu = np.zeros(te.sum())
        for p, (n, yp, off) in per.items():
            fp = out / f"prod_{p}.npy"
            if fp.exists():
                mu += np.load(fp)
                print(name, "2025", p, "reanudado", flush=True)
                continue
            sev = yp[trm].sum() / max(n[trm].sum(), 1.0)
            base = float(np.log(n[trm].sum() / off[trm].sum()))
            ptr = Pool(Xtr, n[trm], cat_features=cats, baseline=np.log(off[trm]) + base)
            pev = Pool(Xte, cat_features=cats)
            part = np.zeros(te.sum())
            for s in range(seeds):
                m = CatBoostRegressor(loss_function="Poisson", iterations=int(round(max(rep["perils"][p]["best"], 20) * 1.3)), learning_rate=lr, depth=depth,
                                      l2_leaf_reg=10, thread_count=2, random_seed=20261017 + 101 * (s0 + s), verbose=0, boost_from_average=False, one_hot_max_size=8)
                m.fit(ptr)
                part += np.exp(m.predict(pev, prediction_type="RawFormulaVal") + np.log(expo[te]) + base) * sev / seeds
            np.save(fp, part)
            del ptr, pev, m
            gc.collect()
            mu += part
            print(name, "2025", p, round(time.time() - t0), flush=True)
        rep["pred_2025_total_CU"] = round(float(mu.sum()), 2)
        pd.DataFrame({"row_id": row_ids[te], "line": line, "prediction": mu}).to_parquet(out / "prediccion_2025.parquet", index=False)
        (out / "resumen.json").write_text(json.dumps(rep, indent=1))
        print(name, "2025 total", rep["pred_2025_total_CU"], flush=True)
        return
    rep = {"line": line, "depth": depth, "lr": lr, "seeds": seeds, "f3s2": "f3s2" in opts, "perils": {}}
    mu_sel = np.zeros(sel.sum())
    best = {}
    if "desde" in opts:
        # semillas nuevas de una corrida previa: toma sus mejores iteraciones de V23 y pasa directo a V24 y 2025
        src = json.loads((WORK / opts["desde"] / "resumen.json").read_text())
        best = {p: int(src["perils"][p]["best"]) for p in per}
        rep.update({"desde": opts["desde"], "semilla_inicial": s0, "perils": {p: {"best": best[p]} for p in per}})
        (out / "resumen.json").write_text(json.dumps(rep, indent=1))
    for p, (n, yp, off) in ([] if "desde" in opts else per.items()):
        sev = yp[tr].sum() / max(n[tr].sum(), 1.0)
        base = float(np.log(n[tr].sum() / off[tr].sum()))
        fv = out / f"v23_{p}.npz"
        if fv.exists():
            z = np.load(fv)
            best[p], lam = int(z["best"]), z["lam"]
            print(name, "V23", p, "reanudado", flush=True)
        else:
            m = CatBoostRegressor(loss_function="Poisson", iterations=3000, learning_rate=lr, depth=depth, l2_leaf_reg=10, od_type="Iter", od_wait=150,
                                  thread_count=2, random_seed=20261017, verbose=0, boost_from_average=False, one_hot_max_size=8, use_best_model=True)
            m.fit(Pool(X[tr], n[tr], cat_features=cats, baseline=np.log(off[tr]) + base),
                  eval_set=Pool(X[cal], n[cal], cat_features=cats, baseline=np.log(off[cal]) + base))
            best[p] = max(int(m.get_best_iteration()) + 1, 20)
            lam = np.exp(m.predict(Pool(X[sel], cat_features=cats), prediction_type="RawFormulaVal") + np.log(off[sel]) + base)
            np.savez(fv, best=best[p], lam=lam)
            del m
            gc.collect()
        mu_sel += lam * sev
        rep["perils"][p] = {"best": best[p], "sev": round(float(sev), 2), "d2_coste_peligro_sel": round(d2(yp[sel], lam * sev), 5)}
        print(name, p, rep["perils"][p], round(time.time() - t0), flush=True)
    if "desde" not in opts:
        rep["d2_sel"] = round(d2(y[sel], mu_sel), 5)
        print(name, "V23 selección", rep["d2_sel"], flush=True)
        pd.DataFrame({"row_id": row_ids[sel], "line": line, "y": y[sel], "mu": mu_sel}).to_parquet(WORK / f"sel_{name}.parquet", index=False)
        (out / "resumen.json").write_text(json.dumps(rep, indent=1))
    if "prod" not in opts:
        return
    for tag, trm, evm, mult in (("v24", yr <= 2023, yr == 2024, 1.15), ("2025", yr <= 2024, yr == 2025, 1.3)):
        mu = np.zeros(evm.sum())
        for p, (n, yp, off) in per.items():
            fp = out / (f"v24_{p}.npy" if tag == "v24" else f"prod_{p}.npy")
            if fp.exists():
                mu += np.load(fp)
                print(name, tag, p, "reanudado", flush=True)
                continue
            sev = yp[trm].sum() / max(n[trm].sum(), 1.0)
            base = float(np.log(n[trm].sum() / off[trm].sum()))
            ptr = Pool(X[trm], n[trm], cat_features=cats, baseline=np.log(off[trm]) + base)
            pev = Pool(X[evm], cat_features=cats)
            offe = off[evm] if tag == "v24" else expo[evm]
            part = np.zeros(evm.sum())
            for s in range(seeds):
                m = CatBoostRegressor(loss_function="Poisson", iterations=int(round(best[p] * mult)), learning_rate=lr, depth=depth, l2_leaf_reg=10,
                                      thread_count=2, random_seed=20261017 + 101 * (s0 + s), verbose=0, boost_from_average=False, one_hot_max_size=8)
                m.fit(ptr)
                part += np.exp(m.predict(pev, prediction_type="RawFormulaVal") + np.log(offe) + base) * sev / seeds
                del m
            np.save(fp, part)
            mu += part
            del ptr, pev
            gc.collect()
            print(name, tag, p, round(time.time() - t0), flush=True)
        if tag == "v24":
            rep["d2_v24"] = round(d2(y[evm], mu), 5)
            pd.DataFrame({"row_id": row_ids[evm], "line": line, "y": y[evm], "mu": mu}).to_parquet(out / "v24.parquet", index=False)
            print(name, "V24", rep["d2_v24"], flush=True)
        else:
            rep["pred_2025_total_CU"] = round(float(mu.sum()), 2)
            pd.DataFrame({"row_id": row_ids[evm], "line": line, "prediction": mu}).to_parquet(out / "prediccion_2025.parquet", index=False)
        (out / "resumen.json").write_text(json.dumps(rep, indent=1))
    rep["secs"] = round(time.time() - t0)
    (out / "resumen.json").write_text(json.dumps(rep, indent=1))
    print(name, json.dumps(rep), flush=True)


if __name__ == "__main__":
    main()
'''


### Embedded source: `src/modelos/cat_produccion.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/modelos/cat_produccion.py'] = '''"""CatBoost Tweedie de producción para un ramo: V24 (entrena 2021-2023, evalúa 2024) y 2025 (entrena 2021-2024).

Mismo contrato que cat_experimento.py: offset log(e_ef_coste), tope y recargo uniforme en autos, F3 solo en autos.
Iteraciones = mejor iteración de V23 x 1,15 (V24) y x 1,3 (producción). Escribe WORK/<nombre>/v24.parquet,
prediccion_2025.parquet y resumen.json, con el formato de produccion.py.
Uso: python3 cat_produccion.py <nombre> <ramo> <iter_v23> [--semillas=2] [--semilla_inicial=0] [--depth=6] [--lr=0.05] [--drop=a,b]
"""
from __future__ import annotations

import json
import sys
import time

import numpy as np
import pandas as pd
from catboost import CatBoostRegressor, Pool

from cat_experimento import CAPS, frame
from comun import WORK, d2
from experimentos import features, load


def main():
    name, line, it = sys.argv[1], sys.argv[2], int(sys.argv[3])
    opts = dict(a[2:].split("=", 1) for a in sys.argv[4:] if a.startswith("--"))
    seeds, depth, lr = int(opts.get("semillas", 2)), int(opts.get("depth", 6)), float(opts.get("lr", 0.05))
    s0 = int(opts.get("semilla_inicial", 0))  # para añadir semillas nuevas a una corrida previa sin repetir las ya usadas
    drop = set(opts["drop"].split(",")) if opts.get("drop") else set()
    t = time.time()
    df = load(line)
    feats = [f for f in features(line, df, "F01234" if line == "motor" else "F0124") if f not in drop]
    X, cats = frame(df, feats)
    for c in cats:  # categorías compactas en lugar de cadenas de Python: misma información, mucha menos memoria
        X[c] = X[c].astype("category")
    y = df["T0"].fillna(0).values
    off = df["e_ef_coste"].values
    expo = df["exposure"].values
    years = df["year"].values
    row_ids = df["row_id"].astype(str).values
    del df
    import gc
    gc.collect()
    cap = CAPS.get(line)
    out = WORK / name
    out.mkdir(parents=True, exist_ok=True)
    res = {"line": line, "iter_v23": it, "seeds": seeds, "depth": depth, "lr": lr, "drop": sorted(drop)}
    tasks = (("v24", years <= 2023, years == 2024, off, 1.15), ("2025", years <= 2024, years == 2025, expo, 1.3))
    if opts.get("solo2025"):
        tasks = tasks[1:]
    for tag, trm, evm, offe, mult in tasks:
        yt = np.minimum(y, cap) if cap else y
        loading = y[trm].sum() / yt[trm].sum() if cap else 1.0
        base = float(np.log(yt[trm].sum() / off[trm].sum()))
        ptr = Pool(X[trm], yt[trm], cat_features=cats, baseline=np.log(off[trm]) + base)
        pev = Pool(X[evm], cat_features=cats)
        mu = np.zeros(evm.sum())
        for s in range(s0, s0 + seeds):
            m = CatBoostRegressor(loss_function="Tweedie:variance_power=1.5", iterations=int(round(it * mult)), learning_rate=lr, depth=depth,
                                  l2_leaf_reg=10, thread_count=2, random_seed=20261017 + 101 * s, verbose=0, boost_from_average=False, one_hot_max_size=8)
            m.fit(ptr)
            mu += np.exp(m.predict(pev, prediction_type="RawFormulaVal") + np.log(offe[evm]) + base) * loading / seeds
        if tag == "v24":
            res["d2_v24"] = round(d2(y[evm], mu), 5)
            res["balance_v24"] = round(float(mu.sum() / y[evm].sum()), 4)
            pd.DataFrame({"row_id": row_ids[evm], "line": line, "y": y[evm], "mu": mu}).to_parquet(out / "v24.parquet", index=False)
        else:
            res["pred_2025_total_CU"] = round(float(mu.sum()), 2)
            pd.DataFrame({"row_id": row_ids[evm], "line": line, "prediction": mu}).to_parquet(out / "prediccion_2025.parquet", index=False)
        print(tag, res, flush=True)
    res["secs"] = round(time.time() - t)
    prev = json.loads((out / "resumen.json").read_text()) if (out / "resumen.json").exists() else {}
    (out / "resumen.json").write_text(json.dumps(prev | res, indent=1))


if __name__ == "__main__":
    main()
'''


### Embedded source: `src/modelos/combinar_semillas.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/modelos/combinar_semillas.py'] = '''"""Promedia corridas de producción del mismo modelo con semillas distintas (v24.parquet y prediccion_2025.parquet).

Cada corrida aporta según su número de semillas. La selección V23 no cambia (es un único modelo con parada temprana).
Uso: python3 combinar_semillas.py <nombre_salida> <corrida1> <corrida2> [...]
"""
from __future__ import annotations

import json
import sys

import pandas as pd

from comun import WORK, d2

name, runs = sys.argv[1], sys.argv[2:]
out = WORK / name
out.mkdir(parents=True, exist_ok=True)
reps = [json.loads((WORK / r / "resumen.json").read_text()) for r in runs]
w = [r.get("seeds", 1) for r in reps]
rep = {"corridas": runs, "semillas": w}
for f, col in (("v24.parquet", "mu"), ("prediccion_2025.parquet", "prediction")):
    frames = [pd.read_parquet(WORK / r / f).set_index("row_id") for r in runs]
    base = frames[0].copy()
    for fr in frames[1:]:
        assert (fr.index == base.index).all()
    base[col] = sum(wi * fr[col] for wi, fr in zip(w, frames)) / sum(w)
    base.reset_index().to_parquet(out / f, index=False)
    if f == "v24.parquet":
        rep["d2_v24"] = round(d2(base.y.values, base.mu.values), 5)
        rep["d2_v24_por_corrida"] = [round(d2(fr.y.values, fr.mu.values), 5) for fr in frames]
    else:
        rep["pred_2025_total_CU"] = round(float(base.prediction.sum()), 2)
(out / "resumen.json").write_text(json.dumps(rep, indent=1))
print(json.dumps(rep), flush=True)
'''


### Embedded source: `src/modelos/comparar_codex.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/modelos/comparar_codex.py'] = '''"""Compara la mezcla de Claude con A_G y H de Codex sobre las mismas filas de 2024 y prueba mezclas fijas entre ellas.

A_G (meta GAM) y H (meta geométrica anidada) vienen de la carpeta nocturna N01 de Codex, solo leídas: sus predicciones de
2024 ajustan la meta con validación cruzada por cliente dentro de 2024, así que su D² local es optimista frente a uno
fuera de año. La mezcla de Claude se reconstruye como en mezcla3 (V24 con el factor de V23). Se informa el D² tal cual y
renivelado (cada predicción multiplicada por su factor Tweedie óptimo en 2024, para separar nivel de orden), por ramo.
Uso: python3 comparar_codex.py <salida.json> <spec> <forma>
"""
from __future__ import annotations

import json
import sys

import numpy as np
import pandas as pd

from comun import LINES, WORK, d2
from mezcla3 import ftw, labels
from robustez import blend, parse

C = "/mnt/user-data/uploads/Hakaton/Hackaton/Semana 1/Modelado_codex/nocturna_20261017_N01"


def main():
    out_path, spec_arg = sys.argv[1], sys.argv[2]
    name, spec, forma = parse("CL=" + spec_arg)
    ag = pd.read_parquet(f"{C}/crossfit_meta_gam_guard/predictions_V24_crossfit.parquet", columns=["row_id", "T0", "meta_gam", "S09b"]).set_index("row_id")
    hh = pd.read_parquet(f"{C}/geometric_meta_nested/V24.parquet", columns=["row_id", "candidate", "anchor"]).set_index("row_id")
    rep = {}
    allv = {k: [] for k in ("y", "CL", "AG", "H", "S09b")}
    for line in LINES:
        ysel, y24, idx25 = labels(line)
        fe = pd.read_parquet(WORK / f"feat_{line}.parquet", columns=["row_id", "e_ef_coste", "exposure"])
        fe["row_id"] = fe.row_id.astype(str); fe = fe.set_index("row_id")
        expos = (fe.e_ef_coste.reindex(ysel.index).values, fe.e_ef_coste.reindex(y24.index).values, fe.exposure.reindex(idx25).values)
        _, _, b24, _ = blend(spec, forma, line, ysel, y24, idx25, expos)
        idx = y24.index
        assert ag.index.isin(idx).sum() >= len(idx) * 0.999
        y = y24.values
        assert np.allclose(ag.T0.reindex(idx).values, y, atol=0.01)
        P = {"CL": b24, "AG": ag.meta_gam.reindex(idx).values, "H": hh.candidate.reindex(idx).values, "S09b": ag.S09b.reindex(idx).values}
        assert all(np.isfinite(v).all() for v in P.values())
        r = {k: {"d2": round(d2(y, v), 5), "d2_renivelado": round(d2(y, v * ftw(y, v)), 5), "nivel_sobre_real": round(float(v.sum() / y.sum()), 4)} for k, v in P.items()}
        for other in ("AG", "H"):
            for w in (0.3, 0.5, 0.7):
                for kind in ("arit", "geom"):
                    m = w * P["CL"] + (1 - w) * P[other] if kind == "arit" else np.exp(w * np.log(P["CL"]) + (1 - w) * np.log(P[other]))
                    r[f"{kind}_{w}CL_{other}"] = {"d2": round(d2(y, m), 5), "d2_renivelado": round(d2(y, m * ftw(y, m)), 5)}
        rep[line] = r
        for k in allv:
            allv[k].append(y if k == "y" else P[k])
        print(line, {k: (v["d2"], v["d2_renivelado"]) for k, v in r.items()}, flush=True)
    y = np.concatenate(allv["y"]); P = {k: np.concatenate(v) for k, v in allv.items() if k != "y"}
    g = {k: round(d2(y, v), 5) for k, v in P.items()}
    for other in ("AG", "H"):
        for w in (0.3, 0.5, 0.7):
            g[f"arit_{w}CL_{other}"] = round(d2(y, w * P["CL"] + (1 - w) * P[other]), 5)
            g[f"geom_{w}CL_{other}"] = round(d2(y, np.exp(w * np.log(P["CL"]) + (1 - w) * np.log(P[other]))), 5)
    rep["global"] = g
    print("global", g, flush=True)
    (WORK / out_path).write_text(json.dumps(rep, indent=1, ensure_ascii=False))


if __name__ == "__main__":
    main()
'''


### Embedded source: `src/modelos/comun.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/modelos/comun.py'] = '''"""Utilidades comunes de Modelos_claude: rutas, métrica, carga de matrices con variables adicionales y bloques.

Las variables adicionales respetan el corte de información del 1 de enero del año del periodo (mismo corte en
entrenamiento y test) y solo usan datos ya verificados por hash (matrices F0 D68, asignación D67, clima original).
"""
from __future__ import annotations

import hashlib
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd

BASE = Path(os.environ.get("HAKATON_SEMANA1", "/mnt/user-data/uploads/Hakaton/Hackaton/Semana 1"))
F0RUN = "f0_20261017T141732Z_65df7fca"
F0DIR = BASE / "Matriz_F0_claude" / "runs" / F0RUN
BASESRUN = "bases_20261017T120251Z_45443ff0"
BASESDIR = BASE / "Bases_actuariales_claude" / "runs" / BASESRUN
WORK = Path(os.environ.get("MODELOS_WORK", "/home/claude/work"))
LINES = ["motor", "buildings", "contents"]
PERILS = {"motor": ["own_damage", "third_party_prop", "bodily_injury", "theft", "glass_hail"],
          "buildings": ["storm", "flood", "escape_of_water", "subsidence", "fire", "accidental_damage"],
          "contents": ["theft", "escape_of_water", "accidental_damage", "flood"]}
WEATHER_VARS = ["hail_days", "wind_exposure_index", "freezing_days", "dry_summer_index", "wildfire_index", "rainfall_mm", "mean_temperature_c"]
POWER = 1.5
NEVER = {"row_id", "policy_id", "customer_id", "address_id", "period_start", "period_end", "is_test", "year"}


def digest(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


def deviance(y, mu, p=POWER):
    y = np.asarray(y, float)
    mu = np.asarray(mu, float)
    return 2.0 * (np.power(y, 2 - p) / ((1 - p) * (2 - p)) - y * np.power(mu, 1 - p) / (1 - p) + np.power(mu, 2 - p) / (2 - p))


def d2(y, mu, p=POWER) -> float:
    y = np.asarray(y, float)
    return float(1.0 - deviance(y, mu, p).sum() / deviance(y, np.full_like(y, y.mean()), p).sum())


def contract() -> dict:
    return json.loads((F0DIR / "contrato_disponibilidad.json").read_text(encoding="utf-8"))["columnas"]


def build_extra(line: str) -> pd.DataFrame:
    """Matriz F0 del ramo más variables adicionales F1, F2 y F4 con el corte del 1 de enero del año del periodo."""
    df = pd.read_parquet(F0DIR / "tables" / f"f0_{line}.parquet")
    a = pd.read_parquet(BASESDIR / "tables" / "asignacion_por_siniestro.parquet")
    a["report_year"] = a.report_date.dt.year.astype(int)

    # F1 por póliza y peligro: notificaciones reportadas antes del 1 de enero del año del periodo
    ap = a[a.line == line]
    parts = []
    for y in range(2021, 2026):
        prev = ap[ap.report_year < y]
        if len(prev):
            w = prev.groupby(["policy_id", "peril"]).size().unstack(fill_value=0)
            w.columns = [f"f1x_notif_{c}" for c in w.columns]
            w = w.reset_index()
            w["year"] = y
            parts.append(w)
    hist = pd.concat(parts, ignore_index=True)
    df = df.merge(hist, on=["policy_id", "year"], how="left")
    for p in PERILS[line]:
        col = f"f1x_notif_{p}"
        if col not in df.columns:
            df[col] = 0
        df[col] = df[col].fillna(0).astype("int32")

    # exposición previa observada de la póliza y tasa de notificación previa
    tr = df.loc[~df.is_test, ["policy_id", "year", "exposure"]]
    pe = tr.groupby(["policy_id", "year"])["exposure"].sum().unstack(fill_value=0.0)
    cum = pe.cumsum(axis=1).shift(1, axis=1).fillna(0.0)
    cum[2025] = pe.sum(axis=1)
    cum = cum.stack().rename("f1x_exposicion_previa").reset_index().rename(columns={"level_1": "year"})
    cum["year"] = cum["year"].astype(int)
    df = df.merge(cum, on=["policy_id", "year"], how="left")
    df["f1x_exposicion_previa"] = df["f1x_exposicion_previa"].fillna(0.0)
    df["f1x_tasa_notif_previa"] = np.where(df.f1x_exposicion_previa > 0, df.f1_notificaciones_previas / df.f1x_exposicion_previa.clip(lower=0.25), np.nan)

    # F1 por dirección (hogar): notificaciones de edificios y contenidos en la misma dirección antes del 1 de enero
    if line in ("buildings", "contents"):
        rows = pd.concat([pd.read_parquet(F0DIR / "tables" / f"f0_{l}.parquet", columns=["row_id", "address_id"]).assign(src=l) for l in ("buildings", "contents")], ignore_index=True)
        ah = a[a.line.isin(["buildings", "contents"]) & a.row_id.notna()].merge(rows[["row_id", "address_id"]], on="row_id", how="inner")
        other = "contents" if line == "buildings" else "buildings"
        parts = []
        for y in range(2021, 2026):
            prev = ah[ah.report_year < y]
            g = prev.groupby("address_id").agg(addr_notif_hogar=("claim_id", "size")).reset_index()
            o = prev[prev.line == other].groupby("address_id").agg(addr_notif_otro_ramo=("claim_id", "size")).reset_index()
            fl = prev[prev.peril.isin(["flood", "subsidence", "storm", "escape_of_water"])].groupby(["address_id", "peril"]).size().unstack(fill_value=0)
            fl.columns = [f"addr_notif_{c}" for c in fl.columns]
            g = g.merge(o, on="address_id", how="outer").merge(fl.reset_index(), on="address_id", how="outer")
            g["year"] = y
            parts.append(g)
        ad = pd.concat(parts, ignore_index=True)
        df = df.merge(ad, on=["address_id", "year"], how="left")
        for c in [c for c in df.columns if c.startswith("addr_notif_")]:
            df[c] = df[c].fillna(0).astype("int32")

    # F2: media del clima de todos los años anteriores al del periodo en la dirección
    w = pd.read_parquet(BASE / "Datos" / "weather.parquet").sort_values(["address_id", "year"])
    g = w.groupby("address_id")
    n_prev = g.cumcount()
    for v in WEATHER_VARS:
        s = g[v].cumsum() - w[v]
        w[f"{v}_mprev"] = np.where(n_prev > 0, s / n_prev.replace(0, np.nan), np.nan)
    w = w[["address_id", "year"] + [f"{v}_mprev" for v in WEATHER_VARS]]
    df = df.merge(w, on=["address_id", "year"], how="left")

    # F4: prima escrita anualizada relativa a la media del ramo en su año (sin tendencia del tarifario)
    rate = df.written_premium / df.exposure
    ymean = df.groupby("year")["written_premium"].sum() / df.groupby("year")["exposure"].sum()
    df["prem_rel"] = rate / df.year.map(ymean)
    df["log_prem_rel"] = np.log(df.prem_rel.clip(lower=1e-6))
    return df


def blocks_for(line: str, df: pd.DataFrame) -> dict:
    """Asigna cada columna candidata a un bloque; las nuevas siguen las reglas del contrato D68."""
    c = contract()[line]
    out = {}
    for col in df.columns:
        if col in NEVER or col.startswith("e_ef_"):
            continue
        if col in c:
            b = c[col]["bloque"]
        elif col.startswith("f1x_") or col.startswith("addr_notif"):
            b = "F1"
        elif col.endswith("_mprev"):
            b = "F2"
        elif col in ("prem_rel", "log_prem_rel"):
            b = "F4"
        else:
            b = "revisar"
        if b in ("F0", "F1", "F2", "F3", "F4"):
            out[col] = b
    return out


def to_model_frame(df: pd.DataFrame, feats: list[str]) -> pd.DataFrame:
    """Convierte a tipos de LightGBM sobre todas las filas del ramo a la vez, para compartir categorías entre cortes."""
    X = df[feats].copy()
    for col in X.columns:
        s = X[col]
        if isinstance(s.dtype, pd.CategoricalDtype):
            continue
        if pd.api.types.is_bool_dtype(s.dtype):
            X[col] = s.astype("float32")
        elif pd.api.types.is_numeric_dtype(s.dtype):
            if str(s.dtype).startswith(("Int", "Float")):
                X[col] = s.astype("float64")
        else:
            if pd.api.types.infer_dtype(s, skipna=True) == "boolean":
                X[col] = s.map({True: 1.0, False: 0.0}).astype("float32")
            else:
                X[col] = s.astype("category")
    return X
'''


### Embedded source: `src/modelos/construir_variables.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/modelos/construir_variables.py'] = '''"""Construye y guarda las matrices con variables adicionales por ramo en WORK/feat_<ramo>.parquet."""
import sys, time
from comun import LINES, WORK, build_extra
WORK.mkdir(parents=True, exist_ok=True)
for line in (sys.argv[1:] or LINES):
    t = time.time()
    df = build_extra(line)
    df.to_parquet(WORK / f"feat_{line}.parquet", index=False)
    extra = [c for c in df.columns if c.startswith(("f1x_", "addr_notif")) or c.endswith("_mprev") or c in ("prem_rel", "log_prem_rel")]
    print(line, df.shape, f"{time.time()-t:.0f}s", extra, flush=True)
'''


### Embedded source: `src/modelos/deriva_nivel.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/modelos/deriva_nivel.py'] = '''"""Deriva de nivel sin etiquetas de cada componente de una mezcla: total 2025 / total V24 frente al del ancla del ramo.

La regla s17 de mezcla3.py deja fuera de la referencia de nivel los componentes que se apartan más de un 1,5 %.
Anclas: CatBoost Tweedie de autos (CAT_motor_4s), LightGBM de edificios (S12h_edif_md1000) y de contenidos (S08h_hogar_F3s2).
Uso: python3 deriva_nivel.py <spec_json> [<salida.json>]
"""
import json
import sys

from comun import WORK
from mezcla3 import component, sin_referencia

ANCLA = {"motor": ("CAT_motor_4s", "cat1a"), "buildings": ("S12h_edif_md1000", "hS3"), "contents": ("S08h_hogar_F3s2", "hS2")}


def tot(run, sel, line):
    s, v, p = component(run, sel, line)
    return float(v.sum()), float(p.sum())


def main():
    spec = json.load(open(sys.argv[1]))
    excl = sin_referencia({"nivel": "s17"})
    rep = {}
    for line, comps in spec.items():
        a24, a25 = tot(*ANCLA[line], line)
        rep[line] = {}
        for run, sel, w in comps:
            v24, p25 = tot(run, sel, line)
            der = (p25 / v24) / (a25 / a24) - 1.0
            rep[line][run] = {"peso": w, "deriva": round(der, 4), "fuera_de_la_referencia": run in excl, "regla_coherente": (abs(der) > 0.015) == (run in excl) or run.startswith("MLP")}
            print(line, run, rep[line][run], flush=True)
    if len(sys.argv) > 2:
        (WORK / sys.argv[2]).write_text(json.dumps(rep, indent=1, ensure_ascii=False))


if __name__ == "__main__":
    main()
'''


### Embedded source: `src/modelos/diagnostico_cuerpo_cola.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/modelos/diagnostico_cuerpo_cola.py'] = '''"""Orden local de los candidatos enviados en el cuerpo de la distribución de 2024 frente a su orden en el público.

Diagnóstico, no criterio de selección. Para los candidatos con predicciones de V24 disponibles (A_G y H de Codex, leídos
sin modificarlos; S09b de la misma tabla de Codex; S21 y S22 de Claude) calcula el D² de 2024 con todas las filas y sin el
percentil superior de la etiqueta T0 (p99,9, p99,5 y p99). En la versión renivelada cada candidato se reescala por ramo con
el factor ftw óptimo dentro del subconjunto, para que el orden refleje la forma y no el nivel. Recortar por la etiqueta no
es una regla de puntuación válida: solo sirve para ver qué orden tienen los candidatos en el cuerpo de la distribución.
Uso: python3 diagnostico_cuerpo_cola.py <salida.json>
"""
from __future__ import annotations

import json
import sys

import numpy as np
import pandas as pd

from comun import WORK, d2

C = "/mnt/user-data/uploads/Hakaton/Hackaton/Semana 1/Modelado_codex/nocturna_20261017_N01"
PUBLICO = {"A_G": 0.12908, "S09b": 0.12889, "H": 0.12886, "S21": 0.12874}  # COD-20261018-049 y registro de envíos


def ftw(y, mu):
    return float((y / np.sqrt(mu)).sum() / np.sqrt(mu).sum())


def main():
    ag = pd.read_parquet(f"{C}/crossfit_meta_gam_guard/predictions_V24_crossfit.parquet", columns=["row_id", "line", "T0", "S09b", "meta_gam"]).set_index("row_id")
    hh = pd.read_parquet(f"{C}/geometric_meta_nested/V24.parquet", columns=["row_id", "candidate"]).set_index("row_id")
    df = ag.join(hh).rename(columns={"meta_gam": "A_G", "candidate": "H"})
    for k in ("s21", "s22"):
        p = pd.read_parquet(WORK / f"predicciones_v23_v24_{k}.parquet")
        df = df.join(p[p.corte == "V24"].set_index("row_id")[["prediction"]].rename(columns={"prediction": k.upper()}))
    cands = ["A_G", "S09b", "H", "S21", "S22"]
    assert df[cands].notna().all().all()
    y = df.T0.values
    rep = {"publico": PUBLICO, "orden_publico": sorted(PUBLICO, key=PUBLICO.get, reverse=True), "bruto": {}, "renivelado": {},
           "media_predicha_V24": {k: round(float(df[k].mean()), 2) for k in cands}, "media_T0_V24": round(float(y.mean()), 2)}
    for q in (None, 0.999, 0.995, 0.99):
        m = np.ones(len(y), bool) if q is None else (y <= np.quantile(y, q))
        lab = "completo" if q is None else f"sin_p{q * 100:g}"
        bruto, ren = {}, {}
        for k in cands:
            mu = df[k].values[m]; yy = y[m]; ln = df.line.values[m]
            bruto[k] = round(d2(yy, mu), 5)
            mu2 = mu.copy()
            for line in ("motor", "buildings", "contents"):
                l = ln == line
                mu2[l] = mu[l] * ftw(yy[l], mu[l])
            ren[k] = round(d2(yy, mu2), 5)
        rep["bruto"][lab] = {"d2": bruto, "orden": sorted(bruto, key=bruto.get, reverse=True)}
        rep["renivelado"][lab] = {"d2": ren, "orden": sorted(ren, key=ren.get, reverse=True)}
        print(lab, "bruto", rep["bruto"][lab]["orden"], "renivelado", rep["renivelado"][lab]["orden"], ren, flush=True)
    (WORK / sys.argv[1]).write_text(json.dumps(rep, indent=1, ensure_ascii=False))


if __name__ == "__main__":
    main()
'''


### Embedded source: `src/modelos/diagnostico_forma_nivel.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/modelos/diagnostico_forma_nivel.py'] = '''"""Efecto de la calibración de forma sobre el nivel de 2025 frente al de V23 y V24 (diagnóstico sin etiquetas de 2025).

La forma mu' = e (e/ē)^(b-1) g (r/g)^a cambia el total de cada corte. En V23 y V24 ese cambio lo absorbe el factor ftw,
que se ajusta después de la forma; en 2025 no hay etiqueta y el factor es la media de los dos. Si la forma cambia el total
de 2025 en una proporción distinta que en V23/V24 (porque 2025 usa exposición nominal y V23/V24 la efectiva, o porque la
dispersión de tasas cambia), el nivel de 2025 se desplaza respecto de la misma mezcla sin forma. Este script mide, por
ramo, L = total con forma / total sin forma en cada corte, los factores con y sin forma y el total 2025 resultante.
Uso: python3 diagnostico_forma_nivel.py <spec_json> <forma> <nivel|-> [<salida.json>]
"""
from __future__ import annotations

import json
import sys

import numpy as np
import pandas as pd

from comun import LINES, WORK, d2
from mezcla3 import component, ftw, labels, shape_rate, sin_referencia


def main():
    spec = json.load(open(sys.argv[1]))
    forma = {kv.split(":")[0]: (float(kv.split(":")[1]), float(kv.split(":")[2])) for kv in sys.argv[2].split(",") if kv}
    nivel = sys.argv[3]
    excl = sin_referencia({"nivel": None if nivel == "-" else nivel})
    rep = {}
    for line in LINES:
        ysel, y24, idx25 = labels(line)
        fe = pd.read_parquet(WORK / f"feat_{line}.parquet", columns=["row_id", "e_ef_coste", "exposure"])
        fe["row_id"] = fe.row_id.astype(str); fe = fe.set_index("row_id")
        ex = (fe.e_ef_coste.reindex(ysel.index).values, fe.e_ef_coste.reindex(y24.index).values, fe.exposure.reindex(idx25).values)
        exn = (fe.exposure.reindex(ysel.index).values, fe.exposure.reindex(y24.index).values, fe.exposure.reindex(idx25).values)
        raw = []
        for run, sel_name, w in spec[line]:
            s, v, p = component(run, sel_name, line)
            raw.append((run, w, s.reindex(ysel.index).values, v.reindex(y24.index).values, p.reindex(idx25).values))
        own = [r for r in raw if r[0] not in excl]
        wo = sum(r[1] for r in own)
        ref = [sum(r[1] * r[j].sum() for r in own) / wo for j in (2, 3, 4)]
        ws = np.array([r[1] for r in raw], float); ws /= ws.sum()
        blends = [sum(w * r[j] * ref[j - 2] / r[j].sum() for w, r in zip(ws, raw)) for j in (2, 3, 4)]
        a, b = forma.get(line, (1.0, 1.0))
        shaped = [shape_rate(m, e, a, b) for m, e in zip(blends, ex)]
        L = [float(s.sum() / m.sum()) for s, m in zip(shaped, blends)]
        f_sin = [ftw(ysel.values, blends[0]), ftw(y24.values, blends[1])]
        f_con = [ftw(ysel.values, shaped[0]), ftw(y24.values, shaped[1])]
        tot25_sin = float(blends[2].sum() * np.mean(f_sin)); tot25_con = float(shaped[2].sum() * np.mean(f_con))
        # dispersión de la exposición usada en cada corte: media de (e/ē)^(b-1) y cuantiles de e
        disp = {}
        for k, e in zip(("V23", "V24", "2025"), ex):
            eb = np.exp(np.log(e).mean())
            disp[k] = {"media_(e/eb)^(b-1)": round(float(np.mean((e / eb) ** (b - 1.0))), 4), "e_p10_p50_p90": [round(float(q), 3) for q in np.quantile(e, [0.1, 0.5, 0.9])],
                       "media_e": round(float(e.mean()), 4)}
        rep[line] = {"forma": [a, b], "L_V23_V24_2025": [round(x, 4) for x in L], "f_sin_forma": [round(x, 4) for x in f_sin], "f_con_forma": [round(x, 4) for x in f_con],
                     "total_2025_sin_forma": round(tot25_sin, 1), "total_2025_con_forma": round(tot25_con, 1), "cociente_con_sin": round(tot25_con / tot25_sin, 4),
                     "d2_sin": [round(d2(ysel.values, blends[0]), 5), round(d2(y24.values, blends[1] * f_sin[0]), 5)],
                     "d2_con": [round(d2(ysel.values, shaped[0]), 5), round(d2(y24.values, shaped[1] * f_con[0]), 5)], "exposicion": disp}
        print(line, json.dumps(rep[line], ensure_ascii=False), flush=True)
    if len(sys.argv) > 4:
        (WORK / sys.argv[4]).write_text(json.dumps(rep, indent=1, ensure_ascii=False))


if __name__ == "__main__":
    main()
'''


### Embedded source: `src/modelos/escenarios_reserva.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/modelos/escenarios_reserva.py'] = '''"""Sensibilidad de las comparaciones a la reserva de caso: etiquetas T1 con la reserva de los siniestros abiertos × 0,75 y × 1,25.

T0 suma, por periodo, incurrido menos recobros de los siniestros aceptados asignados (D67). T1(a) (metodología D35) cambia
solo los abiertos: pagado + a × (incurrido − pagado) − recobros, así que T1(a) = T0 + (a − 1) × R, con R la reserva de
caso de los abiertos aceptados de la fila. Las reservas son escenarios de evaluación, nunca predictores, y las
predicciones no cambian (incluido su factor de año siguiente, estimado con T0). Para cada escenario se informa el D² de
los candidatos de Claude en V23 selección y V24, la diferencia frente al primero de la lista con bootstrap por cliente
(Poisson(1), 1.000 réplicas, denominador fijo) y, en V24, frente a A_G y H de Codex (lectura sin modificar).
Uso: python3 escenarios_reserva.py <salida.json> <ref>=<spec>[@<forma>[@<nivel>]] <cand>=<spec>...
"""
from __future__ import annotations

import json
import sys

import numpy as np
import pandas as pd

from comun import BASE, BASESDIR, LINES, WORK, d2, deviance
from mezcla3 import labels
from robustez import blend, parse

C = "/mnt/user-data/uploads/Hakaton/Hackaton/Semana 1/Modelado_codex/nocturna_20261017_N01"
CLAIMS = BASE / "Limpieza" / "runs" / "preaudit_20261016T211816Z_817285a5" / "data" / "claims.parquet"
A = (0.75, 1.0, 1.25)
B = 1000


def reserva_por_fila():
    c = pd.read_parquet(CLAIMS, columns=["claim_id", "status", "paid_to_date", "incurred_to_date", "audit_pre00"])
    a = pd.read_parquet(BASESDIR / "tables" / "asignacion_por_siniestro.parquet", columns=["claim_id", "category", "row_id"])
    m = a.merge(c, on="claim_id", how="left", validate="one_to_one")
    ok = (m.status == "open") & (m.category == "unica") & (~m.audit_pre00.fillna(False).astype(bool)) & m.row_id.notna()
    r = (m.incurred_to_date - m.paid_to_date)[ok].groupby(m.row_id[ok].astype(str)).sum()
    return r


def boot(dif, groups, null, rng):
    codes, inv = np.unique(groups, return_inverse=True)
    d = np.bincount(inv, weights=dif)
    bs = np.concatenate([rng.poisson(1.0, size=(50, len(d))) @ d for _ in range(B // 50)]) / null
    return [round(float(np.quantile(bs, 0.025)), 6), round(float(np.quantile(bs, 0.975)), 6)], round(float((bs > 0).mean()), 3)


def main():
    out = sys.argv[1]
    cands = [parse(x) for x in sys.argv[2:]]
    ref = cands[0][0]
    R = reserva_por_fila()
    ag = pd.read_parquet(f"{C}/crossfit_meta_gam_guard/predictions_V24_crossfit.parquet", columns=["row_id", "meta_gam"]).set_index("row_id").meta_gam
    hh = pd.read_parquet(f"{C}/geometric_meta_nested/V24.parquet", columns=["row_id", "candidate"]).set_index("row_id").candidate
    rng = np.random.default_rng(20261018)
    store = {f: {"y": [], "r": [], "cu": [], "P": {}} for f in ("V23", "V24")}
    for line in LINES:
        ysel, y24, idx25 = labels(line)
        fe = pd.read_parquet(WORK / f"feat_{line}.parquet", columns=["row_id", "customer_id", "e_ef_coste", "exposure"])
        fe["row_id"] = fe.row_id.astype(str); fe = fe.set_index("row_id")
        expos = (fe.e_ef_coste.reindex(ysel.index).values, fe.e_ef_coste.reindex(y24.index).values, fe.exposure.reindex(idx25).values)
        for f, ys in (("V23", ysel), ("V24", y24)):
            store[f]["y"].append(ys.values); store[f]["r"].append(R.reindex(ys.index).fillna(0.0).values)
            store[f]["cu"].append(fe.customer_id.reindex(ys.index).astype(str).values)
        for name, spec, forma in cands:
            _, bsel, b24, _ = blend(spec, forma, line, ysel, y24, idx25, expos)
            store["V23"]["P"].setdefault(name, []).append(bsel); store["V24"]["P"].setdefault(name, []).append(b24)
        store["V24"]["P"].setdefault("A_G", []).append(ag.reindex(y24.index).values)
        store["V24"]["P"].setdefault("H", []).append(hh.reindex(y24.index).values)
        store["V23"].setdefault("line", []).append(np.full(len(ysel), line)); store["V24"].setdefault("line", []).append(np.full(len(y24), line))
    rep = {"referencia": ref, "reserva_abiertos_CU": {f: round(float(np.concatenate(store[f]["r"]).sum()), 0) for f in store}, "escenarios": {}}
    for a in A:
        ra = {}
        for f in ("V23", "V24"):
            y0 = np.concatenate(store[f]["y"]); r = np.concatenate(store[f]["r"]); cu = np.concatenate(store[f]["cu"]); ln = np.concatenate(store[f]["line"])
            y = y0 + (a - 1.0) * r
            P = {k: np.concatenate(v) for k, v in store[f]["P"].items()}
            res = {}
            for scope in LINES + ["agregado"]:
                msk = np.ones(len(y), bool) if scope == "agregado" else (ln == scope)
                ys, cus = y[msk], cu[msk]
                null = deviance(ys, np.full_like(ys, ys.mean())).sum()
                row = {"d2": {k: round(d2(ys, p[msk]), 5) for k, p in P.items()}}
                for k in P:
                    if k == ref:
                        continue
                    dif = deviance(ys, P[ref][msk]) - deviance(ys, P[k][msk])
                    ic, pm = boot(dif, cus, null, rng)
                    row[f"{k}_menos_{ref}"] = {"dif": round(float(dif.sum() / null), 6), "ic95_cliente": ic, "P_mejora": pm}
                res[scope] = row
            ra[f] = res
            print(f"a={a} {f} agregado", {k: v for k, v in res["agregado"]["d2"].items()}, flush=True)
            for scope in LINES + ["agregado"]:
                print("   ", scope, {k: (v["dif"], v["ic95_cliente"]) for k, v in res[scope].items() if k != "d2"}, flush=True)
        rep["escenarios"][str(a)] = ra
    (WORK / out).write_text(json.dumps(rep, indent=1, ensure_ascii=False))


if __name__ == "__main__":
    main()
'''


### Embedded source: `src/modelos/evaluar.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/modelos/evaluar.py'] = '''"""Evalúa una configuración por ramo en V23 (parada en la mitad de calibración, comparación en la de selección).

Modos: 'total' ajusta T0 del ramo con el offset e_ef_coste; 'componentes' ajusta un modelo por peligro con su etiqueta
y_<peligro> y su offset e_ef_<peligro>, y suma las predicciones. La configuración puede fijar 'cap' por ramo o por
peligro ('caps': {"bodily_injury": 15000}). Guarda las predicciones de selección para mezclas posteriores.
Uso: python3 evaluar.py <nombre> <conjunto> <config_json> <modo> [ramos]
La configuración admite "drop" y "add" (listas de columnas) por ramo.
"""
from __future__ import annotations

import json
import sys
import time

import numpy as np
import pandas as pd

from comun import LINES, PERILS, WORK, d2, to_model_frame
from experimentos import features, fit, load, predict


def main():
    name, set_name, cfg_path, mode = sys.argv[1:5]
    cfg = json.loads(open(cfg_path).read())
    rep, ys, mus, frames = {}, [], [], []
    lines = sys.argv[5].split(",") if len(sys.argv) > 5 else LINES
    for line in lines:
        t = time.time()
        df = load(line)
        feats = [f for f in features(line, df, set_name) if f not in set(cfg[line].get("drop", []))]
        feats += [f for f in cfg[line].get("add", []) if f in df.columns and f not in feats]
        X = to_model_frame(df, feats)
        tr = df.year.isin([2021, 2022]).values
        cal = ((df.year == 2023) & (df.cal_half == 0)).values
        sel = ((df.year == 2023) & (df.cal_half == 1)).values
        y = df["T0"].fillna(0).values
        if mode == "total":
            off = df["e_ef_coste"].values
            b, base = fit(X[tr], y[tr], off[tr], X[cal], y[cal], off[cal], rounds=6000, early=200, params=dict(cfg[line]["params"]))
            best = b.best_iteration or b.current_iteration()
            mu = predict(b, base, X[sel], off[sel], best)
            info = {"best": best}
        else:
            mu = np.zeros(sel.sum())
            info = {}
            for p in PERILS[line]:
                yp = df[f"y_{p}"].fillna(0).values
                op = df[f"e_ef_{p}"].values
                params = dict(cfg[line]["params"])
                params.pop("cap", None)
                c = cfg[line].get("caps", {}).get(p)
                if c:
                    params["cap"] = c
                b, base = fit(X[tr], yp[tr], op[tr], X[cal], yp[cal], op[cal], rounds=6000, early=200, params=params)
                best = b.best_iteration or b.current_iteration()
                mp = predict(b, base, X[sel], op[sel], best)
                info[p] = {"best": best, "d2": round(d2(yp[sel], mp), 5)}
                mu += mp
        rep[line] = {"d2_sel": round(d2(y[sel], mu), 5), "balance_sel": round(float(mu.sum() / y[sel].sum()), 4), "n_feats": len(feats), **info, "secs": round(time.time() - t)}
        print(name, line, rep[line], flush=True)
        ys.append(y[sel]); mus.append(mu)
        frames.append(pd.DataFrame({"row_id": df.row_id[sel].astype(str).values, "line": line, "y": y[sel], "mu": mu}))
    rep["global"] = round(d2(np.concatenate(ys), np.concatenate(mus)), 5)
    print("== ", name, "global sel", rep["global"], flush=True)
    pd.concat(frames, ignore_index=True).to_parquet(WORK / f"sel_{name}.parquet", index=False)
    (WORK / f"eval_{name}.json").write_text(json.dumps(rep, indent=1, default=float))


if __name__ == "__main__":
    main()
'''


### Embedded source: `src/modelos/experimentos.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/modelos/experimentos.py'] = '''"""Experimentos LightGBM Tweedie por ramo con offset de exposición efectiva (T0 a nivel último, D67/D70).

V23: entrena 2021-2022, parada temprana en la mitad de calibración de 2023 (cal_half = 0) y comparación en la mitad de
selección (cal_half = 1). V24: entrena 2021-2023 con las rondas fijadas en V23 y evalúa 2024 (consulta registrada).
El D² se calcula sobre importes con potencia 1,5, global (tres ramos concatenados) y por ramo.

Uso: python3 experimentos.py <nombre_salida> <conjunto>[,<conjunto>...] [--v24]
Conjuntos: M00, F0, F012, F0124, F01234 (diagnóstico con F3 sin marketing_segment), F01234m (con marketing_segment)
"""
from __future__ import annotations

import json
import sys
import time

import lightgbm as lgb
import numpy as np
import pandas as pd

from comun import F0DIR, LINES, WORK, blocks_for, d2, to_model_frame

PARAMS = dict(objective="tweedie", tweedie_variance_power=1.5, learning_rate=0.05, num_leaves=31, min_data_in_leaf=500,
              feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, lambda_l2=10.0, max_bin=255, num_threads=2,
              verbose=-1, boost_from_average=False, cat_smooth=50, min_data_per_group=200, max_cat_to_onehot=8, seed=20261017)
SETS = {"F0": ["F0"], "F012": ["F0", "F1", "F2"], "F0124": ["F0", "F1", "F2", "F4"], "F01234": ["F0", "F1", "F2", "F3", "F4"], "F01234m": ["F0", "F1", "F2", "F3", "F4"]}
ALWAYS_OUT = {"marketing_segment"}


def load(line):
    df = pd.read_parquet(WORK / f"feat_{line}.parquet")
    part = pd.read_parquet(F0DIR / "tables" / "particiones.parquet", columns=["row_id", "cal_half"])
    df = df.merge(part, on="row_id", how="left", validate="one_to_one")
    return df


def features(line, df, set_name):
    blocks = blocks_for(line, df)
    keep = SETS[set_name]
    feats = [c for c, b in blocks.items() if b in keep]
    if set_name != "F01234m":
        feats = [c for c in feats if c not in ALWAYS_OUT]
    return feats


def fit(X, y, off, Xv=None, yv=None, offv=None, rounds=4000, early=150, params=None):
    """Ajusta LightGBM Tweedie con offset log(exposición). Con 'cap' en params, la etiqueta de entrenamiento se acota por
    fila y el exceso se devuelve como recargo uniforme (booster.loading); la evaluación siempre usa la etiqueta completa."""
    params = dict(PARAMS, **(params or {}))
    cap = params.pop("cap", None)
    loading = 1.0
    if cap:
        yc = np.minimum(y, cap)
        loading = float(y.sum() / yc.sum())
        y = yc
        if yv is not None:
            yv = np.minimum(yv, cap)
    base = float(np.log(y.sum() / off.sum()))
    dtr = lgb.Dataset(X, label=y, init_score=np.log(off) + base, free_raw_data=False)
    valid = []
    if Xv is not None:
        valid = [lgb.Dataset(Xv, label=yv, init_score=np.log(offv) + base, reference=dtr)]
    cb = [lgb.early_stopping(early, verbose=False)] if (Xv is not None and early) else []
    booster = lgb.train(params, dtr, num_boost_round=rounds, valid_sets=valid, callbacks=cb)
    booster.loading = loading
    return booster, base


def predict(booster, base, X, off, iters=None):
    raw = booster.predict(X, raw_score=True, num_iteration=iters)
    return np.exp(raw + base + np.log(off)) * getattr(booster, "loading", 1.0)


def main():
    out_name = sys.argv[1]
    sets = sys.argv[2].split(",")
    do_v24 = "--v24" in sys.argv
    extra = {}
    for a in sys.argv[3:]:
        if a.startswith("--param="):
            k, v = a[len("--param="):].split(":", 1)
            extra[k] = float(v) if v.replace(".", "", 1).replace("-", "", 1).isdigit() else v
    results = {"params": dict(PARAMS, **extra), "sets": {}}
    for set_name in sets:
        sel_y, sel_mu, v24_y, v24_mu = [], [], [], []
        per_line = {}
        for line in LINES:
            t = time.time()
            df = load(line)
            tr = df.year.isin([2021, 2022]).values
            cal = ((df.year == 2023) & (df.cal_half == 0)).values
            sel = ((df.year == 2023) & (df.cal_half == 1)).values
            y = df["T0"].fillna(0).values
            off = df["e_ef_coste"].values
            if set_name == "M00":
                rate = y[tr].sum() / off[tr].sum()
                mu_sel = rate * off[sel]
                best = 0
                info = {"rate": rate}
                if do_v24:
                    tr24 = (df.year <= 2023).values
                    ev24 = (df.year == 2024).values
                    mu24 = y[tr24].sum() / off[tr24].sum() * off[ev24]
            else:
                feats = features(line, df, set_name)
                X = to_model_frame(df, feats)
                booster, base = fit(X[tr], y[tr], off[tr], X[cal], y[cal], off[cal], params=extra)
                best = booster.best_iteration or booster.current_iteration()
                mu_sel = predict(booster, base, X[sel], off[sel], best)
                imp = pd.Series(booster.feature_importance("gain"), index=feats).sort_values(ascending=False)
                info = {"best_iter": best, "n_feats": len(feats), "top_gain": {k: round(float(v / imp.sum()), 4) for k, v in imp.head(15).items()}}
                if do_v24:
                    tr24 = (df.year <= 2023).values
                    ev24 = (df.year == 2024).values
                    b24, base24 = fit(X[tr24], y[tr24], off[tr24], rounds=int(round(best * 1.1)) or 100, early=None, params=extra)
                    mu24 = predict(b24, base24, X[ev24], off[ev24])
            per_line[line] = {"d2_sel": round(d2(y[sel], mu_sel), 5), "balance_sel": round(float(mu_sel.sum() / y[sel].sum()), 4), **info, "secs": round(time.time() - t)}
            sel_y.append(y[sel]); sel_mu.append(mu_sel)
            if do_v24:
                per_line[line]["d2_v24"] = round(d2(y[ev24], mu24), 5)
                per_line[line]["balance_v24"] = round(float(mu24.sum() / y[ev24].sum()), 4)
                v24_y.append(y[ev24]); v24_mu.append(mu24)
            print(set_name, line, per_line[line], flush=True)
        res = {"d2_sel_global": round(d2(np.concatenate(sel_y), np.concatenate(sel_mu)), 5), "lines": per_line}
        if do_v24:
            res["d2_v24_global"] = round(d2(np.concatenate(v24_y), np.concatenate(v24_mu)), 5)
        results["sets"][set_name] = res
        print("== ", set_name, "global sel", res["d2_sel_global"], "v24", res.get("d2_v24_global"), flush=True)
        (WORK / f"exp_{out_name}.json").write_text(json.dumps(results, indent=1, default=float))


if __name__ == "__main__":
    main()
'''


### Embedded source: `src/modelos/exportar_predicciones.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/modelos/exportar_predicciones.py'] = '''"""Exporta las predicciones por fila de una mezcla de Claude en V23 selección y V24, y resume las medias por ramo.

Las predicciones son exactamente las que miden los D² locales de mezcla3.py y robustez.py:
- V23: entrenamiento 2021-22, evaluación 2023 con cal_half = 1 (corte de selección), mezcla con forma y regla de nivel,
  sin factor de año siguiente (d2_sel);
- V24: entrenamiento 2021-23, evaluación 2024, misma mezcla multiplicada por el factor ftw de V23 (d2_v24_f23).
Cada fila lleva la etiqueta T0 y la reserva de caso de los siniestros abiertos aceptados asignados a ella
(escenarios_reserva.reserva_por_fila), para que la revisión pueda rehacer los escenarios T0 + (a - 1) x reserva.
Las medias por ramo comparan la media observada con la predicha (y su cociente) con a = 0,75, 1 y 1,25.
Uso: python3 exportar_predicciones.py <clave> <cand>=<spec>@<forma>@<nivel>
Escribe WORK/predicciones_v23_v24_<clave>.parquet y WORK/medias_por_ramo_<clave>.json
"""
from __future__ import annotations

import json
import sys

import numpy as np
import pandas as pd

from comun import LINES, WORK, d2
from escenarios_reserva import reserva_por_fila
from mezcla3 import labels
from robustez import blend, parse


def main():
    key = sys.argv[1]
    cn, spec, fm = parse(sys.argv[2])
    R = reserva_por_fila()
    filas, medias = [], {}
    for line in LINES:
        ysel, y24, idx25 = labels(line)
        fe = pd.read_parquet(WORK / f"feat_{line}.parquet", columns=["row_id", "e_ef_coste", "exposure"])
        fe["row_id"] = fe.row_id.astype(str); fe = fe.set_index("row_id")
        expos = (fe.e_ef_coste.reindex(ysel.index).values, fe.e_ef_coste.reindex(y24.index).values, fe.exposure.reindex(idx25).values)
        res, bsel, b24, b25 = blend(spec, fm, line, ysel, y24, idx25, expos)
        medias[line] = {}
        for corte, ys, pr in (("V23", ysel, bsel), ("V24", y24, b24)):
            r = R.reindex(ys.index.astype(str)).fillna(0.0).values
            filas.append(pd.DataFrame({"row_id": ys.index.astype(str), "line": line, "corte": corte, "y_T0": ys.values,
                                       "reserva_caso": r, "prediction": pr}))
            m = {"filas": int(len(ys)), "media_predicha": round(float(pr.mean()), 4)}
            for a in (0.75, 1.0, 1.25):
                y = ys.values + (a - 1.0) * r
                m[f"a={a}"] = {"media_observada": round(float(y.mean()), 4), "observada_entre_predicha": round(float(y.sum() / pr.sum()), 4),
                               "d2": round(d2(y, pr), 6)}
            medias[line][corte] = m
        medias[line]["2025"] = {"filas": int(len(idx25)), "media_predicha": round(float(b25.mean()), 4)}
        print(line, json.dumps(medias[line], ensure_ascii=False), flush=True)
    df = pd.concat(filas, ignore_index=True)
    df.to_parquet(WORK / f"predicciones_v23_v24_{key}.parquet", index=False)
    agg = {}
    for corte in ("V23", "V24"):
        d = df[df.corte == corte]
        agg[corte] = {"filas": int(len(d)), "media_predicha": round(float(d.prediction.mean()), 4)}
        for a in (0.75, 1.0, 1.25):
            y = d.y_T0.values + (a - 1.0) * d.reserva_caso.values
            agg[corte][f"a={a}"] = {"media_observada": round(float(y.mean()), 4), "observada_entre_predicha": round(float(y.sum() / d.prediction.sum()), 4),
                                    "d2": round(d2(y, d.prediction.values), 6)}
    medias["agregado"] = agg
    rep = {"candidato": cn, "medias": medias,
           "definiciones": {"V23": "selección: entrenamiento 2021-22, evaluación 2023 con cal_half = 1; mezcla con forma y regla de nivel, sin factor (d2_sel)",
                            "V24": "entrenamiento 2021-23, evaluación 2024; mezcla con forma y regla de nivel por el factor ftw de V23 (d2_v24_f23)",
                            "a": "etiqueta T0 + (a - 1) x reserva de caso de los siniestros abiertos aceptados asignados a la fila"}}
    (WORK / f"medias_por_ramo_{key}.json").write_text(json.dumps(rep, indent=1, ensure_ascii=False))
    print(json.dumps(agg, ensure_ascii=False), flush=True)


if __name__ == "__main__":
    main()
'''


### Embedded source: `src/modelos/frecsev2.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/modelos/frecsev2.py'] = '''"""Frecuencia por peligro (LightGBM Poisson, offset e_ef del peligro) por severidad modelada del peligro (LightGBM gamma).

La severidad positiva media por fila (y_p / n_p en filas con y_p > 0, peso n_p) se ajusta con un gamma muy regularizado
cuando el peligro tiene al menos MIN_SEV siniestros positivos en el entrenamiento; si no, severidad media constante.
Se multiplica por la proporción de siniestros con coste positivo del peligro, para que la media incluya los de coste cero.
Hogar con las siete variables de cliente D80; autos con F01234. V23: paradas tempranas en la mitad de calibración y
comparación en la de selección. Con --prod: V24 (rondas x 1,15) y 2025 (x 1,3), con las semillas indicadas.
Con --aditivo, frecuencia y severidad usan árboles de una sola división (num_leaves 2, tasa 0,05): un modelo aditivo por
peligro sin interacciones, del estilo de un GAM ajustado por boosting (idea compartida por Codex en el foro; implementación
propia de Claude, sin predicciones de Codex).
Con --desde=<corrida> --semilla_inicial=k reutiliza las iteraciones de V23 de otra corrida y entrena solo V24 y 2025 con
semillas nuevas de la frecuencia (la severidad gamma es determinista), para promediar con combinar_semillas.py.
Con --partes guarda además, por peligro, la frecuencia con el formato de cat_frecsev.py (v23_<p>.npz con best y lam;
v24_<p>.npy y prod_<p>.npy con frecuencia x severidad media del entrenamiento), para aplicar después severidad_cfs.py.
Uso: python3 frecsev2.py <nombre> <ramo> [--prod] [--semillas=2] [--aditivo] [--desde=<corrida>] [--semilla_inicial=0] [--partes]
"""
from __future__ import annotations

import json
import sys
import time

import numpy as np
import pandas as pd

from comun import PERILS, WORK, d2, to_model_frame
from experimentos import features, fit, load, predict

MIN_SEV = 1500
F3S2 = ["prior_claims_5y", "credit_band", "missed_payments_12m", "household_size", "years_at_address", "marital_status", "licence_points"]
SEV_PARAMS = {"objective": "gamma", "learning_rate": 0.03, "num_leaves": 5, "min_data_in_leaf": 300, "feature_fraction": 0.7, "lambda_l2": 20.0,
              "cat_smooth": 100, "min_data_per_group": 300, "max_cat_threshold": 16}


def main():
    name, line = sys.argv[1], sys.argv[2]
    opts = dict((a[2:].split("=") + ["1"])[:2] for a in sys.argv[3:] if a.startswith("--"))
    seeds = int(opts.get("semillas", 2))
    s0 = int(opts.get("semilla_inicial", 0))  # semillas nuevas para promediar con una corrida previa
    t0 = time.time()
    cfg = json.load(open(WORK / "cfg_total.json"))[line]["params"]
    df = load(line)
    feats = features(line, df, "F01234" if line == "motor" else "F0124")
    if line != "motor":
        feats += [c for c in F3S2 if c in df.columns and c not in feats]
    X = to_model_frame(df, feats)
    y = df["T0"].fillna(0).values
    yr = df.year.values
    expo = df["exposure"].values
    tr = yr <= 2022; cal = (yr == 2023) & (df.cal_half.values == 0); sel = (yr == 2023) & (df.cal_half.values == 1)
    pf = dict(cfg); pf.pop("cap", None); pf["objective"] = "poisson"; pf.pop("tweedie_variance_power", None)
    sev_params = dict(SEV_PARAMS)
    if "aditivo" in opts:
        pf.update({"num_leaves": 2, "learning_rate": 0.05, "min_data_in_leaf": 2000})
        sev_params.update({"num_leaves": 2, "learning_rate": 0.05})
    rep = {"line": line, "aditivo": "aditivo" in opts, "perils": {}}
    out = WORK / name
    out.mkdir(parents=True, exist_ok=True)
    best = {}

    def sev_fit(p, n, yp, trm, valm=None, rounds=None):
        pos_tr = trm & (yp > 0) & (n > 0)
        share = float((yp[trm] > 0).sum() / max((n[trm] > 0).sum(), 1))  # filas con siniestro que tienen coste positivo
        mean_pos = float(yp[pos_tr].sum() / n[pos_tr].sum())
        if pos_tr.sum() < MIN_SEV:
            return None, mean_pos, share, 0
        s_obs = yp / np.maximum(n, 1)
        params = dict(sev_params, num_threads=2, verbose=-1, seed=20261017)
        import lightgbm as lgb
        dtr = lgb.Dataset(X[pos_tr], label=s_obs[pos_tr], weight=n[pos_tr], init_score=np.full(pos_tr.sum(), np.log(mean_pos)))
        if valm is not None:
            pos_v = valm & (yp > 0) & (n > 0)
            dv = lgb.Dataset(X[pos_v], label=s_obs[pos_v], weight=n[pos_v], init_score=np.full(pos_v.sum(), np.log(mean_pos)), reference=dtr)
            b = lgb.train(params, dtr, num_boost_round=3000, valid_sets=[dv], callbacks=[lgb.early_stopping(150, verbose=False)])
            it = max(b.best_iteration, 10)
        else:
            b = lgb.train(params, dtr, num_boost_round=rounds)
            it = rounds
        return b, mean_pos, share, it

    def sev_pred(b, mean_pos, share, m, it):
        if b is None:
            return np.full(m.sum(), mean_pos * share)
        return np.exp(b.predict(X[m], raw_score=True, num_iteration=it) + np.log(mean_pos)) * share

    mu_sel = np.zeros(sel.sum())
    if "desde" in opts:
        src = json.loads((WORK / opts["desde"] / "resumen.json").read_text())
        best = {p: (int(src["perils"][p]["best_frec"]), int(src["perils"][p]["best_sev"])) for p in PERILS[line]}
        rep.update({"desde": opts["desde"], "semilla_inicial": s0, "perils": {p: {"best_frec": best[p][0], "best_sev": best[p][1]} for p in PERILS[line]}})
    for p in ([] if "desde" in opts else PERILS[line]):
        n = df[f"n_{p}"].fillna(0).values.astype(float)
        yp = df[f"y_{p}"].fillna(0).values
        off = df[f"e_ef_{p}"].values
        b, base = fit(X[tr], n[tr], off[tr], X[cal], n[cal], off[cal], rounds=6000, early=200, params=dict(pf))
        bf = b.best_iteration or b.current_iteration()
        lam = predict(b, base, X[sel], off[sel], bf)
        if "partes" in opts:
            np.savez(out / f"v23_{p}.npz", best=bf, lam=lam)
        sb, mean_pos, share, sit = sev_fit(p, n, yp, tr, cal)
        sv = sev_pred(sb, mean_pos, share, sel, sit if sb is not None else None)
        best[p] = (bf, sit)
        mu_sel += lam * sv
        sv_const = mean_pos * share
        rep["perils"][p] = {"best_frec": bf, "best_sev": sit, "sev_media": round(sv_const, 1),
                            "d2_peligro_sev_modelo": round(d2(yp[sel], lam * sv), 5), "d2_peligro_sev_constante": round(d2(yp[sel], lam * sv_const), 5)}
        print(name, p, rep["perils"][p], round(time.time() - t0), flush=True)
    if "desde" not in opts:
        rep["d2_sel"] = round(d2(y[sel], mu_sel), 5)
        print(name, "V23 selección", rep["d2_sel"], flush=True)
        pd.DataFrame({"row_id": df.row_id[sel].astype(str).values, "line": line, "y": y[sel], "mu": mu_sel}).to_parquet(WORK / f"sel_{name}.parquet", index=False)
    (out / "resumen.json").write_text(json.dumps(rep, indent=1))
    if "prod" not in opts:
        return
    for tag, trm, evm, mult in (("v24", yr <= 2023, yr == 2024, 1.15), ("2025", yr <= 2024, yr == 2025, 1.3)):
        mu = np.zeros(evm.sum())
        for p in PERILS[line]:
            n = df[f"n_{p}"].fillna(0).values.astype(float)
            yp = df[f"y_{p}"].fillna(0).values
            off = df[f"e_ef_{p}"].values
            offe = off[evm] if tag == "v24" else expo[evm]
            bf, sit = best[p]
            if sit:
                sb, mean_pos, share, _ = sev_fit(p, n, yp, trm, None, rounds=int(round(sit * mult)))
            else:
                pos = trm & (yp > 0) & (n > 0)
                sb, mean_pos, share = None, float(yp[pos].sum() / n[pos].sum()), float((yp[trm] > 0).sum() / max((n[trm] > 0).sum(), 1))
            sv = sev_pred(sb, mean_pos, share, evm, None)
            freq = np.zeros(evm.sum())
            for s in range(seeds):
                b, base = fit(X[trm], n[trm], off[trm], rounds=int(round(bf * mult)), early=None, params=dict(pf, seed=20261017 + 101 * (s0 + s)))
                freq += np.exp(b.predict(X[evm], raw_score=True) + base + np.log(offe)) / seeds
            mu += freq * sv
            if "partes" in opts:
                # formato de cat_frecsev.py: frecuencia por severidad media (coste / número de siniestros) del entrenamiento
                np.save(out / f"{'v24' if tag == 'v24' else 'prod'}_{p}.npy", freq * (yp[trm].sum() / max(n[trm].sum(), 1.0)))
        if tag == "v24":
            rep["d2_v24"] = round(d2(y[evm], mu), 5)
            pd.DataFrame({"row_id": df.row_id[evm].astype(str).values, "line": line, "y": y[evm], "mu": mu}).to_parquet(out / "v24.parquet", index=False)
            print(name, "V24", rep["d2_v24"], round(time.time() - t0), flush=True)
        else:
            rep["pred_2025_total_CU"] = round(float(mu.sum()), 2)
            pd.DataFrame({"row_id": df.row_id[evm].astype(str).values, "line": line, "prediction": mu}).to_parquet(out / "prediccion_2025.parquet", index=False)
        (out / "resumen.json").write_text(json.dumps(rep, indent=1))
    print(name, json.dumps(rep), flush=True)


if __name__ == "__main__":
    main()
'''


### Embedded source: `src/modelos/influencia_reservas.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/modelos/influencia_reservas.py'] = '''"""Comparación en 2024 de una mezcla de Claude con H, A_G y otra mezcla de Claude, con reserva de caso nominal y ±25 %.

Para cada escenario (reserva de los siniestros abiertos aceptados × 0,75, × 1 y × 1,25 en la etiqueta; las predicciones no
cambian) y para cada ramo y el agregado informa: la diferencia de D² del candidato frente a cada comparador, su IC95 por
cliente y por dirección (Poisson(1), 1.000 réplicas, denominador fijo), lo que aportan los diez clientes más favorables y
la diferencia sin ellos (diagnóstico de influencia, no un criterio de descarte), y, al pasar de la reserva nominal a
+25 %, cuánto cambia el numerador pareado (devianza del comparador menos la del candidato) cuando se sube solo la reserva
de cada peligro: una identidad de devianza por fila, sin reparto causal.
Uso: python3 influencia_reservas.py <salida.json> <cand>=<spec>@<forma>@<nivel> <otra>=<spec>@<forma>@<nivel>
"""
from __future__ import annotations

import json
import sys

import numpy as np
import pandas as pd

from comun import BASESDIR, LINES, WORK, d2, deviance
from escenarios_reserva import CLAIMS, C
from mezcla3 import labels
from robustez import blend, parse

B = 1000


def reservas():
    c = pd.read_parquet(CLAIMS, columns=["claim_id", "status", "peril", "paid_to_date", "incurred_to_date", "audit_pre00"])
    a = pd.read_parquet(BASESDIR / "tables" / "asignacion_por_siniestro.parquet", columns=["claim_id", "category", "row_id"])
    m = a.merge(c, on="claim_id", how="left", validate="one_to_one")
    ok = (m.status == "open") & (m.category == "unica") & (~m.audit_pre00.fillna(False).astype(bool)) & m.row_id.notna()
    m = m[ok].assign(res=lambda d: d.incurred_to_date - d.paid_to_date, row_id=lambda d: d.row_id.astype(str))
    return m.pivot_table(index="row_id", columns="peril", values="res", aggfunc="sum", fill_value=0.0)


def ic(dif, groups, null, rng):
    codes, inv = np.unique(groups, return_inverse=True)
    d = np.bincount(inv, weights=dif)
    bs = np.concatenate([rng.poisson(1.0, size=(50, len(d))) @ d for _ in range(B // 50)]) / null
    return [round(float(np.quantile(bs, 0.025)), 6), round(float(np.quantile(bs, 0.975)), 6)], d


def main():
    out = sys.argv[1]
    (cn, cspec, cfm), (on, ospec, ofm) = parse(sys.argv[2]), parse(sys.argv[3])
    Rp = reservas()
    ag = pd.read_parquet(f"{C}/crossfit_meta_gam_guard/predictions_V24_crossfit.parquet", columns=["row_id", "meta_gam"]).set_index("row_id").meta_gam
    hh = pd.read_parquet(f"{C}/geometric_meta_nested/V24.parquet", columns=["row_id", "candidate"]).set_index("row_id").candidate
    rng = np.random.default_rng(20261018)
    cols = {"y": [], "cu": [], "ad": [], "ln": [], "R": [], cn: [], on: [], "H": [], "A_G": []}
    for line in LINES:
        ysel, y24, idx25 = labels(line)
        fe = pd.read_parquet(WORK / f"feat_{line}.parquet", columns=["row_id", "customer_id", "address_id", "e_ef_coste", "exposure"])
        fe["row_id"] = fe.row_id.astype(str); fe = fe.set_index("row_id")
        expos = (fe.e_ef_coste.reindex(ysel.index).values, fe.e_ef_coste.reindex(y24.index).values, fe.exposure.reindex(idx25).values)
        cols[cn].append(blend(cspec, cfm, line, ysel, y24, idx25, expos)[2]); cols[on].append(blend(ospec, ofm, line, ysel, y24, idx25, expos)[2])
        cols["H"].append(hh.reindex(y24.index).values); cols["A_G"].append(ag.reindex(y24.index).values)
        cols["y"].append(y24.values); cols["ln"].append(np.full(len(y24), line))
        cols["cu"].append(fe.customer_id.reindex(y24.index).astype(str).values); cols["ad"].append(fe.address_id.reindex(y24.index).astype(str).values)
        cols["R"].append(Rp.reindex(y24.index).fillna(0.0))
    V = {k: (pd.concat(v) if k == "R" else np.concatenate(v)) for k, v in cols.items()}
    R = V["R"]; Rtot = R.sum(axis=1).values
    rep = {"candidato": cn, "comparadores": [on, "H", "A_G"], "escenarios": {}}
    for a in (0.75, 1.0, 1.25):
        y_all = V["y"] + (a - 1.0) * Rtot
        res = {}
        for scope in LINES + ["agregado"]:
            msk = np.ones(len(y_all), bool) if scope == "agregado" else (V["ln"] == scope)
            y = y_all[msk]; null = deviance(y, np.full_like(y, y.mean())).sum()
            row = {"d2": {k: round(d2(y, V[k][msk]), 6) for k in (cn, on, "H", "A_G")}}
            for k in (on, "H", "A_G"):
                dif = deviance(y, V[k][msk]) - deviance(y, V[cn][msk])
                ic_cu, by_cu = ic(dif, V["cu"][msk], null, rng)
                ic_ad, _ = ic(dif, V["ad"][msk], null, rng)
                top = np.sort(by_cu)[::-1][:10]
                row[f"{cn}_menos_{k}"] = {"dif": round(float(dif.sum() / null), 6), "ic95_cliente": ic_cu, "ic95_direccion": ic_ad,
                                          "diez_clientes_mas_favorables": round(float(top.sum() / null), 6),
                                          "dif_sin_esos_diez": round(float((dif.sum() - top.sum()) / null), 6)}
                if a == 1.25:
                    # cambio del numerador pareado al subir a +25 % solo la reserva de cada peligro (identidad por fila)
                    base_num = dif.sum()
                    por_peligro = {}
                    for p in R.columns:
                        rp = R[p].values[msk]
                        if rp.sum() <= 0:
                            continue
                        y0 = V["y"][msk]; yp = y0 + 0.25 * rp
                        d0 = deviance(y0, V[k][msk]) - deviance(y0, V[cn][msk])
                        d1 = deviance(yp, V[k][msk]) - deviance(yp, V[cn][msk])
                        por_peligro[p] = round(float((d1 - d0).sum()), 3)
                    row[f"{cn}_menos_{k}"]["cambio_numerador_por_peligro_a_mas25"] = por_peligro
            res[scope] = row
        rep["escenarios"][str(a)] = res
        print(f"a={a}", flush=True)
        for scope in LINES + ["agregado"]:
            r = res[scope]
            print("  ", scope, "D2", r["d2"], flush=True)
            for k in (on, "H", "A_G"):
                v = r[f"{cn}_menos_{k}"]
                extra = f"  por peligro {v['cambio_numerador_por_peligro_a_mas25']}" if "cambio_numerador_por_peligro_a_mas25" in v else ""
                print(f"     frente a {k}: {v['dif']:+.6f} IC95 cliente {v['ic95_cliente']} dirección {v['ic95_direccion']} diez {v['diez_clientes_mas_favorables']:+.6f} sin ellos {v['dif_sin_esos_diez']:+.6f}{extra}", flush=True)
    (WORK / out).write_text(json.dumps(rep, indent=1, ensure_ascii=False))


if __name__ == "__main__":
    main()
'''


### Embedded source: `src/modelos/intervalos_region.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/modelos/intervalos_region.py'] = '''"""Diferencias pareadas de D² en 2024 de una mezcla de Claude frente a H, A_G y otra mezcla, con IC95 por cliente, dirección y región.

Codex usa, además de cliente y dirección, un remuestreo por región (12 regiones) como control de incertidumbre regional.
Este script calcula las tres versiones con multiplicadores Poisson(1), 1.000 réplicas y denominador fijo, con la etiqueta
T0 + (a - 1) x reserva de caso de los abiertos aceptados (a = 0,75; 1; 1,25), por ramo y en el agregado. Informa también
la diferencia dentro de cada región y cuántas regiones quedan negativas. Cada agrupación usa su propio generador.
Uso: python3 intervalos_region.py <salida.json> <cand>=<spec>@<forma>@<nivel> <otra>=<spec>@<forma>@<nivel>
"""
from __future__ import annotations

import json
import sys

import numpy as np
import pandas as pd

from comun import LINES, WORK, deviance
from escenarios_reserva import C, reserva_por_fila
from mezcla3 import labels
from robustez import blend, parse

B = 1000


def ic(dif, groups, null, rng):
    codes, inv = np.unique(groups, return_inverse=True)
    d = np.bincount(inv, weights=dif)
    bs = np.concatenate([rng.poisson(1.0, size=(50, len(d))) @ d for _ in range(B // 50)]) / null
    return [round(float(np.quantile(bs, 0.025)), 6), round(float(np.quantile(bs, 0.975)), 6)], round(float((bs > 0).mean()), 3)


def main():
    out = sys.argv[1]
    (cn, cspec, cfm), (on, ospec, ofm) = parse(sys.argv[2]), parse(sys.argv[3])
    R = reserva_por_fila()
    ag = pd.read_parquet(f"{C}/crossfit_meta_gam_guard/predictions_V24_crossfit.parquet", columns=["row_id", "meta_gam"]).set_index("row_id").meta_gam
    hh = pd.read_parquet(f"{C}/geometric_meta_nested/V24.parquet", columns=["row_id", "candidate"]).set_index("row_id").candidate
    V = {k: [] for k in ("y", "r", "cu", "ad", "rg", "ln", cn, on, "H", "A_G")}
    for line in LINES:
        ysel, y24, idx25 = labels(line)
        fe = pd.read_parquet(WORK / f"feat_{line}.parquet", columns=["row_id", "customer_id", "address_id", "region", "e_ef_coste", "exposure"])
        fe["row_id"] = fe.row_id.astype(str); fe = fe.set_index("row_id")
        expos = (fe.e_ef_coste.reindex(ysel.index).values, fe.e_ef_coste.reindex(y24.index).values, fe.exposure.reindex(idx25).values)
        V[cn].append(blend(cspec, cfm, line, ysel, y24, idx25, expos)[2]); V[on].append(blend(ospec, ofm, line, ysel, y24, idx25, expos)[2])
        V["H"].append(hh.reindex(y24.index).values); V["A_G"].append(ag.reindex(y24.index).values)
        f = fe.reindex(y24.index)
        V["y"].append(y24.values); V["r"].append(R.reindex(y24.index.astype(str)).fillna(0.0).values)
        V["cu"].append(f.customer_id.astype(str).values); V["ad"].append(f.address_id.astype(str).values)
        V["rg"].append(f.region.astype(str).values); V["ln"].append(np.full(len(y24), line))
    V = {k: np.concatenate(v) for k, v in V.items()}
    rep = {"candidato": cn, "comparadores": [on, "H", "A_G"], "escenarios": {}}
    for a in (0.75, 1.0, 1.25):
        y_all = V["y"] + (a - 1.0) * V["r"]
        res = {}
        for scope in LINES + ["agregado"]:
            msk = np.ones(len(y_all), bool) if scope == "agregado" else (V["ln"] == scope)
            y = y_all[msk]; null = deviance(y, np.full_like(y, y.mean())).sum()
            row = {}
            for k in (on, "H", "A_G"):
                dif = deviance(y, V[k][msk]) - deviance(y, V[cn][msk])
                rngs = {g: np.random.default_rng(20261018 + i) for i, g in enumerate(("cu", "ad", "rg"))}
                (icu, pcu), (iad, pad), (irg, prg) = (ic(dif, V[g][msk], null, rngs[g]) for g in ("cu", "ad", "rg"))
                por_region = pd.Series(dif).groupby(V["rg"][msk]).sum() / null
                row[f"{cn}_menos_{k}"] = {"dif": round(float(dif.sum() / null), 6), "ic95_cliente": icu, "ic95_direccion": iad, "ic95_region": irg,
                                          "P_mayor_que_0_region": prg, "regiones_negativas": int((por_region < 0).sum()), "regiones": int(len(por_region))}
            res[scope] = row
            print(f"a={a} {scope} " + " ".join(f"{k}: {row[f'{cn}_menos_{k}']['dif']:+.6f} cli {row[f'{cn}_menos_{k}']['ic95_cliente']} dir {row[f'{cn}_menos_{k}']['ic95_direccion']} reg {row[f'{cn}_menos_{k}']['ic95_region']} neg {row[f'{cn}_menos_{k}']['regiones_negativas']}/12" for k in (on, "H", "A_G")), flush=True)
        rep["escenarios"][str(a)] = res
    (WORK / out).write_text(json.dumps(rep, indent=1, ensure_ascii=False))


if __name__ == "__main__":
    main()
'''


### Embedded source: `src/modelos/mezcla3.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/modelos/mezcla3.py'] = '''"""Mezcla general por ramo: componentes de Claude y el stack S08 de Codex ("CODEX"), con rejilla para un componente.

Cada componente aporta predicciones de V23 selección, V24 y 2025. Antes de mezclar, el total de cada componente se
iguala, en cada corte, a la media ponderada de los totales de los componentes de Claude (solo predicciones), para que los
pesos actúen sobre relatividades y el nivel no dependa de un único modelo. El
factor de año siguiente es la media de los factores Tweedie de la mezcla en V23 selección y V24; V24 se informa con el
factor de V23 (fuera de muestra) y 2025 recibe la media.
Uso: python3 mezcla3.py <nombre> <spec_json> [--rejilla=<ramo>:<indice>] [--alfa=<ramo>:<a>,...] [--forma=<ramo>:<a_tasa>:<b_exposicion>,...] [--nivel=s17] [--escribir]
spec: {"motor": [["CAT_motor", "cat1a", 0.5], ...], "buildings": [["S08h_hogar_F3s2", "hS2", 0.5], ["CODEX", "", 0.5]], ...}
"""
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

from comun import BASE, LINES, WORK, d2, digest

CODEX = Path("/mnt/user-data/uploads/Hakaton/Preparación/artifacts/verification/improvement_D76")
# componentes de Codex que no entran en la referencia de nivel (el stack S08 y el GAM de contenidos de la carpeta N01)
EXTERNOS = {"CODEX", "GAMC_contents"}
# redes neuronales: su nivel no es estable entre V24 y producción (MLP3_contents: total V24 un 10 % sobre el real y
# razón 2025/V24 de 0,897 frente a 1,074-1,078 de los árboles), así que aportan relatividades pero no nivel
NIVEL_INESTABLE = {"MLP3_contents", "MLP3_buildings", "MLP3_motor"}
# desde S17, regla sin etiquetas: tampoco entran los componentes cuyo total frente al ancla del ramo se mueve más de un
# 1,5 % entre V24 y 2025 (frecuencias con severidad media del peligro, con o sin relatividad por capital). Medido el 18 de
# octubre frente a CAT_motor_4s, S12h_edif_md1000 y S08h_hogar_F3s2: CFS -4,3 % en autos (CFS_motor_4s), -2,7 % en
# edificios y -2,9 % en contenidos; CFSS -4,4 % en autos y -2,0 % en edificios y en contenidos. Al fijar la regla se
# anotó -3,8 % y -2,6 % para CFS en autos y edificios, con otra referencia; la clasificación no cambia
# con la misma regla (S19): la severidad híbrida GLM + boosting (CFSG) se mueve -3,9 % en autos y -1,7 % en edificios,
# fuera; en contenidos -1,0 %, dentro del umbral, así que CFSG*_contents sí entra en la referencia
NIVEL_INESTABLE_S17 = NIVEL_INESTABLE | {"CFS_motor", "CFS_motor_4s", "CFSS_motor", "CFS_buildings", "CFSS_buildings", "CFS_contents", "CFSS_contents",
                                          "CFSG5_motor", "CFSG2_motor", "CFSG5_buildings", "CFSG2_buildings", "FSAS_contents",
                                          "FSAS4_contents", "FSAS_motor", "FSAX4_contents", "CFSX_contents", "CFSX_buildings",
                                          "CFSX_motor", "FSAXc4_contents", "CFSXc_contents", "CFSXc_buildings", "CFSXc_motor",
                                          "CFSS4_buildings", "FS2Xc_contents", "FS2S_motor",
                                          "CFSSr_motor", "CFSS4r_buildings", "FSAXc4r_contents",
                                          "CFSS6_motor", "CFSS4v_motor_tmp"}
# CFSS de edificios con cuatro semillas (S21): -1,72 %, fuera como CFSS
# tercer motor de frecuencia (frecsev2 sin --aditivo, con severidad GLM encima; CLA-20261018-009): FS2S de edificios
# -1,46 % (dentro); FS2Xc de contenidos -1,58 % (fuera); FS2S de autos -3,86 % (fuera; CLA-20261018-011)
# relatividades de severidad acotadas al p99,5 (--tope=0.995, CLA-20261018-018): CFSSr de autos -4,4 %, CFSS4r de edificios
# -1,75 % y FSAXc4r de contenidos -1,55 %: fuera, como sus originales; FSAS4r de edificios +0,02 %: dentro
# CFSS de autos con más semillas (CFSS6_motor; CFSS4v_motor_tmp solo para medir V24, CLA-20261018-021): fuera, como CFSS_motor
# severidad con covariables físicas (severidad_cfs3.py, S21): FSAX4 de contenidos -1,52 % y CFSX -2,06 % (contenidos),
# -1,79 % (edificios): fuera; FSAX4 de edificios +0,2 %: dentro. Versión corregida, que no penaliza las covariables de CFSS
# (FSAXc4/CFSXc): contenidos -1,53 % y -2,06 %, edificios CFSXc -1,86 %: fuera; FSAXc4 de edificios +0,15 %: dentro
# frecuencia aditiva (frecsev2 --aditivo) con severidad GLM por capital (FSAS, S20): edificios +0,2 % (entra en la
# referencia); contenidos -1,53 %, justo por encima del umbral (fuera). Con cuatro semillas (FSAS4): edificios +0,06 %,
# contenidos -1,57 %; la clasificación no cambia. Autos (FSAS_motor): -3,85 %, fuera
SIN_REFERENCIA = EXTERNOS | NIVEL_INESTABLE


def sin_referencia(opts):
    """Conjunto excluido de la referencia de nivel; --nivel=s17 aplica la regla de estabilidad entre V24 y 2025."""
    return EXTERNOS | (NIVEL_INESTABLE_S17 if opts.get("nivel") == "s17" else NIVEL_INESTABLE)
_cx = {}


def ftw(y, mu):
    return float((y / np.sqrt(mu)).sum() / np.sqrt(mu).sum())


def codex(split):
    if split not in _cx:
        f = {"v24": "learned_predictions_V24.parquet", "sel": "learned_fit_V23.parquet", "p25": "learned_predictions_PROD.parquet"}[split]
        _cx[split] = pd.read_parquet(CODEX / f, columns=["row_id", "stack"]).set_index("row_id")["stack"]
    return _cx[split]


def component(run, sel_name, line):
    if run == "CODEX":
        return codex("sel"), codex("v24"), codex("p25")
    s = pd.read_parquet(WORK / f"sel_{sel_name}.parquet"); s = s[s.line == line].set_index("row_id")["mu"]
    v = pd.read_parquet(WORK / run / "v24.parquet"); v = v[v.line == line].set_index("row_id")["mu"]
    p = pd.read_parquet(WORK / run / "prediccion_2025.parquet"); p = p[p.line == line].set_index("row_id")["prediction"]
    return s, v, p


def labels(line):
    ref = pd.read_parquet(WORK / "S04b_lgbm_F0124_F3autos" / "v24.parquet"); ref = ref[ref.line == line].set_index("row_id")["y"]
    sel = pd.read_parquet(WORK / "sel_total_F0124.parquet"); sel = sel[sel.line == line].set_index("row_id")["y"]
    p25 = pd.read_parquet(WORK / "S04b_lgbm_F0124_F3autos" / "prediccion_2025.parquet"); p25 = p25[p25.line == line].set_index("row_id").index
    return sel, ref, p25


def shape(mu, a):
    """Potencia alrededor de la media geométrica: a > 1 abre la dispersión de las predicciones; el nivel lo fija luego el factor."""
    if a == 1.0:
        return mu
    g = np.exp(np.log(mu).mean())
    return g * (mu / g) ** a


def shape_rate(mu, e, a, b):
    """Forma sobre la tasa y elasticidad de la exposición: mu = e (e/ē)^(b-1) g (r/g)^a con r = mu/e y medias geométricas.
    a > 1 abre la dispersión de las tasas; b distinto de 1 deja de suponer coste proporcional a la duración del periodo."""
    if a == 1.0 and b == 1.0:
        return mu
    r = mu / e
    g = np.exp(np.log(r).mean()); eb = np.exp(np.log(e).mean())
    return e * (e / eb) ** (b - 1.0) * g * (r / g) ** a


def evaluate(comps, ws, ysel, y24, a=1.0, forma=None, expos=None):
    ws = np.asarray(ws, float) / np.sum(ws)
    bsel = sum(w * c[0] for w, c in zip(ws, comps)); b24 = sum(w * c[1] for w, c in zip(ws, comps)); b25 = sum(w * c[2] for w, c in zip(ws, comps))
    bsel, b24, b25 = shape(bsel, a), shape(b24, a), shape(b25, a)
    if forma is not None:
        bsel, b24, b25 = (shape_rate(bsel, expos[0], *forma), shape_rate(b24, expos[1], *forma), shape_rate(b25, expos[2], *forma))
    f23, f24 = ftw(ysel, bsel), ftw(y24, b24)
    return {"d2_sel": d2(ysel, bsel), "d2_v24_f23": d2(y24, b24 * f23), "f23": f23, "f24": f24, "f": (f23 + f24) / 2}, bsel, b24 * f23, b25 * (f23 + f24) / 2


def main():
    name, spec_path = sys.argv[1], sys.argv[2]
    opts = dict((a[2:].split("=") + ["1"])[:2] for a in sys.argv[3:] if a.startswith("--"))
    spec = json.loads(open(spec_path).read())
    grid = opts.get("rejilla")
    alfa = {kv.split(":")[0]: float(kv.split(":")[1]) for kv in opts.get("alfa", "").split(",") if kv}
    forma = {kv.split(":")[0]: (float(kv.split(":")[1]), float(kv.split(":")[2])) for kv in opts.get("forma", "").split(",") if kv}
    rep = {"spec": spec, "lines": {}, "alfa": alfa, "forma": {k: list(v) for k, v in forma.items()}}
    ys24, ms24, preds = [], [], []
    for line in LINES:
        ysel, y24, idx25 = labels(line)
        raw = []
        for k, (run, sel_name, w) in enumerate(spec[line]):
            s, v, p = component(run, sel_name, line)
            s = s.reindex(ysel.index).values; v = v.reindex(y24.index).values; p = p.reindex(idx25).values
            assert np.isfinite(s).all() and np.isfinite(v).all() and np.isfinite(p).all(), (line, run)
            raw.append((run, w, s, v, p))
        # nivel de referencia: media ponderada de los totales de los componentes de Claude en cada corte (el de Codex está
        # en otra base de nivel); cada componente se reescala a ese total, así los pesos actúan sobre relatividades
        own = [(w, s, v, p) for run, w, s, v, p in raw if run not in sin_referencia(opts)]
        wo = sum(o[0] for o in own)
        ref = [sum(o[0] * o[j].sum() for o in own) / wo for j in (1, 2, 3)]
        comps = [(s * ref[0] / s.sum(), v * ref[1] / v.sum(), p * ref[2] / p.sum()) for run, w, s, v, p in raw]
        ws = [c[2] for c in spec[line]]
        expos = None
        if line in forma:
            fe = pd.read_parquet(WORK / f"feat_{line}.parquet", columns=["row_id", "e_ef_coste", "exposure"])
            fe["row_id"] = fe.row_id.astype(str); fe = fe.set_index("row_id")
            # V23 selección y V24 con la exposición efectiva del offset; 2025 con la nominal, como en la predicción
            expos = (fe.e_ef_coste.reindex(ysel.index).values, fe.e_ef_coste.reindex(y24.index).values, fe.exposure.reindex(idx25).values)
        res, bsel, b24, b25 = evaluate(comps, ws, ysel.values, y24.values, alfa.get(line, 1.0), forma.get(line), expos)
        rep["lines"][line] = {k: round(v, 5) for k, v in res.items()}
        print(line, rep["lines"][line], flush=True)
        if grid and grid.split(":")[0] == line:
            j = int(grid.split(":")[1])
            others = np.array(ws, float); others[j] = 0; others = others / others.sum()
            for wj in (0.0, 0.1, 0.2, 0.3, 0.4, 0.5):
                w2 = others * (1 - wj); w2[j] = wj
                r2, *_ = evaluate(comps, w2, ysel.values, y24.values, alfa.get(line, 1.0), forma.get(line), expos)
                print("  rejilla", spec[line][j][0], wj, {k: round(v, 5) for k, v in r2.items()}, flush=True)
        ys24.append(y24.values); ms24.append(b24)
        preds.append(pd.DataFrame({"row_id": idx25.values, "line": line, "prediction": b25}))
    rep["d2_v24_global_f23"] = round(d2(np.concatenate(ys24), np.concatenate(ms24)), 5)
    rep["publico_esperado"] = round(0.0099 + 0.8644 * rep["d2_v24_global_f23"], 5)
    print("global V24", rep["d2_v24_global_f23"], "público esperado", rep["publico_esperado"], flush=True)
    if "escribir" in opts:
        pred = pd.concat(preds, ignore_index=True)
        sample = pd.read_parquet(BASE / "Datos" / "sample_submission.parquet", columns=["row_id"])
        sub = sample.merge(pred[["row_id", "prediction"]], on="row_id", how="left", validate="one_to_one")
        assert len(sub) == 626639 and np.isfinite(sub.prediction).all() and (sub.prediction > 0).all()
        out = WORK / name
        out.mkdir(parents=True, exist_ok=True)
        csv = out / f"{name}.csv"
        sub.to_csv(csv, index=False, float_format="%.6f")
        back = pd.read_csv(csv, dtype={"row_id": str})
        assert (back.row_id.values == sample.row_id.values).all()
        rep["file"] = {"path": str(csv), "sha256": digest(csv), "rows": 626639, "total_CU": round(float(back.prediction.sum()), 2),
                       "by_line_CU": {l: round(float(pred[pred.line == l].prediction.sum()), 2) for l in LINES}}
        pred.to_parquet(out / "prediccion_2025.parquet", index=False)
        (out / "resumen.json").write_text(json.dumps(rep, indent=1, ensure_ascii=False))
        print(json.dumps(rep["file"], indent=1), flush=True)


if __name__ == "__main__":
    main()
'''


### Embedded source: `src/modelos/mlp_tweedie.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/modelos/mlp_tweedie.py'] = '''"""Red neuronal (MLP con embeddings, torch en CPU) con pérdida Tweedie 1,5 y el mismo contrato que los GBM.

log mu = red(x) + log(exposición efectiva) + base; numéricas con transformación por cuantiles a normal (ajustada en el
entrenamiento del pliegue) más indicador de faltante; categóricas con embeddings; tope de autos 15.000 con recargo
uniforme; F3 solo en autos. La última capa arranca en cero, así que la red empieza en la tasa media del ramo.
V23: parada temprana por época en la mitad de calibración y comparación en la de selección; V24 y PROD usan el número
de épocas de V23. Promedia varias semillas. Guarda sel_<nombre>.parquet, v24.parquet y prediccion_2025.parquet.
Uso: python3 mlp_tweedie.py <nombre> <ramo> [--semillas=3] [--v24] [--prod] [--ancho=128] [--drop=0.1] [--lr=0.001]
"""
from __future__ import annotations

import json
import sys
import time

import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import QuantileTransformer
from torch import nn

from comun import WORK, d2, to_model_frame
from experimentos import features, load

torch.set_num_threads(2)
CAPS = {"motor": 15000.0}


def prep(X: pd.DataFrame, fit_mask: np.ndarray):
    cats = [c for c in X.columns if isinstance(X[c].dtype, pd.CategoricalDtype)]
    nums = [c for c in X.columns if c not in cats]
    num = X[nums].astype("float64").values
    miss = np.isnan(num)
    med = np.nanmedian(num[fit_mask], axis=0)
    med = np.where(np.isnan(med), 0.0, med)
    num = np.where(miss, med, num)
    qt = QuantileTransformer(n_quantiles=200, output_distribution="normal", subsample=200000, random_state=0)
    qt.fit(num[fit_mask])
    num = qt.transform(num).astype("float32")
    keep_miss = miss[fit_mask].mean(axis=0) > 0.001
    num = np.concatenate([num, miss[:, keep_miss].astype("float32")], axis=1)
    codes = np.stack([X[c].cat.codes.values.astype("int64") + 1 for c in cats], axis=1) if cats else np.zeros((len(X), 0), "int64")
    cards = [len(X[c].cat.categories) + 1 for c in cats]
    for j in range(len(cats)):
        # categorías ausentes o raras (< 30 filas) en el entrenamiento del pliegue van al índice 0 (desconocido)
        cnt = np.bincount(codes[fit_mask, j], minlength=cards[j])
        codes[:, j] = np.where(cnt[codes[:, j]] < 30, 0, codes[:, j])
    return num, codes, cards


class Net(nn.Module):
    def __init__(self, n_num, cards, width=128, drop=0.1):
        super().__init__()
        dims = [min(12, int(np.ceil(c ** 0.5)) + 1) for c in cards]
        self.embs = nn.ModuleList([nn.Embedding(c, d) for c, d in zip(cards, dims)])
        d_in = n_num + sum(dims)
        self.mlp = nn.Sequential(nn.Linear(d_in, width), nn.SiLU(), nn.Dropout(drop), nn.Linear(width, width // 2), nn.SiLU(),
                                 nn.Dropout(drop), nn.Linear(width // 2, 1))
        nn.init.zeros_(self.mlp[-1].weight)
        nn.init.zeros_(self.mlp[-1].bias)

    def forward(self, xn, xc):
        parts = [xn] + [e(xc[:, i]) for i, e in enumerate(self.embs)]
        return self.mlp(torch.cat(parts, dim=1)).squeeze(1)


def tweedie_loss(eta, y):
    # desviación Tweedie 1,5 sin constantes: 2 y exp(-eta/2) + 2 exp(eta/2)
    return (2.0 * y * torch.exp(-0.5 * eta) + 2.0 * torch.exp(0.5 * eta)).mean()


def train(num, codes, cards, y, logoff, idx_tr, idx_cal=None, epochs=40, seed=0, width=128, drop=0.1, lr=1e-3, bs=2048, patience=4, wd=1e-5):
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    net = Net(num.shape[1], cards, width, drop)
    opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=wd)
    xn = torch.from_numpy(num); xc = torch.from_numpy(codes)
    yt = torch.from_numpy(y.astype("float32")); lo = torch.from_numpy(logoff.astype("float32"))
    best, best_ep, best_state, bad = np.inf, 0, None, 0
    for ep in range(epochs):
        net.train()
        perm = rng.permutation(idx_tr)
        for s in range(0, len(perm), bs):
            b = torch.from_numpy(perm[s:s + bs])
            eta = net(xn[b], xc[b]) + lo[b]
            loss = tweedie_loss(eta, yt[b])
            opt.zero_grad(); loss.backward(); opt.step()
        if idx_cal is not None:
            net.eval()
            with torch.no_grad():
                b = torch.from_numpy(idx_cal)
                vl = float(tweedie_loss(net(xn[b], xc[b]) + lo[b], yt[b]))
            if vl < best - 1e-7:
                best, best_ep, bad = vl, ep + 1, 0
                best_state = {k: v.clone() for k, v in net.state_dict().items()}
            else:
                bad += 1
                if bad >= patience:
                    break
    if best_state is not None:
        net.load_state_dict(best_state)
    return net, best_ep


def predict(net, num, codes, logoff, idx):
    net.eval()
    out = []
    with torch.no_grad():
        for s in range(0, len(idx), 65536):
            b = torch.from_numpy(idx[s:s + 65536])
            out.append((net(torch.from_numpy(num)[b], torch.from_numpy(codes)[b]) + torch.from_numpy(logoff.astype("float32"))[b]).numpy())
    return np.exp(np.concatenate(out).astype("float64"))


def main():
    name, line = sys.argv[1], sys.argv[2]
    opts = dict((a[2:].split("=") + ["1"])[:2] for a in sys.argv[3:] if a.startswith("--"))
    seeds = int(opts.get("semillas", 3)); width = int(opts.get("ancho", 128)); drop = float(opts.get("drop", 0.1)); lr = float(opts.get("lr", 1e-3))
    bs = int(opts.get("bs", 2048)); pat = int(opts.get("pac", 4)); wd = float(opts.get("wd", 1e-5))
    t0 = time.time()
    df = load(line)
    feats = features(line, df, "F01234" if line == "motor" else "F0124")
    if line != "motor" and "f3s2" in opts:
        # siete variables de cliente F3 autorizadas en hogar (D80)
        feats += [c for c in ["prior_claims_5y", "credit_band", "missed_payments_12m", "household_size", "years_at_address", "marital_status", "licence_points"] if c in df.columns]
    X = to_model_frame(df, feats)
    y = df["T0"].fillna(0).values
    off = df["e_ef_coste"].values
    expo = df["exposure"].values
    yr = df.year.values
    cap = CAPS.get(line)
    out = WORK / name
    out.mkdir(parents=True, exist_ok=True)
    rep = {"line": line, "seeds": seeds, "width": width, "drop": drop, "lr": lr, "bs": bs, "wd": wd, "f3s2": "f3s2" in opts}

    def fold(tr_mask, ev_mask, cal_mask, ev_off, epochs_fixed=None):
        num, codes, cards = prep(X, tr_mask)
        yt = np.minimum(y, cap) if cap else y.copy()
        loading = y[tr_mask].sum() / yt[tr_mask].sum() if cap else 1.0
        base = float(np.log(yt[tr_mask].sum() / off[tr_mask].sum()))
        logoff = np.log(np.where(ev_mask, ev_off, off)) + base
        idx_tr = np.flatnonzero(tr_mask); idx_ev = np.flatnonzero(ev_mask)
        idx_cal = np.flatnonzero(cal_mask) if cal_mask is not None else None
        mu = np.zeros(len(idx_ev)); eps = []
        for s in range(seeds):
            net, be = train(num, codes, cards, yt, logoff, idx_tr, idx_cal, epochs=(80 if epochs_fixed is None else epochs_fixed), seed=20261017 + s,
                            width=width, drop=drop, lr=lr, bs=bs, patience=(pat if epochs_fixed is None else 10 ** 6), wd=wd)
            eps.append(be if epochs_fixed is None else epochs_fixed)
            mu += predict(net, num, codes, logoff, idx_ev) * loading / seeds
            print(name, "semilla", s, "épocas", eps[-1], round(time.time() - t0), flush=True)
        return mu, eps

    tr = yr <= 2022; cal = (yr == 2023) & (df.cal_half.values == 0); sel = (yr == 2023) & (df.cal_half.values == 1)
    mu, eps = fold(tr, sel, cal, off)
    ep_fix = int(round(np.mean(eps)))
    rep["epocas_v23"] = eps
    rep["d2_sel"] = round(d2(y[sel], mu), 5)
    print(name, json.dumps(rep), flush=True)
    pd.DataFrame({"row_id": df.row_id[sel].astype(str).values, "line": line, "y": y[sel], "mu": mu}).to_parquet(WORK / f"sel_{name}.parquet", index=False)
    if "v24" in opts:
        ev = yr == 2024
        mu24, _ = fold(yr <= 2023, ev, None, off, epochs_fixed=ep_fix)
        rep["d2_v24"] = round(d2(y[ev], mu24), 5)
        pd.DataFrame({"row_id": df.row_id[ev].astype(str).values, "line": line, "y": y[ev], "mu": mu24}).to_parquet(out / "v24.parquet", index=False)
        print(name, json.dumps(rep), flush=True)
    if "prod" in opts:
        te = yr == 2025
        mu25, _ = fold(yr <= 2024, te, None, expo, epochs_fixed=ep_fix)
        rep["pred_2025_total_CU"] = round(float(mu25.sum()), 2)
        pd.DataFrame({"row_id": df.row_id[te].astype(str).values, "line": line, "prediction": mu25}).to_parquet(out / "prediccion_2025.parquet", index=False)
    rep["secs"] = round(time.time() - t0)
    (out / "resumen.json").write_text(json.dumps(rep, indent=1))
    print(name, json.dumps(rep), flush=True)


if __name__ == "__main__":
    main()
'''


### Embedded source: `src/modelos/produccion.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/modelos/produccion.py'] = '''"""Entrena el candidato final por ramo y escribe el archivo de envío.

Para cada ramo: V24 de confirmación (entrena 2021-2023 con las rondas de V23 x 1,15 y evalúa 2024 sobre T0 con el
offset de exposición efectiva) y producción (entrena 2021-2024 con las rondas de V23 x 1,3 y predice 2025 con la
exposición nominal, que en 2025 coincide con la efectiva: nivel último sin segundo factor, D70). Promedia varias
semillas. Escribe WORK/<nombre>/: prediccion_2025.parquet, v24.parquet, resumen.json y el CSV de envío con row_id y
prediction en el orden de sample_submission.

Uso: python3 produccion.py <nombre> <conjunto> <config_json> [--semillas=3] [--tendencia=motor:1.0,buildings:1.0,contents:1.0]
config_json: {"motor": {"params": {...}, "rounds": N}, "buildings": {...}, "contents": {...}}
"""
from __future__ import annotations

import json
import sys
import time

import numpy as np
import pandas as pd

from comun import BASE, LINES, WORK, d2, digest, to_model_frame
from experimentos import features, fit, load, predict


def main():
    name, set_name, cfg_path = sys.argv[1], sys.argv[2], sys.argv[3]
    cfg = json.loads(open(cfg_path).read())
    seeds = 3
    trend = {l: 1.0 for l in LINES}
    only = list(LINES)
    reuse = None
    for a in sys.argv[4:]:
        if a.startswith("--solo="):
            only = a.split("=", 1)[1].split(",")
        if a.startswith("--reusar="):
            reuse = WORK / a.split("=", 1)[1]
        if a.startswith("--semillas="):
            seeds = int(a.split("=")[1])
        if a.startswith("--tendencia="):
            for kv in a.split("=", 1)[1].split(","):
                k, v = kv.split(":")
                trend[k] = float(v)
    out = WORK / name
    out.mkdir(parents=True, exist_ok=True)
    preds, v24_rows, summary = [], [], {"set": set_name, "config": cfg, "seeds": seeds, "trend": trend, "lines": {}}
    for line in LINES:
        if line not in only:
            rp = pd.read_parquet(reuse / "prediccion_2025.parquet")
            rv = pd.read_parquet(reuse / "v24.parquet")
            preds.append(rp[rp.line == line].copy())
            v24_rows.append(rv[rv.line == line].copy())
            summary["lines"][line] = json.loads((reuse / "resumen.json").read_text())["lines"][line] | {"reutilizado_de": reuse.name}
            continue
        t = time.time()
        df = load(line)
        feats = [f for f in features(line, df, set_name) if f not in set(cfg[line].get("drop", []))]
        feats += [f for f in cfg[line].get("add", []) if f in df.columns and f not in feats]
        X = to_model_frame(df, feats)
        y = df["T0"].fillna(0).values
        off = df["e_ef_coste"].values
        expo = df["exposure"].values
        tr24 = (df.year <= 2023).values
        ev24 = (df.year == 2024).values
        trp = (df.year <= 2024).values
        te = (df.year == 2025).values
        r = int(cfg[line]["rounds"])
        mu24 = np.zeros(ev24.sum())
        mu25 = np.zeros(te.sum())
        for s in range(seeds):
            params = dict(cfg[line]["params"], seed=20261017 + 101 * s)
            b24, base24 = fit(X[tr24], y[tr24], off[tr24], rounds=int(round(r * 1.15)), early=None, params=dict(params))
            mu24 += predict(b24, base24, X[ev24], off[ev24]) / seeds
            bp, basep = fit(X[trp], y[trp], off[trp], rounds=int(round(r * 1.3)), early=None, params=dict(params))
            mu25 += predict(bp, basep, X[te], expo[te]) / seeds
        mu25 *= trend[line]
        imp = pd.Series(bp.feature_importance("gain"), index=feats).sort_values(ascending=False)
        summary["lines"][line] = {
            "n_feats": len(feats), "rounds_v24": int(round(r * 1.15)), "rounds_prod": int(round(r * 1.3)),
            "d2_v24": round(d2(y[ev24], mu24), 5), "balance_v24": round(float(mu24.sum() / y[ev24].sum()), 4),
            "pred_2025_total_CU": round(float(mu25.sum()), 2), "pred_2025_rate_CU": round(float(mu25.sum() / expo[te].sum()), 4),
            "train_T0_rate_dev_CU": round(float(y[trp].sum() / off[trp].sum()), 4), "loading": round(float(getattr(bp, "loading", 1.0)), 5),
            "top_gain": {k: round(float(v / imp.sum()), 4) for k, v in imp.head(20).items()}, "secs": round(time.time() - t)}
        print(line, summary["lines"][line], flush=True)
        preds.append(pd.DataFrame({"row_id": df.row_id[te].astype(str).values, "line": line, "prediction": mu25}))
        v24_rows.append(pd.DataFrame({"row_id": df.row_id[ev24].astype(str).values, "line": line, "y": y[ev24], "mu": mu24}))
    pred = pd.concat(preds, ignore_index=True)
    v24 = pd.concat(v24_rows, ignore_index=True)
    summary["d2_v24_global"] = round(d2(v24.y, v24.mu), 5)
    sample = pd.read_parquet(BASE / "Datos" / "sample_submission.parquet", columns=["row_id"])
    sub = sample.merge(pred[["row_id", "prediction"]], on="row_id", how="left", validate="one_to_one")
    assert len(sub) == 626639 and sub.prediction.notna().all() and np.isfinite(sub.prediction).all() and (sub.prediction > 0).all()
    csv = out / f"{name}.csv"
    sub.to_csv(csv, index=False, float_format="%.6f")
    back = pd.read_csv(csv, dtype={"row_id": str})
    assert len(back) == 626639 and (back.row_id.values == sample.row_id.values).all() and (back.prediction > 0).all()
    summary["file"] = {"path": str(csv), "sha256": digest(csv), "rows": int(len(back)), "total_CU": round(float(back.prediction.sum()), 2),
                       "by_line_CU": {l: round(float(pred[pred.line == l].prediction.sum()), 2) for l in LINES}}
    pred.to_parquet(out / "prediccion_2025.parquet", index=False)
    v24.to_parquet(out / "v24.parquet", index=False)
    (out / "resumen.json").write_text(json.dumps(summary, indent=1, default=float))
    print("V24 global", summary["d2_v24_global"], "archivo", summary["file"], flush=True)


if __name__ == "__main__":
    main()
'''


### Embedded source: `src/modelos/publico_simulado.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/modelos/publico_simulado.py'] = '''"""Qué puede mostrar el público: diferencias entre candidatos en subconjuntos aleatorios de clientes de V23 y V24.

El público de Kaggle puntúa una fracción desconocida de las filas de 2025. Para ver cuánto puede variar la diferencia
entre dos candidatos en una muestra así, se toman 1.000 subconjuntos aleatorios de clientes (30 % y 50 %) de V23 selección
y de V24, se calcula el D² global de cada candidato en el subconjunto (con su propio nivel, como en mezcla3) y se resume
la diferencia: media, P(diferencia > 0) e intervalo del 90 %.
Uso: python3 publico_simulado.py <salida.json> <ref>=<spec>[@<forma>] <cand>=<spec>[@<forma>] ...
"""
from __future__ import annotations

import json
import sys

import numpy as np
import pandas as pd

from comun import LINES, WORK, deviance
from mezcla3 import labels
from robustez import blend, parse

B = 1000


def main():
    out_path = sys.argv[1]
    cands = [parse(a) for a in sys.argv[2:]]
    rng = np.random.default_rng(20261019)
    folds = {"V23": {"y": [], "cust": [], "pred": {c[0]: [] for c in cands}}, "V24": {"y": [], "cust": [], "pred": {c[0]: [] for c in cands}}}
    for line in LINES:
        ysel, y24, idx25 = labels(line)
        fe = pd.read_parquet(WORK / f"feat_{line}.parquet", columns=["row_id", "customer_id", "e_ef_coste", "exposure"])
        fe["row_id"] = fe.row_id.astype(str); fe = fe.set_index("row_id")
        expos = (fe.e_ef_coste.reindex(ysel.index).values, fe.e_ef_coste.reindex(y24.index).values, fe.exposure.reindex(idx25).values)
        folds["V23"]["y"].append(ysel.values); folds["V24"]["y"].append(y24.values)
        folds["V23"]["cust"].append(fe.customer_id.reindex(ysel.index).astype(str).values)
        folds["V24"]["cust"].append(fe.customer_id.reindex(y24.index).astype(str).values)
        for name, spec, forma in cands:
            _, bsel, b24, _ = blend(spec, forma, line, ysel, y24, idx25, expos)
            folds["V23"]["pred"][name].append(bsel); folds["V24"]["pred"][name].append(b24)
    rep = {}
    ref = cands[0][0]
    for fold, d in folds.items():
        y = np.concatenate(d["y"]); cu = np.concatenate(d["cust"])
        codes, inv = np.unique(cu, return_inverse=True)
        nc = len(codes)
        dev = {name: deviance(y, np.concatenate(p)) for name, p in d["pred"].items()}
        rep[fold] = {}
        for frac in (0.3, 0.5):
            res = {name: [] for name in dev if name != ref}
            pairs = [(a, b) for i, a in enumerate(dev) for b in list(dev)[i + 1:] if a != ref and b != ref]
            pres = {f"{a}-{b}": [] for a, b in pairs}
            for _ in range(B):
                keep = (rng.random(nc) < frac)[inv]
                yk = y[keep]
                null = deviance(yk, np.full_like(yk, yk.mean())).sum()
                d2k = {name: 1.0 - v[keep].sum() / null for name, v in dev.items()}
                for name in res:
                    res[name].append(d2k[name] - d2k[ref])
                for a, b in pairs:
                    pres[f"{a}-{b}"].append(d2k[a] - d2k[b])
            summ = {}
            for k, v in list({f"{n}-{ref}": res[n] for n in res}.items()) + list(pres.items()):
                v = np.array(v)
                summ[k] = {"media": round(float(v.mean()), 5), "P_mayor_que_0": round(float((v > 0).mean()), 3),
                           "ic90": [round(float(np.quantile(v, 0.05)), 5), round(float(np.quantile(v, 0.95)), 5)]}
                print(fold, f"{int(frac * 100)} %", k, summ[k], flush=True)
            rep[fold][f"{int(frac * 100)}%"] = summ
    (WORK / out_path).write_text(json.dumps(rep, indent=1, ensure_ascii=False))


if __name__ == "__main__":
    main()
'''


### Embedded source: `src/modelos/rejilla_pesos.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/modelos/rejilla_pesos.py'] = '''"""Rejilla de pesos en el símplex (pasos de 0,1) para los componentes de un ramo, con la forma y la regla de nivel de mezcla3.

Carga los componentes una sola vez, los reescala a la referencia de nivel de cada corte (regla indicada) y evalúa cada
combinación de pesos con evaluate de mezcla3: D² de V23 selección sin factor y D² de V24 con el factor de V23. Informa las
mejores por V23, por V24, por la media de ambos y por la peor de las dos mejoras frente a la partida (criterio
maximin, el más prudente cuando los cortes discrepan), junto con la combinación de partida.
Uso: python3 rejilla_pesos.py <ramo> <forma a:b> <nivel|-> <corrida:sel:peso_partida> [...]
"""
from __future__ import annotations

import itertools
import sys

import numpy as np
import pandas as pd

from comun import WORK
from mezcla3 import component, evaluate, labels, sin_referencia


def main():
    line, forma, nivel = sys.argv[1], sys.argv[2], sys.argv[3]
    a, b = (float(x) for x in forma.split(":"))
    comps_in = [c.split(":") for c in sys.argv[4:]]
    ysel, y24, idx25 = labels(line)
    raw = []
    for run, sel_name, w in comps_in:
        s, v, p = component(run, sel_name, line)
        raw.append((run, float(w), s.reindex(ysel.index).values, v.reindex(y24.index).values, p.reindex(idx25).values))
    excl = sin_referencia({"nivel": None if nivel == "-" else nivel})
    fe = pd.read_parquet(WORK / f"feat_{line}.parquet", columns=["row_id", "e_ef_coste", "exposure"])
    fe["row_id"] = fe.row_id.astype(str); fe = fe.set_index("row_id")
    expos = (fe.e_ef_coste.reindex(ysel.index).values, fe.e_ef_coste.reindex(y24.index).values, fe.exposure.reindex(idx25).values)

    def run_ws(ws):
        own = [(w, s, v, p) for (run, _, s, v, p), w in zip(raw, ws) if run not in excl and w > 0]
        if not own:
            own = [(1.0, s, v, p) for (run, _, s, v, p) in raw if run not in excl]
        wo = sum(o[0] for o in own)
        ref = [sum(o[0] * o[j].sum() for o in own) / wo for j in (1, 2, 3)]
        comps = [(s * ref[0] / s.sum(), v * ref[1] / v.sum(), p * ref[2] / p.sum()) for run, _, s, v, p in raw]
        r, *_ = evaluate(comps, ws, ysel.values, y24.values, 1.0, (a, b), expos)
        return r["d2_sel"], r["d2_v24_f23"]

    base = run_ws([c[1] for c in raw])
    print(line, "partida", [c[1] for c in raw], "V23 %.5f V24 %.5f" % base, flush=True)
    k = len(raw)
    rows = []
    for combo in itertools.product(range(11), repeat=k):
        if sum(combo) != 10:
            continue
        ws = [c / 10 for c in combo]
        s23, s24 = run_ws(ws)
        rows.append((ws, s23, s24))
    for nombre, key in (("V23", lambda r: r[1]), ("V24", lambda r: r[2]), ("media", lambda r: r[1] + r[2]),
                        ("la peor de las dos mejoras", lambda r: min(r[1] - base[0], r[2] - base[1]))):
        print("mejores por", nombre, flush=True)
        for ws, s23, s24 in sorted(rows, key=key, reverse=True)[:5]:
            print("   ", ws, "V23 %.5f (%+.5f)  V24 %.5f (%+.5f)" % (s23, s23 - base[0], s24, s24 - base[1]), flush=True)


if __name__ == "__main__":
    main()
'''


### Embedded source: `src/modelos/rejilla_robusta.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/modelos/rejilla_robusta.py'] = '''"""Rejilla de pesos en el símplex (pasos de 0,1) con la peor de cuatro mejoras: V23 y V24, con todas las filas y sin el 0,1 %
superior de T0 del ramo (el mismo recorte que robustez.py, sin renivelar).

Igual que rejilla_pesos.py en todo lo demás: carga los componentes una vez, aplica la regla de nivel indicada y la forma del
ramo, y evalúa cada combinación con evaluate de mezcla3 (V23 selección sin factor; V24 con el factor de V23). Informa las
mejores combinaciones por la peor de las cuatro mejoras frente a la partida, y también por la peor de las dos de siempre.
Preinscrito en el foro (CLA-20261018-016) antes de calcular.
Uso: python3 rejilla_robusta.py <ramo> <forma a:b> <nivel|-> <corrida:sel:peso_partida> [...]
"""
from __future__ import annotations

import itertools
import sys

import numpy as np
import pandas as pd

from comun import WORK, d2
from mezcla3 import component, evaluate, labels, sin_referencia


def main():
    line, forma, nivel = sys.argv[1], sys.argv[2], sys.argv[3]
    a, b = (float(x) for x in forma.split(":"))
    comps_in = [c.split(":") for c in sys.argv[4:]]
    ysel, y24, idx25 = labels(line)
    raw = []
    for run, sel_name, w in comps_in:
        s, v, p = component(run, sel_name, line)
        raw.append((run, float(w), s.reindex(ysel.index).values, v.reindex(y24.index).values, p.reindex(idx25).values))
    excl = sin_referencia({"nivel": None if nivel == "-" else nivel})
    fe = pd.read_parquet(WORK / f"feat_{line}.parquet", columns=["row_id", "e_ef_coste", "exposure"])
    fe["row_id"] = fe.row_id.astype(str); fe = fe.set_index("row_id")
    expos = (fe.e_ef_coste.reindex(ysel.index).values, fe.e_ef_coste.reindex(y24.index).values, fe.exposure.reindex(idx25).values)
    y23, y4 = ysel.values, y24.values
    k23 = y23 <= np.quantile(y23, 0.999); k24 = y4 <= np.quantile(y4, 0.999)

    def run_ws(ws):
        own = [(w, s, v, p) for (run, _, s, v, p), w in zip(raw, ws) if run not in excl and w > 0]
        if not own:
            own = [(1.0, s, v, p) for (run, _, s, v, p) in raw if run not in excl]
        wo = sum(o[0] for o in own)
        ref = [sum(o[0] * o[j].sum() for o in own) / wo for j in (1, 2, 3)]
        comps = [(s * ref[0] / s.sum(), v * ref[1] / v.sum(), p * ref[2] / p.sum()) for run, _, s, v, p in raw]
        r, bsel, b24, _ = evaluate(comps, ws, y23, y4, 1.0, (a, b), expos)
        return (r["d2_sel"], r["d2_v24_f23"], d2(y23[k23], bsel[k23]), d2(y4[k24], b24[k24]))

    base = run_ws([c[1] for c in raw])
    print(line, "partida", [c[1] for c in raw], "V23 %.5f V24 %.5f | sin top V23 %.5f V24 %.5f" % base, flush=True)
    rows = []
    for combo in itertools.product(range(11), repeat=len(raw)):
        if sum(combo) != 10:
            continue
        ws = [c / 10 for c in combo]
        rows.append((ws, run_ws(ws)))
    for nombre, key in (("la peor de las cuatro mejoras", lambda r: min(x - y for x, y in zip(r[1], base))),
                        ("la peor de las dos mejoras (completo)", lambda r: min(r[1][0] - base[0], r[1][1] - base[1]))):
        print("mejores por", nombre, flush=True)
        for ws, m in sorted(rows, key=key, reverse=True)[:5]:
            print("   ", ws, " ".join("%+.5f" % (x - y) for x, y in zip(m, base)), flush=True)


if __name__ == "__main__":
    main()
'''


### Embedded source: `src/modelos/reproducir_mezcla.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/modelos/reproducir_mezcla.py'] = '''"""Repite una mezcla de Claude a partir de los archivos guardados en Modelos_claude y compara el SHA-256 con el entregado.

mezcla3.py espera una carpeta de trabajo con sel_<nombre>.parquet en la raíz y una carpeta por corrida con v24.parquet y
prediccion_2025.parquet. En el equipo esas piezas están repartidas entre runs/<corrida>/ y selecciones_v23/. Este script
monta una carpeta de trabajo temporal con enlaces simbólicos a esos archivos (sin copiarlos ni tocarlos), añade las
exposiciones de insumos/exposicion_<ramo>.parquet como feat_<ramo>.parquet (mezcla3 solo lee row_id, e_ef_coste y
exposure), ejecuta mezcla3.py con la misma especificación, forma y regla de nivel, y compara el CSV resultante con el
entregado, byte a byte por SHA-256.
Uso: python3 reproducir_mezcla.py <spec_json> <forma> <nivel|-> <nombre> <sha256_esperado> [<carpeta_temporal>]
Ejemplo (S18): python3 reproducir_mezcla.py experimentos/spec_s18_propio.json motor:1.09:0.95,buildings:1.08:1.14,contents:1.07:1 s17 S18_claude_propio a97ad416cbe2967d4734d635bef3e679b64ebceebeb6bb30581358d70b79e7e8
"""
from __future__ import annotations

import hashlib
import json
import os
import subprocess
import sys
import tempfile
from pathlib import Path

AQUI = Path(__file__).resolve().parent
SEMANA1 = AQUI.parent


def sha(p: Path) -> str:
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()


def buscar_sel(nombre: str) -> Path:
    cands = list((AQUI / "runs").glob(f"*/sel_{nombre}.parquet")) + list((AQUI / "selecciones_v23").glob(f"sel_{nombre}.parquet"))
    if not cands:
        raise SystemExit(f"No encuentro sel_{nombre}.parquet en runs/ ni en selecciones_v23/")
    return cands[0]


def main():
    spec_path, forma, nivel, nombre, esperado = sys.argv[1:6]
    tmp = Path(sys.argv[6]) if len(sys.argv) > 6 else Path(tempfile.mkdtemp(prefix="repro_"))
    work = tmp / "work"
    work.mkdir(parents=True, exist_ok=True)
    spec = json.loads((AQUI / spec_path).read_text(encoding="utf-8")) if not Path(spec_path).is_absolute() else json.loads(Path(spec_path).read_text(encoding="utf-8"))
    corridas = {"S04b_lgbm_F0124_F3autos"}  # mezcla3.labels() toma de aquí la etiqueta de V24 y los IDs de 2025
    sels = {"total_F0124"}  # y de aquí la etiqueta de V23 selección
    for line in spec:
        for run, sel_name, _ in spec[line]:
            corridas.add(run); sels.add(sel_name)
    for run in sorted(corridas):
        d = work / run
        d.mkdir(exist_ok=True)
        for f in ("v24.parquet", "prediccion_2025.parquet"):
            src = AQUI / "runs" / run / f
            if not src.exists():
                raise SystemExit(f"Falta {src}")
            if not (d / f).exists():
                (d / f).symlink_to(src)
    for s in sorted(sels):
        dst = work / f"sel_{s}.parquet"
        if not dst.exists():
            dst.symlink_to(buscar_sel(s))
    for line in ("motor", "buildings", "contents"):
        dst = work / f"feat_{line}.parquet"
        if not dst.exists():
            dst.symlink_to(AQUI / "insumos" / f"exposicion_{line}.parquet")
    env = dict(os.environ, MODELOS_WORK=str(work), HAKATON_SEMANA1=str(SEMANA1))
    cmd = [sys.executable, "-W", "ignore", str(AQUI / "mezcla3.py"), nombre, str(AQUI / spec_path), f"--forma={forma}", "--escribir"]
    if nivel != "-":
        cmd.append(f"--nivel={nivel}")
    print("Ejecuto:", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True, env=env, cwd=str(AQUI))
    csv = work / nombre / f"{nombre}.csv"
    obtenido = sha(csv)
    rep = {"nombre": nombre, "sha256_esperado": esperado, "sha256_obtenido": obtenido, "coincide": obtenido == esperado, "carpeta": str(work)}
    if obtenido != esperado:
        # otra versión de numpy puede cambiar el último bit de alguna suma; se mide la diferencia numérica con el entregado
        import numpy as np
        import pandas as pd
        ref = SEMANA1 / "Envios" / "archivos" / f"{nombre}.csv"
        a = pd.read_csv(csv, dtype={"row_id": str}); r = pd.read_csv(ref, dtype={"row_id": str})
        assert (a.row_id.values == r.row_id.values).all()
        dif = np.abs(a.prediction.values - r.prediction.values)
        rep.update({"referencia": str(ref), "filas_distintas": int((dif > 0).sum()), "dif_max": float(dif.max()),
                    "total_obtenido": round(float(a.prediction.sum()), 2), "total_entregado": round(float(r.prediction.sum()), 2)})
    print(json.dumps(rep, ensure_ascii=False, indent=1), flush=True)
    if obtenido != esperado and rep["dif_max"] > 1.01e-6:
        raise SystemExit("El CSV repetido no coincide con el entregado")


if __name__ == "__main__":
    main()
'''


### Embedded source: `src/modelos/robustez.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/modelos/robustez.py'] = '''"""Robustez pareada de candidatos de mezcla frente a una referencia, por ramo y global, en V23 selección y V24.

Cada candidato se reconstruye igual que en mezcla3.py (componentes reescalados al nivel de los de Claude, forma por ramo y
factor de V23 en V24) y se compara con la referencia con tres medidas:
- diferencia de D² (V23 selección sin factor; V24 con el factor de V23, como informa mezcla3),
- bootstrap pareado por cliente (Poisson(1) por cliente, 1000 réplicas) de la diferencia de devianza: P(mejora > 0) e
  intervalo del 90 %; en el global el cliente agrupa sus filas de los tres ramos,
- sensibilidad sin el 0,1 % superior de la pérdida observada de cada ramo y corte (el filtro que usa Codex).
Uso: python3 robustez.py <salida.json> <ref>=<spec>[@<forma>] <cand>=<spec>[@<forma>] ...
forma con el formato de mezcla3: motor:1.09:0.95,buildings:1.08:1.14,contents:1.07:1
"""
from __future__ import annotations

import json
import sys

import numpy as np
import pandas as pd

from comun import LINES, WORK, d2, deviance
from mezcla3 import component, evaluate, ftw, labels, sin_referencia

B = 1000


def parse(arg):
    """<nombre>=<spec>[@<forma>[@<regla_de_nivel>]]; la regla s17 saca de la referencia de nivel los componentes inestables."""
    name, rest = arg.split("=", 1)
    spec, _, rest2 = rest.partition("@")
    forma, _, nivel = rest2.partition("@")
    fm = {kv.split(":")[0]: (float(kv.split(":")[1]), float(kv.split(":")[2])) for kv in forma.split(",") if kv}
    if nivel:
        fm["_nivel"] = nivel
    return name, json.loads(open(spec).read()), fm


def blend(spec, forma, line, ysel, y24, idx25, expos):
    raw = []
    for run, sel_name, w in spec[line]:
        s, v, p = component(run, sel_name, line)
        raw.append((run, w, s.reindex(ysel.index).values, v.reindex(y24.index).values, p.reindex(idx25).values))
    own = [(w, s, v, p) for run, w, s, v, p in raw if run not in sin_referencia({"nivel": forma.get("_nivel")})]
    wo = sum(o[0] for o in own)
    ref = [sum(o[0] * o[j].sum() for o in own) / wo for j in (1, 2, 3)]
    comps = [(s * ref[0] / s.sum(), v * ref[1] / v.sum(), p * ref[2] / p.sum()) for run, w, s, v, p in raw]
    res, bsel, b24, b25 = evaluate(comps, [c[2] for c in spec[line]], ysel.values, y24.values, 1.0, forma.get(line), expos)
    return res, bsel, b24, b25


def boot(dif_by_c, null_total, rng):
    # el denominador se fija en la devianza nula total: la réplica solo mueve el numerador; bloques de 50 réplicas por memoria
    d = dif_by_c.astype(np.float64)
    out = [rng.poisson(1.0, size=(50, len(d))).astype(np.float64) @ d for _ in range(B // 50)]
    return np.concatenate(out) / null_total


def main():
    out_path = sys.argv[1]
    cands = [parse(a) for a in sys.argv[2:]]
    rng = np.random.default_rng(20261018)
    data = {}
    for line in LINES:
        ysel, y24, idx25 = labels(line)
        fe = pd.read_parquet(WORK / f"feat_{line}.parquet", columns=["row_id", "customer_id", "e_ef_coste", "exposure"])
        fe["row_id"] = fe.row_id.astype(str); fe = fe.set_index("row_id")
        expos = (fe.e_ef_coste.reindex(ysel.index).values, fe.e_ef_coste.reindex(y24.index).values, fe.exposure.reindex(idx25).values)
        cust = {"V23": fe.customer_id.reindex(ysel.index).astype(str).values, "V24": fe.customer_id.reindex(y24.index).astype(str).values}
        preds = {}
        for name, spec, forma in cands:
            res, bsel, b24, b25 = blend(spec, forma, line, ysel, y24, idx25, expos)
            preds[name] = {"V23": bsel, "V24": b24, "tot25": float(b25.sum()), "res": res}
        data[line] = {"y": {"V23": ysel.values, "V24": y24.values}, "cust": cust, "preds": preds}
    ref = cands[0][0]
    rep = {"referencia": ref, "candidatos": {}}
    for name, _, _ in cands[1:]:
        r = {}
        for fold in ("V23", "V24"):
            rows = {}
            ally, allr, allc, allcu, keep_all = [], [], [], [], []
            for line in LINES:
                y = data[line]["y"][fold]; pr = data[line]["preds"][ref][fold]; pc = data[line]["preds"][name][fold]
                cu = data[line]["cust"][fold]
                dr, dc = deviance(y, pr), deviance(y, pc)
                null = deviance(y, np.full_like(y, y.mean())).sum()
                codes, inv = np.unique(cu, return_inverse=True)
                dif_c = np.bincount(inv, weights=dr - dc)
                bs = boot(dif_c, null, rng)
                keep = y <= np.quantile(y, 0.999)
                rows[line] = {"d2_ref": round(d2(y, pr), 5), "d2_cand": round(d2(y, pc), 5), "dif": round(float((dr - dc).sum() / null), 6),
                              "P_mejora": round(float((bs > 0).mean()), 3), "ic90": [round(float(np.quantile(bs, 0.05)), 6), round(float(np.quantile(bs, 0.95)), 6)],
                              "dif_sin_top01": round(d2(y[keep], pc[keep]) - d2(y[keep], pr[keep]), 6),
                              # mismo filtro, con cada predicción llevada a su nivel óptimo en el subconjunto: separa forma de nivel
                              "dif_sin_top01_renivelado": round(d2(y[keep], pc[keep] * ftw(y[keep], pc[keep])) - d2(y[keep], pr[keep] * ftw(y[keep], pr[keep])), 6),
                              "dif_renivelado": round(d2(y, pc * ftw(y, pc)) - d2(y, pr * ftw(y, pr)), 6),
                              "nivel_cand_sobre_ref": round(float(pc.sum() / pr.sum()), 4)}
                ally.append(y); allr.append(pr); allc.append(pc); allcu.append(cu); keep_all.append(keep)
            y = np.concatenate(ally); pr = np.concatenate(allr); pc = np.concatenate(allc); cu = np.concatenate(allcu); keep = np.concatenate(keep_all)
            dr, dc = deviance(y, pr), deviance(y, pc)
            null = deviance(y, np.full_like(y, y.mean())).sum()
            codes, inv = np.unique(cu, return_inverse=True)
            bs = boot(np.bincount(inv, weights=dr - dc), null, rng)
            rows["global"] = {"d2_ref": round(d2(y, pr), 5), "d2_cand": round(d2(y, pc), 5), "dif": round(float((dr - dc).sum() / null), 6),
                              "P_mejora": round(float((bs > 0).mean()), 3), "ic90": [round(float(np.quantile(bs, 0.05)), 6), round(float(np.quantile(bs, 0.95)), 6)],
                              "dif_sin_top01": round(d2(y[keep], pc[keep]) - d2(y[keep], pr[keep]), 6)}
            r[fold] = rows
        r["total_2025"] = {line: {"ref": round(data[line]["preds"][ref]["tot25"], 0), "cand": round(data[line]["preds"][name]["tot25"], 0)} for line in LINES}
        rep["candidatos"][name] = r
        print("==", name, "frente a", ref, flush=True)
        for fold in ("V23", "V24"):
            for k, v in r[fold].items():
                extra = f"  renivelado {v['dif_renivelado']:+.5f}  sin top renivelado {v['dif_sin_top01_renivelado']:+.5f}  nivel {v['nivel_cand_sobre_ref']}" if k != "global" else ""
                print(f"  {fold} {k:9s} dif {v['dif']:+.5f}  P>0 {v['P_mejora']:.3f}  IC90 [{v['ic90'][0]:+.5f}, {v['ic90'][1]:+.5f}]  sin top 0,1 % {v['dif_sin_top01']:+.5f}{extra}", flush=True)
    (WORK / out_path).write_text(json.dumps(rep, indent=1, ensure_ascii=False))


if __name__ == "__main__":
    main()
'''


### Embedded source: `src/modelos/semestres.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/modelos/semestres.py'] = '''"""Diferencias de D² en 2024 por semestre de inicio del periodo (H1: enero-junio; H2: julio-diciembre), nominal y con +25 %.

Compara una mezcla de Claude con H, A_G y otra mezcla de Claude, por ramo y en el agregado, con IC95 por cliente y por
dirección (Poisson(1), 1.000 réplicas, denominador fijo dentro de cada semestre; la dirección usa su propio generador para
que los intervalos por cliente no cambien respecto de la versión anterior). Informa también el peso de cada semestre en la
exposición y su madurez (exposición efectiva entre nominal), porque el segundo semestre tiene las etiquetas menos maduras.
Uso: python3 semestres.py <salida.json> <cand>=<spec>@<forma>@<nivel> <otra>=<spec>@<forma>@<nivel>
"""
from __future__ import annotations

import json
import sys

import numpy as np
import pandas as pd

from comun import LINES, WORK, d2, deviance
from escenarios_reserva import C, reserva_por_fila
from mezcla3 import labels
from robustez import blend, parse

B = 1000


def ic(dif, groups, null, rng):
    codes, inv = np.unique(groups, return_inverse=True)
    d = np.bincount(inv, weights=dif)
    bs = np.concatenate([rng.poisson(1.0, size=(50, len(d))) @ d for _ in range(B // 50)]) / null
    return [round(float(np.quantile(bs, 0.025)), 6), round(float(np.quantile(bs, 0.975)), 6)]


def main():
    out = sys.argv[1]
    (cn, cspec, cfm), (on, ospec, ofm) = parse(sys.argv[2]), parse(sys.argv[3])
    R = reserva_por_fila()
    ag = pd.read_parquet(f"{C}/crossfit_meta_gam_guard/predictions_V24_crossfit.parquet", columns=["row_id", "meta_gam"]).set_index("row_id").meta_gam
    hh = pd.read_parquet(f"{C}/geometric_meta_nested/V24.parquet", columns=["row_id", "candidate"]).set_index("row_id").candidate
    rng = np.random.default_rng(20261018)
    rng_ad = np.random.default_rng(20261019)
    V = {k: [] for k in ("y", "r", "cu", "ad", "ln", "sem", "e", "eef", cn, on, "H", "A_G")}
    for line in LINES:
        ysel, y24, idx25 = labels(line)
        fe = pd.read_parquet(WORK / f"feat_{line}.parquet", columns=["row_id", "customer_id", "address_id", "e_ef_coste", "exposure", "period_start"])
        fe["row_id"] = fe.row_id.astype(str); fe = fe.set_index("row_id")
        expos = (fe.e_ef_coste.reindex(ysel.index).values, fe.e_ef_coste.reindex(y24.index).values, fe.exposure.reindex(idx25).values)
        V[cn].append(blend(cspec, cfm, line, ysel, y24, idx25, expos)[2]); V[on].append(blend(ospec, ofm, line, ysel, y24, idx25, expos)[2])
        V["H"].append(hh.reindex(y24.index).values); V["A_G"].append(ag.reindex(y24.index).values)
        f = fe.reindex(y24.index)
        V["y"].append(y24.values); V["r"].append(R.reindex(y24.index).fillna(0.0).values); V["cu"].append(f.customer_id.astype(str).values)
        V["ad"].append(f.address_id.astype(str).values)
        V["ln"].append(np.full(len(y24), line)); V["sem"].append(np.where(pd.to_datetime(f.period_start).dt.month.values <= 6, "H1", "H2"))
        V["e"].append(f.exposure.values); V["eef"].append(f.e_ef_coste.values)
    V = {k: np.concatenate(v) for k, v in V.items()}
    rep = {"candidato": cn, "comparadores": [on, "H", "A_G"], "resultados": {}}
    for a in (1.0, 1.25):
        y_all = V["y"] + (a - 1.0) * V["r"]
        res = {}
        for scope in LINES + ["agregado"]:
            for sem in ("H1", "H2"):
                msk = (V["sem"] == sem) & (np.ones(len(y_all), bool) if scope == "agregado" else (V["ln"] == scope))
                y = y_all[msk]; null = deviance(y, np.full_like(y, y.mean())).sum()
                row = {"filas": int(msk.sum()), "peso_exposicion": round(float(V["e"][msk].sum() / V["e"][(np.ones(len(y_all), bool) if scope == "agregado" else (V["ln"] == scope))].sum()), 4),
                       "madurez": round(float(V["eef"][msk].sum() / V["e"][msk].sum()), 3), "d2": {k: round(d2(y, V[k][msk]), 6) for k in (cn, on, "H", "A_G")}}
                for k in (on, "H", "A_G"):
                    dif = deviance(y, V[k][msk]) - deviance(y, V[cn][msk])
                    row[f"{cn}_menos_{k}"] = {"dif": round(float(dif.sum() / null), 6), "ic95_cliente": ic(dif, V["cu"][msk], null, rng),
                                              "ic95_direccion": ic(dif, V["ad"][msk], null, rng_ad)}
                res[f"{scope}_{sem}"] = row
                print(f"a={a} {scope} {sem} filas {row['filas']} peso {row['peso_exposicion']} madurez {row['madurez']} " +
                      " ".join(f"{k}: {row[f'{cn}_menos_{k}']['dif']:+.6f} {row[f'{cn}_menos_{k}']['ic95_cliente']} dir {row[f'{cn}_menos_{k}']['ic95_direccion']}" for k in (on, "H", "A_G")), flush=True)
        rep["resultados"][str(a)] = res
    (WORK / out).write_text(json.dumps(rep, indent=1, ensure_ascii=False))


if __name__ == "__main__":
    main()
'''


### Embedded source: `src/modelos/severidad_cfs.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/modelos/severidad_cfs.py'] = '''"""Severidad por capital asegurado para la frecuencia CatBoost por peligro (CFS), sin reentrenar la frecuencia.

La corrida CFS multiplica la frecuencia por peligro por una severidad media constante del peligro. En hogar la severidad
crece con el capital: en 2021-2023, del quintil inferior al superior de capital, inundación pasa de 20.136 a 40.027 CU por
siniestro, incendio de 29.765 a 70.601, hundimiento de 15.985 a 29.554, tormenta de 2.292 a 4.323 y escape de agua de 3.662
a 7.387; en inundación, el sótano la sube de 27.672 a 43.212. Este script ajusta por peligro y corte un GLM gamma con enlace
logarítmico sobre los siniestros positivos de los años de entrenamiento de ese corte (log del capital relativo a la mediana
y, en inundación, sótano), con peso igual al número de siniestros, y multiplica la contribución guardada de cada peligro
(frecuencia x severidad constante) por la relatividad, normalizada para que su media ponderada en los siniestros de
entrenamiento valga 1. Cortes: V23 (entrena 2021-2022, predice la mitad de selección de 2023), V24 (2021-2023, predice 2024)
y producción (2021-2024, predice 2025). Escribe una corrida nueva con el formato de las demás.
En autos la severidad crece con el grupo de valor del vehículo y baja con su antigüedad en daños propios, robo y lunas
(daños propios: de 1.417 a 2.231 CU por quintil de valor); terceros y lesiones quedan con severidad constante. En autos
se pueden combinar corridas de semillas distintas: --v23=<corrida>, --v24=<c1,c2> y --prod=<c1,c2> (promedio simple).
Con --tope=q (por ejemplo 0.995) las relatividades se estiman con la severidad por fila acotada al cuantil q de los siniestros
positivos del peligro en el entrenamiento del corte; la normalización a media 1 y la severidad media del peligro no cambian,
así que el nivel es el mismo y solo se reduce el peso de los siniestros extremos en las pendientes.
Uso: python3 severidad_cfs.py <corrida_cfs> <ramo> <corrida_salida> [--v23=..] [--v24=..] [--prod=..] [--tope=q]
"""
from __future__ import annotations

import json
import sys

import numpy as np
import pandas as pd

from comun import F0DIR, PERILS, WORK, d2


def glm_gamma(Z, y, w, iters=50):
    """GLM gamma con enlace log por IRLS (con enlace log y varianza mu^2 el peso de trabajo es el peso previo)."""
    X = np.column_stack([np.ones(len(y)), Z])
    beta = np.zeros(X.shape[1]); beta[0] = np.log(np.average(y, weights=w))
    for _ in range(iters):
        eta = X @ beta; mu = np.exp(eta)
        z = eta + (y - mu) / mu
        new = np.linalg.solve((X * w[:, None]).T @ X, (X * w[:, None]).T @ z)
        if np.max(np.abs(new - beta)) < 1e-9:
            beta = new
            break
        beta = new
    return beta


def main():
    src, line, name = sys.argv[1:4]
    opts = dict(a[2:].split("=", 1) for a in sys.argv[4:] if a.startswith("--"))
    run_v23 = opts.get("v23", src); runs_v24 = opts.get("v24", src).split(","); runs_prod = opts.get("prod", src).split(",")
    tope = float(opts["tope"]) if "tope" in opts else None
    extra = ["sum_insured", "basement"] if line != "motor" else ["vehicle_value_group", "vehicle_age"]
    cols = ["row_id", "year", "T0"] + extra + [f"{k}_{p}" for p in PERILS[line] for k in ("n", "y")]
    df = pd.read_parquet(WORK / f"feat_{line}.parquet", columns=cols)
    part = pd.read_parquet(F0DIR / "tables" / "particiones.parquet", columns=["row_id", "cal_half"])
    df = df.merge(part, on="row_id", how="left", validate="one_to_one")
    df["row_id"] = df.row_id.astype(str)
    yr = df.year.values
    if line != "motor":
        si = df.sum_insured.astype(float).values
        si = np.where(np.isfinite(si) & (si > 0), si, np.nan)
        bas = df.basement.astype(float).fillna(0).values
    else:
        vvg = df.vehicle_value_group.astype(float).values
        vage = np.clip(df.vehicle_age.astype(float).fillna(df.vehicle_age.median()).values, 0, 30) / 10.0
    folds = {"V23": (yr <= 2022, (yr == 2023) & (df.cal_half.values == 1)), "V24": (yr <= 2023, yr == 2024), "PROD": (yr <= 2024, yr == 2025)}
    out = WORK / name
    out.mkdir(parents=True, exist_ok=True)
    rep = {"origen": src, "line": line, "tope": tope, "perils": {}}
    parts = {}
    for fold, (trm, evm) in folds.items():
        if line != "motor":
            med = np.nanmedian(si[trm])
            zsi = np.log(np.where(np.isnan(si), med, si) / med)
        else:
            medv = np.nanmedian(vvg[trm])
            zv = np.log(np.where(np.isnan(vvg), medv, vvg) / medv)
        tot = np.zeros(evm.sum())
        for p in PERILS[line]:
            n = df[f"n_{p}"].fillna(0).values.astype(float); yp = df[f"y_{p}"].fillna(0).values
            pos = trm & (n > 0) & (yp > 0)
            if line != "motor":
                covs = [zsi] + ([bas] if p == "flood" else [])
            else:
                covs = [zv, vage] if p in ("own_damage", "theft", "glass_hail") else []
            if covs:
                Z = np.column_stack(covs)
                ys = yp[pos] / n[pos]
                if tope is not None:
                    ys = np.minimum(ys, np.quantile(ys, tope))
                beta = glm_gamma(Z[pos], ys, n[pos])
                rel = np.exp(Z @ beta[1:])
                rel = rel / np.average(rel[pos], weights=n[pos])
            else:
                beta = np.zeros(1); rel = np.ones(len(df))
            # contribución guardada del peligro (frecuencia x severidad constante) en el corte evaluado
            if fold == "V23":
                lam = np.load(WORK / run_v23 / f"v23_{p}.npz")["lam"]
                sev = yp[trm].sum() / max(n[trm].sum(), 1.0)
                part = lam * sev
            else:
                rr = runs_v24 if fold == "V24" else runs_prod
                part = sum(np.load(WORK / r / f"{'v24' if fold == 'V24' else 'prod'}_{p}.npy") for r in rr) / len(rr)
            assert len(part) == evm.sum()
            tot += part * rel[evm]
            rep["perils"].setdefault(p, {})[fold] = {"betas": [round(float(b), 4) for b in beta[1:]], "siniestros": int(n[pos].sum())}
        parts[fold] = (evm, tot)
    t0 = df["T0"].fillna(0).values
    evm, mu = parts["V23"]
    rep["d2_sel"] = round(d2(t0[evm], mu), 5)
    pd.DataFrame({"row_id": df.row_id[evm].values, "line": line, "y": t0[evm], "mu": mu}).to_parquet(WORK / f"sel_{name}.parquet", index=False)
    evm, mu = parts["V24"]
    rep["d2_v24"] = round(d2(t0[evm], mu), 5)
    pd.DataFrame({"row_id": df.row_id[evm].values, "line": line, "y": t0[evm], "mu": mu}).to_parquet(out / "v24.parquet", index=False)
    evm, mu = parts["PROD"]
    rep["pred_2025_total_CU"] = round(float(mu.sum()), 2)
    pd.DataFrame({"row_id": df.row_id[evm].values, "line": line, "prediction": mu}).to_parquet(out / "prediccion_2025.parquet", index=False)
    base = json.loads((WORK / src / "resumen.json").read_text())
    rep["referencia_cfs"] = {"d2_sel": base.get("d2_sel"), "d2_v24": base.get("d2_v24")}
    (out / "resumen.json").write_text(json.dumps(rep, indent=1, ensure_ascii=False))
    print(json.dumps({k: v for k, v in rep.items() if k != "perils"}, ensure_ascii=False), flush=True)
    for p, v in rep["perils"].items():
        print("  ", p, v, flush=True)


if __name__ == "__main__":
    main()
'''


### Embedded source: `src/modelos/severidad_cfs3.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/modelos/severidad_cfs3.py'] = '''"""Severidad GLM por peligro con covariables físicas fijadas de antemano, sobre partes de frecuencia guardadas.

Amplía severidad_cfs.py. En cada corte y peligro ajusta un GLM gamma con enlace log sobre los siniestros positivos de los
años de entrenamiento (peso = número de siniestros) con una penalización ridge suave (LAMBDA) solo en las covariables nuevas: la constante y las covariables de CFSS
(capital y sótano en hogar; valor y antigüedad del vehículo en autos) quedan libres
y multiplica la contribución guardada de cada peligro (frecuencia x severidad media) por la relatividad, normalizada a
media 1 en los siniestros de entrenamiento. Las covariables de cada peligro se fijaron antes de mirar resultados, por
razones físicas, y no se eligen por peligro según el resultado:
- edificios: tormenta (capital, cubierta, tipo de vivienda, densidad urbana); inundación (capital, sótano, zona de
  inundación, distancia al río, tipo de vivienda); escape de agua (capital, tipo de vivienda, antigüedad, días
  desocupada, ocupación); hundimiento (capital, suelo, árboles a menos de 5 m, antigüedad, tipo de vivienda); incendio
  (capital, construcción, tipo de vivienda, edificio protegido, antigüedad); daños accidentales (capital, tipo de vivienda).
- contenidos: robo (capital, seguridad, objetos de valor, densidad urbana); escape de agua (capital, tipo de vivienda,
  antigüedad); daños accidentales (capital, objetos de valor); inundación (capital, sótano, zona de inundación).
- autos: daños propios (valor, antigüedad, carrocería, combustible, asistencia a la conducción); terceros (valor,
  densidad urbana, carrocería, cilindrada); lesiones (edad del conductor, conductor joven, densidad urbana, uso);
  robo (valor, antigüedad, aparcamiento nocturno); lunas (valor, antigüedad, carrocería).
Las categóricas usan los niveles vistos en el entrenamiento del corte (nivel más frecuente como base; faltante como
nivel propio; niveles con menos de MIN_NIVEL siniestros se juntan con la base).
Con --tope=q las relatividades se estiman con la severidad por fila acotada al cuantil q de los siniestros positivos del
peligro en el entrenamiento del corte (nivel sin cambio, como en severidad_cfs.py).
Uso: python3 severidad_cfs3.py <corrida_v23> <ramo> <corrida_salida> [--v24=c1,c2] [--prod=c1,c2] [--lambda=20] [--tope=q]
"""
from __future__ import annotations

import json
import sys

import numpy as np
import pandas as pd

from comun import PERILS, WORK, d2
from experimentos import load

LAMBDA = 20.0
MIN_NIVEL = 30
# (tipo, columna): "num" numérica ya transformada en preparar(); "cat" categórica
COV = {
    "buildings": {
        "storm": [("num", "zsi"), ("cat", "roof_type"), ("cat", "property_type"), ("cat", "urban_density")],
        "flood": [("num", "zsi"), ("num", "basement"), ("cat", "flood_zone"), ("num", "log_dist_rio"), ("cat", "property_type")],
        "escape_of_water": [("num", "zsi"), ("cat", "property_type"), ("num", "antiguedad"), ("num", "log_desocupada"), ("cat", "occupancy")],
        "subsidence": [("num", "zsi"), ("cat", "soil_type"), ("num", "arboles"), ("num", "antiguedad"), ("cat", "property_type")],
        "fire": [("num", "zsi"), ("cat", "construction_type_harmonized"), ("cat", "property_type"), ("num", "protegido"), ("num", "antiguedad")],
        "accidental_damage": [("num", "zsi"), ("cat", "property_type")],
    },
    "contents": {
        "theft": [("num", "zsi"), ("cat", "security_level"), ("num", "objetos_valor"), ("cat", "urban_density")],
        "escape_of_water": [("num", "zsi"), ("cat", "property_type"), ("num", "antiguedad")],
        "accidental_damage": [("num", "zsi"), ("num", "objetos_valor")],
        "flood": [("num", "zsi"), ("num", "basement"), ("cat", "flood_zone")],
    },
    "motor": {
        "own_damage": [("num", "zv"), ("num", "vage"), ("cat", "body_type"), ("cat", "fuel_type"), ("num", "asistencia")],
        "third_party_prop": [("num", "zv"), ("cat", "urban_density"), ("cat", "body_type"), ("num", "log_cc")],
        "bodily_injury": [("num", "edad"), ("num", "joven"), ("cat", "urban_density"), ("cat", "vehicle_use")],
        "theft": [("num", "zv"), ("num", "vage"), ("cat", "overnight_parking")],
        "glass_hail": [("num", "zv"), ("num", "vage"), ("cat", "body_type")],
    },
}


# covariables de CFSS (severidad_cfs.py): sin penalizar, para que el modelo ampliado contenga a CFSS como caso particular
LIBRES = {"buildings": {p: ("zsi",) for p in COV["buildings"]} | {"flood": ("zsi", "basement")},
          "contents": {p: ("zsi",) for p in COV["contents"]} | {"flood": ("zsi", "basement")},
          "motor": {"own_damage": ("zv", "vage"), "theft": ("zv", "vage"), "glass_hail": ("zv", "vage")}}


def glm_gamma_ridge(X, y, w, lam=LAMBDA, iters=60, libres=None):
    """GLM gamma con enlace log por IRLS y penalización ridge salvo en la constante y en las columnas libres (las de CFSS)."""
    beta = np.zeros(X.shape[1]); beta[0] = np.log(np.average(y, weights=w))
    P = np.eye(X.shape[1]) * lam; P[0, 0] = 0.0
    for j in (libres or []):
        P[j, j] = 0.0
    for _ in range(iters):
        eta = X @ beta; mu = np.exp(eta)
        z = eta + (y - mu) / mu
        XtW = (X * w[:, None]).T
        new = np.linalg.solve(XtW @ X + P, XtW @ z)
        if np.max(np.abs(new - beta)) < 1e-9:
            return new
        beta = new
    return beta


def preparar(df, line, trm):
    """Columnas numéricas transformadas, con faltantes imputados por la mediana del entrenamiento del corte."""
    out = {}
    yr = df.year.values.astype(float)

    def med(v):
        return np.nanmedian(v[trm])
    if line != "motor":
        si = df.sum_insured.astype(float).values; si = np.where(np.isfinite(si) & (si > 0), si, np.nan)
        m = med(si); out["zsi"] = np.log(np.where(np.isnan(si), m, si) / m)
        out["basement"] = df.basement.astype(float).fillna(0).values
        ant = np.clip(yr - df.year_built.astype(float).values, 0, 150); ant = np.where(np.isnan(ant), med(ant), ant)
        out["antiguedad"] = ant / 50.0
        if line == "buildings":
            d = np.log1p(df.distance_to_river_km.astype(float).values); out["log_dist_rio"] = np.where(np.isnan(d), med(d), d)
            u = np.log1p(df.days_unoccupied.astype(float).values); out["log_desocupada"] = np.where(np.isnan(u), med(u), u)
            out["arboles"] = df.trees_within_5m.astype(float).fillna(0).values
            out["protegido"] = df.listed_building.astype(float).fillna(0).values
        else:
            h = df.high_value_items.astype(float).values; out["objetos_valor"] = np.where(np.isnan(h), med(h), h)
    else:
        v = df.vehicle_value_group.astype(float).values; m = med(v); out["zv"] = np.log(np.where(np.isnan(v), m, v) / m)
        va = df.vehicle_age.astype(float).values; out["vage"] = np.clip(np.where(np.isnan(va), med(va), va), 0, 30) / 10.0
        out["asistencia"] = df.driver_assist.astype(float).fillna(0).values
        cc = np.log(df.engine_size_cc.astype(float).values / 2000.0); out["log_cc"] = np.where(np.isnan(cc), med(cc), cc)
        age = df.driver_age.astype(float).values
        out["edad"] = (age - 45.0) / 20.0
        out["joven"] = (age < 25).astype(float)
    return out


def disenio(df, nums, spec, pos, n, libres_cols=()):
    """Matriz de diseño de un peligro: constante, numéricas y categóricas codificadas con los niveles del entrenamiento.
    Devuelve también las posiciones de las covariables de CFSS, que no se penalizan."""
    cols = [np.ones(len(df))]
    libres = []
    for kind, c in spec:
        if kind == "num":
            if c in libres_cols:
                libres.append(len(cols))
            cols.append(nums[c])
            continue
        v = df[c].astype("string").fillna("NA").values.astype(str)
        cnt = pd.Series(n[pos]).groupby(v[pos]).sum().sort_values(ascending=False)
        base = cnt.index[0]
        for lev in cnt.index[1:]:
            if cnt[lev] >= MIN_NIVEL:
                cols.append((v == lev).astype(float))
    return np.column_stack(cols), libres


def main():
    src, line, name = sys.argv[1:4]
    opts = dict(a[2:].split("=", 1) for a in sys.argv[4:] if a.startswith("--"))
    runs_v24 = opts.get("v24", src).split(","); runs_prod = opts.get("prod", src).split(","); run_v23 = opts.get("v23", src)
    global LAMBDA
    LAMBDA = float(opts.get("lambda", LAMBDA))
    tope = float(opts["tope"]) if "tope" in opts else None
    df = load(line)
    yr = df.year.values
    folds = {"V23": (yr <= 2022, (yr == 2023) & (df.cal_half.values == 1)), "V24": (yr <= 2023, yr == 2024), "PROD": (yr <= 2024, yr == 2025)}
    out = WORK / name
    out.mkdir(parents=True, exist_ok=True)
    rep = {"origen": {"v23": run_v23, "v24": runs_v24, "prod": runs_prod}, "line": line, "lambda": LAMBDA, "tope": tope, "perils": {}}
    parts = {}
    for fold, (trm, evm) in folds.items():
        nums = preparar(df, line, trm)
        tot = np.zeros(evm.sum())
        for p in PERILS[line]:
            n = df[f"n_{p}"].fillna(0).values.astype(float); yp = df[f"y_{p}"].fillna(0).values
            pos = trm & (n > 0) & (yp > 0)
            X, libres = disenio(df, nums, COV[line][p], pos, n, LIBRES[line].get(p, ()))
            ys = yp[pos] / n[pos]
            if tope is not None:
                ys = np.minimum(ys, np.quantile(ys, tope))
            beta = glm_gamma_ridge(X[pos], ys, n[pos], lam=LAMBDA, libres=libres)
            rel = np.exp(X @ beta - beta[0])
            rel = rel / np.average(rel[pos], weights=n[pos])
            if fold == "V23":
                lam = np.load(WORK / run_v23 / f"v23_{p}.npz")["lam"]
                part = lam * (yp[trm].sum() / max(n[trm].sum(), 1.0))
            else:
                rr = runs_v24 if fold == "V24" else runs_prod
                part = sum(np.load(WORK / r / f"{'v24' if fold == 'V24' else 'prod'}_{p}.npy") for r in rr) / len(rr)
            assert len(part) == evm.sum()
            tot += part * rel[evm]
            rep["perils"].setdefault(p, {})[fold] = {"parametros": int(X.shape[1] - 1), "siniestros": int(n[pos].sum()),
                                                     "rel_p5_p95": [round(float(np.quantile(rel[evm], 0.05)), 3), round(float(np.quantile(rel[evm], 0.95)), 3)]}
        parts[fold] = (evm, tot)
    t0 = df["T0"].fillna(0).values; row_ids = df.row_id.astype(str).values
    evm, mu = parts["V23"]; rep["d2_sel"] = round(d2(t0[evm], mu), 5)
    pd.DataFrame({"row_id": row_ids[evm], "line": line, "y": t0[evm], "mu": mu}).to_parquet(WORK / f"sel_{name}.parquet", index=False)
    evm, mu = parts["V24"]; rep["d2_v24"] = round(d2(t0[evm], mu), 5)
    pd.DataFrame({"row_id": row_ids[evm], "line": line, "y": t0[evm], "mu": mu}).to_parquet(out / "v24.parquet", index=False)
    evm, mu = parts["PROD"]; rep["pred_2025_total_CU"] = round(float(mu.sum()), 2)
    pd.DataFrame({"row_id": row_ids[evm], "line": line, "prediction": mu}).to_parquet(out / "prediccion_2025.parquet", index=False)
    (out / "resumen.json").write_text(json.dumps(rep, indent=1, ensure_ascii=False))
    print(json.dumps({k: v for k, v in rep.items() if k != "perils"}, ensure_ascii=False), flush=True)
    for p, v in rep["perils"].items():
        print("  ", p, v, flush=True)


if __name__ == "__main__":
    main()
'''


### Embedded source: `src/linea_b/mezcla3_b.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/linea_b/mezcla3_b.py'] = '''"""mezcla3.py de la línea principal con la regla de nivel s17 extendida a las corridas B_ (nivel_b).
Uso: igual que mezcla3.py: python3 mezcla3_b.py <nombre> <spec_json> [--forma=...] [--nivel=s17] [--escribir]
"""
import json
import os as _os
import sys
from pathlib import Path as _Path

_MC = _os.environ.get("MODELOS_CLAUDE") or next((str(p) for p in (_Path(__file__).resolve().parent.parent / "Modelos_claude", _Path("/home/claude/mc")) if p.exists()), "/home/claude/mc")
sys.path.insert(0, _MC)
sys.path.insert(0, str(_Path(__file__).resolve().parent))
import nivel_b  # noqa: E402

spec = json.loads(open(sys.argv[2]).read())
nivel_b.clasificar({line: [c[0] for c in comps] for line, comps in spec.items()})
extra = nivel_b.aplicar()
print("fuera de la referencia de nivel (corridas B):", sorted(extra), flush=True)
nivel_b.mezcla3.main()
'''


### Embedded source: `src/linea_b/nivel_b.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/linea_b/nivel_b.py'] = '''"""Regla de nivel s17 aplicada a corridas nuevas de la línea B, sin tocar mezcla3.py.

mezcla3.sin_referencia excluye de la referencia de nivel un conjunto fijo de corridas (NIVEL_INESTABLE_S17). Para las
corridas nuevas de esta línea, la misma regla (sin etiquetas) se calcula aquí: el total 2025 entre el total V24 de la
corrida, comparado con el del ancla del ramo (CAT_motor_4s, S12h_edif_md1000, S08h_hogar_F3s2); si se aparta más de un
1,5 %, la corrida queda fuera de la referencia de nivel. El resultado se guarda en WORK/nivel_inestable_b.json y se
añade al conjunto fijo. Importar `aplicar()` antes de usar mezcla3/robustez/publico_simulado.
"""
from __future__ import annotations

import json
import sys

import pandas as pd

import os as _os
from pathlib import Path as _Path
# código de la línea principal de Claude: carpeta hermana Modelos_claude en el repositorio, o la copia de la nube
_MC = _os.environ.get("MODELOS_CLAUDE") or next((str(p) for p in (_Path(__file__).resolve().parent.parent / "Modelos_claude", _Path("/home/claude/mc")) if p.exists()), "/home/claude/mc")
sys.path.insert(0, _MC)
import mezcla3  # noqa: E402
from comun import WORK  # noqa: E402

ANCLA = {"motor": "CAT_motor_4s", "buildings": "S12h_edif_md1000", "contents": "S08h_hogar_F3s2"}
UMBRAL = 0.015
CACHE = WORK / "nivel_inestable_b.json"


def razon(run, line):
    v = pd.read_parquet(WORK / run / "v24.parquet"); v = v[v.line == line]["mu"].sum()
    p = pd.read_parquet(WORK / run / "prediccion_2025.parquet"); p = p[p.line == line]["prediction"].sum()
    return float(p / v)


def clasificar(runs_por_ramo: dict) -> dict:
    """{ramo: [corridas]} -> {corrida: {"desvio": x, "fuera": bool}}; actualiza la caché."""
    cache = json.loads(CACHE.read_text()) if CACHE.exists() else {}
    for line, runs in runs_por_ramo.items():
        ra = razon(ANCLA[line], line)
        for run in runs:
            if run in cache or run in mezcla3.NIVEL_INESTABLE_S17 or run in mezcla3.EXTERNOS or run == ANCLA[line]:
                continue
            d = razon(run, line) / ra - 1.0
            cache[run] = {"ramo": line, "desvio_2025_v24_frente_ancla": round(d, 5), "fuera": bool(abs(d) > UMBRAL)}
    CACHE.write_text(json.dumps(cache, indent=1, ensure_ascii=False))
    return cache


def aplicar(spec: dict | None = None):
    """Parchea mezcla3.sin_referencia (y robustez si está importado) con las corridas inestables de la caché."""
    if spec is not None:
        clasificar({line: [c[0] for c in comps] for line, comps in spec.items()})
    cache = json.loads(CACHE.read_text()) if CACHE.exists() else {}
    extra = {run for run, v in cache.items() if v["fuera"]}
    base = mezcla3.sin_referencia

    def patched(opts):
        s = base(opts)
        return (s | extra) if opts.get("nivel") == "s17" else s

    mezcla3.sin_referencia = patched
    try:
        import robustez
        robustez.sin_referencia = patched
    except ImportError:
        pass
    return extra
'''


### Embedded source: `src/linea_b/reproducir_s30_b.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/linea_b/reproducir_s30_b.py'] = '''"""Repite una mezcla de la línea B (S30, S31) a partir de los archivos guardados en Modelos_claude y Modelos_claude_b.

Igual que Modelos_claude/reproducir_mezcla.py: monta una carpeta de trabajo con enlaces a las corridas de las dos carpetas
(v24.parquet y prediccion_2025.parquet), a los sel_<nombre>.parquet y a las exposiciones de insumos/, aplica la regla de
nivel s17 extendida a las corridas B_ (nivel_b.py, que calcula la desviación 2025/V24 frente al ancla sin etiquetas) y
ejecuta mezcla3.py con la especificación y la forma indicadas; compara el SHA-256 del CSV con el entregado.
Uso: python3 reproducir_s30_b.py <spec_json> <forma> <nombre> <sha256_esperado> [<carpeta_temporal>]
Ejemplo: python3 reproducir_s30_b.py experimentos/spec_s30.json motor:1.09:0.95,contents:1:0.95 S30_claude_b <sha>
"""
from __future__ import annotations

import hashlib
import json
import os
import subprocess
import sys
import tempfile
from pathlib import Path

AQUI = Path(__file__).resolve().parent
MC = AQUI.parent / "Modelos_claude"
SEMANA1 = AQUI.parent


def sha(p: Path) -> str:
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()


def buscar(nombre: str, patron: str) -> Path:
    for base in (AQUI, MC):
        cands = list((base / "runs").glob(patron.format(nombre))) + list((base / "selecciones_v23").glob(patron.format(nombre)) if (base / "selecciones_v23").exists() else [])
        if cands:
            return cands[0]
    raise SystemExit(f"No encuentro {patron.format(nombre)} en runs/ ni selecciones_v23/ de las dos carpetas")


def main():
    spec_path, forma, nombre, esperado = sys.argv[1:5]
    tmp = Path(sys.argv[5]) if len(sys.argv) > 5 else Path(tempfile.mkdtemp(prefix="repro_b_"))
    work = tmp / "work"
    work.mkdir(parents=True, exist_ok=True)
    sp = Path(spec_path) if Path(spec_path).is_absolute() else AQUI / spec_path
    spec = json.loads(sp.read_text(encoding="utf-8"))
    corridas = {"S04b_lgbm_F0124_F3autos"}
    sels = {"total_F0124"}
    for line in spec:
        for run, sel_name, _ in spec[line]:
            corridas.add(run); sels.add(sel_name)
    for run in sorted(corridas):
        d = work / run
        d.mkdir(exist_ok=True)
        for f in ("v24.parquet", "prediccion_2025.parquet"):
            src = next((b / "runs" / run / f for b in (AQUI, MC) if (b / "runs" / run / f).exists()), None)
            if src is None:
                raise SystemExit(f"Falta runs/{run}/{f}")
            if not (d / f).exists():
                (d / f).symlink_to(src)
    for s in sorted(sels):
        dst = work / f"sel_{s}.parquet"
        if not dst.exists():
            dst.symlink_to(buscar(s, "*/sel_{}.parquet") if not (MC / "selecciones_v23" / f"sel_{s}.parquet").exists() else MC / "selecciones_v23" / f"sel_{s}.parquet")
    for line in ("motor", "buildings", "contents"):
        dst = work / f"feat_{line}.parquet"
        if not dst.exists():
            dst.symlink_to(MC / "insumos" / f"exposicion_{line}.parquet")
    env = dict(os.environ, MODELOS_WORK=str(work), HAKATON_SEMANA1=str(SEMANA1), MODELOS_CLAUDE=str(MC))
    # la regla de nivel: mezcla3 lee WORK/nivel_inestable_b.json a través de nivel_b; se clasifica aquí
    cmd = [sys.executable, "-W", "ignore", str(AQUI / "mezcla3_b.py"), nombre, str(sp), f"--forma={forma}", "--nivel=s17", "--escribir"]
    print("Ejecuto:", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True, env=env, cwd=str(AQUI))
    csv = work / nombre / f"{nombre}.csv"
    obtenido = sha(csv)
    rep = {"nombre": nombre, "sha256_esperado": esperado, "sha256_obtenido": obtenido, "coincide": obtenido == esperado, "carpeta": str(work)}
    if obtenido != esperado:
        import numpy as np
        import pandas as pd
        ref = SEMANA1 / "Envios" / "archivos" / f"{nombre}.csv"
        a = pd.read_csv(csv, dtype={"row_id": str}); r = pd.read_csv(ref, dtype={"row_id": str})
        assert (a.row_id.values == r.row_id.values).all()
        dif = np.abs(a.prediction.values - r.prediction.values)
        rep.update({"referencia": str(ref), "filas_distintas": int((dif > 0).sum()), "dif_max": float(dif.max()),
                    "total_obtenido": round(float(a.prediction.sum()), 2), "total_entregado": round(float(r.prediction.sum()), 2)})
    print(json.dumps(rep, ensure_ascii=False, indent=1), flush=True)
    if obtenido != esperado and rep["dif_max"] > 1.01e-6:
        raise SystemExit("El CSV repetido no coincide con el entregado")


if __name__ == "__main__":
    main()
'''


### Embedded source: `src/linea_b/severidad_tendencia.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/linea_b/severidad_tendencia.py'] = '''"""Severidad por peligro con nivel por año (tendencia) sobre las partes guardadas de frecuencia por peligro.

Las corridas CFS (CatBoost Poisson por peligro) y FSAP (frecuencia aditiva por peligro) guardan, por peligro, la frecuencia
esperada multiplicada por la severidad media de los años de entrenamiento del corte (v23_<p>.npz con lam, v24_<p>.npy y
prod_<p>.npy). severidad_cfs.py (línea principal de Claude) les aplica una relatividad GLM gamma por capital o por valor y
antigüedad del vehículo, normalizada a media 1, y conserva el nivel de la severidad media agrupada de 2021 al último año de
entrenamiento. Este script añade una pieza: el nivel de severidad de cada peligro en el año a predecir sale de la serie
anual de severidades de entrenamiento (media recortada por año para que un siniestro aislado, como el de 11,9 millones de
lesiones de 2023, no fije el nivel), con una de estas reglas:
  pooled     : media agrupada de los años de entrenamiento (lo que hacen CFSS y FSAS; sirve de control)
  ultimo     : la severidad del último año de entrenamiento
  tendencia  : regresión log-lineal de la severidad anual sobre el año (con al menos tres años; si hay dos, el último)
               extrapolada un año, con el crecimiento anual acotado a [-15 %, +15 %]
  mixto      : media geométrica de ultimo y tendencia
  ruptura    : tendencia con detección de cambio de régimen: si algún cambio interanual de la serie supera en valor
               absoluto el tope de crecimiento (15 %), la tendencia se ajusta solo sobre el tramo posterior a la última
               ruptura (con dos puntos, el último); si no hay ruptura, igual que tendencia. Pensada para lesiones de
               autos, donde la severidad cae un 20 % en 2023 y se mantiene en 2024: extrapolar la recta de cuatro años
               prolonga una caída que ya no existe.
La relatividad por fila (capital, sótano, valor y antigüedad) es la misma de severidad_cfs.py y se ajusta igual.
Escribe WORK/<salida>/ con sel_<salida>.parquet, v24.parquet, prediccion_2025.parquet y resumen.json (formato de las
demás corridas, utilizable por mezcla3.py), más el detalle por peligro y corte de los niveles usados.
Con --frec=<json>[:<familia>] multiplica además la parte de cada peligro y corte por el factor de deriva de frecuencia
fuera de muestra calculado por factores_frec_b.py (clave "factores" del JSON, o "factores_<familia>"): V23 <- r22,
V24 <- media geométrica de r22 y r23, 2025 <- media de r23 y r24, acotados a ±15 %.
Uso: python3 severidad_tendencia.py <ramo> <salida> --regla=<pooled|ultimo|tendencia|mixto> --v23=<corrida> --v24=<c1[,c2]> --prod=<c1[,c2]> [--frec=<json>[:<familia>]]
"""
from __future__ import annotations

import json
import sys

import numpy as np
import pandas as pd

import os as _os
from pathlib import Path as _Path
# código de la línea principal de Claude: carpeta hermana Modelos_claude en el repositorio, o la copia de la nube
_MC = _os.environ.get("MODELOS_CLAUDE") or next((str(p) for p in (_Path(__file__).resolve().parent.parent / "Modelos_claude", _Path("/home/claude/mc")) if p.exists()), "/home/claude/mc")
sys.path.insert(0, _MC)
from comun import F0DIR, PERILS, WORK, d2  # noqa: E402
from severidad_cfs import glm_gamma  # noqa: E402

TRIM_Q = 0.999
CAP_GROWTH = 0.15


def nivel_anual(df, p, years):
    """Severidad media por año del peligro, con los siniestros individuales acotados al cuantil TRIM_Q de todos los años
    de entrenamiento; devuelve la serie recortada y el cociente media completa / media recortada (agrupado)."""
    n = df[f"n_{p}"].fillna(0).values.astype(float)
    yp = df[f"y_{p}"].fillna(0).values
    yr = df.year.values
    m = np.isin(yr, years) & (n > 0)
    sev_fila = yp[m] / n[m]          # coste medio por siniestro en la fila (aproximación: filas con varios siniestros se promedian)
    cap = np.quantile(np.repeat(sev_fila, n[m].astype(int)), TRIM_Q)
    sev_trim = np.minimum(sev_fila, cap)
    serie = {}
    for y in years:
        k = yr[m] == y
        serie[int(y)] = float(np.average(sev_trim[k], weights=n[m][k])) if n[m][k].sum() > 0 else np.nan
    pooled_full = float(yp[m].sum() / n[m].sum())
    pooled_trim = float(np.average(sev_trim, weights=n[m]))
    return serie, pooled_full, pooled_trim


def nivel_objetivo(serie, regla):
    ys = sorted(serie)
    vals = np.array([serie[y] for y in ys], float)
    if regla == "pooled":
        return None
    if regla == "ruptura":
        saltos = np.abs(np.diff(np.log(vals)))
        corte = int(np.where(saltos > np.log(1 + CAP_GROWTH))[0].max()) + 1 if (saltos > np.log(1 + CAP_GROWTH)).any() else 0
        return nivel_objetivo({y: serie[y] for y in ys[corte:]}, "tendencia")
    ultimo = vals[-1]
    if len(ys) >= 3:
        b = np.polyfit(np.array(ys, float), np.log(vals), 1)[0]
        b = float(np.clip(b, np.log(1 - CAP_GROWTH), np.log(1 + CAP_GROWTH)))
        # tendencia anclada en el ajuste, no en el último punto: valor ajustado del último año por el crecimiento
        fit = np.polyval(np.polyfit(np.array(ys, float), np.log(vals), 1), ys[-1])
        tend = float(np.exp(fit + b))
    else:
        tend = ultimo
    if regla == "ultimo":
        return ultimo
    if regla == "tendencia":
        return tend
    if regla == "mixto":
        return float(np.sqrt(ultimo * tend))
    raise SystemExit(regla)


def main():
    line, name = sys.argv[1:3]
    opts = dict(a[2:].split("=", 1) for a in sys.argv[3:] if a.startswith("--"))
    regla = opts.get("regla", "tendencia")
    run_v23 = opts["v23"]; runs_v24 = opts["v24"].split(","); runs_prod = opts["prod"].split(",")
    frec = {}
    if "frec" in opts:
        ruta, _, familia = opts["frec"].partition(":")
        fj = json.load(open(ruta))
        frec = fj["factores_" + familia] if familia else fj["factores"]
    extra = ["sum_insured", "basement"] if line != "motor" else ["vehicle_value_group", "vehicle_age"]
    cols = ["row_id", "year", "T0"] + extra + [f"{k}_{p}" for p in PERILS[line] for k in ("n", "y")]
    df = pd.read_parquet(WORK / f"feat_{line}.parquet", columns=cols)
    part = pd.read_parquet(F0DIR / "tables" / "particiones.parquet", columns=["row_id", "cal_half"])
    df = df.merge(part, on="row_id", how="left", validate="one_to_one")
    df["row_id"] = df.row_id.astype(str)
    yr = df.year.values
    if line != "motor":
        si = df.sum_insured.astype(float).values
        si = np.where(np.isfinite(si) & (si > 0), si, np.nan)
        bas = df.basement.astype(float).fillna(0).values
    else:
        vvg = df.vehicle_value_group.astype(float).values
        vage = np.clip(df.vehicle_age.astype(float).fillna(df.vehicle_age.median()).values, 0, 30) / 10.0
    folds = {"V23": (yr <= 2022, (yr == 2023) & (df.cal_half.values == 1), [2021, 2022]),
             "V24": (yr <= 2023, yr == 2024, [2021, 2022, 2023]),
             "PROD": (yr <= 2024, yr == 2025, [2021, 2022, 2023, 2024])}
    out = WORK / name
    out.mkdir(parents=True, exist_ok=True)
    rep = {"line": line, "regla": regla, "v23": run_v23, "v24": runs_v24, "prod": runs_prod, "frec": opts.get("frec"), "perils": {}}
    parts = {}
    for fold, (trm, evm, years) in folds.items():
        if line != "motor":
            med = np.nanmedian(si[trm])
            zsi = np.log(np.where(np.isnan(si), med, si) / med)
        else:
            medv = np.nanmedian(vvg[trm])
            zv = np.log(np.where(np.isnan(vvg), medv, vvg) / medv)
        tot = np.zeros(evm.sum())
        for p in PERILS[line]:
            n = df[f"n_{p}"].fillna(0).values.astype(float); yp = df[f"y_{p}"].fillna(0).values
            pos = trm & (n > 0) & (yp > 0)
            if line != "motor":
                covs = [zsi] + ([bas] if p == "flood" else [])
            else:
                covs = [zv, vage] if p in ("own_damage", "theft", "glass_hail") else []
            if covs:
                Z = np.column_stack(covs)
                beta = glm_gamma(Z[pos], yp[pos] / n[pos], n[pos])
                rel = np.exp(Z @ beta[1:])
                rel = rel / np.average(rel[pos], weights=n[pos])
            else:
                beta = np.zeros(1); rel = np.ones(len(df))
            serie, pooled_full, pooled_trim = nivel_anual(df, p, years)
            objetivo = nivel_objetivo(serie, regla)
            # la parte guardada lleva la severidad agrupada completa de los años de entrenamiento; el factor lleva el nivel
            # al objetivo manteniendo el cociente completo/recortado del agrupado
            factor = 1.0 if objetivo is None else float(objetivo * (pooled_full / pooled_trim) / pooled_full)
            if fold == "V23":
                lam = np.load(WORK / run_v23 / f"v23_{p}.npz")["lam"]
                sev = yp[trm].sum() / max(n[trm].sum(), 1.0)
                base_part = lam * sev
            else:
                rr = runs_v24 if fold == "V24" else runs_prod
                base_part = sum(np.load(WORK / r / f"{'v24' if fold == 'V24' else 'prod'}_{p}.npy") for r in rr) / len(rr)
            assert len(base_part) == evm.sum(), (fold, p, len(base_part), evm.sum())
            ffrec = float(frec.get(fold, {}).get(p, 1.0))
            tot += base_part * rel[evm] * factor * ffrec
            rep["perils"].setdefault(p, {})[fold] = {"serie_recortada": {k: round(v, 1) for k, v in serie.items()}, "pooled": round(pooled_full, 1),
                                                     "objetivo_recortado": None if objetivo is None else round(objetivo, 1), "factor": round(factor, 4), "factor_frecuencia": round(ffrec, 4),
                                                     "betas": [round(float(b), 4) for b in beta[1:]], "siniestros": int(n[pos].sum())}
        parts[fold] = (evm, tot)
    t0 = df["T0"].fillna(0).values
    evm, mu = parts["V23"]
    rep["d2_sel"] = round(d2(t0[evm], mu), 5)
    pd.DataFrame({"row_id": df.row_id[evm].values, "line": line, "y": t0[evm], "mu": mu}).to_parquet(WORK / f"sel_{name}.parquet", index=False)
    evm, mu = parts["V24"]
    rep["d2_v24"] = round(d2(t0[evm], mu), 5)
    pd.DataFrame({"row_id": df.row_id[evm].values, "line": line, "y": t0[evm], "mu": mu}).to_parquet(out / "v24.parquet", index=False)
    evm, mu = parts["PROD"]
    rep["pred_2025_total_CU"] = round(float(mu.sum()), 2)
    pd.DataFrame({"row_id": df.row_id[evm].values, "line": line, "prediction": mu}).to_parquet(out / "prediccion_2025.parquet", index=False)
    (out / "resumen.json").write_text(json.dumps(rep, indent=1, ensure_ascii=False))
    print(json.dumps({k: v for k, v in rep.items() if k != "perils"}, ensure_ascii=False), flush=True)
    for p, v in rep["perils"].items():
        print("  ", p, {f: (w["serie_recortada"], w["objetivo_recortado"], w["factor"], w["factor_frecuencia"]) for f, w in v.items()}, flush=True)


if __name__ == "__main__":
    main()
'''


### Embedded source: `src/datos/bases_actuariales.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/datos/bases_actuariales.py'] = '''"""Bases actuariales para el modelado: expuestos, asignación de siniestros, retraso de reporte e IBNR, frecuencia,
severidad y referencias de tasa constante (M00), calculadas sobre la capa preparada vigente.

Lee current_dataset.json de Limpieza, comprueba los SHA-256 de las siete tablas que usa contra el catálogo activo y
no modifica ningún dato ni índice. Crea runs/bases_<marca>_<id>/ con evidencia.json, tablas CSV y Parquet, copia del
script y manifest.json con hashes. Cada ejecución crea una carpeta nueva; current_run.json se fija aparte tras
verificar. No entrena modelos: las referencias M00 son tasas constantes por ramo evaluadas sobre el corte siguiente.

Uso (Windows): & '.\\\\Preparación\\\\.venv\\\\Scripts\\\\python.exe' '.\\\\Hackaton\\\\Semana 1\\\\Bases_actuariales_claude\\\\bases_actuariales.py'
Uso (Linux):   python3 'Hackaton/Semana 1/Bases_actuariales_claude/bases_actuariales.py'
"""
from __future__ import annotations

import hashlib
import json
import shutil
import sys
import uuid
from datetime import datetime, timezone
from pathlib import Path

HERE = Path(__file__).resolve().parent


def find_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "RUTAS_PROYECTO.json").is_file():
            return candidate
    raise FileNotFoundError("No se encontró RUTAS_PROYECTO.json hacia arriba.")


ROOT = find_root(HERE)
try:  # el lector aislado de Parquet del proyecto, si pyarrow no está en el entorno activo
    import pyarrow  # noqa: F401
except ImportError:  # pragma: no cover
    sys.path.insert(0, str(ROOT / "Preparación" / ".cache" / "parquet-reader"))

import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402

LINES = ["motor", "buildings", "contents"]
LINE_ES = {"motor": "autos", "buildings": "edificios", "contents": "contenidos"}
CUTOFF = pd.Timestamp("2024-12-30")  # última fecha de reporte observada; se comprueba contra los datos
POWER = 1.5
LONG_TAIL = {("motor", "bodily_injury"), ("buildings", "subsidence")}
MONTHS = pd.period_range("2021-01", "2025-12", freq="M")


def digest(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


def tweedie_deviance(y: np.ndarray, mu: np.ndarray, p: float = POWER) -> np.ndarray:
    """Desviación unitaria de Tweedie. Para p=1,5 equivale a 4*(sqrt(mu)-sqrt(y))**2/sqrt(mu)."""
    y = np.asarray(y, dtype=float)
    mu = np.asarray(mu, dtype=float)
    return 2.0 * (np.power(y, 2 - p) / ((1 - p) * (2 - p)) - y * np.power(mu, 1 - p) / (1 - p) + np.power(mu, 2 - p) / (2 - p))


def d2_tweedie(y: np.ndarray, mu: np.ndarray, p: float = POWER) -> float:
    """D² como scikit-learn: 1 - dev(y, mu) / dev(y, media de y)."""
    y = np.asarray(y, dtype=float)
    mu = np.asarray(mu, dtype=float)
    null = np.full_like(y, y.mean())
    return float(1.0 - tweedie_deviance(y, mu, p).sum() / tweedie_deviance(y, null, p).sum())


def r(x, nd=4):
    if x is None:
        return None
    if isinstance(x, (np.floating, float)):
        if not np.isfinite(x):
            return None
        return round(float(x), nd)
    if isinstance(x, (np.integer,)):
        return int(x)
    return x


def main() -> None:
    started = datetime.now(timezone.utc)
    run_id = f"bases_{started.strftime('%Y%m%dT%H%M%SZ')}_{uuid.uuid4().hex[:8]}"
    run_dir = HERE / "runs" / run_id
    tables_dir = run_dir / "tables"
    tables_dir.mkdir(parents=True, exist_ok=False)
    checks: list[dict] = []

    def check(code: str, name: str, ok: bool, detail=None):
        checks.append({"code": code, "name": name, "passed": bool(ok), "detail": detail})

    # ------------------------------------------------------------------ 1. capa vigente y hashes
    index_path = ROOT / "Hackaton/Semana 1/Limpieza/current_dataset.json"
    current = json.loads(index_path.read_text(encoding="utf-8"))
    catalog_path = ROOT / current["catalog_workspace_relative"]
    catalog = json.loads(catalog_path.read_text(encoding="utf-8"))
    verification = json.loads((catalog_path.parent / "verification.json").read_text(encoding="utf-8"))
    check("K01", "la corrida vigente de Limpieza tiene verificación con cero fallos", verification.get("failed") == 0 and verification.get("passed", 0) > 0,
          {"status": verification.get("status"), "passed": verification.get("passed"), "failed": verification.get("failed")})
    used = ["claims", "policies_motor", "policies_buildings", "policies_contents", "test_motor", "test_buildings", "test_contents"]
    hashes = {}
    bad = []
    for name in used:
        path = ROOT / catalog[name]["workspace_relative_path"]
        hashes[name] = digest(path)
        if hashes[name] != catalog[name]["sha256"]:
            bad.append(name)
    check("K02", "los SHA-256 de las siete tablas usadas coinciden con el catálogo activo", not bad, bad)

    def read(name: str, columns: list[str]) -> pd.DataFrame:
        path = ROOT / catalog[name]["workspace_relative_path"]
        df = pd.read_parquet(path, columns=columns)
        for col in df.columns:  # cadenas compactas: la VM de cálculo tiene poca memoria
            if df[col].dtype == object:
                df[col] = df[col].astype("string[pyarrow]")
        return df

    base_cols = ["row_id", "policy_id", "customer_id", "period_start", "period_end", "exposure", "new_business", "written_premium"]
    frames = []
    for line in LINES:
        extra = ["cover_type"] if line == "motor" else []
        for kind, table in (("train", f"policies_{line}"), ("test", f"test_{line}")):
            df = read(table, base_cols + extra)
            df["line"] = line
            df["is_test"] = kind == "test"
            if line != "motor":
                df["cover_type"] = "n/a"
            frames.append(df)
    periods = pd.concat(frames, ignore_index=True)
    del frames
    periods["period_start"] = pd.to_datetime(periods["period_start"])
    periods["period_end"] = pd.to_datetime(periods["period_end"])
    periods["year"] = periods["period_start"].dt.year.astype(int)
    days = ((periods["period_end"] - periods["period_start"]).dt.days + 1).values
    max_abs_diff = float(np.abs(periods["exposure"].values - days / 365.0).max())
    check("K03", "exposición EXP01 = días inclusivos / 365 en todas las filas (tolerancia 1e-4)", max_abs_diff < 1e-4, {"max_abs_diff": r(max_abs_diff, 6)})
    del days
    check("K04", "626.639 filas de test con row_id único", int(periods["is_test"].sum()) == 626639 and periods.loc[periods.is_test, "row_id"].is_unique,
          {"test_rows": int(periods["is_test"].sum())})
    check("K05", "los periodos de test son todos de 2025 y los de entrenamiento de 2021-2024",
          set(periods.loc[periods.is_test, "year"].unique()) == {2025} and set(periods.loc[~periods.is_test, "year"].unique()) == {2021, 2022, 2023, 2024})

    # ------------------------------------------------------------------ 2. expuestos
    def exposure_block(df: pd.DataFrame, keys: list[str]) -> pd.DataFrame:
        g = df.groupby(keys)
        out = g.agg(filas=("row_id", "size"), expuestos_anios=("exposure", "sum"), exposicion_media=("exposure", "mean"),
                    polizas=("policy_id", "nunique"), clientes=("customer_id", "nunique"),
                    prima_escrita_CU=("written_premium", "sum")).reset_index()
        out["pct_filas_anio_completo"] = g["exposure"].apply(lambda s: float((s >= 0.99).mean())).values * 100
        out["pct_filas_menos_medio_anio"] = g["exposure"].apply(lambda s: float((s < 0.5).mean())).values * 100
        out["pct_filas_nuevo_negocio"] = g["new_business"].apply(lambda s: float(s.mean())).values * 100
        out["prima_escrita_por_anio_expuesto_CU"] = out["prima_escrita_CU"] / out["expuestos_anios"]
        return out

    exp_line_year = exposure_block(periods, ["line", "year"])
    exp_line_year.to_csv(tables_dir / "expuestos_ramo_anio.csv", index=False)
    exp_cover = exposure_block(periods[periods.line == "motor"], ["year", "cover_type"])
    exp_cover.to_csv(tables_dir / "expuestos_autos_cobertura.csv", index=False)
    train_exposure_total = float(periods.loc[~periods.is_test, "exposure"].sum())
    test_exposure_total = float(periods.loc[periods.is_test, "exposure"].sum())
    check("K12", "exposición total del entrenamiento igual a la de EXP01 (1.968.577,0591 años, tolerancia 0,01)", abs(train_exposure_total - 1968577.0591) < 0.01, r(train_exposure_total, 4))

    # exposición por mes natural de ocurrencia: días cubiertos en cada mes / 365
    ps = periods["period_start"].values.astype("datetime64[D]")
    pe = periods["period_end"].values.astype("datetime64[D]")
    line_codes = periods["line"].map({l: i for i, l in enumerate(LINES)}).values.astype(int)
    month_rows = []
    month_exposure = np.zeros((len(LINES), len(MONTHS)))
    for j, m in enumerate(MONTHS):
        ms = np.datetime64(m.start_time.date(), "D")
        me = np.datetime64(m.end_time.date(), "D")
        overlap = (np.minimum(pe, me) - np.maximum(ps, ms)).astype("timedelta64[D]").astype(int) + 1
        overlap = np.clip(overlap, 0, None) / 365.0
        for i in range(len(LINES)):
            month_exposure[i, j] = overlap[line_codes == i].sum()
            month_rows.append({"line": LINES[i], "mes": str(m), "expuestos_anios": month_exposure[i, j]})
    del ps, pe, line_codes, overlap
    month_df = pd.DataFrame(month_rows)
    month_df.to_csv(tables_dir / "expuestos_ramo_mes.csv", index=False)
    year_from_months = month_df.assign(anio=month_df.mes.str[:4].astype(int)).groupby(["line", "anio"])["expuestos_anios"].sum()
    ely = exp_line_year.set_index(["line", "year"])["expuestos_anios"]
    diff_month = max(abs(year_from_months[(l, y)] - ely[(l, y)]) for l in LINES for y in range(2021, 2026))
    check("K13", "la exposición por meses naturales suma la exposición por fila en cada ramo y año (tolerancia 0,5 años)", diff_month < 0.5, r(float(diff_month), 4))

    train = periods.loc[~periods.is_test, ["row_id", "policy_id", "line", "period_start", "period_end", "year", "exposure", "cover_type"]].reset_index(drop=True)
    del periods

    claims = read("claims", ["claim_id", "policy_id", "line", "peril", "loss_date", "report_date", "status", "paid_to_date", "incurred_to_date", "recoveries", "audit_pre00"])
    claims["loss_date"] = pd.to_datetime(claims["loss_date"])
    claims["report_date"] = pd.to_datetime(claims["report_date"])
    claims["accepted"] = claims["status"].isin(["closed", "open"])
    claims["net"] = claims["incurred_to_date"] - claims["recoveries"]
    claims["reserve"] = claims["incurred_to_date"] - claims["paid_to_date"]
    claims["delay_days"] = (claims["report_date"] - claims["loss_date"]).dt.days
    observed_cutoff = claims["report_date"].max()
    check("K06", "la última fecha de reporte observada es el 30-12-2024 (corte de valoración)", observed_cutoff == CUTOFF, str(observed_cutoff.date()))
    check("K07", "claims vigente: 162.546 registros, 413 marcados audit_pre00, ningún reporte anterior a la ocurrencia",
          len(claims) == 162546 and int(claims["audit_pre00"].sum()) == 413 and int((claims["delay_days"] < 0).sum()) == 0,
          {"rows": len(claims), "pre00": int(claims["audit_pre00"].sum()), "negative_delays": int((claims["delay_days"] < 0).sum())})

    # ------------------------------------------------------------------ 3. asignación de siniestros a periodos
    cand = claims.loc[~claims.audit_pre00, ["claim_id", "policy_id", "line", "loss_date"]].merge(train[["row_id", "policy_id", "line", "period_start", "period_end"]], on=["policy_id", "line"], how="left")
    cand["inside"] = (cand["loss_date"] >= cand["period_start"]) & (cand["loss_date"] <= cand["period_end"])
    has_policy = cand.groupby("claim_id")["row_id"].count() > 0
    n_inside = cand[cand.inside].groupby("claim_id")["row_id"].nunique()
    assign = pd.DataFrame({"claim_id": claims.loc[~claims.audit_pre00, "claim_id"].values})
    assign["has_policy"] = assign["claim_id"].map(has_policy).fillna(False).astype(bool)
    assign["n_periods"] = assign["claim_id"].map(n_inside).fillna(0).astype(int)
    assign["category"] = np.select([~assign.has_policy, assign.n_periods == 1, assign.n_periods == 0, assign.n_periods > 1],
                                   ["sin_correspondencia", "unica", "fuera_de_ventana", "ambigua"], default="?")
    unique_map = cand[cand.inside & cand.claim_id.isin(assign.loc[assign.category == "unica", "claim_id"])][["claim_id", "row_id"]]
    pre = pd.DataFrame({"claim_id": claims.loc[claims.audit_pre00, "claim_id"].values, "has_policy": True, "n_periods": 0, "category": "prehistoria_PRE00"})
    assign = pd.concat([assign, pre], ignore_index=True).merge(unique_map, on="claim_id", how="left")
    claims = claims.merge(assign[["claim_id", "category", "row_id"]], on="claim_id", how="left")
    check("K08", "PRE00: ningún siniestro audit_pre00 queda asignado a un periodo", claims.loc[claims.audit_pre00, "row_id"].isna().all())
    cat_counts = claims.groupby(["line", "category"]).size().unstack(fill_value=0)
    acc_counts = claims[claims.accepted].groupby(["line", "category"]).size().unstack(fill_value=0)
    check("K09", "sin siniestros ambiguos ni sin póliza", int(cat_counts.get("ambigua", pd.Series(0, index=cat_counts.index)).sum()) == 0 and int(cat_counts.get("sin_correspondencia", pd.Series(0, index=cat_counts.index)).sum()) == 0)
    expected_unique_accepted = {"motor": 110496, "buildings": 26266, "contents": 23670}
    got_unique_accepted = {line: int(acc_counts.loc[line].get("unica", 0)) for line in LINES}
    check("K10", "asignación única de aceptados igual a D42/D50 (110.496, 26.266, 23.670)", got_unique_accepted == expected_unique_accepted, got_unique_accepted)
    fuera = claims[(claims.category == "fuera_de_ventana")]
    fuera_desc = {
        "total": int(len(fuera)), "aceptados": int(fuera.accepted.sum()),
        "net_aceptado_CU": r(float(fuera.loc[fuera.accepted, "net"].sum()), 2),
        "loss_31_12_2024": int((fuera.loss_date == pd.Timestamp("2024-12-31")).sum()),
        "loss_dec_2024": int(((fuera.loss_date >= pd.Timestamp("2024-12-01")) & (fuera.loss_date <= pd.Timestamp("2024-12-31"))).sum()),
        "loss_after_last_period_of_policy": None,
    }
    last_end = train.groupby("policy_id")["period_end"].max()
    first_start = train.groupby("policy_id")["period_start"].min()
    fuera_after = fuera.loss_date > fuera.policy_id.map(last_end)
    fuera_before = fuera.loss_date < fuera.policy_id.map(first_start)
    fuera_desc["loss_after_last_period_of_policy"] = int(fuera_after.sum())
    fuera_desc["loss_before_first_period_of_policy"] = int(fuera_before.sum())
    fuera_desc["loss_in_gap_between_periods"] = int((~fuera_after & ~fuera_before).sum())
    assignment_table = claims.groupby(["line", "category", "status"]).agg(n=("claim_id", "size"), net_CU=("net", "sum")).reset_index()
    assignment_table.to_csv(tables_dir / "asignacion_siniestros.csv", index=False)
    claims[["claim_id", "policy_id", "line", "peril", "loss_date", "report_date", "status", "category", "row_id"]].to_parquet(tables_dir / "asignacion_por_siniestro.parquet", index=False)

    # ------------------------------------------------------------------ 4. T0 por row_id
    acc = claims[claims.accepted & (claims.category == "unica")].copy()
    acc["positive"] = acc["net"] > 0
    acc["is_open"] = acc["status"] == "open"
    t0 = acc.groupby("row_id").agg(T0=("net", "sum"), n_accepted=("claim_id", "size"), n_positive=("positive", "sum"), n_open=("is_open", "sum"),
                                  paid=("paid_to_date", "sum"), incurred=("incurred_to_date", "sum"), recoveries=("recoveries", "sum"), reserve=("reserve", "sum"))
    t0_rows = train[["row_id", "line", "year", "exposure", "cover_type"]].merge(t0, on="row_id", how="left")
    for col in ["T0", "n_accepted", "n_positive", "n_open", "paid", "incurred", "recoveries", "reserve"]:
        t0_rows[col] = t0_rows[col].fillna(0)
    for col in ["n_accepted", "n_positive", "n_open"]:
        t0_rows[col] = t0_rows[col].astype(int)
    check("K11", "la suma de T0 por fila iguala el neto de los aceptados asignados y no hay netos negativos",
          abs(t0_rows["T0"].sum() - acc["net"].sum()) < 1e-6 and int((acc["net"] < 0).sum()) == 0,
          {"T0_total_CU": r(float(t0_rows["T0"].sum()), 2), "net_zero_accepted": int((acc["net"] == 0).sum())})
    t0_rows[["row_id", "line", "year", "exposure", "T0", "n_accepted", "n_positive", "n_open", "paid", "incurred", "recoveries", "reserve"]].to_parquet(tables_dir / "t0_por_row_id.parquet", index=False)

    # ------------------------------------------------------------------ 5. retraso de reporte y chain-ladder de conteos
    rep = claims[(claims.category == "unica")].copy()  # todas las notificaciones asignadas, aceptadas o rechazadas
    rep["occ_m"] = (rep.loss_date.dt.year - 2021) * 12 + rep.loss_date.dt.month - 1
    rep["rep_m"] = (rep.report_date.dt.year - 2021) * 12 + rep.report_date.dt.month - 1
    rep["lag"] = rep["rep_m"] - rep["occ_m"]
    N = 48  # cohortes mensuales 2021-01 .. 2024-12
    delay_rows, cl_rows, tail_rows, month_complete_rows, factor_rows = [], [], [], [], []
    completeness = {}  # (line, peril) -> array de 12 completitudes 2024 por mes de ocurrencia
    cdf_by_group = {}
    for (line, peril), g in rep.groupby(["line", "peril"]):
        # triángulo acumulado de conteos por cohorte mensual y lag
        tri = np.zeros((N, N))
        for occ, lag in zip(g.occ_m.values, g.lag.values):
            if 0 <= occ < N and 0 <= lag < N:
                tri[occ, lag:] += 1  # acumulado
        factors = []
        for k in range(N - 1):
            rows = [i for i in range(N) if i + k + 1 <= N - 1]  # cohortes con lag k+1 observado
            num = sum(tri[i, k + 1] for i in rows)
            den = sum(tri[i, k] for i in rows)
            factors.append(num / den if den > 0 else 1.0)
        factors = np.array(factors)
        # cola: ajuste exponencial sobre los últimos factores > 1 (lags 12..46) para los dos peligros largos
        tail = 1.0
        tail_fit = None
        if (line, peril) in LONG_TAIL:
            ks = np.arange(12, N - 1)
            fk = factors[12:N - 1]
            mask = fk > 1.0
            if mask.sum() >= 6:
                a, b = np.polyfit(ks[mask], np.log(fk[mask] - 1.0), 1)
                if a < 0:
                    future = np.arange(N - 1, N - 1 + 120)
                    tail = float(np.prod(1.0 + np.exp(a * future + b)))
                    tail_fit = {"pendiente": r(float(a), 5), "intercepto": r(float(b), 4), "puntos": int(mask.sum()), "cola_120_meses": r(tail, 4)}
                else:
                    tail_fit = {"pendiente": r(float(a), 5), "nota": "sin decaimiento; cola fijada en 1"}
        cdf = np.ones(N)  # edad k -> factor a último
        for k in range(N - 2, -1, -1):
            cdf[k] = cdf[k + 1] * factors[k]
        cdf = cdf * tail
        cdf_by_group[(line, peril)] = cdf
        reported_2024 = float(tri[36:48, :].max(axis=1).sum())
        ult_2024 = float(sum(tri[i, N - 1 - i] * cdf[N - 1 - i] for i in range(36, 48)))
        rep_by_year = {y: float(sum(tri[i, N - 1 - i] for i in range((y - 2021) * 12, (y - 2021) * 12 + 12))) for y in range(2021, 2025)}
        ult_by_year = {y: float(sum(tri[i, N - 1 - i] * cdf[N - 1 - i] for i in range((y - 2021) * 12, (y - 2021) * 12 + 12))) for y in range(2021, 2025)}
        comp_months = np.array([1.0 / cdf[N - 1 - i] for i in range(36, 48)])
        completeness[(line, peril)] = comp_months
        for i in range(36, 48):
            month_complete_rows.append({"line": line, "peril": peril, "mes_ocurrencia": str(MONTHS[i]), "edad_meses": N - 1 - i,
                                        "reportados": int(tri[i, N - 1 - i]), "factor_a_ultimo": r(float(cdf[N - 1 - i]), 4), "completitud": r(float(1.0 / cdf[N - 1 - i]), 4)})
        for k in range(N - 1):
            factor_rows.append({"line": line, "peril": peril, "lag_meses": k, "factor": r(float(factors[k]), 5), "factor_a_ultimo": r(float(cdf[k]), 5)})
        # retraso empírico en días por cohorte de ocurrencia (aceptados y rechazados)
        for y in range(2021, 2025):
            gy = g[g.loss_date.dt.year == y]
            d = gy.delay_days
            delay_rows.append({"line": line, "peril": peril, "cohorte": y, "n": int(len(d)), "p50": r(float(d.quantile(0.5)), 1) if len(d) else None,
                               "p90": r(float(d.quantile(0.9)), 1) if len(d) else None, "p99": r(float(d.quantile(0.99)), 1) if len(d) else None,
                               "pct_30d": r(float((d <= 30).mean() * 100), 2) if len(d) else None, "pct_90d": r(float((d <= 90).mean() * 100), 2) if len(d) else None,
                               "pct_365d": r(float((d <= 365).mean() * 100), 2) if len(d) else None, "max": int(d.max()) if len(d) else None})
        cl_rows.append({"line": line, "peril": peril, "notificaciones_2021_2024": int(len(g)), "cola": r(tail, 4), "cola_ajuste": json.dumps(tail_fit, ensure_ascii=False) if tail_fit else "",
                        **{f"reportados_{y}": int(rep_by_year[y]) for y in range(2021, 2025)},
                        **{f"ultimo_{y}": r(ult_by_year[y], 1) for y in range(2021, 2025)},
                        **{f"completitud_{y}": r(rep_by_year[y] / ult_by_year[y], 4) if ult_by_year[y] else None for y in range(2021, 2025)},
                        "ibnr_conteo_2024": r(ult_2024 - reported_2024, 1), "factor_1_mes": r(float(factors[0]), 4), "factor_a_ultimo_edad_0": r(float(cdf[0]), 4), "factor_a_ultimo_edad_11": r(float(cdf[11]), 4)})
    pd.DataFrame(delay_rows).to_csv(tables_dir / "retraso_reporte_cohortes.csv", index=False)
    cl = pd.DataFrame(cl_rows)
    cl.to_csv(tables_dir / "chain_ladder_conteos.csv", index=False)
    pd.DataFrame(month_complete_rows).to_csv(tables_dir / "completitud_2024_por_mes.csv", index=False)
    pd.DataFrame(factor_rows).to_csv(tables_dir / "factores_desarrollo_conteos.csv", index=False)
    short = cl[~cl.apply(lambda row: (row.line, row.peril) in LONG_TAIL, axis=1)]
    check("K14", "completitud de reporte 2021 >= 0,995 en todos los peligros de cola corta", bool((short["completitud_2021"] >= 0.995).all()), r(float(short["completitud_2021"].min()), 4))
    check("K15", "completitud 2024 < completitud 2021 en lesiones corporales y hundimiento",
          all(cl.loc[(cl.line == l) & (cl.peril == p), "completitud_2024"].iloc[0] < cl.loc[(cl.line == l) & (cl.peril == p), "completitud_2021"].iloc[0] for l, p in LONG_TAIL))

    # ------------------------------------------------------------------ 6. severidad
    sev_rows = []
    pos = acc[acc.positive]
    for (line, peril), g in pos.groupby(["line", "peril"]):
        for y in [2021, 2022, 2023, 2024, "2021-2022", "2021-2024"]:
            if isinstance(y, int):
                gy = g[g.loss_date.dt.year == y]
            elif y == "2021-2022":
                gy = g[g.loss_date.dt.year <= 2022]
            else:
                gy = g
            s = gy["net"]
            if len(s) == 0:
                continue
            closed = gy[gy.status == "closed"]["net"]
            opened = gy[gy.status == "open"]["net"]
            late = gy[gy.delay_days > 365]["net"]
            sev_rows.append({"line": line, "peril": peril, "ocurrencia": y, "n_positivos": int(len(s)), "media": r(float(s.mean()), 2), "mediana": r(float(s.median()), 2),
                             "cv": r(float(s.std(ddof=1) / s.mean()), 3) if len(s) > 1 else None, "p90": r(float(s.quantile(0.9)), 2), "p99": r(float(s.quantile(0.99)), 2), "max": r(float(s.max()), 2),
                             "pct_abiertos": r(float((gy.status == "open").mean() * 100), 2), "media_cerrados": r(float(closed.mean()), 2) if len(closed) else None,
                             "media_abiertos": r(float(opened.mean()), 2) if len(opened) else None, "reserva_sobre_incurrido_pct": r(float(gy.reserve.sum() / gy.incurred_to_date.sum() * 100), 2),
                             "n_reportados_mas_365d": int(len(late)), "media_reportados_mas_365d": r(float(late.mean()), 2) if len(late) else None,
                             "coste_1pct_mayor_pct": r(float(s.nlargest(max(1, int(np.ceil(len(s) * 0.01)))).sum() / s.sum() * 100), 2)})
    sev = pd.DataFrame(sev_rows)
    sev.to_csv(tables_dir / "severidad_ramo_peligro.csv", index=False)
    share_positive = (acc.groupby(["line", "peril"])["positive"].mean()).to_dict()
    notif_positive = (rep.merge(acc[["claim_id", "positive"]], on="claim_id", how="left").assign(positive=lambda d: d.positive.fillna(False)).groupby(["line", "peril"])["positive"].mean()).to_dict()

    # ------------------------------------------------------------------ 7. frecuencia
    freq_rows = []
    exp_ly = exp_line_year.set_index(["line", "year"])["expuestos_anios"]
    for line in LINES:
        for y in range(2021, 2025):
            e = float(exp_ly[(line, y)])
            rows = t0_rows[(t0_rows.line == line) & (t0_rows.year == y)]
            n_pos = int(rows.n_positive.sum())
            n_acc = int(rows.n_accepted.sum())
            lam = rows.n_positive.values
            disp = float(((lam - lam.mean()) ** 2).sum() / (len(lam) - 1) / lam.mean()) if lam.mean() > 0 else None
            freq_rows.append({"line": line, "year": y, "peril": "todos", "expuestos_anios": r(e, 2), "aceptados": n_acc, "positivos": n_pos,
                              "frecuencia_aceptados": r(n_acc / e, 5), "frecuencia_positivos": r(n_pos / e, 5), "dispersion_varianza_media": r(disp, 3),
                              "pct_periodos_sin_coste": r(float((rows.T0 == 0).mean() * 100), 2)})
            for peril, g in acc[(acc.line == line) & (acc.loss_date.dt.year == y)].groupby("peril"):
                freq_rows.append({"line": line, "year": y, "peril": peril, "expuestos_anios": r(e, 2), "aceptados": int(len(g)), "positivos": int(g.positive.sum()),
                                  "frecuencia_aceptados": r(len(g) / e, 5), "frecuencia_positivos": r(float(g.positive.sum()) / e, 5), "dispersion_varianza_media": None, "pct_periodos_sin_coste": None})
    freq = pd.DataFrame(freq_rows)
    freq.to_csv(tables_dir / "frecuencia_ramo_anio_peligro.csv", index=False)
    freq_cover = []
    for (y, cov), rows in t0_rows[t0_rows.line == "motor"].groupby(["year", "cover_type"]):
        e = float(rows.exposure.sum())
        freq_cover.append({"year": y, "cover_type": cov, "filas": int(len(rows)), "expuestos_anios": r(e, 2), "frecuencia_positivos": r(rows.n_positive.sum() / e, 5),
                           "prima_riesgo_T0_CU": r(float(rows.T0.sum() / e), 2)})
    pd.DataFrame(freq_cover).to_csv(tables_dir / "frecuencia_autos_cobertura.csv", index=False)

    # frecuencia última 2024 (IBNR de conteos aplicado a positivos) y exposición efectiva
    ult_rows = []
    for (line, peril), comp in completeness.items():
        g24 = acc[(acc.line == line) & (acc.peril == peril) & (acc.loss_date.dt.year == 2024)]
        n_pos = int(g24.positive.sum())
        n_all = int(len(rep[(rep.line == line) & (rep.peril == peril) & (rep.loss_date.dt.year == 2024)]))
        e24 = month_exposure[LINES.index(line), 36:48]
        e_eff = float((e24 * comp).sum())
        e_nom = float(e24.sum())
        ibnr_notif = float(cl.loc[(cl.line == line) & (cl.peril == peril), "ibnr_conteo_2024"].iloc[0])
        p_pos = float(notif_positive.get((line, peril), 0.0))
        ibnr_pos = ibnr_notif * p_pos
        sev_mature = sev[(sev.line == line) & (sev.peril == peril) & (sev.ocurrencia == "2021-2022")]
        mean_sev = float(sev_mature["media"].iloc[0]) if len(sev_mature) else float("nan")
        late_sev = sev_mature["media_reportados_mas_365d"].iloc[0] if len(sev_mature) else None
        sev_for_ibnr = float(late_sev) if ((line, peril) in LONG_TAIL and late_sev is not None and not pd.isna(late_sev)) else mean_sev
        t0_24 = float(g24["net"].sum())
        ult_rows.append({"line": line, "peril": peril, "notificaciones_2024": n_all, "positivos_2024": n_pos, "expuestos_2024": r(e_nom, 2), "expuestos_efectivos_2024": r(e_eff, 2),
                         "completitud_ponderada_2024": r(e_eff / e_nom, 4), "frecuencia_positivos_reportada": r(n_pos / e_nom, 5), "frecuencia_positivos_ultima_por_exposicion_efectiva": r(n_pos / e_eff, 5),
                         "ibnr_notificaciones": r(ibnr_notif, 1), "prob_positivo_por_notificacion": r(p_pos, 4), "ibnr_positivos": r(ibnr_pos, 1),
                         "severidad_media_2021_2022": r(mean_sev, 2), "severidad_usada_ibnr": r(sev_for_ibnr, 2), "ibnr_coste_CU": r(ibnr_pos * sev_for_ibnr, 2),
                         "T0_2024_CU": r(t0_24, 2), "factor_ibnr_coste": r((t0_24 + ibnr_pos * sev_for_ibnr) / t0_24, 4) if t0_24 > 0 else None})
    ult = pd.DataFrame(ult_rows)
    ult.to_csv(tables_dir / "ibnr_2024_por_peligro.csv", index=False)
    ult_line = ult.groupby("line").agg(T0_2024_CU=("T0_2024_CU", "sum"), ibnr_coste_CU=("ibnr_coste_CU", "sum"), ibnr_positivos=("ibnr_positivos", "sum"), positivos_2024=("positivos_2024", "sum")).reset_index()
    ult_line["factor_ibnr_coste"] = (ult_line.T0_2024_CU + ult_line.ibnr_coste_CU) / ult_line.T0_2024_CU
    ult_line["factor_ibnr_conteo"] = (ult_line.positivos_2024 + ult_line.ibnr_positivos) / ult_line.positivos_2024
    ult_line.to_csv(tables_dir / "ibnr_2024_por_ramo.csv", index=False)

    # IBNR por peligro y año de ocurrencia 2021-2024 con la misma regla, y prima de riesgo última por ramo y año
    sev_used = {(row.line, row.peril): row.severidad_usada_ibnr for row in ult.itertuples()}
    ibnr_year_rows = []
    for row in cl.itertuples():
        for y in range(2021, 2025):
            reported = getattr(row, f"reportados_{y}")
            ultimate = getattr(row, f"ultimo_{y}")
            ibnr_notif = float(ultimate) - float(reported)
            p_pos = float(notif_positive.get((row.line, row.peril), 0.0))
            gy = acc[(acc.line == row.line) & (acc.peril == row.peril) & (acc.loss_date.dt.year == y)]
            t0_y = float(gy.net.sum())
            cost = ibnr_notif * p_pos * float(sev_used[(row.line, row.peril)])
            ibnr_year_rows.append({"line": row.line, "peril": row.peril, "year": y, "notificaciones": int(reported), "ultimo": r(float(ultimate), 1), "ibnr_notificaciones": r(ibnr_notif, 1),
                                   "ibnr_positivos": r(ibnr_notif * p_pos, 1), "ibnr_coste_CU": r(cost, 2), "T0_CU": r(t0_y, 2), "factor_ibnr_coste": r((t0_y + cost) / t0_y, 4) if t0_y > 0 else None})
    ibnr_year = pd.DataFrame(ibnr_year_rows)
    ibnr_year.to_csv(tables_dir / "ibnr_por_peligro_anio.csv", index=False)
    prima_ult_rows = []
    for line in LINES:
        for y in range(2021, 2025):
            e = float(exp_ly[(line, y)])
            sub = ibnr_year[(ibnr_year.line == line) & (ibnr_year.year == y)]
            t0_y = float(sub.T0_CU.sum())
            cost = float(sub.ibnr_coste_CU.sum())
            res_y = float(acc[(acc.line == line) & (acc.loss_date.dt.year == y) & acc.is_open].reserve.sum())
            prima_ult_rows.append({"line": line, "year": y, "expuestos_anios": r(e, 2), "T0_CU": r(t0_y, 2), "ibnr_coste_CU": r(cost, 2), "reserva_abiertos_CU": r(res_y, 2),
                                   "prima_riesgo_T0_CU": r(t0_y / e, 2), "prima_riesgo_T0_mas_IBNR_CU": r((t0_y + cost) / e, 2), "factor_ibnr_coste": r((t0_y + cost) / t0_y, 4),
                                   "prima_T1_a075_mas_IBNR_CU": r((t0_y - 0.25 * res_y + cost) / e, 2), "prima_T1_a125_mas_IBNR_CU": r((t0_y + 0.25 * res_y + cost) / e, 2)})
    prima_ult = pd.DataFrame(prima_ult_rows)
    prima_ult.to_csv(tables_dir / "prima_riesgo_ultima_ramo_anio.csv", index=False)

    # ------------------------------------------------------------------ 8. reservas de caso y sensibilidad T1
    res_rows = []
    for (line, peril, y), g in acc.groupby(["line", "peril", acc.loss_date.dt.year.rename("year")]):
        op = g[g.is_open]
        res_rows.append({"line": line, "peril": peril, "year": int(y), "aceptados": int(len(g)), "abiertos": int(len(op)), "pct_abiertos": r(float(len(op) / len(g) * 100), 2),
                         "T0_CU": r(float(g.net.sum()), 2), "reserva_abiertos_CU": r(float(op.reserve.sum()), 2), "reserva_sobre_T0_pct": r(float(op.reserve.sum() / g.net.sum() * 100), 2) if g.net.sum() > 0 else None,
                         "T1_a075_CU": r(float(g.net.sum() - 0.25 * op.reserve.sum()), 2), "T1_a125_CU": r(float(g.net.sum() + 0.25 * op.reserve.sum()), 2)})
    res = pd.DataFrame(res_rows)
    res.to_csv(tables_dir / "reservas_caso_T1.csv", index=False)
    res_line_year = res.groupby(["line", "year"]).agg(T0_CU=("T0_CU", "sum"), reserva_abiertos_CU=("reserva_abiertos_CU", "sum"), abiertos=("abiertos", "sum"), aceptados=("aceptados", "sum")).reset_index()
    res_line_year["reserva_sobre_T0_pct"] = res_line_year.reserva_abiertos_CU / res_line_year.T0_CU * 100
    res_line_year["T1_rango_pct"] = 0.25 * res_line_year["reserva_sobre_T0_pct"]
    res_line_year.to_csv(tables_dir / "reservas_caso_ramo_anio.csv", index=False)

    # ------------------------------------------------------------------ 9. prima de riesgo y referencias M00
    pr_rows = []
    for line in LINES:
        for y in range(2021, 2025):
            rows = t0_rows[(t0_rows.line == line) & (t0_rows.year == y)]
            e = float(rows.exposure.sum())
            n_pos = float(rows.n_positive.sum())
            pr_rows.append({"line": line, "year": y, "expuestos_anios": r(e, 2), "T0_CU": r(float(rows.T0.sum()), 2), "prima_riesgo_T0_CU": r(float(rows.T0.sum() / e), 2),
                            "frecuencia_positivos": r(n_pos / e, 5), "severidad_media_positivos_CU": r(float(rows.T0.sum() / n_pos), 2) if n_pos else None,
                            "identidad_freq_x_sev": r(float(n_pos / e * rows.T0.sum() / n_pos), 2) if n_pos else None})
    pr = pd.DataFrame(pr_rows)
    pr.to_csv(tables_dir / "prima_riesgo_ramo_anio.csv", index=False)
    check("K16", "identidad prima de riesgo = frecuencia positiva x severidad media positiva en todos los ramos y años",
          bool(np.allclose(pr["prima_riesgo_T0_CU"], pr["identidad_freq_x_sev"], rtol=1e-6)))

    # comprobación de la desviación cerrada para p=1,5
    yy = np.array([0.0, 0.0, 12.5, 3000.0, 1.0])
    mm = np.array([100.0, 2.0, 10.0, 2500.0, 0.5])
    closed_form = 4.0 * (np.sqrt(mm) - np.sqrt(yy)) ** 2 / np.sqrt(mm)
    check("K17", "la desviación Tweedie p=1,5 coincide con la forma cerrada 4(sqrt(mu)-sqrt(y))^2/sqrt(mu)", bool(np.allclose(tweedie_deviance(yy, mm), closed_form)))

    m00_rows = []
    splits = {"V22": ([2021], 2022), "V23": ([2021, 2022], 2023), "V24": ([2021, 2022, 2023], 2024)}
    for split, (train_years, eval_year) in splits.items():
        tr = t0_rows[t0_rows.year.isin(train_years)]
        ev = t0_rows[t0_rows.year == eval_year].copy()
        q_bal = tr.groupby("line")["T0"].sum() / tr.groupby("line")["exposure"].sum()
        q_tw = (tr.T0 / np.sqrt(tr.exposure)).groupby(tr.line).sum() / np.sqrt(tr.exposure).groupby(tr.line).sum()
        q_glob_bal = tr.T0.sum() / tr.exposure.sum()
        ev["m_bal"] = ev.exposure * ev.line.map(q_bal)
        ev["m_tw"] = ev.exposure * ev.line.map(q_tw)
        ev["m_glob"] = ev.exposure * q_glob_bal
        ev["m_const"] = tr.T0.mean()
        rowd = {"split": split, "train": "-".join(map(str, train_years)), "eval": eval_year, "filas_eval": int(len(ev)),
                "d2_constante_por_fila": r(d2_tweedie(ev.T0.values, ev.m_const.values), 5),
                "d2_tasa_global_x_exposicion": r(d2_tweedie(ev.T0.values, ev.m_glob.values), 5),
                "d2_tasa_ramo_balance_x_exposicion": r(d2_tweedie(ev.T0.values, ev.m_bal.values), 5),
                "d2_tasa_ramo_tweedie_x_exposicion": r(d2_tweedie(ev.T0.values, ev.m_tw.values), 5)}
        for line in LINES:
            evl = ev[ev.line == line]
            rowd[f"q_balance_{line}"] = r(float(q_bal[line]), 4)
            rowd[f"q_tweedie_{line}"] = r(float(q_tw[line]), 4)
            rowd[f"d2_ramo_{line}_balance"] = r(d2_tweedie(evl.T0.values, evl.m_bal.values), 5)
            rowd[f"d2_ramo_{line}_tweedie"] = r(d2_tweedie(evl.T0.values, evl.m_tw.values), 5)
            rowd[f"balance_{line}_pred_sobre_obs"] = r(float(evl.m_bal.sum() / evl.T0.sum()), 4)
        m00_rows.append(rowd)
    prod = t0_rows
    q_bal_prod = prod.groupby("line")["T0"].sum() / prod.groupby("line")["exposure"].sum()
    q_tw_prod = (prod.T0 / np.sqrt(prod.exposure)).groupby(prod.line).sum() / np.sqrt(prod.exposure).groupby(prod.line).sum()
    m00 = pd.DataFrame(m00_rows)
    m00.to_csv(tables_dir / "referencias_M00.csv", index=False)
    check("K18", "en los tres cortes: D² de la constante por fila dentro de ±0,01 y D² de tasa por ramo x exposición > tasa global x exposición > 0",
          bool((m00["d2_constante_por_fila"].abs() < 0.01).all() and (m00["d2_tasa_ramo_balance_x_exposicion"] > m00["d2_tasa_global_x_exposicion"]).all() and (m00["d2_tasa_global_x_exposicion"] > 0).all()),
          m00[["split", "d2_constante_por_fila", "d2_tasa_global_x_exposicion", "d2_tasa_ramo_balance_x_exposicion"]].to_dict(orient="records"))

    # ------------------------------------------------------------------ 10. evidencia y manifiesto
    evidence = {
        "run_id": run_id,
        "created_utc": started.isoformat(timespec="seconds"),
        "author": "Claude (Cowork), a solicitud del participante; cálculo propio asistido por IA sin revisión humana ni independiente",
        "scope": {"data_modified": False, "indices_modified": False, "models_trained": False, "remote_execution": False, "external_providers": False,
                  "description": "Bases actuariales (expuestos, asignación, T0, retraso e IBNR de conteos, frecuencia, severidad, reservas de caso, referencias M00) sobre la capa preparada vigente"},
        "provenance": {"active_index": "Hackaton/Semana 1/Limpieza/current_dataset.json", "active_run": current["workspace_relative_run"], "active_status": current["status"],
                       "catalog_sha256": digest(catalog_path), "tables_sha256": hashes, "cutoff_observed": str(observed_cutoff.date()), "power_eval": POWER},
        "assignment": {"by_line_category": {line: {cat: int(cat_counts.loc[line].get(cat, 0)) for cat in cat_counts.columns} for line in LINES},
                       "accepted_unique_by_line": got_unique_accepted, "accepted_unique_total": int(sum(got_unique_accepted.values())),
                       "accepted_reported_by_line": {line: int(claims[(claims.line == line) & claims.accepted].shape[0]) for line in LINES},
                       "out_of_window": fuera_desc, "pre00_claims": 413, "pre00_accepted_net_CU": r(float(claims.loc[claims.audit_pre00 & claims.accepted, "net"].sum()), 2)},
        "t0": {"total_CU": r(float(t0_rows.T0.sum()), 2), "by_line_CU": {line: r(float(t0_rows.loc[t0_rows.line == line, "T0"].sum()), 2) for line in LINES},
               "rows_train": int(len(t0_rows)), "rows_with_positive_cost": int((t0_rows.T0 > 0).sum()), "accepted_net_zero": int((acc.net == 0).sum()),
               "open_claims_assigned": int(acc.is_open.sum()), "reserve_open_total_CU": r(float(acc.loc[acc.is_open, "reserve"].sum()), 2)},
        "exposure": {"train_total_years": r(train_exposure_total, 4), "test_total_years": r(test_exposure_total, 4),
                     "by_line_year": exp_line_year.round(4).to_dict(orient="records"), "motor_by_cover": exp_cover.round(4).to_dict(orient="records"),
                     "by_line_month_2024": {line: [r(float(v), 2) for v in month_exposure[LINES.index(line), 36:48]] for line in LINES},
                     "by_line_month_2025": {line: [r(float(v), 2) for v in month_exposure[LINES.index(line), 48:60]] for line in LINES}},
        "reporting": {"chain_ladder_by_peril": cl.to_dict(orient="records"), "delay_by_cohort": delay_rows, "long_tail_perils": [list(x) for x in sorted(LONG_TAIL)],
                      "method": "chain-ladder de conteos acumulados por cohorte mensual de ocurrencia y lag mensual de reporte, factores ponderados por volumen, cola exponencial solo en lesiones y hundimiento"},
        "ibnr_2024": {"by_peril": ult.to_dict(orient="records"), "by_line": ult_line.round(4).to_dict(orient="records"),
                      "note": "IBNR de notificaciones por chain-ladder; positivos = notificaciones x probabilidad de coste positivo (2021-2024); coste = positivos x severidad media 2021-2022 (en lesiones y hundimiento, media de los reportados a más de 365 días)"},
        "ibnr_by_year": {"by_peril_year": ibnr_year.to_dict(orient="records"), "risk_premium_ultimate_by_line_year": prima_ult.to_dict(orient="records")},
        "frequency": {"by_line_year_peril": freq.to_dict(orient="records"), "motor_by_cover": freq_cover},
        "severity": {"by_line_peril": sev.to_dict(orient="records"), "share_positive_among_accepted": {f"{k[0]}|{k[1]}": r(float(v), 4) for k, v in share_positive.items()}},
        "case_reserves": {"by_line_peril_year": res.to_dict(orient="records"), "by_line_year": res_line_year.round(4).to_dict(orient="records")},
        "risk_premium": {"by_line_year": pr.to_dict(orient="records")},
        "m00": {"splits": m00.to_dict(orient="records"), "production_2021_2024": {line: {"q_balance": r(float(q_bal_prod[line]), 4), "q_tweedie": r(float(q_tw_prod[line]), 4)} for line in LINES},
                "deviance_closed_form_p15": "4*(sqrt(mu)-sqrt(y))^2/sqrt(mu)"},
        "checks": checks,
        "checks_summary": {"checks_total": len(checks), "checks_passed": sum(c["passed"] for c in checks), "all_passed": all(c["passed"] for c in checks)},
    }
    (run_dir / "evidencia.json").write_text(json.dumps(evidence, ensure_ascii=False, indent=1, default=lambda o: r(o) if isinstance(o, (np.floating, np.integer)) else str(o)), encoding="utf-8")
    pd.DataFrame(checks).to_csv(tables_dir / "controles.csv", index=False)
    shutil.copy2(Path(__file__), run_dir / Path(__file__).name)
    manifest = {}
    for path in sorted(run_dir.rglob("*")):
        if path.is_file() and path.name != "manifest.json":
            manifest[str(path.relative_to(run_dir)).replace("\\\\", "/")] = digest(path)
    (run_dir / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=1), encoding="utf-8")
    summary = evidence["checks_summary"]
    print(f"Corrida {run_id}: {summary['checks_passed']}/{summary['checks_total']} controles; T0 total {evidence['t0']['total_CU']:,} CU")
    for c in checks:
        print(("OK  " if c["passed"] else "FALLA ") + c["code"] + " " + c["name"] + ("" if c["passed"] else f" -> {c['detail']}"))
    if not summary["all_passed"]:
        sys.exit(1)


if __name__ == "__main__":
    main()
'''


### Embedded source: `src/datos/construir_f0.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/datos/construir_f0.py'] = '''"""Construye la matriz de predictores por ramo (entrenamiento 2021-2024 y test 2025) con contrato de disponibilidad por
columna, las etiquetas T0 por fila y por peligro, y las particiones persistentes V22/V23/V24/producción.

Entregables 2, 3 y 4 de METODOLOGIA_ACTUARIAL.md en versión preliminar de Claude (D68). Lee la capa vigente que señala
current_dataset.json, comprueba los SHA-256 de las diez tablas que usa, toma T0, la asignación y los factores de
desarrollo de la corrida vigente de Bases_actuariales_claude (comprobando su manifiesto) y no modifica ningún dato ni
índice. Crea runs/f0_<marca>_<id>/ con tables/f0_<ramo>.parquet (un grupo de filas por año), tables/particiones.parquet,
contrato_disponibilidad.json, evidencia.json, copia del script y manifest.json.

Reglas (metodología 1.11, D50, D66 y plan D67):
- Bloques: F0 contractual y físico (disponible al inicio del periodo bajo estabilidad de maestros); F1 historia de
  notificaciones de la póliza conocida al 1 de enero del año del periodo (mismo corte en entrenamiento y test);
  F2 clima del año anterior en la dirección; F3 información condicionada sin fecha de referencia (prueba aparte);
  F4 tarifario existente (ablación aparte); excluida (nunca predictor); clave; etiqueta.
- Capital (D66): sum_insured = sum_insured_raw/1000 si supera el umbral del ramo; marcas audit_capital_scale_adjusted
  y audit_capital_scale_confirmed, más audit_capital_cap01, excluidas de los predictores.
- construction_type: t y w se armonizan en construction_type_harmonized (tw_code); el original queda excluido.
- Exposición efectiva por peligro: exposure x (suma por mes natural de días del periodo en el mes x completitud del
  peligro para ese mes de ocurrencia al 30-12-2024) / días cubiertos; la completitud es la inversa del factor a
  último del chain-ladder D67 y vale 1 en 2025. e_ef_coste pondera los peligros por su participación en T0 + IBNR
  2021-2024 del ramo.
- Particiones: V22 (train 2021, eval 2022), V23 (train 2021-2022, eval 2023 dividida por el hash de cliente cal_v1
  en calibración y selección), V24 (train 2021-2023, eval 2024), PROD (train 2021-2024, predict 2025).

Uso completo (Windows, sin límite de tiempo):
  & '.\\\\Preparación\\\\.venv\\\\Scripts\\\\python.exe' '.\\\\Hackaton\\\\Semana 1\\\\Matriz_F0_claude\\\\construir_f0.py'
Uso por partes (máquinas con poca memoria o llamadas con tiempo limitado):
  python3 construir_f0.py --ramo motor              # crea la corrida e imprime su id
  python3 construir_f0.py --run <id> --ramo buildings,contents
  python3 construir_f0.py --run <id> --finalizar
"""
from __future__ import annotations

import argparse
import gc
import hashlib
import json
import shutil
import sys
import uuid
from datetime import datetime, timezone
from pathlib import Path

HERE = Path(__file__).resolve().parent


def find_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "RUTAS_PROYECTO.json").is_file():
            return candidate
    raise FileNotFoundError("No se encontró RUTAS_PROYECTO.json hacia arriba.")


ROOT = find_root(HERE)
try:
    import pyarrow  # noqa: F401
except ImportError:  # pragma: no cover
    sys.path.insert(0, str(ROOT / "Preparación" / ".cache" / "parquet-reader"))

import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import pyarrow as pa  # noqa: E402
import pyarrow.parquet as pq  # noqa: E402

LINES = ["motor", "buildings", "contents"]
PERILS = {"motor": ["own_damage", "third_party_prop", "bodily_injury", "theft", "glass_hail"],
          "buildings": ["storm", "flood", "escape_of_water", "subsidence", "fire", "accidental_damage"],
          "contents": ["theft", "escape_of_water", "accidental_damage", "flood"]}
THRESHOLD = {"buildings": 10_000_000.0, "contents": 1_000_000.0}
MONTHS = pd.period_range("2021-01", "2025-12", freq="M")
CUTOFF_MONTH_INDEX = 47  # diciembre de 2024 con 2021-01 = 0
CAL_SALT = "cal_v1"
TRAIN_ROWS = 2409311
TEST_ROWS = 626639
STRING_MAPPER = {pa.string(): pd.StringDtype("pyarrow"), pa.large_string(): pd.StringDtype("pyarrow")}.get
AMOUNT_LABELS = ["T0", "paid", "incurred", "recoveries", "reserve"]
COUNT_LABELS = ["n_accepted", "n_positive", "n_open"]
EXCLUDED_ALWAYS = ["customer_service_contacts", "renewal_decision", "audit_ren00", "surname", "construction_type", "sum_insured_raw",
                   "audit_capital_scale_adjusted", "audit_capital_scale_confirmed", "audit_capital_cap01"]
BLOCKS = ["F0", "F1", "F2", "F3", "F4", "excluida", "clave", "etiqueta"]

CONTRACT_COMMON = {
    "row_id": ("clave", "identificador de la fila; nunca predictor"),
    "policy_id": ("clave", "identificador; nunca predictor"),
    "customer_id": ("clave", "identificador; solo para hash de calibración y agrupación"),
    "address_id": ("clave", "identificador; solo para agrupación y cruces"),
    "period_start": ("clave", "fecha de inicio; define año y mes de ocurrencia"),
    "period_end": ("clave", "fecha de fin"),
    "is_test": ("clave", "indicador de fila de test 2025"),
    "exposure": ("F0", "exposición nominal EXP01 en años; offset o multiplicador en predicción"),
    "year": ("F0", "año del periodo; efecto de nivel con extrapolación declarada a 2025"),
    "mes_inicio": ("F0", "mes de inicio del periodo; estacionalidad y duración"),
    "periodo_corto": ("F0", "indicador de exposición menor que 0,5 años; efecto de duración (D37)"),
    "new_business": ("F0", "nuevo negocio frente a renovación; conocido al inicio"),
    "renewal_count": ("F0", "número de renovaciones; conocido al inicio"),
    "channel": ("F0", "canal de venta; contractual"),
    "payment_method": ("F0", "forma de pago; contractual"),
    "voluntary_excess": ("F0", "franquicia voluntaria; contractual"),
    "compulsory_excess": ("F0", "franquicia obligatoria; contractual"),
    "days_quote_to_inception": ("F0", "días de cotización a inicio; conocido al inicio"),
    "missed_payments_12m": ("F3", "impagos de los últimos doce meses sin fecha de referencia; prueba aparte"),
    "discount_applied": ("F4", "descuento comercial; ablación del tarifario"),
    "written_premium": ("F4", "prima escrita del tarifario existente; ablación aparte, nunca en la matriz base"),
    "customer_service_contacts": ("excluida", "mide el siniestro del propio periodo y cambia de distribución en el test (D50)"),
    "renewal_decision": ("excluida", "solo existe en entrenamiento; decisión posterior al periodo (D45)"),
    "audit_ren00": ("excluida", "marca de auditoría REN00"),
    "anios_como_cliente": ("F0", "año del periodo menos customer_since_year; conocido al inicio"),
    "surname": ("excluida", "nombre personal"),
    "occupation": ("F3", "atributo del maestro de clientes sin fecha de referencia"),
    "marital_status": ("F3", "atributo del maestro sin fecha de referencia"),
    "household_size": ("F3", "atributo del maestro sin fecha de referencia"),
    "credit_band": ("F3", "banda de crédito sin fecha de referencia"),
    "prior_claims_5y": ("F3", "siniestros previos declarados sin fecha de referencia; posible solape con F1"),
    "licence_points": ("F3", "puntos de licencia sin fecha de referencia"),
    "years_at_address": ("F3", "años en el domicilio sin fecha de referencia"),
    "email_domain": ("F3", "atributo comercial"),
    "marketing_consent": ("F3", "atributo comercial"),
    "document_preference": ("F3", "atributo comercial"),
    "previous_insurer": ("F3", "atributo comercial sin fecha"),
    "campaign_code": ("F3", "atributo comercial sin fecha"),
    "branch_id": ("F3", "sucursal; atributo comercial"),
    "marketing_segment": ("F3", "segmento comercial de la aseguradora; posible proxy de edad y renta"),
    "region": ("F0", "región de la dirección; física y estable"),
    "urban_density": ("F0", "densidad urbana; física y estable"),
    "flood_zone": ("F0", "zona de inundación 1-5; nula en 2,9 % de direcciones, el nulo se conserva como ausencia"),
    "distance_to_river_km": ("F0", "distancia al río; física"),
    "elevation_m": ("F0", "elevación; física"),
    "soil_type": ("F0", "tipo de suelo; físico"),
    "year_built": ("F0", "año de construcción; estable"),
    "property_type": ("F0", "tipo de inmueble; estable"),
    "roof_type": ("F0", "tipo de tejado; estable"),
    "basement": ("F0", "sótano; estable"),
    "trees_within_5m": ("F0", "árboles a menos de 5 m; físico"),
    "listed_building": ("F0", "edificio protegido; estable"),
    "bedrooms": ("F0", "habitaciones; estable"),
    "energy_rating": ("F0", "calificación energética; estable bajo supuesto de maestro"),
    "solar_panels": ("F0", "paneles solares; estable bajo supuesto de maestro"),
}
CONTRACT_LINE = {
    "motor": {
        "driver_age": ("F0", "edad del conductor principal; contractual"),
        "years_licensed": ("F0", "años de licencia (VD34 abierto: 3.717 edades implícitas menores de 16)"),
        "ncb_years": ("F0", "años de bonificación (161.822 filas con ncb mayor que licencia, sin recortar)"),
        "vehicle_value_group": ("F0", "grupo de valor del vehículo"),
        "vehicle_age": ("F0", "antigüedad del vehículo"),
        "make_model": ("F0", "marca y modelo; alta cardinalidad, codificar dentro del fold"),
        "engine_size_cc": ("F0", "cilindrada; nula en 2.533 eléctricos (VD35), se conserva nula"),
        "body_type": ("F0", "carrocería"),
        "transmission": ("F0", "transmisión"),
        "seats": ("F0", "plazas"),
        "vehicle_colour": ("F0", "color; señal dudosa, candidata a ablación"),
        "fuel_type": ("F0", "combustible"),
        "driver_assist": ("F0", "asistencia a la conducción"),
        "cover_type": ("F0", "cobertura; estructura contractual con elegibilidad por peligro"),
        "annual_mileage": ("F0", "kilometraje declarado"),
        "overnight_parking": ("F0", "estacionamiento nocturno"),
        "vehicle_use": ("F0", "uso del vehículo"),
        "named_young_driver": ("F0", "conductor joven designado"),
        "telematics_opt_in": ("F3", "adhesión a telemática; bloque condicional"),
        "telematics_score": ("F3", "score telemático; puede generarse durante la cobertura; ausente = sin adhesión, no cero"),
    },
    "buildings": {
        "policyholder_age": ("F0", "edad AGE01 al inicio del periodo"),
        "sum_insured_raw": ("excluida", "capital leído de la capa vigente; se conserva para auditoría"),
        "sum_insured": ("F0", "capital bajo el contrato de F0 de D66 (entre mil sobre 10.000.000 CU)"),
        "log_sum_insured": ("F0", "logaritmo del capital del contrato; curva suave candidata"),
        "construction_type": ("excluida", "t y w son alias temporales (D39); usar construction_type_harmonized"),
        "construction_type_harmonized": ("F0", "construcción con t y w unificados en tw_code (equivalente a CON01, no activado en Limpieza)"),
        "occupancy": ("F0", "ocupación del inmueble; contractual"),
        "days_unoccupied": ("F0", "días desocupado declarados; contractual"),
        "smoke_alarms": ("F0", "detectores de humo; nulo en 2,5 %, nunca cambia dentro de una dirección (D63)"),
        "accidental_damage_cover": ("F0", "cobertura opcional de daño accidental"),
        "mortgage": ("F0", "hipoteca; contractual"),
        "audit_capital_scale_adjusted": ("excluida", "marca D66: capital dividido entre mil en F0"),
        "audit_capital_scale_confirmed": ("excluida", "marca D66: división respaldada por historia (rama CAP02R-CAP06R o lista D66)"),
        "audit_capital_cap01": ("excluida", "fila corregida por CAP01 en la base"),
    },
    "contents": {
        "policyholder_age": ("F0", "edad AGE01 al inicio del periodo"),
        "sum_insured_raw": ("excluida", "capital leído de la capa vigente; se conserva para auditoría"),
        "sum_insured": ("F0", "capital bajo el contrato de F0 de D66 (entre mil sobre 1.000.000 CU)"),
        "log_sum_insured": ("F0", "logaritmo del capital del contrato; curva suave candidata"),
        "tenure": ("F0", "tenencia; contractual"),
        "security_level": ("F0", "nivel de seguridad; nulo en 2,5 %"),
        "high_value_items": ("F0", "bienes de alto valor; nulo en 2,5 %"),
        "accidental_damage_cover": ("F0", "cobertura opcional"),
        "works_from_home": ("F0", "trabajo desde casa; contractual"),
        "lodgers": ("F0", "huéspedes; contractual"),
        "bicycle_cover": ("F0", "cobertura de bicicleta"),
        "audit_capital_scale_adjusted": ("excluida", "marca D66"),
        "audit_capital_scale_confirmed": ("excluida", "marca D66"),
        "audit_capital_cap01": ("excluida", "fila corregida por CAP01 en la base"),
    },
}
WEATHER_VARS = ["hail_days", "wind_exposure_index", "freezing_days", "dry_summer_index", "wildfire_index", "rainfall_mm", "mean_temperature_c"]


def digest(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


def read_table(path: Path, columns=None, filters=None) -> pd.DataFrame:
    return pq.read_table(path, columns=columns, filters=filters).to_pandas(types_mapper=STRING_MAPPER)


def cal_half(customer_id: str) -> int:
    return int(hashlib.sha256(f"{CAL_SALT}|{customer_id}".encode("utf-8")).hexdigest()[:16], 16) % 2


def classify(col: str, line: str):
    if col in CONTRACT_LINE[line]:
        return CONTRACT_LINE[line][col]
    if col in CONTRACT_COMMON:
        return CONTRACT_COMMON[col]
    if col.endswith("_prev1"):
        return ("F2", "clima del año anterior en la dirección (nulo en 2021); años posteriores al corte nunca")
    if col.startswith("f1_"):
        return ("F1", "historia de notificaciones de la póliza conocida al 1 de enero del año del periodo, sin filtrar por estado final")
    if col.startswith("e_ef_"):
        return ("F0", "exposición efectiva por completitud de reporte: offset de entrenamiento; en predicción se usa exposure")
    if col in AMOUNT_LABELS + COUNT_LABELS or col.startswith("y_") or col.startswith("n_"):
        return ("etiqueta", "coste o conteo del propio periodo valorado al 30-12-2024; nunca predictor; nulo en test")
    return ("revisar", "columna sin contrato declarado")


class Context:
    """Carga común: capa vigente, hashes, bases D67, bitácoras de capital, maestros y completitud por peligro."""

    def __init__(self):
        self.checks = []
        current = json.loads((ROOT / "Hackaton/Semana 1/Limpieza/current_dataset.json").read_text(encoding="utf-8"))
        self.current = current
        self.catalog_path = ROOT / current["catalog_workspace_relative"]
        self.catalog = json.loads(self.catalog_path.read_text(encoding="utf-8"))
        used = ["policies_motor", "policies_buildings", "policies_contents", "test_motor", "test_buildings", "test_contents", "customers", "addresses", "weather", "claims"]
        self.hashes = {n: digest(ROOT / self.catalog[n]["workspace_relative_path"]) for n in used}
        bad = [n for n in used if self.hashes[n] != self.catalog[n]["sha256"]]
        self.check("K01", "SHA-256 de las diez tablas usadas iguales al catálogo activo", not bad, bad)

        self.bases_index = json.loads((ROOT / "Hackaton/Semana 1/Bases_actuariales_claude/current_run.json").read_text(encoding="utf-8"))
        self.bases_run = ROOT / self.bases_index["workspace_relative_run"]
        manifest = json.loads((self.bases_run / "manifest.json").read_text(encoding="utf-8"))
        self.bases_evidence = json.loads((self.bases_run / "evidencia.json").read_text(encoding="utf-8"))
        needed = ["tables/t0_por_row_id.parquet", "tables/asignacion_por_siniestro.parquet", "tables/factores_desarrollo_conteos.csv", "tables/ibnr_por_peligro_anio.csv"]
        self.check("K02", "la corrida vigente de Bases_actuariales_claude usa la misma capa y sus tablas conservan el hash",
                   self.bases_evidence["provenance"]["active_run"] == current["workspace_relative_run"] and all(digest(self.bases_run / n) == manifest[n] for n in needed),
                   self.bases_index["workspace_relative_run"])

        # The historical ledgers are not redistributed. This is an explicit retrospective
        # reconstruction of the audit flags, not the original authorization process.
        self.branch_index = {"workspace_relative_run": "audit-ledger-not-included"}
        self.cap01_index = {"workspace_relative_run": current["workspace_relative_run"]}
        self.d66_index = {"workspace_relative_run": "audit-ledger-not-included"}
        reconstructed = json.loads((ROOT / current["workspace_relative_run"] / "evidence/cap01_row_ids.json").read_text(encoding="utf-8"))
        self.cap01_ids = {str(row) for rows in reconstructed.values() for row in rows}
        self.confirmed_ids = set()
        for line, threshold in (("buildings", 10000000), ("contents", 1000000)):
            frames = [read_table(ROOT / self.catalog[f"{kind}_{line}"]["workspace_relative_path"], columns=["row_id", "policy_id", "address_id", "sum_insured"]) for kind in ("policies", "test")]
            rows = pd.concat(frames, ignore_index=True)
            normal = rows.sum_insured.le(threshold)
            grouping = [rows.policy_id, rows.address_id] if line == "buildings" else rows.policy_id
            has_normal = normal.groupby(grouping, dropna=False).transform("sum").gt(0)
            self.confirmed_ids.update(rows.loc[rows.sum_insured.gt(threshold) & has_normal, "row_id"].astype(str))
        self.check("K03_CAP01", "CAP01 identifiers reconstructed from originals", len(self.cap01_ids) == 2073)
        self.check("K03_RECONSTRUCTED", "Retrospective reconstruction of confirmation flag", len(self.confirmed_ids) == 1305)

        factors = pd.read_csv(self.bases_run / "tables/factores_desarrollo_conteos.csv")
        self.comp = {}
        for (line, peril), g in factors.groupby(["line", "peril"]):
            cdf = np.ones(48)
            for row in g.itertuples():
                cdf[int(row.lag_meses)] = float(row.factor_a_ultimo)
            c = np.ones(len(MONTHS))
            for m in range(CUTOFF_MONTH_INDEX + 1):
                c[m] = 1.0 / cdf[CUTOFF_MONTH_INDEX - m]
            self.comp[(line, peril)] = c
        ibnr_year = pd.read_csv(self.bases_run / "tables/ibnr_por_peligro_anio.csv")
        self.weights = {}
        for line in LINES:
            sub = ibnr_year[ibnr_year.line == line].groupby("peril").agg(T0=("T0_CU", "sum"), ibnr=("ibnr_coste_CU", "sum"))
            tot = sub.T0 + sub.ibnr
            self.weights[line] = (tot / tot.sum()).to_dict()

        self.customers = read_table(ROOT / self.catalog["customers"]["workspace_relative_path"])
        self.addresses = read_table(ROOT / self.catalog["addresses"]["workspace_relative_path"])
        weather = read_table(ROOT / self.catalog["weather"]["workspace_relative_path"])
        weather = weather.rename(columns={v: f"{v}_prev1" for v in WEATHER_VARS})
        weather["year"] = weather["year"].astype(int) + 1  # clima del año Y para los periodos del año Y+1
        self.weather = weather
        claims = read_table(ROOT / self.catalog["claims"]["workspace_relative_path"], columns=["claim_id", "policy_id", "line", "peril", "report_date", "status", "incurred_to_date", "recoveries"])
        claims["report_date"] = pd.to_datetime(claims["report_date"])
        claims["net"] = claims["incurred_to_date"] - claims["recoveries"]
        self.claims = claims
        assignment = read_table(self.bases_run / "tables/asignacion_por_siniestro.parquet", columns=["claim_id", "row_id", "category"])
        cr = claims.merge(assignment[assignment.category == "unica"][["claim_id", "row_id"]], on="claim_id", how="inner")
        cr = cr[cr.status.isin(["closed", "open"])].copy()
        cr["positive"] = (cr.net > 0).astype(int)
        self.claim_rows = cr
        self.half_cache: dict[str, int] = {}

    def check(self, code, name, ok, detail=None):
        self.checks.append({"code": code, "name": name, "passed": bool(ok), "detail": detail})


def process_line(line: str, ctx: Context, run_dir: Path) -> dict:
    tables_dir = run_dir / "tables"
    (tables_dir / "partes").mkdir(parents=True, exist_ok=True)
    checks = []

    def check(code, name, ok, detail=None):
        checks.append({"code": code, "name": name, "passed": bool(ok), "detail": detail})

    # F1: historia de notificaciones por póliza conocida al 1 de enero de cada año
    cl = ctx.claims[ctx.claims.line == line][["policy_id", "report_date"]].copy()
    cl["report_year"] = cl.report_date.dt.year.astype(int)
    hist_parts = []
    for y in range(2021, 2026):
        prev = cl[cl.report_year < y]
        h = prev.groupby("policy_id").agg(f1_notificaciones_previas=("report_date", "size"), f1_ultima_notificacion=("report_date", "max")).reset_index()
        h["year"] = y
        hist_parts.append(h)
    hist = pd.concat(hist_parts, ignore_index=True)

    # etiquetas del ramo
    t0 = read_table(ctx.bases_run / "tables/t0_por_row_id.parquet", filters=[("line", "==", line)]).drop(columns=["line", "year", "exposure"])
    per = ctx.claim_rows[ctx.claim_rows.line == line].groupby(["row_id", "peril"]).agg(y=("net", "sum"), n=("positive", "sum"))
    wide = per.unstack("peril")
    wide.columns = [f"{a}_{b}" for a, b in wide.columns]
    for p in PERILS[line]:
        for pre in ("y", "n"):
            if f"{pre}_{p}" not in wide.columns:
                wide[f"{pre}_{p}"] = 0
    wide = wide.reset_index()
    labels = t0.merge(wide, on="row_id", how="left")
    amount_cols = AMOUNT_LABELS + [f"y_{p}" for p in PERILS[line]]
    count_cols = COUNT_LABELS + [f"n_{p}" for p in PERILS[line]]
    for c in amount_cols:
        labels[c] = labels[c].fillna(0.0).astype("float64")
    for c in count_cols:
        labels[c] = labels[c].fillna(0).astype("int64").astype("Int64")
    del t0, per, wide

    acc = {"rows": {"train": 0, "test": 0}, "nulls": {}, "n_total": 0, "cap_ok": True, "cap": {"adjusted": 0, "confirmed": 0, "cap01": 0, "adjusted_test": 0, "confirmed_test": 0},
           "tw_ok": True, "tw_values": set(), "leak": 0, "f1_rows": 0, "f1_test_rows": 0, "ef_ok": True, "test_ef_ok": True, "ef_sum": {}, "exp_sum": {},
           "label_ok": True, "T0_train": 0.0, "test_label_null_ok": True, "weather_rows": 0, "chunk_unique_ok": True}
    out_path = tables_dir / f"f0_{line}.parquet"
    writer = None
    schema = None
    columns = None
    part_pieces = []
    for kind, table in (("train", f"policies_{line}"), ("test", f"test_{line}")):
        tab = pq.read_table(ROOT / ctx.catalog[table]["workspace_relative_path"])
        years_arr = pd.to_datetime(tab.column("period_start").to_pandas()).dt.year.to_numpy()
        for y in sorted(set(years_arr.tolist())):
            df = tab.filter(pa.array(years_arr == y)).to_pandas(types_mapper=STRING_MAPPER)
            is_test = kind == "test"
            df["is_test"] = is_test
            if "renewal_decision" not in df.columns:
                df["renewal_decision"] = pd.array([None] * len(df), dtype="string[pyarrow]")
            df["period_start"] = pd.to_datetime(df["period_start"])
            df["period_end"] = pd.to_datetime(df["period_end"])
            df["year"] = df.period_start.dt.year.astype("int64")
            df["mes_inicio"] = df.period_start.dt.month.astype("int64")
            df["periodo_corto"] = df.exposure < 0.5
            df = df.merge(ctx.customers, on="customer_id", how="left")
            df = df.merge(ctx.addresses, on="address_id", how="left")
            df["anios_como_cliente"] = (df.year - df.customer_since_year).astype("float64")
            df = df.merge(ctx.weather[ctx.weather.year == y], on=["address_id", "year"], how="left")
            df = df.drop(columns=["date_of_birth", "customer_since_year"])
            acc["weather_rows"] += int(df[f"{WEATHER_VARS[0]}_prev1"].notna().sum())

            if line in THRESHOLD:
                thr = THRESHOLD[line]
                df["sum_insured_raw"] = df["sum_insured"]
                adj = df["sum_insured_raw"] > thr
                df["audit_capital_scale_adjusted"] = adj
                df["sum_insured"] = np.where(adj, df["sum_insured_raw"] / 1000.0, df["sum_insured_raw"])
                ids = df["row_id"].astype(str)
                df["audit_capital_scale_confirmed"] = adj & ids.isin(ctx.confirmed_ids)
                df["audit_capital_cap01"] = ids.isin(ctx.cap01_ids)
                df["log_sum_insured"] = np.log(df["sum_insured"])
                idem = np.where(df["sum_insured"] > thr, df["sum_insured"] / 1000.0, df["sum_insured"])
                acc["cap_ok"] &= bool(np.allclose(idem, df["sum_insured"])) and bool((df["sum_insured"] <= thr).all()) and bool((df.audit_capital_scale_confirmed <= adj).all())
                acc["cap"]["adjusted"] += int(adj.sum())
                acc["cap"]["confirmed"] += int(df.audit_capital_scale_confirmed.sum())
                acc["cap"]["cap01"] += int(df.audit_capital_cap01.sum())
                if is_test:
                    acc["cap"]["adjusted_test"] += int(adj.sum())
                    acc["cap"]["confirmed_test"] += int(df.audit_capital_scale_confirmed.sum())
            if line == "buildings":
                df["construction_type_harmonized"] = df["construction_type"].replace({"t": "tw_code", "w": "tw_code"})
                acc["tw_ok"] &= not bool(df["construction_type_harmonized"].isin(["t", "w"]).any())
                acc["tw_values"] |= set(df["construction_type_harmonized"].dropna().astype(str).unique())

            df = df.merge(hist[hist.year == y], on=["policy_id", "year"], how="left")
            df["f1_notificaciones_previas"] = df["f1_notificaciones_previas"].fillna(0).astype("int64")
            df["f1_dias_desde_ultima_notificacion"] = (df.period_start - df.f1_ultima_notificacion).dt.days.astype("float64")
            acc["leak"] += int((df.f1_ultima_notificacion.notna() & (df.f1_ultima_notificacion >= pd.Timestamp(f"{y}-01-01"))).sum())
            acc["f1_rows"] += int((df.f1_notificaciones_previas > 0).sum())
            if is_test:
                acc["f1_test_rows"] += int((df.f1_notificaciones_previas > 0).sum())
            df = df.drop(columns=["f1_ultima_notificacion"])

            ps = df["period_start"].values.astype("datetime64[D]")
            pe = df["period_end"].values.astype("datetime64[D]")
            # completitud media del periodo ponderada por días, aplicada a la exposición EXP01 para conservar su redondeo
            e_ef = {p: np.zeros(len(df)) for p in PERILS[line]}
            e_cost = np.zeros(len(df))
            covered = np.zeros(len(df))
            for j, m in enumerate(MONTHS):
                ms = np.datetime64(m.start_time.date(), "D")
                me = np.datetime64(m.end_time.date(), "D")
                if ms > pe.max() or me < ps.min():
                    continue
                overlap = np.clip((np.minimum(pe, me) - np.maximum(ps, ms)).astype("timedelta64[D]").astype(int) + 1, 0, None).astype(float)
                covered += overlap
                for p in PERILS[line]:
                    c = ctx.comp[(line, p)][j]
                    e_ef[p] += overlap * c
                    e_cost += overlap * c * ctx.weights[line][p]
            expo = df["exposure"].to_numpy(dtype=float)
            for p in PERILS[line]:
                df[f"e_ef_{p}"] = expo * e_ef[p] / covered
            df["e_ef_coste"] = expo * e_cost / covered
            ef_cols = [f"e_ef_{p}" for p in PERILS[line]] + ["e_ef_coste"]
            acc["ef_ok"] &= all(bool(((df[c] <= df.exposure * (1 + 1e-12)) & (df[c] > 0)).all()) for c in ef_cols) and bool((covered > 0).all())
            if is_test:
                acc["test_ef_ok"] &= all(bool(np.allclose(df[c], df["exposure"], rtol=1e-12, atol=0)) for c in ef_cols)
            acc["ef_sum"][y] = acc["ef_sum"].get(y, 0.0) + float(df.e_ef_coste.sum())
            acc["exp_sum"][y] = acc["exp_sum"].get(y, 0.0) + float(df.exposure.sum())

            if is_test:
                for c in amount_cols:
                    df[c] = pd.array([np.nan] * len(df), dtype="float64")
                for c in count_cols:
                    df[c] = pd.array([pd.NA] * len(df), dtype="Int64")
                acc["test_label_null_ok"] &= bool(df["T0"].isna().all())
            else:
                df = df.merge(labels, on="row_id", how="left")
                for c in amount_cols:
                    df[c] = df[c].fillna(0.0).astype("float64")
                for c in count_cols:
                    df[c] = df[c].fillna(0).astype("Int64")
                ysum = sum(df[f"y_{p}"] for p in PERILS[line])
                acc["label_ok"] &= bool(np.allclose(ysum, df["T0"], atol=1e-6))
                acc["T0_train"] += float(df["T0"].sum())

            acc["chunk_unique_ok"] &= bool(df.row_id.is_unique)
            acc["rows"][kind] += len(df)
            acc["n_total"] += len(df)
            for c in df.columns:
                acc["nulls"][c] = acc["nulls"].get(c, 0) + int(df[c].isna().sum())

            uniq = df.customer_id.astype(str).unique()
            for cid in uniq:
                if cid not in ctx.half_cache:
                    ctx.half_cache[cid] = cal_half(cid)
            piece = pd.DataFrame({"row_id": df.row_id.astype(str).values, "line": line, "year": df.year.values, "is_test": is_test,
                                  "cal_half": df.customer_id.astype(str).map(ctx.half_cache).astype("int8").values})
            part_pieces.append(piece)

            if columns is None:
                columns = list(df.columns)
            df = df[columns]
            table_out = pa.Table.from_pandas(df, preserve_index=False)
            if writer is None:
                schema = table_out.schema
                writer = pq.ParquetWriter(out_path, schema)
            else:
                table_out = table_out.cast(schema)
            writer.write_table(table_out)
            del df, table_out, piece, e_ef, e_cost, ps, pe, covered, expo
            gc.collect()
        del tab, years_arr
        gc.collect()
    writer.close()
    pd.concat(part_pieces, ignore_index=True).to_parquet(tables_dir / "partes" / f"particiones_{line}.parquet", index=False)

    if line in THRESHOLD:
        check(f"K04_{line}", f"{line}: contrato de capital idempotente, ningún capital del modelo supera el umbral y confirmadas contenidas en ajustadas", acc["cap_ok"], acc["cap"])
    if line == "buildings":
        check("K05", "buildings: construction_type_harmonized sin t ni w y con cuatro categorías", acc["tw_ok"] and len(acc["tw_values"]) == 4, sorted(acc["tw_values"]))
    check(f"K06_{line}", f"{line}: ninguna notificación de F1 posterior o igual al 1 de enero del año del periodo", acc["leak"] == 0,
          {"filas_con_historia": acc["f1_rows"], "test_con_historia": acc["f1_test_rows"]})
    ratio = {int(y): round(acc["ef_sum"][y] / acc["exp_sum"][y], 4) for y in sorted(acc["ef_sum"])}
    check(f"K07_{line}", f"{line}: exposición efectiva positiva, nunca mayor que la nominal e igual a la nominal en 2025", acc["ef_ok"] and acc["test_ef_ok"], ratio)
    t0_expected = ctx.bases_evidence["t0"]["by_line_CU"][line]
    check(f"K08_{line}", f"{line}: etiquetas por peligro suman T0 en cada fila, T0 total igual a D67 y etiquetas nulas en test",
          acc["label_ok"] and abs(acc["T0_train"] - t0_expected) < 0.01 and acc["test_label_null_ok"], {"T0_CU": round(acc["T0_train"], 2), "esperado": t0_expected})
    contract = {}
    for field in schema:
        block, reason = classify(field.name, line)
        contract[field.name] = {"bloque": block, "motivo": reason, "tipo": str(field.type), "nulos_pct": round(acc["nulls"].get(field.name, 0) / acc["n_total"] * 100, 3)}
    blocks = {b: sum(1 for v in contract.values() if v["bloque"] == b) for b in BLOCKS}
    check(f"K09_{line}", f"{line}: todas las columnas con contrato y las de exclusión obligatoria marcadas como excluidas",
          not any(v["bloque"] == "revisar" for v in contract.values()) and all(contract[c]["bloque"] == "excluida" for c in EXCLUDED_ALWAYS if c in contract),
          {"bloques": blocks, "revisar": [k for k, v in contract.items() if v["bloque"] == "revisar"]})
    src_train = pq.read_metadata(ROOT / ctx.catalog[f"policies_{line}"]["workspace_relative_path"]).num_rows
    src_test = pq.read_metadata(ROOT / ctx.catalog[f"test_{line}"]["workspace_relative_path"]).num_rows
    check(f"K10_{line}", f"{line}: filas de entrenamiento y test iguales a la capa vigente y row_id único por año",
          acc["rows"]["train"] == src_train and acc["rows"]["test"] == src_test and acc["chunk_unique_ok"], acc["rows"])
    evidence = {"rows_train": acc["rows"]["train"], "rows_test": acc["rows"]["test"], "columns": len(schema), "blocks": blocks,
                "weather_prev1_rows": acc["weather_rows"], "f1_rows_with_history": acc["f1_rows"], "f1_test_rows_with_history": acc["f1_test_rows"],
                "e_ef_coste_over_exposure_by_year": ratio, "capital": acc["cap"] if line in THRESHOLD else None,
                "peril_cost_weights": {k: round(v, 4) for k, v in ctx.weights[line].items()}, "T0_train_CU": round(acc["T0_train"], 2),
                "file": f"tables/f0_{line}.parquet", "row_groups_by_year": True}
    partial = {"line": line, "common_checks": ctx.checks, "checks": checks, "evidence": evidence, "contract": contract}
    (run_dir / f"parcial_{line}.json").write_text(json.dumps(partial, ensure_ascii=False, indent=1, default=str), encoding="utf-8")
    print(f"{line}: {sum(c['passed'] for c in checks)}/{len(checks)} controles del ramo; {acc['rows']}")
    return partial


def finalize(run_dir: Path, ctx_meta: dict) -> dict:
    tables_dir = run_dir / "tables"
    partials = {}
    for line in LINES:
        path = run_dir / f"parcial_{line}.json"
        if not path.is_file():
            raise FileNotFoundError(f"Falta {path.name}; ejecute --ramo {line} antes de --finalizar")
        partials[line] = json.loads(path.read_text(encoding="utf-8"))
    common = partials["motor"]["common_checks"]
    checks = list(common)
    same_common = all([(c["code"], c["passed"]) for c in partials[l]["common_checks"]] == [(c["code"], c["passed"]) for c in common] for l in LINES)
    for line in LINES:
        checks.extend(partials[line]["checks"])
    part = pd.concat([pd.read_parquet(tables_dir / "partes" / f"particiones_{l}.parquet") for l in LINES], ignore_index=True)
    part["V22"] = np.select([part.year == 2021, part.year == 2022], ["train", "eval"], default="none")
    part["V23"] = np.select([part.year <= 2022, (part.year == 2023) & (part.cal_half == 0), (part.year == 2023) & (part.cal_half == 1)], ["train", "eval_calibracion", "eval_seleccion"], default="none")
    part["V24"] = np.select([part.year <= 2023, part.year == 2024], ["train", "eval"], default="none")
    part["PROD"] = np.select([part.year <= 2024, part.year == 2025], ["train", "predict"], default="none")
    part.to_parquet(tables_dir / "particiones.parquet", index=False)
    half_share = float(part.loc[part.year == 2023, "cal_half"].mean())

    def check(code, name, ok, detail=None):
        checks.append({"code": code, "name": name, "passed": bool(ok), "detail": detail})

    check("K11", "las tres partes usaron la misma capa y los mismos controles comunes", same_common)
    check("K12", "particiones: V24 entrena 2021-2023 y evalúa 2024; PROD predice las 626.639 filas de 2025; row_id único en 3.035.950 filas",
          int((part.V24 == "train").sum()) == int((part.year <= 2023).sum()) and int((part.V24 == "eval").sum()) == int((part.year == 2024).sum())
          and int((part.PROD == "predict").sum()) == TEST_ROWS and int((part.PROD == "train").sum()) == TRAIN_ROWS and part.row_id.is_unique,
          {"filas": int(len(part)), "predict": int((part.PROD == "predict").sum()), "train": int((part.PROD == "train").sum())})
    check("K13", "mitades de calibración y selección de V23 entre 45 % y 55 % de las filas de 2023", 0.45 <= half_share <= 0.55, round(half_share, 4))
    known = "05180324"  # cliente del maestro; la mitad se recalcula aquí con la regla publicada
    check("K14", "regla de hash reproducible: SHA-256('cal_v1|customer_id'), primeros 16 hex, módulo 2", cal_half(known) in (0, 1),
          {"customer_id": known, "half": cal_half(known), "sha256_prefix": hashlib.sha256(f"{CAL_SALT}|{known}".encode()).hexdigest()[:16]})

    contract = {line: partials[line]["contract"] for line in LINES}
    (run_dir / "contrato_disponibilidad.json").write_text(json.dumps({
        "version": "F0 v1 (D68)",
        "bloques": {"F0": "contractual y físico, disponible al inicio del periodo bajo estabilidad de maestros; matriz base",
                    "F1": "historia de notificaciones de la póliza al 1 de enero del año del periodo; bloque adicional",
                    "F2": "clima del año anterior; bloque adicional", "F3": "información condicionada sin fecha de referencia; prueba separada",
                    "F4": "tarifario existente; ablación aparte", "excluida": "nunca predictor", "clave": "identificadores y fechas",
                    "etiqueta": "T0 y componentes; solo entrenamiento, nulo en test"},
        "capital": "sum_insured = sum_insured_raw/1000 si supera el umbral del ramo (10.000.000 CU edificios, 1.000.000 contenidos); marcas audit_* fuera de los predictores (D66)",
        "exposicion_efectiva": "exposure x completitud media del periodo ponderada por días; completitud del peligro para el mes de ocurrencia al 30-12-2024 (chain-ladder D67); 1 en 2025",
        "lectura": "las columnas de predictores se toman por bloque desde este contrato; el selector debe rechazar 'excluida', 'clave' y 'etiqueta' además del guard 1.0",
        "columnas": contract}, ensure_ascii=False, indent=1), encoding="utf-8")
    evidence = {
        "run_id": run_dir.name, "created_utc": ctx_meta["started"],
        "author": "Claude (Cowork), a solicitud del participante; cálculo propio asistido por IA sin revisión humana ni independiente",
        "scope": {"data_modified": False, "indices_modified": False, "models_trained": False, "remote_execution": False, "external_providers": False,
                  "description": "Matriz F0 por ramo con contrato de disponibilidad, etiquetas T0 y por peligro, exposición efectiva y particiones V22/V23/V24/PROD"},
        "provenance": ctx_meta["provenance"],
        "lines": {line: partials[line]["evidence"] for line in LINES},
        "partitions": {"rows": int(len(part)), "by_year": {int(y): int(n) for y, n in part.year.value_counts().sort_index().items()}, "cal_half_share_2023": round(half_share, 4),
                       "rule": "SHA-256 del texto UTF-8 'cal_v1|customer_id', primeros 16 hex como entero, módulo 2: 0 calibración, 1 selección",
                       **{v: {k: int(n) for k, n in part[v].value_counts().items()} for v in ["V22", "V23", "V24", "PROD"]}},
        "checks": checks,
        "checks_summary": {"checks_total": len(checks), "checks_passed": sum(c["passed"] for c in checks), "all_passed": all(c["passed"] for c in checks)},
    }
    (run_dir / "evidencia.json").write_text(json.dumps(evidence, ensure_ascii=False, indent=1, default=str), encoding="utf-8")
    pd.DataFrame(checks).to_csv(tables_dir / "controles.csv", index=False)
    shutil.copy2(Path(__file__), run_dir / Path(__file__).name)
    manifest = {str(p.relative_to(run_dir)).replace("\\\\", "/"): digest(p) for p in sorted(run_dir.rglob("*")) if p.is_file() and p.name != "manifest.json"}
    (run_dir / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=1), encoding="utf-8")
    s = evidence["checks_summary"]
    print(f"Corrida {run_dir.name}: {s['checks_passed']}/{s['checks_total']} controles")
    for c in checks:
        print(("OK  " if c["passed"] else "FALLA ") + c["code"] + " " + c["name"] + ("" if c["passed"] else f" -> {c['detail']}"))
    return evidence


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    parser.add_argument("--run", help="id de una corrida existente en runs/")
    parser.add_argument("--ramo", help="ramos a procesar, separados por comas (motor, buildings, contents)")
    parser.add_argument("--finalizar", action="store_true", help="reúne los parciales, construye particiones, evidencia y manifiesto")
    args = parser.parse_args()
    if args.run:
        run_dir = HERE / "runs" / args.run
        if not run_dir.is_dir():
            raise FileNotFoundError(run_dir)
    else:
        started = datetime.now(timezone.utc)
        run_dir = HERE / "runs" / f"f0_{started.strftime('%Y%m%dT%H%M%SZ')}_{uuid.uuid4().hex[:8]}"
        (run_dir / "tables").mkdir(parents=True, exist_ok=False)
        (run_dir / "inicio.json").write_text(json.dumps({"started": started.isoformat(timespec="seconds")}), encoding="utf-8")
        print(f"Corrida creada: {run_dir.name}")
    started_iso = json.loads((run_dir / "inicio.json").read_text(encoding="utf-8"))["started"]
    lines = LINES if not (args.ramo or args.finalizar) else ([l.strip() for l in args.ramo.split(",")] if args.ramo else [])
    ctx = None
    if lines or not args.finalizar:
        ctx = Context()
        for line in lines:
            process_line(line, ctx, run_dir)
            gc.collect()
    if args.finalizar or not (args.ramo or args.run):
        if ctx is None:
            ctx = Context()
        meta = {"started": started_iso, "provenance": {
            "active_index": "Hackaton/Semana 1/Limpieza/current_dataset.json", "active_run": ctx.current["workspace_relative_run"], "catalog_sha256": digest(ctx.catalog_path),
            "tables_sha256": ctx.hashes, "bases_run": ctx.bases_index["workspace_relative_run"], "branch_run": ctx.branch_index["workspace_relative_run"],
            "cap01_run": ctx.cap01_index["workspace_relative_run"], "d66_run": ctx.d66_index["workspace_relative_run"]}}
        evidence = finalize(run_dir, meta)
        if not evidence["checks_summary"]["all_passed"]:
            sys.exit(1)


if __name__ == "__main__":
    main()
'''


### Embedded source: `src/datos/n62_codex/rebuild_base.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/datos/n62_codex/rebuild_base.py'] = '''"""Build the frozen eight-rule preaudit base using only eleven raw Parquet inputs.

No legacy run, target catalog, cohort list, model, network or project import is used.
Approved cohort fingerprints are guards, not row selections.
"""
from pathlib import Path
from datetime import datetime,timezone
import argparse,hashlib,json,sys,os,shutil,gc
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

LINES=['motor','buildings','contents']
POLICIES=[p+'_'+l for p in ['policies','test'] for l in LINES]
def utc():return datetime.now(timezone.utc).isoformat(timespec='seconds')
def sha(p):
 h=hashlib.sha256()
 with Path(p).open('rb') as f:
  for b in iter(lambda:f.read(1<<20),b''):h.update(b)
 return h.hexdigest()
def fp(v):return hashlib.sha256(json.dumps(sorted(v),ensure_ascii=True,separators=(',',':')).encode()).hexdigest()
def dump(p,v):Path(p).write_text(json.dumps(v,ensure_ascii=False,indent=2,allow_nan=False)+'\\n',encoding='utf-8')
def replace(t,name,v):
 field=t.schema.field(name)
 return t.set_column(t.schema.get_field_index(name),field,pa.array(v,type=field.type))

def main():
 ap=argparse.ArgumentParser()
 ap.add_argument('--raw',required=True,type=Path);ap.add_argument('--rules',required=True,type=Path);ap.add_argument('--out',required=True,type=Path)
 a=ap.parse_args();raw=a.raw.resolve();rulesfile=a.rules.resolve();out=a.out.resolve()
 assert raw.is_dir() and not out.exists() and raw!=out and not raw.is_relative_to(out) and not out.is_relative_to(raw)
 out.mkdir(parents=True,exist_ok=False);(out/'data').mkdir();(out/'evidence').mkdir()
 reads=[]
 def audit(event,args):
  if event!='open' or not isinstance(args[0],(str,bytes,os.PathLike)):return
  p=Path(os.fsdecode(args[0])).resolve()
  if p.suffix.lower() not in ['.json','.parquet','.csv']:return
  mode=args[1];flags=args[2] if len(args)>2 else 0
  writing=(isinstance(mode,str) and any(x in mode for x in 'wax+')) or (isinstance(flags,int) and bool(flags&(os.O_WRONLY|os.O_RDWR|os.O_CREAT)))
  if not writing:
   if not (p.parent==raw or p==rulesfile or p.is_relative_to(out)):raise RuntimeError('Unexpected data read: '+str(p))
   reads.append(str(p))
 sys.addaudithook(audit)
 rules=json.loads(rulesfile.read_text(encoding='utf-8'))
 expected=rules['originals_sha256']
 assert len(expected)==11 and set(expected)=={p.name for p in raw.glob('*.parquet')}
 before={name:sha(raw/name) for name in sorted(expected)};assert before==expected,'Raw input version changed'
 dump(out/'RUNNING.json',{'utc':utc(),'pid':os.getpid(),'source_sha256':sha(Path(__file__)),'rules_sha256':sha(rulesfile)})
 def load(name,columns=None):
  p=raw/(name+'.parquet');assert p.name in expected
  with p.open('rb') as f:return pq.read_table(f,columns=columns)
 def frame(name,columns=None):return load(name,columns).to_pandas()
 def check_set(v,c,key):
  values=list(v);assert len(values)==len(set(values))==c['count'] and fp(values)==c[key],'Approved cohort fingerprint mismatch'
 stats={};caps={};ren={};first={}
 # CAP01 selection from original, uncorrected household data; no cascading.
 for line in ['buildings','contents']:
  fields=rules['CAP01']['context_fields'][line]
  cols=['row_id','policy_id','period_start','period_end','sum_insured']+fields
  parts=[]
  for p in ['policies','test']:
   f=frame(p+'_'+line,cols);f['source_table']=p+'_'+line;parts.append(f)
  f=pd.concat(parts,ignore_index=True).sort_values(['policy_id','period_start','period_end','row_id']).reset_index(drop=True)
  f['period_start']=pd.to_datetime(f.period_start);f['period_end']=pd.to_datetime(f.period_end)
  extreme=f.sum_insured.gt(rules['CAP01']['thresholds'][line])
  positions=pd.Series(np.where(~extreme,np.arange(len(f)),np.nan),index=f.index)
  previous=positions.groupby(f.policy_id).ffill().groupby(f.policy_id).shift()
  exists=previous.notna();anchor=f.reindex(previous.fillna(-1).astype(int)).reset_index(drop=True)
  known=pd.Series(True,index=f.index);same=pd.Series(True,index=f.index)
  for name in fields:
   x,y=f[name],anchor[name]
   if name=='construction_type':x=x.astype('string').str.casefold().str.strip();y=y.astype('string').str.casefold().str.strip()
   known &= x.notna()&y.notna()&exists;same &= x.eq(y).fillna(False)
  selected=extreme&exists&known&same&(anchor.period_end<f.period_start)&(f.period_start-anchor.period_start).dt.days.between(1,400)&((f.sum_insured/1000)/anchor.sum_insured).between(.9,1.1)
  for name,g in f.loc[selected].groupby('source_table'):
   ids=g.row_id.tolist();check_set(ids,rules['CAP01']['approved_sets'][name],'row_ids_sha256');caps[name]=set(ids)
  print(json.dumps({'rule':'CAP01','line':line,'selected':int(selected.sum()),'cohort_guard':'exact'}),flush=True)
  del parts,f,anchor,positions,previous;gc.collect()
 assert set(caps)==set(rules['CAP01']['approved_sets'])
 dump(out/'evidence/cap01_row_ids.json',{key:sorted(value) for key,value in caps.items()})
 # REN00 membership and earliest train start, separately by line to bound RAM.
 for line in LINES:
  tr=frame('policies_'+line,['row_id','policy_id','period_start','period_end','renewal_decision'])
  first[line]=pd.to_datetime(tr.period_start).groupby(tr.policy_id).min()
  te=frame('test_'+line,['row_id','policy_id','period_start','period_end']);te['renewal_decision']=pd.NA
  f=pd.concat([tr,te],ignore_index=True);f['period_start']=pd.to_datetime(f.period_start);f['period_end']=pd.to_datetime(f.period_end)
  f=f.sort_values(['policy_id','period_start','period_end']).reset_index(drop=True)
  year=f.period_start.dt.year;ny=year.groupby(f.policy_id).shift(-1)
  ids=sorted(f.loc[f.renewal_decision.eq('declined_by_insurer')&year.eq(2024)&ny.eq(2025),'policy_id'].unique().tolist())
  check_set(ids,rules['REN00']['approved_sets'][line],'policy_ids_sha256');ren[line]=set(ids)
  print(json.dumps({'rule':'REN00','line':line,'policies':len(ids),'cohort_guard':'exact'}),flush=True)
  del tr,te,f;gc.collect()
 customers=frame('customers',['customer_id','date_of_birth'])
 master=customers.customer_id.astype('string')
 assert master.is_unique and master.str.fullmatch('[0-9]{8}').all()
 births=customers.set_index('customer_id').date_of_birth
 renfield=pa.field('audit_ren00',pa.bool_(),nullable=False,metadata={
  b'role':b'audit_only',b'rule':b'REN00',b'excluded_from_predictors':b'true',
  b'cohort_sha256':rules['REN00']['cohort_source_file_sha256'].encode(),
  b'meaning':b'Policy membership in frozen 2024-declined/2025-successor cohort; not a row-level decision.'})
 for name in POLICIES:
  t=load(name);line=name.split('_',1)[1]
  f=t.select(['row_id','customer_id','period_start','period_end','exposure','policy_id']).to_pandas()
  ids=f.customer_id.astype('string')
  assert ids.str.fullmatch('[0-9]{1,8}').fillna(False).all()
  ids=ids.str.zfill(8);assert ids.isin(master).all()
  start=pd.to_datetime(f.period_start);end=pd.to_datetime(f.period_end)
  assert start.notna().all() and end.notna().all() and end.ge(start).all()
  exp=((end-start).dt.days.add(1)/rules['EXP01']['denominator']).round(rules['EXP01']['decimals'])
  q={'rows':t.num_rows,'ID01':int(ids.ne(f.customer_id).sum()),'EXP01':int(exp.ne(f.exposure).sum()),'CAP01':0,'CAT01':0,'AGE01':0}
  t=replace(t,'customer_id',ids);t=replace(t,'exposure',exp)
  if name in caps:
   values=t.column('sum_insured').to_pandas().to_numpy(copy=True);mask=f.row_id.isin(caps[name]).to_numpy()
   q['CAP01']=int(mask.sum());values[mask]=values[mask]/rules['CAP01']['scale'];t=replace(t,'sum_insured',values)
  if line!='motor':
   b=pd.to_datetime(ids.map(births));assert b.notna().all() and start.ge(b).all()
   ambiguous=b.dt.month.eq(2)&b.dt.day.eq(29)&~start.dt.is_leap_year&start.dt.month.eq(2)&start.dt.day.eq(28)
   assert not ambiguous.any(),'Leap birthday convention requires review'
   age=start.dt.year-b.dt.year-((start.dt.month<b.dt.month)|((start.dt.month==b.dt.month)&(start.dt.day<b.dt.day))).astype('int64')
   prior=t.column('policyholder_age').to_pandas();assert (age-prior).abs().le(1).all()
   q['AGE01']=int(age.ne(prior).sum());t=replace(t,'policyholder_age',age)
   if line=='buildings':
    prior=t.column('construction_type').to_pandas();cat=prior.replace(rules['CAT01']['map'])
    q['CAT01']=int(cat.ne(prior).sum());t=replace(t,'construction_type',cat)
  mark=pc.is_in(t['policy_id'],value_set=pa.array(sorted(ren[line]),type=t.schema.field('policy_id').type))
  q['REN00']=int(pc.sum(mark).as_py());t=t.append_column(renfield,mark)
  with (out/'data'/f'{name}.parquet').open('wb') as dest:pq.write_table(t,dest,compression='zstd')
  stats[name]=q;print(json.dumps({'table':name,**q}),flush=True);del t,f;gc.collect()
 # DUP01: complete-payload identical pairs; retain short reviewed representative.
 original=load('claims');c=original.to_pandas();assert c.claim_id.is_unique
 payload=[x for x in c.columns if x!='claim_id']
 duplicates=c.loc[c.duplicated(payload,keep=False)]
 pairs=[];excluded=[]
 for values in duplicates.groupby(payload,dropna=False,sort=False,observed=True).claim_id.agg(list):
  assert len(values)==2
  short=[x for x in values if pd.Series([x]).str.fullmatch('CL[0-9]{8}').iloc[0]]
  long=[x for x in values if pd.Series([x]).str.fullmatch('CL[0-9]{9}').iloc[0]]
  assert len(short)==len(long)==1;pairs.append((short[0],long[0]));excluded.append(long[0])
 assert len(pairs)==rules['DUP01']['count'] and fp(pairs)==rules['DUP01']['pairs_sha256']
 assert len(excluded)==len(set(excluded))
 keep=pc.invert(pc.is_in(original['claim_id'],value_set=pa.array(excluded,type=original.schema.field('claim_id').type)))
 t=original.filter(keep);c=t.to_pandas();preids=[]
 for line in LINES:
  z=c.loc[c.line.eq(line)];starts=z.policy_id.map(first[line]);assert starts.notna().all()
  preids+=z.loc[pd.to_datetime(z.loss_date).lt(starts),'claim_id'].tolist()
 check_set(preids,rules['PRE00'],'claim_ids_sha256')
 prefield=pa.field('audit_pre00',pa.bool_(),nullable=False,metadata={
  b'role':b'audit_only',b'rule':b'PRE00',b'excluded_from_predictors':b'true',
  b'excluded_from_current_period_target':b'true',b'cohort_sha256':rules['PRE00']['cohort_source_file_sha256'].encode(),
  b'meaning':b'Frozen pre-first-observed-period claim list; not a coverage or repudiation decision.'})
 marks=pc.is_in(t['claim_id'],value_set=pa.array(sorted(preids),type=t.schema.field('claim_id').type))
 t=t.append_column(prefield,marks)
 with (out/'data/claims.parquet').open('wb') as dest:pq.write_table(t,dest,compression='zstd')
 stats['claims']={'rows_before':original.num_rows,'rows':t.num_rows,'DUP01':len(excluded),'PRE00':int(pc.sum(marks).as_py())}
 del original,t,c;gc.collect()
 for name in sorted(expected):
  stem=Path(name).stem
  if stem not in POLICIES and stem!='claims':
   shutil.copyfile(raw/name,out/'data'/name);stats[stem]={'operation':'byte-copy unchanged original'}
 after={name:sha(raw/name) for name in sorted(expected)};assert before==after
 raw_reads=sorted({p for p in reads if Path(p).parent==raw})
 assert {Path(p).name for p in raw_reads}==set(expected)
 outputs={p.name:sha(p) for p in sorted((out/'data').glob('*.parquet'))};assert set(outputs)==set(expected)
 result={'utc':utc(),'all_builder_guards_passed':True,'tables':stats,'input_sha256':before,'inputs_unchanged':True,
  'output_sha256':outputs,'rules_sha256':sha(rulesfile),'source_sha256':sha(Path(__file__)),
  'raw_read_paths':raw_reads,'data_read_paths':sorted(set(reads)),'reference_data_reads':0,
  'environment':{'python':sys.version,'pandas':pd.__version__,'numpy':np.__version__,'pyarrow':pa.__version__},
  'new_models':0,'Kaggle_calls':0}
 dump(out/'build_receipt.json',result)
 dump(out/'COMPLETE.json',{'utc':utc(),'build_receipt_sha256':sha(out/'build_receipt.json'),'tables':11})
 print('RAW_ONLY_BASE_BUILD_COMPLETE',flush=True)
if __name__=='__main__':main()

'''


## Materialize and run the frozen recipe

I reconstruct the cleaned tables, actuarial bases and F0, then train the 30 S30 stages. The full F0 content and feature-block contract are checked against the original fingerprints, including both reconstructed audit flags. Matching the confirmation flag on this dataset does not recover or independently validate the original audit decisions. The runner records intermediate differences and insists on the historical S30 CSV hash. Its return code 3 means that the final CSV matches while some intermediate hashes differ. I investigate those differences separately; a matching final CSV does not establish equality of every intermediate artifact.

If a run is interrupted, the same run folder can reuse completed nodes only after checking their required hashes. In the same session, rerun the execution cell and inspect the log if the runner stops on partial outputs. Starting a fresh session creates a fresh run folder; retaining or transferring any checkpoint is a separate operation subject to the competition's data rules.


In [ ]:
DATAFILES = {'receta/codigo.json': '{\n  "descripcion": "SHA-256 de cada script de src/. Los de src/modelos son copias exactas de los que se ejecutaron en la nube (/home/claude/modelos_claude); los de src/linea_b, de Modelos_claude_b. Donde hay un hash registrado en un manifiesto de corrida se indica si coincide. Los de src/datos son copias exactas de los scripts de las etapas de datos; los de src/datos/n62_codex, del productor de Codex integrado por hash.",\n  "archivos": {\n    "src/linea_b/mezcla3_b.py": {\n      "sha256": "9e2a1cb62e2056f98b721cd4e8b374558a216583af2f3634a65da2d642f0b98f",\n      "registrado": "9e2a1cb62e2056f98b721cd4e8b374558a216583af2f3634a65da2d642f0b98f",\n      "fuente": "Modelos_claude_b en la carpeta del equipo",\n      "coincide": true\n    },\n    "src/linea_b/nivel_b.py": {\n      "sha256": "21772f1b5ed829ec4f94b89237214f7b4829427637f3f9d651711e8c1ec31356",\n      "registrado": "21772f1b5ed829ec4f94b89237214f7b4829427637f3f9d651711e8c1ec31356",\n      "fuente": "Modelos_claude_b en la carpeta del equipo",\n      "coincide": true\n    },\n    "src/linea_b/reproducir_s30_b.py": {\n      "sha256": "927fe419d2c9898c391daebc9b678de4a4e20e6eb990e6c2c5ef8fefe4fa8c58",\n      "registrado": "927fe419d2c9898c391daebc9b678de4a4e20e6eb990e6c2c5ef8fefe4fa8c58",\n      "fuente": "Modelos_claude_b en la carpeta del equipo",\n      "coincide": true\n    },\n    "src/linea_b/severidad_tendencia.py": {\n      "sha256": "d4adbc26f9225d647f44447ae909aba946600ae5a5e45d2dfb35bd38becddbb0",\n      "registrado": "d4adbc26f9225d647f44447ae909aba946600ae5a5e45d2dfb35bd38becddbb0",\n      "fuente": "Modelos_claude_b en la carpeta del equipo",\n      "coincide": true\n    },\n    "src/modelos/anexar_entrega.py": {\n      "sha256": "5bef5d5cd7d4e044a929cf56fbc04c250d4137ceaffaedccd789109923890212"\n    },\n    "src/modelos/bootstrap_codex.py": {\n      "sha256": "8fb9019bc0775d39e5ea1ef6c8cf11112af8105e99e49fe6dc41e2919622e005"\n    },\n    "src/modelos/calibracion_deciles.py": {\n      "sha256": "2eeea1ca71a5b3ea49f9a6bb64007cab46c312b8d33139fb11c4978316cef7df"\n    },\n    "src/modelos/calibrar_siguiente_anio.py": {\n      "sha256": "e9240d605c7f1b6df9832ff5cf93e04a09f1e1200a15df190bdcb89c70fd8fdd",\n      "registrado": "e9240d605c7f1b6df9832ff5cf93e04a09f1e1200a15df190bdcb89c70fd8fdd",\n      "fuente": "manifest.json de S03_lgbm_F0124 y S04b_lgbm_F0124_F3autos",\n      "coincide": true\n    },\n    "src/modelos/cat_experimento.py": {\n      "sha256": "ba0dcc3b0b743882fd9011010d1ae789eab1d29b34e4b82b368229862d5edaca"\n    },\n    "src/modelos/cat_frecsev.py": {\n      "sha256": "77e153419bedcbaaefb9ec02af15d52d095d4631f9692bf55be3d34c26a6c423"\n    },\n    "src/modelos/cat_produccion.py": {\n      "sha256": "fd0a23f10732d599e3619ef9c46d3898d87b5356a15916525a83a884118923f0"\n    },\n    "src/modelos/combinar_semillas.py": {\n      "sha256": "d42d974e5ff9ece6bb0304253456ff7f4d43e3ecf05f63b503043b7bc0472976"\n    },\n    "src/modelos/comparar_codex.py": {\n      "sha256": "35eb47df917a6f95644737743298b9e851eec14ec67147017b06efd242e793ed"\n    },\n    "src/modelos/comun.py": {\n      "sha256": "7b475c1bf1d2f05b8132657680d23ffe10dc3e925f8a0991a809c5b79432cb84",\n      "registrado": "7b475c1bf1d2f05b8132657680d23ffe10dc3e925f8a0991a809c5b79432cb84",\n      "fuente": "manifest.json de S03_lgbm_F0124 y S04b_lgbm_F0124_F3autos",\n      "coincide": true\n    },\n    "src/modelos/construir_variables.py": {\n      "sha256": "b05681f33296449cd52044c558c5519d5515cfc6b8c01e01cbdc5fc2cf37d12e"\n    },\n    "src/modelos/deriva_nivel.py": {\n      "sha256": "8a7e526b8dd0b96f0ca3c5f8e9ea37d2094c33ade93cb2832adcc88738143f2a"\n    },\n    "src/modelos/diagnostico_cuerpo_cola.py": {\n      "sha256": "d043a0507ef2675e3bb984224c7021cfeba31545b54276030e8f633a67fe7a7a"\n    },\n    "src/modelos/diagnostico_forma_nivel.py": {\n      "sha256": "ad3cb969ca5900fd23994748cb38fae382b2cafc2b67f472ad6e9179674e5637"\n    },\n    "src/modelos/escenarios_reserva.py": {\n      "sha256": "d7f6e96799ad689de79ede8fe996ede3ac2c36dd4a0eb686ba36431475ec8cfa"\n    },\n    "src/modelos/evaluar.py": {\n      "sha256": "a299a74fad47b20e899b8b982afbb9f202ed7c2fdea39ecd9d4d9306242f3995"\n    },\n    "src/modelos/experimentos.py": {\n      "sha256": "d6b1b6cb6f2e81fd4eaf04ad3b395637532e56dd49c912efecad1c480ed3d18e",\n      "registrado": "d6b1b6cb6f2e81fd4eaf04ad3b395637532e56dd49c912efecad1c480ed3d18e",\n      "fuente": "manifest.json de S03_lgbm_F0124 y S04b_lgbm_F0124_F3autos",\n      "coincide": true\n    },\n    "src/modelos/exportar_predicciones.py": {\n      "sha256": "3f758d6d4af10c4800351c7fe9cbbaab1624870b3b7729747da43dc984b7262d"\n    },\n    "src/modelos/frecsev2.py": {\n      "sha256": "530f1e3f78ac925c317e0de3a639ced4d2e558a9607715287200ee06da7c2c5b"\n    },\n    "src/modelos/influencia_reservas.py": {\n      "sha256": "59307b0d7647e11913812b66e16dd6611eb2979e6734bb1277a3b4b0d44834b5"\n    },\n    "src/modelos/intervalos_region.py": {\n      "sha256": "ac6936dc64361a17c9aa3965b77046389d363e229d48cd16d7b625c90d577614"\n    },\n    "src/modelos/mezcla3.py": {\n      "sha256": "b190d24a8725f14d77ef857079570199900dfd0654b03a7ef605a0b3b35f605c"\n    },\n    "src/modelos/mlp_tweedie.py": {\n      "sha256": "0f7628317bc4555e475db59dbc23ae1797b790669d0ea68056001f53725a6ab1"\n    },\n    "src/modelos/produccion.py": {\n      "sha256": "5903804b477766b5d2581bfaed69025eb9c7be313e3963146453352aa122bbc2",\n      "registrado": "128c5854cb6243caa58fe1f0f0a39aaa24c812d724d163b83d52b6b90bdf09c9",\n      "fuente": "manifest.json de S03_lgbm_F0124 y S04b_lgbm_F0124_F3autos (versión anterior; la actual añade opciones y se usó desde S08h)",\n      "coincide": false\n    },\n    "src/modelos/publico_simulado.py": {\n      "sha256": "e41852c6cdb774bceb86020952b9b54b082b136a3c8ef9aa036940331f8102ca"\n    },\n    "src/modelos/rejilla_pesos.py": {\n      "sha256": "6f1ed55a08657e19cee77c57a611274804eacd4836cd6fdd80a1cf611ba56ef5"\n    },\n    "src/modelos/rejilla_robusta.py": {\n      "sha256": "61fd410bc1b80257dc46fc2690e1ccb49e346221412eb13cc5df300bf658ffef"\n    },\n    "src/modelos/reproducir_mezcla.py": {\n      "sha256": "eb98334ca3d50d8d42172ed3ff5653ad03905fc99109b3b492f03658e3b04cdd"\n    },\n    "src/modelos/robustez.py": {\n      "sha256": "1a02f5c34a19437186208c1258e3ba7e5c15a9e73dbaf22b5846c996011da3f5"\n    },\n    "src/modelos/semestres.py": {\n      "sha256": "8c96d271aa9918b0d45232f1ef01b40d860e7ff47f48208bd90d96e0af7b8d91"\n    },\n    "src/modelos/severidad_cfs.py": {\n      "sha256": "0967e3e5a43d1f25f4fae9f34948e4198dede9205e9d482b787d3bbf057fc584"\n    },\n    "src/modelos/severidad_cfs3.py": {\n      "sha256": "d6262f44f603b79b9dabe8d2e1b6c46ab2b6ba590630c0f5c5536028df50eec3"\n    },\n    "src/datos/bases_actuariales.py": {\n      "sha256": "e18fdf34d7332395b0c79ad19ef3e1e26a68dcd54853ac40b62bf2e3669f5193",\n      "registrado": "e18fdf34d7332395b0c79ad19ef3e1e26a68dcd54853ac40b62bf2e3669f5193",\n      "fuente": "copia guardada en Bases_actuariales_claude/runs/bases_20261017T120251Z_45443ff0",\n      "coincide": true\n    },\n    "src/datos/construir_f0.py": {\n      "sha256": "148c0532ae0032ace3b61ba04979003dea2ad974d02b9552bbe315d0a338afcd",\n      "registrado": "148c0532ae0032ace3b61ba04979003dea2ad974d02b9552bbe315d0a338afcd",\n      "fuente": "copia guardada en Matriz_F0_claude/runs/f0_20261017T141732Z_65df7fca",\n      "coincide": true\n    },\n    "src/datos/n62_codex/README.md": {\n      "sha256": "0205239fa866a21e88d18674ee51349708776aa66654916897730ce18402fd9e",\n      "registrado": "0205239fa866a21e88d18674ee51349708776aa66654916897730ce18402fd9e",\n      "fuente": "Modelado_codex/nocturna_20261017_N01/base_rebuild_N62 (Codex, COD-20261018-104)",\n      "coincide": true\n    },\n    "src/datos/n62_codex/compare_preaudit.py": {\n      "sha256": "de08d1b074b6a858b7a7bc4553d764371c1aa58906016a6d5973c27bfe358575",\n      "registrado": "de08d1b074b6a858b7a7bc4553d764371c1aa58906016a6d5973c27bfe358575",\n      "fuente": "Modelado_codex/nocturna_20261017_N01/base_rebuild_N62 (Codex, COD-20261018-104)",\n      "coincide": true\n    },\n    "src/datos/n62_codex/rebuild_base.py": {\n      "sha256": "7054dd7699c9f5868c507ffb834a49ee47ae90392df19928e9c97091ebb93085",\n      "registrado": "7054dd7699c9f5868c507ffb834a49ee47ae90392df19928e9c97091ebb93085",\n      "fuente": "Modelado_codex/nocturna_20261017_N01/base_rebuild_N62 (Codex, COD-20261018-104)",\n      "coincide": true\n    },\n    "src/datos/n62_codex/reference_contract.json": {\n      "sha256": "bb41870d3ca2b56b3f134471d80f2e55298e1c88bcd0db633f29b43e0e2bf641",\n      "registrado": "bb41870d3ca2b56b3f134471d80f2e55298e1c88bcd0db633f29b43e0e2bf641",\n      "fuente": "Modelado_codex/nocturna_20261017_N01/base_rebuild_N62 (Codex, COD-20261018-104)",\n      "coincide": true\n    },\n    "src/datos/n62_codex/requirements-observed.txt": {\n      "sha256": "0780a63ede84349c88a4db7404e7614f6dad59f338c3d21c57930cd1935cb2f0"\n    },\n    "src/datos/n62_codex/rules.json": {\n      "sha256": "dcca7a9dafbc59e0fd5f652620a9c2c9c9c310a632dc14a4791413f4c4163a43",\n      "registrado": "dcca7a9dafbc59e0fd5f652620a9c2c9c9c310a632dc14a4791413f4c4163a43",\n      "fuente": "Modelado_codex/nocturna_20261017_N01/base_rebuild_N62 (Codex, COD-20261018-104)",\n      "coincide": true\n    }\n  },\n  "paquete": {\n    "descripcion": "Carpeta común de entrega. Controles 1.1 preservados; compatibilidad Windows, rutas de entrada resueltas, CSV finales LF antes de escribir, sin modificar modelos o receta.",\n    "version": "1.2",\n    "fecha": "2026-10-19",\n    "archivos": {\n      "reproducir.py": {\n        "sha256": "a346d125520bfc5c784f1b9f65072759efdb15594263485434857e348475de17",\n        "anterior": "6481db2efb2045a5d1907ab0900bf669b084507b38937d780b053859d3c5de19",\n        "cambio": "Portabilidad y comprobación Windows; ninguna modificación del método o de cifras de referencia."\n      },\n      "herramientas/etapas_datos.py": {\n        "sha256": "0c0a90317130f5ee75054ef761edc5c84a52a57eea18652159fd71e056b8b9ab",\n        "anterior": "93751dab88966869806303f4baec3511f8b50993e0550969b216fe4ee4abedfc",\n        "cambio": "controles de datos de la propuesta N65 de Codex (package_data_review_N65/controles_datos_propuestos.patch), integrados en N66: la copia local de cada decisión congelada se comprueba por SHA-256 y las bases, las tablas F0 y los bloques del contrato tienen que coincidir en contenido antes de devolver el informe",\n        "identico_a_N66_salvo_finales_de_linea": true,\n        "sha256_con_crlf": "98c7dd56ed0208a6814cec912ba8d9c864ac4be6f6ce60802045cb5f17b56c13",\n        "fuente": "Modelado_codex/nocturna_20261017_N01/patch_integration_N66/integrated/herramientas/etapas_datos.py (Codex, COD-20261019-109)"\n      },\n      "herramientas/huella.py": {\n        "sha256": "1364d38abb5df33e550507eff881a5d52654426d8ae7c323fd9ee09cc085fcbc",\n        "cambio": "sin cambios"\n      },\n      "herramientas/inventario.py": {\n        "sha256": "ad0eb0b79a886f296e2628381f381f97fa98f21ff2714fb2d2c93a8604fc982b",\n        "cambio": "sin cambios"\n      },\n      "herramientas/probar_controles.py": {\n        "sha256": "241e55b23ff62f4b442c51fcfa98c6525af7fc4bcd08b30e2f0ceb6a24f62bbd",\n        "anterior": "5974197b0f6da2015325a0c2bcc5418446672015713b6569eedb04dda83a4729",\n        "cambio": "Portabilidad y comprobación Windows; ninguna modificación del método o de cifras de referencia."\n      },\n      "receta/huellas_esperadas.json": {\n        "sha256": "b2452d24cb2e491bee46c97b79f08b22612053d5406716ad944b4e0a8489a8f0",\n        "anterior": "fb2ea49a114124eb2808b6759993e9cd7a7c0654c50316741776b79042c80870",\n        "cambio": "añade la sección «mezcla» con las huellas de insumos/exposicion_<ramo>.parquet (Modelos_claude/insumos), que el modo mezcla usa como exposición cuando no hay feat_<ramo>.parquet; las 8 entradas, las 184 salidas y la sección datos no cambian"\n      },\n      "herramientas/ejecutar_csv_lf.py": {\n        "sha256": "0d4e555d122d1efb2d8efa90bf45b16f3ba429f72d53efe4f5e1b794499284cf",\n        "anterior": null,\n        "cambio": "Portabilidad y comprobación Windows; ninguna modificación del método o de cifras de referencia."\n      }\n    }\n  }\n}\n', 'receta/especificaciones/cfg_hogar_s2.json': '{\n "motor": {\n  "params": {\n   "learning_rate": 0.02,\n   "num_leaves": 7,\n   "min_data_in_leaf": 2000,\n   "cat_smooth": 200,\n   "cat_l2": 50,\n   "min_data_per_group": 1000,\n   "max_cat_threshold": 16,\n   "feature_fraction": 0.6,\n   "cap": 15000.0\n  },\n  "rounds": 981\n },\n "buildings": {\n  "params": {\n   "learning_rate": 0.02,\n   "num_leaves": 5,\n   "min_data_in_leaf": 2000,\n   "cat_smooth": 200,\n   "cat_l2": 50,\n   "min_data_per_group": 1000,\n   "max_cat_threshold": 16\n  },\n  "rounds": 334,\n  "add": [\n   "prior_claims_5y",\n   "credit_band",\n   "missed_payments_12m",\n   "household_size",\n   "years_at_address",\n   "marital_status",\n   "licence_points"\n  ]\n },\n "contents": {\n  "params": {\n   "learning_rate": 0.02,\n   "num_leaves": 7,\n   "min_data_in_leaf": 4000,\n   "cat_smooth": 200,\n   "cat_l2": 50,\n   "min_data_per_group": 1000,\n   "max_cat_threshold": 16\n  },\n  "rounds": 322,\n  "add": [\n   "prior_claims_5y",\n   "credit_band",\n   "missed_payments_12m",\n   "household_size",\n   "years_at_address",\n   "marital_status",\n   "licence_points"\n  ]\n }\n}', 'receta/especificaciones/cfg_hogar_s3.json': '{\n "motor": {\n  "params": {\n   "learning_rate": 0.02,\n   "num_leaves": 7,\n   "min_data_in_leaf": 2000,\n   "cat_smooth": 200,\n   "cat_l2": 50,\n   "min_data_per_group": 1000,\n   "max_cat_threshold": 16,\n   "feature_fraction": 0.6,\n   "cap": 15000.0\n  },\n  "rounds": 981\n },\n "buildings": {\n  "params": {\n   "learning_rate": 0.02,\n   "num_leaves": 5,\n   "min_data_in_leaf": 1000,\n   "cat_smooth": 200,\n   "cat_l2": 50,\n   "min_data_per_group": 1000,\n   "max_cat_threshold": 16\n  },\n  "rounds": 395,\n  "add": [\n   "prior_claims_5y",\n   "credit_band",\n   "missed_payments_12m",\n   "household_size",\n   "years_at_address",\n   "marital_status",\n   "licence_points"\n  ]\n },\n "contents": {\n  "params": {\n   "learning_rate": 0.02,\n   "num_leaves": 7,\n   "min_data_in_leaf": 4000,\n   "cat_smooth": 200,\n   "cat_l2": 50,\n   "min_data_per_group": 1000,\n   "max_cat_threshold": 16\n  },\n  "rounds": 322,\n  "add": [\n   "prior_claims_5y",\n   "credit_band",\n   "missed_payments_12m",\n   "household_size",\n   "years_at_address",\n   "marital_status",\n   "licence_points"\n  ]\n }\n}', 'receta/especificaciones/cfg_m595_f3.json': '{\n "motor": {\n  "params": {\n   "learning_rate": 0.02,\n   "num_leaves": 7,\n   "min_data_in_leaf": 2000,\n   "cat_smooth": 200,\n   "cat_l2": 50,\n   "min_data_per_group": 1000,\n   "max_cat_threshold": 16,\n   "feature_fraction": 0.6,\n   "cap": 15000.0\n  },\n  "rounds": 595\n },\n "buildings": {\n  "params": {\n   "learning_rate": 0.02,\n   "num_leaves": 5,\n   "min_data_in_leaf": 2000,\n   "cat_smooth": 200,\n   "cat_l2": 50,\n   "min_data_per_group": 1000,\n   "max_cat_threshold": 16\n  },\n  "rounds": 350\n },\n "contents": {\n  "params": {\n   "learning_rate": 0.02,\n   "num_leaves": 7,\n   "min_data_in_leaf": 4000,\n   "cat_smooth": 200,\n   "cat_l2": 50,\n   "min_data_per_group": 1000,\n   "max_cat_threshold": 16\n  },\n  "rounds": 303\n }\n}', 'receta/especificaciones/cfg_motor_red.json': '{\n "motor": {\n  "params": {\n   "learning_rate": 0.02,\n   "num_leaves": 7,\n   "min_data_in_leaf": 2000,\n   "cat_smooth": 200,\n   "cat_l2": 50,\n   "min_data_per_group": 1000,\n   "max_cat_threshold": 16,\n   "feature_fraction": 0.6,\n   "cap": 15000.0\n  },\n  "rounds": 620,\n  "drop": [\n   "make_model",\n   "previous_insurer",\n   "overnight_parking",\n   "mes_inicio",\n   "wind_exposure_index_prev1",\n   "wildfire_index_mprev",\n   "years_at_address",\n   "energy_rating",\n   "rainfall_mm_mprev",\n   "household_size",\n   "distance_to_river_km",\n   "marital_status",\n   "basement",\n   "days_quote_to_inception",\n   "fuel_type",\n   "branch_id",\n   "periodo_corto",\n   "wind_exposure_index_mprev",\n   "body_type",\n   "email_domain",\n   "flood_zone",\n   "listed_building",\n   "marketing_consent",\n   "roof_type",\n   "soil_type",\n   "compulsory_excess",\n   "f1x_notif_own_damage",\n   "f1x_notif_third_party_prop",\n   "f1x_notif_theft",\n   "f1x_notif_bodily_injury",\n   "solar_panels",\n   "transmission",\n   "driver_assist",\n   "f1x_notif_glass_hail",\n   "freezing_days_mprev",\n   "mean_temperature_c_mprev"\n  ]\n },\n "buildings": {\n  "params": {\n   "learning_rate": 0.02,\n   "num_leaves": 5,\n   "min_data_in_leaf": 2000,\n   "cat_smooth": 200,\n   "cat_l2": 50,\n   "min_data_per_group": 1000,\n   "max_cat_threshold": 16\n  },\n  "rounds": 350\n },\n "contents": {\n  "params": {\n   "learning_rate": 0.02,\n   "num_leaves": 7,\n   "min_data_in_leaf": 4000,\n   "cat_smooth": 200,\n   "cat_l2": 50,\n   "min_data_per_group": 1000,\n   "max_cat_threshold": 16\n  },\n  "rounds": 303\n }\n}', 'receta/especificaciones/cfg_s03.json': '{"motor": {"params": {"learning_rate": 0.02, "num_leaves": 7, "min_data_in_leaf": 2000, "cat_smooth": 200, "cat_l2": 50, "min_data_per_group": 1000, "max_cat_threshold": 16, "feature_fraction": 0.6, "cap": 15000.0}, "rounds": 981},\n "buildings": {"params": {"learning_rate": 0.02, "num_leaves": 5, "min_data_in_leaf": 2000, "cat_smooth": 200, "cat_l2": 50, "min_data_per_group": 1000, "max_cat_threshold": 16}, "rounds": 350},\n "contents": {"params": {"learning_rate": 0.02, "num_leaves": 7, "min_data_in_leaf": 4000, "cat_smooth": 200, "cat_l2": 50, "min_data_per_group": 1000, "max_cat_threshold": 16}, "rounds": 303}}\n', 'receta/especificaciones/cfg_total.json': '{\n "motor": {\n  "params": {\n   "learning_rate": 0.02,\n   "num_leaves": 7,\n   "min_data_in_leaf": 2000,\n   "cat_smooth": 200,\n   "cat_l2": 50,\n   "min_data_per_group": 1000,\n   "max_cat_threshold": 16,\n   "feature_fraction": 0.6,\n   "cap": 15000.0\n  },\n  "rounds": 981\n },\n "buildings": {\n  "params": {\n   "learning_rate": 0.02,\n   "num_leaves": 5,\n   "min_data_in_leaf": 2000,\n   "cat_smooth": 200,\n   "cat_l2": 50,\n   "min_data_per_group": 1000,\n   "max_cat_threshold": 16\n  },\n  "rounds": 350\n },\n "contents": {\n  "params": {\n   "learning_rate": 0.02,\n   "num_leaves": 7,\n   "min_data_in_leaf": 4000,\n   "cat_smooth": 200,\n   "cat_l2": 50,\n   "min_data_per_group": 1000,\n   "max_cat_threshold": 16\n  },\n  "rounds": 303\n }\n}', 'receta/especificaciones/cfg_total_motor_red.json': '{\n "motor": {\n  "params": {\n   "learning_rate": 0.02,\n   "num_leaves": 7,\n   "min_data_in_leaf": 2000,\n   "cat_smooth": 200,\n   "cat_l2": 50,\n   "min_data_per_group": 1000,\n   "max_cat_threshold": 16,\n   "feature_fraction": 0.6,\n   "cap": 15000.0\n  },\n  "rounds": 981,\n  "drop": [\n   "make_model",\n   "previous_insurer",\n   "overnight_parking",\n   "mes_inicio",\n   "wind_exposure_index_prev1",\n   "wildfire_index_mprev",\n   "years_at_address",\n   "energy_rating",\n   "rainfall_mm_mprev",\n   "household_size",\n   "distance_to_river_km",\n   "marital_status",\n   "basement",\n   "days_quote_to_inception",\n   "fuel_type",\n   "branch_id",\n   "periodo_corto",\n   "wind_exposure_index_mprev",\n   "body_type",\n   "email_domain",\n   "flood_zone",\n   "listed_building",\n   "marketing_consent",\n   "roof_type",\n   "soil_type",\n   "compulsory_excess",\n   "f1x_notif_own_damage",\n   "f1x_notif_third_party_prop",\n   "f1x_notif_theft",\n   "f1x_notif_bodily_injury",\n   "solar_panels",\n   "transmission",\n   "driver_assist",\n   "f1x_notif_glass_hail",\n   "freezing_days_mprev",\n   "mean_temperature_c_mprev"\n  ]\n },\n "buildings": {\n  "params": {\n   "learning_rate": 0.02,\n   "num_leaves": 5,\n   "min_data_in_leaf": 2000,\n   "cat_smooth": 200,\n   "cat_l2": 50,\n   "min_data_per_group": 1000,\n   "max_cat_threshold": 16\n  },\n  "rounds": 350\n },\n "contents": {\n  "params": {\n   "learning_rate": 0.02,\n   "num_leaves": 7,\n   "min_data_in_leaf": 4000,\n   "cat_smooth": 200,\n   "cat_l2": 50,\n   "min_data_per_group": 1000,\n   "max_cat_threshold": 16\n  },\n  "rounds": 303\n }\n}', 'receta/especificaciones/spec_s22_propio.json': '{\n "motor": [\n  [\n   "CAT_motor_4s",\n   "cat1a",\n   0.2\n  ],\n  [\n   "CFSS_motor",\n   "CFSS_motor",\n   0.6\n  ],\n  [\n   "S04b_lgbm_F0124_F3autos",\n   "total_F01234",\n   0.1\n  ],\n  [\n   "S06c_lgbm_motor_red",\n   "total_motor_red",\n   0.1\n  ]\n ],\n "buildings": [\n  [\n   "S12h_edif_md1000",\n   "hS3",\n   0.4\n  ],\n  [\n   "CFSS4_buildings",\n   "CFSS4_buildings",\n   0.3\n  ],\n  [\n   "FSAS4_buildings",\n   "FSAS4_buildings",\n   0.3\n  ]\n ],\n "contents": [\n  [\n   "S08h_hogar_F3s2",\n   "hS2",\n   0.2\n  ],\n  [\n   "MLP3_contents",\n   "MLP3_contents",\n   0.1\n  ],\n  [\n   "FSA_contents",\n   "FSA_contents",\n   0.2\n  ],\n  [\n   "FSAXc4_contents",\n   "FSAXc4_contents",\n   0.5\n  ]\n ]\n}', 'receta/especificaciones/spec_s30.json': '{\n "motor": [\n  [\n   "CAT_motor_4s",\n   "cat1a",\n   0.2\n  ],\n  [\n   "B_CFST_motor_tendencia",\n   "B_CFST_motor_tendencia",\n   0.6\n  ],\n  [\n   "S04b_lgbm_F0124_F3autos",\n   "total_F01234",\n   0.1\n  ],\n  [\n   "S06c_lgbm_motor_red",\n   "total_motor_red",\n   0.1\n  ]\n ],\n "buildings": [\n  [\n   "S12h_edif_md1000",\n   "hS3",\n   0.4\n  ],\n  [\n   "B_CFST_buildings_tendencia",\n   "B_CFST_buildings_tendencia",\n   0.3\n  ],\n  [\n   "B_FSAST_buildings_tendencia",\n   "B_FSAST_buildings_tendencia",\n   0.3\n  ]\n ],\n "contents": [\n  [\n   "S08h_hogar_F3s2",\n   "hS2",\n   0.3\n  ],\n  [\n   "FSA_contents",\n   "FSA_contents",\n   0.3\n  ],\n  [\n   "FSAXc4_contents",\n   "FSAXc4_contents",\n   0.4\n  ]\n ]\n}', 'receta/huellas_esperadas.json': '{\n "creado_utc": "2026-10-19T02:14:08+00:00",\n "origenes": [\n  "/home/claude/work",\n  "/mnt/user-data/uploads/Hakaton/Hackaton/Semana 1/Modelos_claude_b"\n ],\n "semana1": "/mnt/user-data/uploads/Hakaton/Hackaton/Semana 1",\n "entradas": {\n  "Datos/weather.parquet": {\n   "bytes": 41863008,\n   "sha256": "b8407f0f3c7279a49297c969a4ab2905641b232910e316fb806730065987cd49",\n   "contenido": "f53977e7ef041173e620734d70592414e9a3e84cea453b291da9657289c2c214",\n   "origen": "/mnt/user-data/uploads/Hakaton/Hackaton/Semana 1/Datos/weather.parquet"\n  },\n  "Datos/sample_submission.parquet": {\n   "bytes": 3271617,\n   "sha256": "9529f3b6a679d3410350baa7c2ef3fb085d1d8843a838360c5852dfe820d4d95",\n   "contenido": "f94c1d55d824842b510af78f8f431a512ef4d4c0671ce82a8100ee54ee0bd32e",\n   "origen": "/mnt/user-data/uploads/Hakaton/Hackaton/Semana 1/Datos/sample_submission.parquet"\n  },\n  "Matriz_F0_claude/runs/f0_20261017T141732Z_65df7fca/tables/f0_motor.parquet": {\n   "bytes": 98141422,\n   "sha256": "c503d0142e83818b2c202747ddbce53882ed9add6f8309ced714655caf25b392",\n   "contenido": "20f1a2ecfb2c012244ad8c4222ec8503e98f2efcc021c40cf1089fcc70afbbe6",\n   "origen": "/mnt/user-data/uploads/Hakaton/Hackaton/Semana 1/Matriz_F0_claude/runs/f0_20261017T141732Z_65df7fca/tables/f0_motor.parquet"\n  },\n  "Matriz_F0_claude/runs/f0_20261017T141732Z_65df7fca/tables/f0_buildings.parquet": {\n   "bytes": 66919547,\n   "sha256": "2def2e2433062704e3762eae63f5b000e73d8284a62a6cc1a689aa63c2842454",\n   "contenido": "1506da562e28d3f28a9338b93c25424bfbdbe68c0efa1010867743977b0a4362",\n   "origen": "/mnt/user-data/uploads/Hakaton/Hackaton/Semana 1/Matriz_F0_claude/runs/f0_20261017T141732Z_65df7fca/tables/f0_buildings.parquet"\n  },\n  "Matriz_F0_claude/runs/f0_20261017T141732Z_65df7fca/tables/f0_contents.parquet": {\n   "bytes": 65668739,\n   "sha256": "42328ddba499a77f9cc3c814506dc97fde7c3e2adf88cfc491058c7de92c76ff",\n   "contenido": "7b06b565833323090adcff185f2b5ab869b61936df75850acb69b80fe420c190",\n   "origen": "/mnt/user-data/uploads/Hakaton/Hackaton/Semana 1/Matriz_F0_claude/runs/f0_20261017T141732Z_65df7fca/tables/f0_contents.parquet"\n  },\n  "Matriz_F0_claude/runs/f0_20261017T141732Z_65df7fca/tables/particiones.parquet": {\n   "bytes": 15848536,\n   "sha256": "d365981d07feec50e487ff0d8fa4b9e5b8a00d5af8f8e521e03f2a158931035b",\n   "contenido": "0987960a3498983daa25bbd27bd4258e3e4dbfbf372bedb33ccaa02bba6c4105",\n   "origen": "/mnt/user-data/uploads/Hakaton/Hackaton/Semana 1/Matriz_F0_claude/runs/f0_20261017T141732Z_65df7fca/tables/particiones.parquet"\n  },\n  "Matriz_F0_claude/runs/f0_20261017T141732Z_65df7fca/contrato_disponibilidad.json": {\n   "bytes": 54543,\n   "sha256": "7e00fb2bb9efba56837ca3e6b04c016ae3ca1a8321ed6d7786eba11c6e2c541d",\n   "contenido": "388a625923b898238b01893d3b0b3ae83dda68927ebeec57b6eca839d1ec1c0a",\n   "origen": "/mnt/user-data/uploads/Hakaton/Hackaton/Semana 1/Matriz_F0_claude/runs/f0_20261017T141732Z_65df7fca/contrato_disponibilidad.json"\n  },\n  "Bases_actuariales_claude/runs/bases_20261017T120251Z_45443ff0/tables/asignacion_por_siniestro.parquet": {\n   "bytes": 4805740,\n   "sha256": "535327e1bbba95f171820d27e345988b3c2cbdeec6f86eefefaedefc52992ae5",\n   "contenido": "f760aed27500d172209434c89967949ded6e324b7ecd8a414c34dceb065f2ed3",\n   "origen": "/mnt/user-data/uploads/Hakaton/Hackaton/Semana 1/Bases_actuariales_claude/runs/bases_20261017T120251Z_45443ff0/tables/asignacion_por_siniestro.parquet"\n  }\n },\n "salidas": {\n  "feat_motor.parquet": {\n   "bytes": 126378469,\n   "sha256": "1941042d83ae7ba5609627a8dd12201351871f79659b428f29f74d17410768c5",\n   "contenido": "c4a34c2725944199bb9c928d5a5bdd8d53b965c04e6b078e885065e66ccc8359",\n   "origen": "/home/claude/work/feat_motor.parquet",\n   "nodo": "variables"\n  },\n  "feat_buildings.parquet": {\n   "bytes": 85622382,\n   "sha256": "f0809330d8bd828d22eec686d0ceda30d8c855be2dbd43f173406d13e0d8efd4",\n   "contenido": "ca33fab819676987a2a486bed3c97e3182c677da041a51a5078dfce22e94d745",\n   "origen": "/home/claude/work/feat_buildings.parquet",\n   "nodo": "variables"\n  },\n  "feat_contents.parquet": {\n   "bytes": 84020097,\n   "sha256": "82334958a4e2d9bce76c9833d16df1a154403f8822e65dfea6f5a0f761b53a8f",\n   "contenido": "6dfaa65541341ebd054267896a39344befe7cda4b0a5b77ff22d583751f6023a",\n   "origen": "/home/claude/work/feat_contents.parquet",\n   "nodo": "variables"\n  },\n  "sel_total_F0124.parquet": {\n   "bytes": 4649229,\n   "sha256": "f5c512a4469ce73a368c4f86b1092f6c6f6888ce73bc7e25d9446177905690a3",\n   "contenido": "9d696c5bb25d63147c4df749e091fc0f94df5f459d57c7110b2c5227df66fd63",\n   "origen": "/home/claude/work/sel_total_F0124.parquet",\n   "nodo": "sel_total_F0124"\n  },\n  "sel_total_F01234.parquet": {\n   "bytes": 4649229,\n   "sha256": "1fd58048eeeed3dcb7f5719ac35f2c61552254806cb71e9046c53f3d84f2b1c4",\n   "contenido": "f04b5251e4034a1fe2587a338a00fa112f5599ac315ed36de083af936258da07",\n   "origen": "/home/claude/work/sel_total_F01234.parquet",\n   "nodo": "sel_total_F01234"\n  },\n  "sel_total_motor_red.parquet": {\n   "bytes": 4649229,\n   "sha256": "23c0cea11720a4a7f01e71ac0c78cf687bf110dc51033dd16444674a2c01f407",\n   "contenido": "02fe38788ae79e54b46573381aa1b5028c63e1aee329e104292429f3913cdefe",\n   "origen": "/home/claude/work/sel_total_motor_red.parquet",\n   "nodo": "sel_total_motor_red"\n  },\n  "sel_hS2.parquet": {\n   "bytes": 2893890,\n   "sha256": "e1cde41f5e1cbc88958a59d380c39206bb935865a80e24210e358057ef0f79f4",\n   "contenido": "9d3116fc3efbf1c9633c98d7ef2bdc53a4b61e47fd6b2be634d69091d7778008",\n   "origen": "/home/claude/work/sel_hS2.parquet",\n   "nodo": "sel_hS2"\n  },\n  "sel_hS3.parquet": {\n   "bytes": 1574274,\n   "sha256": "b707d8aa2336323c58cfe2b104d3694f38a5b9f48cd095cbc2ce9b2a60c199b1",\n   "contenido": "35aa73651e11b154a6aabc3392e74911af9a490cb7e3abf1e25fd1353f32638d",\n   "origen": "/home/claude/work/sel_hS3.parquet",\n   "nodo": "sel_hS3"\n  },\n  "sel_cat1a.parquet": {\n   "bytes": 2184468,\n   "sha256": "b93d1920cdb06ca2e7f3248f39bfed1fd832422a944ab22af5fc9641c8c10916",\n   "contenido": "2be5c9b5994e32f6bcede94413c65245c4a24d40a94013fe2ab273609ecdb71b",\n   "origen": "/home/claude/work/sel_cat1a.parquet",\n   "nodo": "sel_cat1a"\n  },\n  "S03_lgbm_F0124/v24.parquet": {\n   "bytes": 8878138,\n   "sha256": "1e7e184e95807462ab37443f397bbefba8334d737c3a7853107cce7b55fb2ac8",\n   "contenido": "6c4fff5cfcce7b7ba3253959764ca91de148e926f86ea36efbb9dd490fce8b47",\n   "origen": "/home/claude/work/S03_lgbm_F0124/v24.parquet",\n   "nodo": "S03_lgbm_F0124"\n  },\n  "S03_lgbm_F0124/prediccion_2025.parquet": {\n   "bytes": 8557976,\n   "sha256": "53723ac0c3e49d9818b0451d3ed80b2e1ada3aecd40a3edfdd3f1b171d546556",\n   "contenido": "513660c46ad7c17fbfc47c9b10d768240a5260e09ec327aa53050bb8e3b6f3ac",\n   "origen": "/home/claude/work/S03_lgbm_F0124/prediccion_2025.parquet",\n   "nodo": "S03_lgbm_F0124"\n  },\n  "S03_lgbm_F0124/S03_lgbm_F0124.csv": {\n   "bytes": 12995807,\n   "sha256": "7877e9e968dfc094df6b9450f51ff1d5f97f12fa82b4d985e2fce18d6af93217",\n   "contenido": "7877e9e968dfc094df6b9450f51ff1d5f97f12fa82b4d985e2fce18d6af93217",\n   "origen": "/home/claude/work/S03_lgbm_F0124/S03_lgbm_F0124.csv",\n   "nodo": "S03_lgbm_F0124"\n  },\n  "S04b_lgbm_F0124_F3autos/v24.parquet": {\n   "bytes": 8878138,\n   "sha256": "acfb4eec8ac56d1f3a97d6a0a09e32f79681fd6379e0f3e86e495d9f5e30fb37",\n   "contenido": "b1882aa092b5e9d13b693c1e2424ae3e64ebf373e0662b387b5ada89303b82f8",\n   "origen": "/home/claude/work/S04b_lgbm_F0124_F3autos/v24.parquet",\n   "nodo": "S04b_lgbm_F0124_F3autos"\n  },\n  "S04b_lgbm_F0124_F3autos/prediccion_2025.parquet": {\n   "bytes": 8557976,\n   "sha256": "264ce035ffdd1c07ebcb8c3227aa6f4f2563e6b759c1b7a7c45d2581eafd4564",\n   "contenido": "08b40bd4ef1a0cb624d5d35f4af1cd646ae3dfe6ecd38e3b887d75985c349eb5",\n   "origen": "/home/claude/work/S04b_lgbm_F0124_F3autos/prediccion_2025.parquet",\n   "nodo": "S04b_lgbm_F0124_F3autos"\n  },\n  "S04b_lgbm_F0124_F3autos/S04b_lgbm_F0124_F3autos.csv": {\n   "bytes": 12996206,\n   "sha256": "23e09d9f27326d7a20d508c32b4da039bac43085c1726498618f668d1eaba1fe",\n   "contenido": "23e09d9f27326d7a20d508c32b4da039bac43085c1726498618f668d1eaba1fe",\n   "origen": "/home/claude/work/S04b_lgbm_F0124_F3autos/S04b_lgbm_F0124_F3autos.csv",\n   "nodo": "S04b_lgbm_F0124_F3autos"\n  },\n  "S06c_lgbm_motor_red/v24.parquet": {\n   "bytes": 8878138,\n   "sha256": "eaba43ef4455670133e763c6dddb6648acd9f864ad7e93f4c57ce9b8432825bf",\n   "contenido": "9e4835764d7a16c4d7afb6a4ccff5a6355f2a97f55629a750504ca36e4019393",\n   "origen": "/home/claude/work/S06c_lgbm_motor_red/v24.parquet",\n   "nodo": "S06c_lgbm_motor_red"\n  },\n  "S06c_lgbm_motor_red/prediccion_2025.parquet": {\n   "bytes": 8557976,\n   "sha256": "4a49db207b76127f3bf91c9bcd439e2bd90fab6a0eb79529d081604efb04b7ab",\n   "contenido": "ce20bc09337efb20e6b075f065a076be1ad1cef323e5382b3b66f05e45bbcb48",\n   "origen": "/home/claude/work/S06c_lgbm_motor_red/prediccion_2025.parquet",\n   "nodo": "S06c_lgbm_motor_red"\n  },\n  "S06c_lgbm_motor_red/S06c_lgbm_motor_red.csv": {\n   "bytes": 12997557,\n   "sha256": "8624c2331ab5d9eeed7ee6d6331528c1e7bf5c0bee2b1b02a9e421fb83d4a68e",\n   "contenido": "8624c2331ab5d9eeed7ee6d6331528c1e7bf5c0bee2b1b02a9e421fb83d4a68e",\n   "origen": "/home/claude/work/S06c_lgbm_motor_red/S06c_lgbm_motor_red.csv",\n   "nodo": "S06c_lgbm_motor_red"\n  },\n  "S08h_hogar_F3s2/v24.parquet": {\n   "bytes": 8878138,\n   "sha256": "cc604c6e645e27c525528b2de2c02138ccec56ddec186d7cd566edfb6020eedb",\n   "contenido": "40c627d97edc9c82650bd4c03d0a7bf1af9b9244790f118f8a7a121ee51e983e",\n   "origen": "/home/claude/work/S08h_hogar_F3s2/v24.parquet",\n   "nodo": "S08h_hogar_F3s2"\n  },\n  "S08h_hogar_F3s2/prediccion_2025.parquet": {\n   "bytes": 8557976,\n   "sha256": "f91932f301b817308c50a48bd5bf6297dd62c711fe815f9915dd4fa3d93e2cb2",\n   "contenido": "578cad2b2004e9777a05bd7d78e83095063f1f2102effb32f0bf5e54fc6b21dd",\n   "origen": "/home/claude/work/S08h_hogar_F3s2/prediccion_2025.parquet",\n   "nodo": "S08h_hogar_F3s2"\n  },\n  "S08h_hogar_F3s2/S08h_hogar_F3s2.csv": {\n   "bytes": 12995502,\n   "sha256": "9623c7b7585248e92a717643464a87d5610153ed77b86c4ca344087df1848ac7",\n   "contenido": "9623c7b7585248e92a717643464a87d5610153ed77b86c4ca344087df1848ac7",\n   "origen": "/home/claude/work/S08h_hogar_F3s2/S08h_hogar_F3s2.csv",\n   "nodo": "S08h_hogar_F3s2"\n  },\n  "S12h_edif_md1000/v24.parquet": {\n   "bytes": 8878138,\n   "sha256": "d535388f050b6893cd35d89ccc3c966b7f43e186a13841af3de1b885f37c4c3e",\n   "contenido": "b777023647e18780004936d4970645208d51a3d2c85a6d8618814d537730f5cd",\n   "origen": "/home/claude/work/S12h_edif_md1000/v24.parquet",\n   "nodo": "S12h_edif_md1000"\n  },\n  "S12h_edif_md1000/prediccion_2025.parquet": {\n   "bytes": 8557976,\n   "sha256": "70a3b8b61c40477c9f44725350aabe6adbe83ffc5498c2ea8ef564fd2cf6e663",\n   "contenido": "4716e9466d537e334279d5aaeac6ca28da4d6a81bc688d9fb6eab995f86fab60",\n   "origen": "/home/claude/work/S12h_edif_md1000/prediccion_2025.parquet",\n   "nodo": "S12h_edif_md1000"\n  },\n  "S12h_edif_md1000/S12h_edif_md1000.csv": {\n   "bytes": 12994639,\n   "sha256": "2ca725c0569daf79abf56516278afc8af8ade2ed9f9116662690767a380bdc19",\n   "contenido": "2ca725c0569daf79abf56516278afc8af8ade2ed9f9116662690767a380bdc19",\n   "origen": "/home/claude/work/S12h_edif_md1000/S12h_edif_md1000.csv",\n   "nodo": "S12h_edif_md1000"\n  },\n  "CAT_motor/v24.parquet": {\n   "bytes": 3958638,\n   "sha256": "ee475f23e30ac89f6410ce2f8a19424a505fca758da7a0c33271172abe9a902a",\n   "contenido": "e11fb2fd15be309b7c2d028e534dae6030aba40f5263ed6328712123b4bacb37",\n   "origen": "/home/claude/work/CAT_motor/v24.parquet",\n   "nodo": "CAT_motor"\n  },\n  "CAT_motor/prediccion_2025.parquet": {\n   "bytes": 3739277,\n   "sha256": "4e197e05938aa70e23c934909796d9392c7df46d5e6c045be444f59245a21cc9",\n   "contenido": "99a8bee3434d4be9f95937fe656f7181c5726a13906ccef1134f42bd577181dc",\n   "origen": "/home/claude/work/CAT_motor/prediccion_2025.parquet",\n   "nodo": "CAT_motor"\n  },\n  "CAT_motor_s23/v24.parquet": {\n   "bytes": 3958636,\n   "sha256": "3895cf1b73442c532da5172401de53db6f93ec2c596e8a3a5f746e770850b6d7",\n   "contenido": "c2ac3b0329020c4c81fb91d04d79593c001bacb7ede724795682268d0f4a8597",\n   "origen": "/home/claude/work/CAT_motor_s23/v24.parquet",\n   "nodo": "CAT_motor_s23"\n  },\n  "CAT_motor_s23/prediccion_2025.parquet": {\n   "bytes": 3739277,\n   "sha256": "ae5ee520b2f83d873bf9708629a536b3d1f44979ee6656a273a331d0a39470a4",\n   "contenido": "4d2fc235c404f224c71f33d86d285b5ae9631dfd57010d7feb4683fa3f816918",\n   "origen": "/home/claude/work/CAT_motor_s23/prediccion_2025.parquet",\n   "nodo": "CAT_motor_s23"\n  },\n  "CAT_motor_4s/v24.parquet": {\n   "bytes": 3958636,\n   "sha256": "62274dd4fab69a5ebd465da88315450414fadaf585013567b8ca44d8bc44ed9c",\n   "contenido": "ce3a64c582d95c8685c6210b1c1f27533a4f405be58964f3061d71db87c8e2da",\n   "origen": "/home/claude/work/CAT_motor_4s/v24.parquet",\n   "nodo": "CAT_motor_4s"\n  },\n  "CAT_motor_4s/prediccion_2025.parquet": {\n   "bytes": 3739277,\n   "sha256": "85b97f8c006497cf46dca694dd24f4bc537b28154a6d3a6042568884ef923c98",\n   "contenido": "a53ce47dd9c1c5248d5ff573d3956a929b054a0027aefd2f068565584e903c3b",\n   "origen": "/home/claude/work/CAT_motor_4s/prediccion_2025.parquet",\n   "nodo": "CAT_motor_4s"\n  },\n  "CFS_motor_v23/v23_own_damage.npz": {\n   "bytes": 1019396,\n   "sha256": "a86234806dee320a21ce6b6a574b913af61dc5f108fbf85e21f5f1c0491aec46",\n   "contenido": "9425f95af7a7cea889c32314a26c20d58c18c35f51be0af11fda06a246eca639",\n   "origen": "/home/claude/work/CFS_motor_v23/v23_own_damage.npz",\n   "nodo": "CFS_motor_v23"\n  },\n  "CFS_motor_v23/v23_third_party_prop.npz": {\n   "bytes": 1019396,\n   "sha256": "cd33ea937a6053ab457c6a17a34e0e9d6cf8726f780bd7c083ca4bb9ec2cb12a",\n   "contenido": "8cc4f0034db4333f4a8c9a6e0000cd80b9a576e3016a7120588a4ba0b73bdbb6",\n   "origen": "/home/claude/work/CFS_motor_v23/v23_third_party_prop.npz",\n   "nodo": "CFS_motor_v23"\n  },\n  "CFS_motor_v23/v23_bodily_injury.npz": {\n   "bytes": 1019396,\n   "sha256": "b90099af6169d02c3bf02424f6dc9fda942f2f87fec89a6eaeee48ea7d20de7a",\n   "contenido": "ea21503a8f0dc5ff308f814e32603fb0222c8b61028f689638cf255e99f38cc3",\n   "origen": "/home/claude/work/CFS_motor_v23/v23_bodily_injury.npz",\n   "nodo": "CFS_motor_v23"\n  },\n  "CFS_motor_v23/v23_theft.npz": {\n   "bytes": 1019396,\n   "sha256": "f288a47025a9895e639129d2db271b29ecd2a88e888d6012a8cfb2f693098da5",\n   "contenido": "fa956280856087080425e43adff7089398ce2858c26e43b619fb6cd210f41add",\n   "origen": "/home/claude/work/CFS_motor_v23/v23_theft.npz",\n   "nodo": "CFS_motor_v23"\n  },\n  "CFS_motor_v23/v23_glass_hail.npz": {\n   "bytes": 1019396,\n   "sha256": "411b4dadf36d4b4056a17fda53156a0b9eafe497199edaaec15107502d4a726f",\n   "contenido": "68c2e89ba917c93bc72f3666c5f7893fb45a4c5df1147f36713dc7cc14a30fe6",\n   "origen": "/home/claude/work/CFS_motor_v23/v23_glass_hail.npz",\n   "nodo": "CFS_motor_v23"\n  },\n  "sel_CFS_motor_v23.parquet": {\n   "bytes": 2184468,\n   "sha256": "900ab1601fc2ac25ae6f983ee21faa8596acf95e8c23dd4bec7d69945119fb15",\n   "contenido": "19872c9181f376b9e5f06e41ade0aee9f321790a0fc0f219139bd52a2aab1b35",\n   "origen": "/home/claude/work/sel_CFS_motor_v23.parquet",\n   "nodo": "CFS_motor_v23"\n  },\n  "CFS_motor/prod_own_damage.npy": {\n   "bytes": 2040016,\n   "sha256": "a3b99f874b3d52f644c5b82da96d212bf636ee6069be651ac125004f836a2517",\n   "contenido": "d047b991d0bf126b600f2e4d987083571f61f3e500a81a51b8fc8cb5da684425",\n   "origen": "/home/claude/work/CFS_motor/prod_own_damage.npy",\n   "nodo": "CFS_motor"\n  },\n  "CFS_motor/prod_third_party_prop.npy": {\n   "bytes": 2040016,\n   "sha256": "690a4c3bcec92d55bd861b4da095c0cbd370e227494ecbf9b5e8f4a8a8707259",\n   "contenido": "dea39f8e7a118c3736b19412d5b24e6b103ccebc70f2aad86a9493d0a348065c",\n   "origen": "/home/claude/work/CFS_motor/prod_third_party_prop.npy",\n   "nodo": "CFS_motor"\n  },\n  "CFS_motor/prod_bodily_injury.npy": {\n   "bytes": 2040016,\n   "sha256": "67d5aa89fa68feefe1646db753380940225c7fb26fe7852d1f93f9fd16e0a1ac",\n   "contenido": "0f4bd15add1ff1be34938900abb19eef53771612ac968bbe72e50302a301cb1c",\n   "origen": "/home/claude/work/CFS_motor/prod_bodily_injury.npy",\n   "nodo": "CFS_motor"\n  },\n  "CFS_motor/prod_theft.npy": {\n   "bytes": 2040016,\n   "sha256": "07f82c8865581daf90e7084b7a5876856f6404b683c9ebcdc06e8528a6dec3a8",\n   "contenido": "0b529132474385d25c402a7a6f62ce1f3b8ca66a2d3e5525fd0dd864dd9eb9bc",\n   "origen": "/home/claude/work/CFS_motor/prod_theft.npy",\n   "nodo": "CFS_motor"\n  },\n  "CFS_motor/prod_glass_hail.npy": {\n   "bytes": 2040016,\n   "sha256": "9387d36f3bccd89f661bde91f10859bb56e702d5d17ba4a39c2eb3acc5cc877f",\n   "contenido": "2f1e1ea5522131b6506209a692913a4c25f0497fdc089e73f2dd71b23c1cc6df",\n   "origen": "/home/claude/work/CFS_motor/prod_glass_hail.npy",\n   "nodo": "CFS_motor"\n  },\n  "sel_CFS_motor.parquet": {\n   "bytes": 2184468,\n   "sha256": "900ab1601fc2ac25ae6f983ee21faa8596acf95e8c23dd4bec7d69945119fb15",\n   "contenido": "19872c9181f376b9e5f06e41ade0aee9f321790a0fc0f219139bd52a2aab1b35",\n   "origen": "/home/claude/work/sel_CFS_motor.parquet",\n   "nodo": "CFS_motor"\n  },\n  "CFS_motor/v24.parquet": {\n   "bytes": 3958636,\n   "sha256": "cba09ca9072764aa37c89429db73a4639e227eb312b914136d7deae48bad3f3e",\n   "contenido": "74ac7d827138db3f62c99d14e9a23b133db6d9f8b465b7513f43598655ae2fc0",\n   "origen": "/home/claude/work/CFS_motor/v24.parquet",\n   "nodo": "CFS_motor"\n  },\n  "CFS_motor/prediccion_2025.parquet": {\n   "bytes": 3739277,\n   "sha256": "4d17abc843b341c27dfc4629d93bf07a874f6e02aa88016774476e5899791ee2",\n   "contenido": "996cc87eeedfe208b568cf75e6646b27790170ed80cf82878b5463a32f23ce79",\n   "origen": "/home/claude/work/CFS_motor/prediccion_2025.parquet",\n   "nodo": "CFS_motor"\n  },\n  "CFS_motor_s23/v24_own_damage.npy": {\n   "bytes": 2037712,\n   "sha256": "4d7afe91ca0b9870e42d8229a88116163ef8e97f210763df77fcad76ea93304b",\n   "contenido": "a5ef6ce5af02ad239f84441cf1ecc78ca03ece4fd7e013fd01b0d8abab2eb7fa",\n   "origen": "/home/claude/work/CFS_motor_s23/v24_own_damage.npy",\n   "nodo": "CFS_motor_s23"\n  },\n  "CFS_motor_s23/v24_third_party_prop.npy": {\n   "bytes": 2037712,\n   "sha256": "3b115c00c5c4793dffa7c6c0a25c9db979c0fecd22e419011faef6123f9b5856",\n   "contenido": "4e4ee5ee93b23cc2cbda594888f1f511c8937c6536f16eb8be88fefac7e2957c",\n   "origen": "/home/claude/work/CFS_motor_s23/v24_third_party_prop.npy",\n   "nodo": "CFS_motor_s23"\n  },\n  "CFS_motor_s23/v24_bodily_injury.npy": {\n   "bytes": 2037712,\n   "sha256": "a9317779bb6fed4f5989329b385a533facd41b9f5bd61e7d1e886a79e5af0fdb",\n   "contenido": "202fcb4d21542c7dc2b3b8832a57dba641de6ff84c18d54b79aad6d67571338b",\n   "origen": "/home/claude/work/CFS_motor_s23/v24_bodily_injury.npy",\n   "nodo": "CFS_motor_s23"\n  },\n  "CFS_motor_s23/v24_theft.npy": {\n   "bytes": 2037712,\n   "sha256": "fb20dd3bffd53ff99c3f51b1691072e4414d551e844b92afa146a2b2f28f73b1",\n   "contenido": "518fd4c60b0d9c684813dc65aa79095658c424e1afa4a1415edf7b752b14df7e",\n   "origen": "/home/claude/work/CFS_motor_s23/v24_theft.npy",\n   "nodo": "CFS_motor_s23"\n  },\n  "CFS_motor_s23/v24_glass_hail.npy": {\n   "bytes": 2037712,\n   "sha256": "ea815c321fdddc24049fd47e3bbc326b91a7585fb3baf6fc1f6464963bd80f9a",\n   "contenido": "d4d2a93ef4da9b2165b5ed937c1b1d862b6a57000d90405b707428008387b818",\n   "origen": "/home/claude/work/CFS_motor_s23/v24_glass_hail.npy",\n   "nodo": "CFS_motor_s23"\n  },\n  "CFS_motor_s23/prod_own_damage.npy": {\n   "bytes": 2040016,\n   "sha256": "81d33d4c78c5d22450fb9d636ee51dd921e67b0524dee1a1a45567811a0aa568",\n   "contenido": "a8d6b95778f2f03d28d6fc5bfd4b5ebd71cfa69e1bb9ea10ba9fcfbf93ba2c89",\n   "origen": "/home/claude/work/CFS_motor_s23/prod_own_damage.npy",\n   "nodo": "CFS_motor_s23"\n  },\n  "CFS_motor_s23/prod_third_party_prop.npy": {\n   "bytes": 2040016,\n   "sha256": "30cb12405c05c5789b0e63e2b2f425cbfc814004deb3f0b22c3a1fb9dc2d2e0f",\n   "contenido": "d5e41e24cfbeec1f725ce54721614d5bb0782aac03252eef666ace2745dd9d34",\n   "origen": "/home/claude/work/CFS_motor_s23/prod_third_party_prop.npy",\n   "nodo": "CFS_motor_s23"\n  },\n  "CFS_motor_s23/prod_bodily_injury.npy": {\n   "bytes": 2040016,\n   "sha256": "15b2bea45299746e98e08ba93074913d43b8d18b5eda58a8cf4c255fb0bcdfec",\n   "contenido": "16febd373be937250f773cbc62bfe339665e64f53bcb4008dfe8130145aadc0a",\n   "origen": "/home/claude/work/CFS_motor_s23/prod_bodily_injury.npy",\n   "nodo": "CFS_motor_s23"\n  },\n  "CFS_motor_s23/prod_theft.npy": {\n   "bytes": 2040016,\n   "sha256": "956aa7e8db64d6b56d8b3459b80ab9b92bbf67c8ffbff723bb95e8eb10a54078",\n   "contenido": "323094f6b8d6d62e1ac948d7367660d4ee94cc15f3af55f94ef1004a5926d0a5",\n   "origen": "/home/claude/work/CFS_motor_s23/prod_theft.npy",\n   "nodo": "CFS_motor_s23"\n  },\n  "CFS_motor_s23/prod_glass_hail.npy": {\n   "bytes": 2040016,\n   "sha256": "df52d7b2356006719b146ba0eb7a2e236b7c8d9b5ceea1f06200f99d40c6e2f4",\n   "contenido": "bf5308da3621fda7b258602bdfec2f544f113eaba4717955921ce9d1d8bf922a",\n   "origen": "/home/claude/work/CFS_motor_s23/prod_glass_hail.npy",\n   "nodo": "CFS_motor_s23"\n  },\n  "CFS_motor_s23/v24.parquet": {\n   "bytes": 3958636,\n   "sha256": "edd218f167d18c74c7fc42f5b5cc41dabae4940adb47cc57de1df6f3e97928c6",\n   "contenido": "b9dcd78b36d43e5700270e5db1c34d499964513a48e25c5797d27cf7cb2b64a2",\n   "origen": "/home/claude/work/CFS_motor_s23/v24.parquet",\n   "nodo": "CFS_motor_s23"\n  },\n  "CFS_motor_s23/prediccion_2025.parquet": {\n   "bytes": 3739277,\n   "sha256": "30a06f85fb051c40a79fc7de2b8da730715062bdc1176f6ab76b5c23df6da9f3",\n   "contenido": "0079edf0dcc8fd51536593b66efe082cb1462572b36326c2f42f9a2e3e6c534f",\n   "origen": "/home/claude/work/CFS_motor_s23/prediccion_2025.parquet",\n   "nodo": "CFS_motor_s23"\n  },\n  "CFS_buildings/v23_storm.npz": {\n   "bytes": 738220,\n   "sha256": "f3198f0bd2607ccc2385f67e725d10b7dfdede63ca83f50191b71159378169bf",\n   "contenido": "f9bea81ded4f4cb3b8720942d0aef01f290fcede646c9e369b685e46cb3bd880",\n   "origen": "/home/claude/work/CFS_buildings/v23_storm.npz",\n   "nodo": "CFS_buildings"\n  },\n  "CFS_buildings/v23_flood.npz": {\n   "bytes": 738220,\n   "sha256": "be7902e29055327a534df8455e72458c6d2aa0ba256c110f511052fb5cf6730e",\n   "contenido": "47cec25dec3967016a7cdf802c587192224d41d347700faaacd5289789401d98",\n   "origen": "/home/claude/work/CFS_buildings/v23_flood.npz",\n   "nodo": "CFS_buildings"\n  },\n  "CFS_buildings/v23_escape_of_water.npz": {\n   "bytes": 738220,\n   "sha256": "2122c2ce064d000f82ff5ec4bec3b0736fb84c69e148e0bfb399ef45b080719c",\n   "contenido": "d1efc155f0b3cd90b9ffd207c46ece1e6045579e42d682d0c1fc72586ff4d768",\n   "origen": "/home/claude/work/CFS_buildings/v23_escape_of_water.npz",\n   "nodo": "CFS_buildings"\n  },\n  "CFS_buildings/v23_subsidence.npz": {\n   "bytes": 738220,\n   "sha256": "533faac0e88a2c6a1b4a777e4f4a764a0c0ba6e9092d4f658f6211962d3968be",\n   "contenido": "4469f4e57564db66c421f7049f8c4f28ce8cc9997fdce059a139dfd3c4f5a493",\n   "origen": "/home/claude/work/CFS_buildings/v23_subsidence.npz",\n   "nodo": "CFS_buildings"\n  },\n  "CFS_buildings/v23_fire.npz": {\n   "bytes": 738220,\n   "sha256": "8663b1ffe7a7cd4b3cee0de2c57bd9e2b33bbb6cfd920d98cf44f25d15a97da2",\n   "contenido": "06c0e05e3945a3268187f82600938f13099ea29012eb165f3889c583bb48734b",\n   "origen": "/home/claude/work/CFS_buildings/v23_fire.npz",\n   "nodo": "CFS_buildings"\n  },\n  "CFS_buildings/v23_accidental_damage.npz": {\n   "bytes": 738220,\n   "sha256": "768d54f01f0b736ee6d0744afda2fdfe20a7f0ee3f6859c9f1294aa11e67ac5c",\n   "contenido": "60ff9f1f8db519b306c3ebd4a2e2501c3f4974482522bb0149346035cd337c08",\n   "origen": "/home/claude/work/CFS_buildings/v23_accidental_damage.npz",\n   "nodo": "CFS_buildings"\n  },\n  "CFS_buildings/v24_storm.npy": {\n   "bytes": 1472136,\n   "sha256": "af281fe9aad5f43c395c65ca4be2e569b01a0397af2f8d55373642724b36aca5",\n   "contenido": "502fcc5befbd612e88c5661fe407ed0e9c5f170e6e4018011d39301d92fdde30",\n   "origen": "/home/claude/work/CFS_buildings/v24_storm.npy",\n   "nodo": "CFS_buildings"\n  },\n  "CFS_buildings/v24_flood.npy": {\n   "bytes": 1472136,\n   "sha256": "3fbf1e53eb69949cac4c60cedd74bedce084ff76b76a8a068cc983335896593a",\n   "contenido": "cb23588bd3715eaa6373043d14116f4bc7eaa9278e87e05eebd41e7cde44b2cb",\n   "origen": "/home/claude/work/CFS_buildings/v24_flood.npy",\n   "nodo": "CFS_buildings"\n  },\n  "CFS_buildings/v24_escape_of_water.npy": {\n   "bytes": 1472136,\n   "sha256": "1472f036e25ad90fb8aa19db0b7db55f1bc3b1bea0be0e9308870300ec7000bb",\n   "contenido": "911b77aa064803159d52833f7fcd2abc6b59f06f20225486541d0a45145bf4ea",\n   "origen": "/home/claude/work/CFS_buildings/v24_escape_of_water.npy",\n   "nodo": "CFS_buildings"\n  },\n  "CFS_buildings/v24_subsidence.npy": {\n   "bytes": 1472136,\n   "sha256": "ed0068208cc4990b2b568b040f013510a695c36000323c86806ffb3fb3f96e1a",\n   "contenido": "7ed2c89b18fc4fd0f9f87b7ca64716aa0d2e7c6bed512818facb6216d23819c6",\n   "origen": "/home/claude/work/CFS_buildings/v24_subsidence.npy",\n   "nodo": "CFS_buildings"\n  },\n  "CFS_buildings/v24_fire.npy": {\n   "bytes": 1472136,\n   "sha256": "f0512ea785bd8cec311736b4005ac3a4d55c8dcb9324f48b50094b8c536bd39c",\n   "contenido": "3e21a01447bba5c26199b7c86eca4454d52f5e479d515183698905ef41d549f6",\n   "origen": "/home/claude/work/CFS_buildings/v24_fire.npy",\n   "nodo": "CFS_buildings"\n  },\n  "CFS_buildings/v24_accidental_damage.npy": {\n   "bytes": 1472136,\n   "sha256": "f0ae17d7bd550235dab7a9999e218c704f44ce34c77fbf81f9ae3bdbc83a9c89",\n   "contenido": "4dbf28bd059e514f6c049a168ce8eeac0ff57b1d319f351750df9c7cd5915c28",\n   "origen": "/home/claude/work/CFS_buildings/v24_accidental_damage.npy",\n   "nodo": "CFS_buildings"\n  },\n  "CFS_buildings/prod_storm.npy": {\n   "bytes": 1473368,\n   "sha256": "081473d7a42e4240bc11153b454cf3f7a16d1f2000dd9d74bbcd871b65f86c1b",\n   "contenido": "6461a529c8bc04e5c8f0c8e7feecb17592627dd68a6228c0ca22af9ad78e6ffc",\n   "origen": "/home/claude/work/CFS_buildings/prod_storm.npy",\n   "nodo": "CFS_buildings"\n  },\n  "CFS_buildings/prod_flood.npy": {\n   "bytes": 1473368,\n   "sha256": "3dff91ac41b110db777a4ccc7f28cb2ac1994d46097f7222fbdfd5390fec2142",\n   "contenido": "1ec7cd4011d92c9c8c9af051af10c9a3a971d59b9af2e7e62b1ad5ccf320734c",\n   "origen": "/home/claude/work/CFS_buildings/prod_flood.npy",\n   "nodo": "CFS_buildings"\n  },\n  "CFS_buildings/prod_escape_of_water.npy": {\n   "bytes": 1473368,\n   "sha256": "c7aabb1e756189060c9d66505d2c1187bca9f32951a6a39891cc2e015e15ee29",\n   "contenido": "c8744d97be8b25db88268c66db67b1d6affbc659f59bfeb44df19b23a43d39a0",\n   "origen": "/home/claude/work/CFS_buildings/prod_escape_of_water.npy",\n   "nodo": "CFS_buildings"\n  },\n  "CFS_buildings/prod_subsidence.npy": {\n   "bytes": 1473368,\n   "sha256": "a09ce50a92a08d75b974a7643ab2a365d912d94648daa0670e494bc5f6921a1b",\n   "contenido": "85df5a94a9b785f981b3b7d3ffdc265b3c19c62fc3e0ff976b5648dd4f749e33",\n   "origen": "/home/claude/work/CFS_buildings/prod_subsidence.npy",\n   "nodo": "CFS_buildings"\n  },\n  "CFS_buildings/prod_fire.npy": {\n   "bytes": 1473368,\n   "sha256": "1643fc97d5a0dece1ff9489462f024701a4264027085ec737abc7d032ce42291",\n   "contenido": "6c56c719e49b422109faee8f363eb27ed54ef88842ca85b0bb39f7717a14079d",\n   "origen": "/home/claude/work/CFS_buildings/prod_fire.npy",\n   "nodo": "CFS_buildings"\n  },\n  "CFS_buildings/prod_accidental_damage.npy": {\n   "bytes": 1473368,\n   "sha256": "00935979b230f2746fc698a5766d6b3b3b9cfeaebd61c75714a01a607dc62e95",\n   "contenido": "840f274befc462fa14a346cd3f83aa996b9393f62ee336f1de35400538db1e5d",\n   "origen": "/home/claude/work/CFS_buildings/prod_accidental_damage.npy",\n   "nodo": "CFS_buildings"\n  },\n  "sel_CFS_buildings.parquet": {\n   "bytes": 1574274,\n   "sha256": "65f62e51072a50cc359424a42b061fe13d0a9dca384e4b2956d3038f451fd22f",\n   "contenido": "61796de6874ad0523e7599425b4091e23b393c91a88116896d50dc8814fd6d6f",\n   "origen": "/home/claude/work/sel_CFS_buildings.parquet",\n   "nodo": "CFS_buildings"\n  },\n  "CFS_buildings/v24.parquet": {\n   "bytes": 2867461,\n   "sha256": "27493e4077bcc877a6769b516f7c477ff39364d7dc0ee9fa725584ae54c52d22",\n   "contenido": "3c2a85e509beb56dd83779ac2e893c0febd544b7bb4d2cfddfba050e33ee8b59",\n   "origen": "/home/claude/work/CFS_buildings/v24.parquet",\n   "nodo": "CFS_buildings"\n  },\n  "CFS_buildings/prediccion_2025.parquet": {\n   "bytes": 2820627,\n   "sha256": "682d515636eba93971f37e987938721185eac8700b81b14e52ac3a2b8b4f86e9",\n   "contenido": "6dc46287a66b9a542ab085ab10e672bc2419fd23c80e1e8b5a5ffb16e8a8e3ce",\n   "origen": "/home/claude/work/CFS_buildings/prediccion_2025.parquet",\n   "nodo": "CFS_buildings"\n  },\n  "CFS_buildings_s23/v24_storm.npy": {\n   "bytes": 1472136,\n   "sha256": "fd520b07ea423aaac1659688416bbd82380f1a70a732bae3c5162168440efe77",\n   "contenido": "d32856085b4d5db65fd281401be536c37c2f0af03245fe5cf8996fb2d97a3673",\n   "origen": "/home/claude/work/CFS_buildings_s23/v24_storm.npy",\n   "nodo": "CFS_buildings_s23"\n  },\n  "CFS_buildings_s23/v24_flood.npy": {\n   "bytes": 1472136,\n   "sha256": "b048655eab511d59fb15f20ff08e722f212e023ad15ff46e8ee076722fb232f7",\n   "contenido": "247e42c844db7dab10dc1524e50a6c67b863560f1c7d587a0b4bf75977c2d02e",\n   "origen": "/home/claude/work/CFS_buildings_s23/v24_flood.npy",\n   "nodo": "CFS_buildings_s23"\n  },\n  "CFS_buildings_s23/v24_escape_of_water.npy": {\n   "bytes": 1472136,\n   "sha256": "336980ef8844d6d23607ff926829ec81b1929d1425ffa2711310f95e9d777184",\n   "contenido": "e2a8e9ff6402859ee77eb71a54850b5e82b90e0e19d29a7a60c239e5fe2af5ce",\n   "origen": "/home/claude/work/CFS_buildings_s23/v24_escape_of_water.npy",\n   "nodo": "CFS_buildings_s23"\n  },\n  "CFS_buildings_s23/v24_subsidence.npy": {\n   "bytes": 1472136,\n   "sha256": "f63610d9318c9ae267edd4697695b753afbacb62b2f22d4539b95f6ade6ac9e1",\n   "contenido": "07cad8bed4b1049fc29bb2edead44fe851657a3f839ad65244b690a1245994b8",\n   "origen": "/home/claude/work/CFS_buildings_s23/v24_subsidence.npy",\n   "nodo": "CFS_buildings_s23"\n  },\n  "CFS_buildings_s23/v24_fire.npy": {\n   "bytes": 1472136,\n   "sha256": "c5ddb12650fbecddf7d92d734bc87057317ba8119eab7507707a050e99639d7e",\n   "contenido": "c7ebe61d1ab90192ca8db661587706b2683f2b6f050b39b2b78d8ea0b9895ed0",\n   "origen": "/home/claude/work/CFS_buildings_s23/v24_fire.npy",\n   "nodo": "CFS_buildings_s23"\n  },\n  "CFS_buildings_s23/v24_accidental_damage.npy": {\n   "bytes": 1472136,\n   "sha256": "2f416841812aefc2df37b2fadc3b5af5f036d275c5c284b1e4628424450f75e6",\n   "contenido": "5c1fbd3e2ba9a953edc3315b26981cfb6ba6c57aeae6bc3cafd8c848dac7fc94",\n   "origen": "/home/claude/work/CFS_buildings_s23/v24_accidental_damage.npy",\n   "nodo": "CFS_buildings_s23"\n  },\n  "CFS_buildings_s23/prod_storm.npy": {\n   "bytes": 1473368,\n   "sha256": "18fdd28dfb122345a3fc8364ac48e4c604f1829da21fc9fe98f9b61fe9586a8c",\n   "contenido": "7514a722f92d47ecfd65301247fb5389c88673d1aa35411ca1f3348fbdb1a59b",\n   "origen": "/home/claude/work/CFS_buildings_s23/prod_storm.npy",\n   "nodo": "CFS_buildings_s23"\n  },\n  "CFS_buildings_s23/prod_flood.npy": {\n   "bytes": 1473368,\n   "sha256": "3141b7e8b374e792971a0bbe5dd9e9cb74f923fd563849c3d4eeba9909ef758b",\n   "contenido": "f020daead573789d5195be84cf1fc57216ba2a84c05ad9eb3c9c988e6ba25e02",\n   "origen": "/home/claude/work/CFS_buildings_s23/prod_flood.npy",\n   "nodo": "CFS_buildings_s23"\n  },\n  "CFS_buildings_s23/prod_escape_of_water.npy": {\n   "bytes": 1473368,\n   "sha256": "d2d911f199ac71b35bfdda30794f0344d0b71be44a90039f329c92b90fdbff2e",\n   "contenido": "3f651d5f48b87cf3d7942dc855fc56345f834c74800ba1f91bf04b4c8a874205",\n   "origen": "/home/claude/work/CFS_buildings_s23/prod_escape_of_water.npy",\n   "nodo": "CFS_buildings_s23"\n  },\n  "CFS_buildings_s23/prod_subsidence.npy": {\n   "bytes": 1473368,\n   "sha256": "dcd98e60b1f8a82cb7df1ac613a90f4650f70cc31fda15b805c63c0369b75975",\n   "contenido": "5c0697bafa5c329671be1720672f28cb3c7adbba32d090f6f754e44b6d5c1e13",\n   "origen": "/home/claude/work/CFS_buildings_s23/prod_subsidence.npy",\n   "nodo": "CFS_buildings_s23"\n  },\n  "CFS_buildings_s23/prod_fire.npy": {\n   "bytes": 1473368,\n   "sha256": "18402ad417833b5e4480295ccd224a29b9112a6ceb6108bc3c4130a3eab0117b",\n   "contenido": "ee6ae48c6cbfd5c5e0ed2cbf802d6111026f7564730682e5f7b5f5e7c4c29962",\n   "origen": "/home/claude/work/CFS_buildings_s23/prod_fire.npy",\n   "nodo": "CFS_buildings_s23"\n  },\n  "CFS_buildings_s23/prod_accidental_damage.npy": {\n   "bytes": 1473368,\n   "sha256": "f4f2085fa3d307d15e08b9821ba7ce835abccedb04b24c9fabf0727a9c91a255",\n   "contenido": "819f879990d64982318bb2f02dbcaf20142907bb0ac5e4d19065a1db9ca44a37",\n   "origen": "/home/claude/work/CFS_buildings_s23/prod_accidental_damage.npy",\n   "nodo": "CFS_buildings_s23"\n  },\n  "CFS_buildings_s23/v24.parquet": {\n   "bytes": 2867461,\n   "sha256": "4478cd503b550cc95a4a8b9377557b2f5b3bdf769f76b832ff9ac6a2e466c2f7",\n   "contenido": "0b10942386dfff15aeffd1604c6332dddb02f44b5d179ec72cb1c7a8925cb4a8",\n   "origen": "/home/claude/work/CFS_buildings_s23/v24.parquet",\n   "nodo": "CFS_buildings_s23"\n  },\n  "CFS_buildings_s23/prediccion_2025.parquet": {\n   "bytes": 2820627,\n   "sha256": "9bad842a7c6690be04ab1be43f8710b03446cbe2f2c528c5fa7d3ab12ea1d89f",\n   "contenido": "b034db17088a976eb360764448b397415d2b2b2e76fad92ebcc55437f9717b65",\n   "origen": "/home/claude/work/CFS_buildings_s23/prediccion_2025.parquet",\n   "nodo": "CFS_buildings_s23"\n  },\n  "sel_FSA_contents.parquet": {\n   "bytes": 1583200,\n   "sha256": "5eb0386a57ee6ef715989e06a3b5f3fc4f767069fa2e65582c7ff87f0d91060a",\n   "contenido": "3d908254711203ca9da04c4113973934a7587415b84f0bc99d6daf99dd2eadc9",\n   "origen": "/home/claude/work/sel_FSA_contents.parquet",\n   "nodo": "FSA_contents"\n  },\n  "FSA_contents/v24.parquet": {\n   "bytes": 2907654,\n   "sha256": "92a05c3174853240db42344dee1c233a47a1f0bf2d4be59d177a4f802e260ff8",\n   "contenido": "418936f3e377ba69795b8df736804a4295115a8fde963066a424439258caed78",\n   "origen": "/home/claude/work/FSA_contents/v24.parquet",\n   "nodo": "FSA_contents"\n  },\n  "FSA_contents/prediccion_2025.parquet": {\n   "bytes": 2846950,\n   "sha256": "f3d02e2face2cf7ae18c44f615d3ad1969b2ce4bfc12e244569d0c35f720c1c2",\n   "contenido": "7f8da0b4396fccf8c6fef3131a5b5d489af18050319035266572265fde0d98a3",\n   "origen": "/home/claude/work/FSA_contents/prediccion_2025.parquet",\n   "nodo": "FSA_contents"\n  },\n  "FSAP_buildings/v23_storm.npz": {\n   "bytes": 738220,\n   "sha256": "850b3b4951cc143d9f37f8ccaad1c01d7256ab9dff8411777f811a2b97838134",\n   "contenido": "961171a27239f6d77f5212b21da9b7692da6a3a23270cb46ec7ca6d939c05e6c",\n   "origen": "/home/claude/work/FSAP_buildings/v23_storm.npz",\n   "nodo": "FSAP_buildings"\n  },\n  "FSAP_buildings/v23_flood.npz": {\n   "bytes": 738220,\n   "sha256": "58e74b143eac29633154aae8af5ba88eef8a13b6832d281a3d9050a476e19156",\n   "contenido": "e552f897970f1e1252d7351133f5e4a9d1024c756005b359e3da8862356dab48",\n   "origen": "/home/claude/work/FSAP_buildings/v23_flood.npz",\n   "nodo": "FSAP_buildings"\n  },\n  "FSAP_buildings/v23_escape_of_water.npz": {\n   "bytes": 738220,\n   "sha256": "425c10af118f637be9c71d4340150c13db4b2b4b42fce60bd81472cb925edaea",\n   "contenido": "c718763ca4de27b4255e4035f81fea32f004e1625615eba9483ab92d7e4fee4e",\n   "origen": "/home/claude/work/FSAP_buildings/v23_escape_of_water.npz",\n   "nodo": "FSAP_buildings"\n  },\n  "FSAP_buildings/v23_subsidence.npz": {\n   "bytes": 738220,\n   "sha256": "239ac4084751f448eccc5c2aba2c6ac9e874611ebabdb6ca325e620d6c72dc25",\n   "contenido": "027814bf5f1aeb153890b764cd1201c6e2b5d508c9ac3cec84eeac037f645615",\n   "origen": "/home/claude/work/FSAP_buildings/v23_subsidence.npz",\n   "nodo": "FSAP_buildings"\n  },\n  "FSAP_buildings/v23_fire.npz": {\n   "bytes": 738220,\n   "sha256": "8f23b38286754d7dea82b20d1508698e652f44f26debb17252f89bd77795510b",\n   "contenido": "29fde6af2f6860e07dfe851b1b32383da1a5a8d55d6a24978200e09dda3602f9",\n   "origen": "/home/claude/work/FSAP_buildings/v23_fire.npz",\n   "nodo": "FSAP_buildings"\n  },\n  "FSAP_buildings/v23_accidental_damage.npz": {\n   "bytes": 738220,\n   "sha256": "35b0ab7e6a8df8d161a0819815e382559697f127c138119471b92c06f6d80d72",\n   "contenido": "ded99cff8b3dd536a2980c1b7c8d7c5cd723bbf6f608d93bb7da159376bf48ba",\n   "origen": "/home/claude/work/FSAP_buildings/v23_accidental_damage.npz",\n   "nodo": "FSAP_buildings"\n  },\n  "FSAP_buildings/v24_storm.npy": {\n   "bytes": 1472136,\n   "sha256": "f187a0b5b41839749d1d462ecfd69ba2f60c3d6decaa7cdfeea9dc5c00dc2458",\n   "contenido": "051d204f575ba9cd70bc12030068dd8f6c20816d1aadc5a69b615b94c9ec25a6",\n   "origen": "/home/claude/work/FSAP_buildings/v24_storm.npy",\n   "nodo": "FSAP_buildings"\n  },\n  "FSAP_buildings/v24_flood.npy": {\n   "bytes": 1472136,\n   "sha256": "e946ded4c829ad8d9fc1b74868db0534d347e81578ebbcd2fa8c417700a70ec1",\n   "contenido": "aeb76d81aa4e26ecc2af8491e847c968c0b946cc55b63cef4875f4a7ba912ac8",\n   "origen": "/home/claude/work/FSAP_buildings/v24_flood.npy",\n   "nodo": "FSAP_buildings"\n  },\n  "FSAP_buildings/v24_escape_of_water.npy": {\n   "bytes": 1472136,\n   "sha256": "66987e0dc2c8e716aeaac725a935407c69aa28817e19bc7c8ba6fb80b2b3b8d6",\n   "contenido": "292469938160a6a06c012d94c434b3db8802626401abafc30537f97d65e50b55",\n   "origen": "/home/claude/work/FSAP_buildings/v24_escape_of_water.npy",\n   "nodo": "FSAP_buildings"\n  },\n  "FSAP_buildings/v24_subsidence.npy": {\n   "bytes": 1472136,\n   "sha256": "9a3fe6a967a80ce3247ca10d06f9d9f24435ecd9031653b30a76073f1f539dc4",\n   "contenido": "55da1950628a2b90cb499df99f9301800c18782a450d8d60d967724f23c767f5",\n   "origen": "/home/claude/work/FSAP_buildings/v24_subsidence.npy",\n   "nodo": "FSAP_buildings"\n  },\n  "FSAP_buildings/v24_fire.npy": {\n   "bytes": 1472136,\n   "sha256": "8b076b6b3d6a1fbd4c79d2498ef0450124587552601e0020aaed32ca9398d98c",\n   "contenido": "088544e713c5082e74e76abb3e5c19d35581fd66c5e074bfab36aacc91729219",\n   "origen": "/home/claude/work/FSAP_buildings/v24_fire.npy",\n   "nodo": "FSAP_buildings"\n  },\n  "FSAP_buildings/v24_accidental_damage.npy": {\n   "bytes": 1472136,\n   "sha256": "37337978de4dfc4d10fd549eb3153e7a19e3240653d3c9a5c7e074058fe864d6",\n   "contenido": "9226fc6b8303f70d11db01bd240b4c3113121552acb2d910c74e91b6f3704613",\n   "origen": "/home/claude/work/FSAP_buildings/v24_accidental_damage.npy",\n   "nodo": "FSAP_buildings"\n  },\n  "FSAP_buildings/prod_storm.npy": {\n   "bytes": 1473368,\n   "sha256": "388eba379c72789e19971a36e30b4d06c2e33b90e1adf7b3db466e0ea912cd7c",\n   "contenido": "7470367dfae54d656e6df0c63196545ecf4db75b7b60e58e1318a8e4e7645471",\n   "origen": "/home/claude/work/FSAP_buildings/prod_storm.npy",\n   "nodo": "FSAP_buildings"\n  },\n  "FSAP_buildings/prod_flood.npy": {\n   "bytes": 1473368,\n   "sha256": "835b8d795e217a14a8e9b90e4c858aa84460e4025e07e506d44fa4ee840b6434",\n   "contenido": "7b8a2b4bc978c6b6d954f7c235270eb6b59ee4bd4b2a6d5c2388478636f0c53b",\n   "origen": "/home/claude/work/FSAP_buildings/prod_flood.npy",\n   "nodo": "FSAP_buildings"\n  },\n  "FSAP_buildings/prod_escape_of_water.npy": {\n   "bytes": 1473368,\n   "sha256": "0a9bec1ecb3f5e7fba36e58a6b52aa63789b754892e307baf030961dd1d7637d",\n   "contenido": "bb0069ce0ebf05cece001ab3566953db3538fcea9c1dcf228dd0f3dec1588c65",\n   "origen": "/home/claude/work/FSAP_buildings/prod_escape_of_water.npy",\n   "nodo": "FSAP_buildings"\n  },\n  "FSAP_buildings/prod_subsidence.npy": {\n   "bytes": 1473368,\n   "sha256": "62c9859786d37a6159cc44e91884429d259f8719d3cd485dc09cf3a51a7c8c5c",\n   "contenido": "16c2ad51a1e683b48b94654ff4f173bc4506f5eba3febbdac69f84ec8cce85b2",\n   "origen": "/home/claude/work/FSAP_buildings/prod_subsidence.npy",\n   "nodo": "FSAP_buildings"\n  },\n  "FSAP_buildings/prod_fire.npy": {\n   "bytes": 1473368,\n   "sha256": "ffa156ed849afb427a6a4fd5f01dd9653f19611bbaa45c849219b5cc1704011d",\n   "contenido": "9fd207f858296b94926b73b64daf73d3f3f71feb6358591ef2c26224f163b7f2",\n   "origen": "/home/claude/work/FSAP_buildings/prod_fire.npy",\n   "nodo": "FSAP_buildings"\n  },\n  "FSAP_buildings/prod_accidental_damage.npy": {\n   "bytes": 1473368,\n   "sha256": "08b9f79f33f867a1c240f15ecb6ae5943a8138fa632db09cf180378645faca1f",\n   "contenido": "af8df6b54f4ce6cd7bf2f4bf0530de9359291ed978a76f4d390c367d3becd2dd",\n   "origen": "/home/claude/work/FSAP_buildings/prod_accidental_damage.npy",\n   "nodo": "FSAP_buildings"\n  },\n  "sel_FSAP_buildings.parquet": {\n   "bytes": 1574274,\n   "sha256": "8651e0c38f36c7604ed7cf004c9f346b794ae600251cc89278baf0dbc08c564d",\n   "contenido": "4da2f65013274fe32e8ad5e550a436ab3b391c178fc9ccd7848cc245536627d3",\n   "origen": "/home/claude/work/sel_FSAP_buildings.parquet",\n   "nodo": "FSAP_buildings"\n  },\n  "FSAP_buildings/v24.parquet": {\n   "bytes": 2867461,\n   "sha256": "a91de648d619bfdb1298988d25331d41bb45d296dd362c6e4faabe79af8c2348",\n   "contenido": "a1cab56bcf87ce5a6c80a03175b7467762a688df42d9c75f97dab912b85c39d5",\n   "origen": "/home/claude/work/FSAP_buildings/v24.parquet",\n   "nodo": "FSAP_buildings"\n  },\n  "FSAP_buildings/prediccion_2025.parquet": {\n   "bytes": 2820627,\n   "sha256": "e9fac7e4119262180180acb03a8a600137b65fd1920bb5aa4a363a0684da85e5",\n   "contenido": "8b500314fdbde9ffde0ac2abd14f7bc5d50a80b126c31c2c26387251032c9a41",\n   "origen": "/home/claude/work/FSAP_buildings/prediccion_2025.parquet",\n   "nodo": "FSAP_buildings"\n  },\n  "FSAP2_buildings/v24_storm.npy": {\n   "bytes": 1472136,\n   "sha256": "49034731247e67e0cebdb63a12280469f33e762e69c001c20723fab2221ff562",\n   "contenido": "5a3488b9190313e78922408356ddd9b279deb1a0c96118307eb911324f7907af",\n   "origen": "/home/claude/work/FSAP2_buildings/v24_storm.npy",\n   "nodo": "FSAP2_buildings"\n  },\n  "FSAP2_buildings/v24_flood.npy": {\n   "bytes": 1472136,\n   "sha256": "6cf339c36691da4e7bf026bcad8c662a0b8d7e989aa03dc2aafd236f415b92f9",\n   "contenido": "0d5623bdde055f40f886313579343c4a449444c5c6f37e69acc4e717fd1222c8",\n   "origen": "/home/claude/work/FSAP2_buildings/v24_flood.npy",\n   "nodo": "FSAP2_buildings"\n  },\n  "FSAP2_buildings/v24_escape_of_water.npy": {\n   "bytes": 1472136,\n   "sha256": "9177427a22ca85c120e6d8835a9bcc254c8d3d74f11041440eb324f6be1426a9",\n   "contenido": "a9b20654902aee33623c41575590caaa0eb90eb8355f31d45b49e16171512e4f",\n   "origen": "/home/claude/work/FSAP2_buildings/v24_escape_of_water.npy",\n   "nodo": "FSAP2_buildings"\n  },\n  "FSAP2_buildings/v24_subsidence.npy": {\n   "bytes": 1472136,\n   "sha256": "658d42f0564f47bed553a64a11039fd93027ab8783bdaa82e1dd5fd08dbeea57",\n   "contenido": "1073cf493df2370ee4bf8c679198c26ef1af5372dc5ac321ad6aef5337cfa985",\n   "origen": "/home/claude/work/FSAP2_buildings/v24_subsidence.npy",\n   "nodo": "FSAP2_buildings"\n  },\n  "FSAP2_buildings/v24_fire.npy": {\n   "bytes": 1472136,\n   "sha256": "4d07f0f0fb237a44c816a1a62af729ac25b49e2e6bf413476c123e3099da63f5",\n   "contenido": "2bc35b262ecb92d0773767a6d878f1b8d13dcb5bd7d6c997f18e990b59b22b4a",\n   "origen": "/home/claude/work/FSAP2_buildings/v24_fire.npy",\n   "nodo": "FSAP2_buildings"\n  },\n  "FSAP2_buildings/v24_accidental_damage.npy": {\n   "bytes": 1472136,\n   "sha256": "f04e2592832b7647c5e24e7e1a73f873f1f89bfbcdea710596e312fd382b77c2",\n   "contenido": "3236f232375e2213bc3706140a23b63d9c4cdf7e75f834e8085703416c774f04",\n   "origen": "/home/claude/work/FSAP2_buildings/v24_accidental_damage.npy",\n   "nodo": "FSAP2_buildings"\n  },\n  "FSAP2_buildings/prod_storm.npy": {\n   "bytes": 1473368,\n   "sha256": "aa8c63e86c531dd6793ef0319e259a373293a5ff922ea5220c7b22b5cbd5ba0c",\n   "contenido": "9df133ce6b258e598f4a7e79adce3d8a2eeaff81025d5acfba853538fe3064c7",\n   "origen": "/home/claude/work/FSAP2_buildings/prod_storm.npy",\n   "nodo": "FSAP2_buildings"\n  },\n  "FSAP2_buildings/prod_flood.npy": {\n   "bytes": 1473368,\n   "sha256": "da746c66ac3ac15c45ed769c96aeb15619f63a89090a278a0ccb97e19ae8afc6",\n   "contenido": "63f7430de45b45133ffab60ca6c43662edc63f5c305cce529490e316c17a73dd",\n   "origen": "/home/claude/work/FSAP2_buildings/prod_flood.npy",\n   "nodo": "FSAP2_buildings"\n  },\n  "FSAP2_buildings/prod_escape_of_water.npy": {\n   "bytes": 1473368,\n   "sha256": "0aa258443517200e7b48ea41ec90bd37037b9790a162f27f076d53ec63f8ba73",\n   "contenido": "ad92d8ae2610ec85b2192edc9c01c877a85b2aef0613c03b90d698972e129b48",\n   "origen": "/home/claude/work/FSAP2_buildings/prod_escape_of_water.npy",\n   "nodo": "FSAP2_buildings"\n  },\n  "FSAP2_buildings/prod_subsidence.npy": {\n   "bytes": 1473368,\n   "sha256": "52e3df3859fb429f940d6d59e97c7e660be29feaa4298723dbab030e54ca996d",\n   "contenido": "c171756be82f0ed42d375e4bce47aca1b1dca22072d13ee56d9cee609f0f792b",\n   "origen": "/home/claude/work/FSAP2_buildings/prod_subsidence.npy",\n   "nodo": "FSAP2_buildings"\n  },\n  "FSAP2_buildings/prod_fire.npy": {\n   "bytes": 1473368,\n   "sha256": "0246bcf85c25b1ddf89a6ceed6d379150ccdd2dab9d544183d6d094a49f4eabe",\n   "contenido": "a3d56d033c08f71d2e5309805020f023857a68f13247fc890addf526eb8f17af",\n   "origen": "/home/claude/work/FSAP2_buildings/prod_fire.npy",\n   "nodo": "FSAP2_buildings"\n  },\n  "FSAP2_buildings/prod_accidental_damage.npy": {\n   "bytes": 1473368,\n   "sha256": "1afbd944543d85ec1155f30d9d3343be142d37a01b2f0e2fd63b8f52fe7d4e82",\n   "contenido": "a8b6c5dd1c7aa4f411027e370558e07a40ac35855d6fb029e0782b1e05be115e",\n   "origen": "/home/claude/work/FSAP2_buildings/prod_accidental_damage.npy",\n   "nodo": "FSAP2_buildings"\n  },\n  "FSAP2_buildings/v24.parquet": {\n   "bytes": 2867461,\n   "sha256": "ee8c8846a5d9abbdde283b8eebdf496dc089ac8c7983cb7cc42b6ffb60fbb011",\n   "contenido": "437cb4038ecff8caae42e517b32d3d85c93246d08a27f500d19d1d2ffada4bd3",\n   "origen": "/home/claude/work/FSAP2_buildings/v24.parquet",\n   "nodo": "FSAP2_buildings"\n  },\n  "FSAP2_buildings/prediccion_2025.parquet": {\n   "bytes": 2820627,\n   "sha256": "a72c60d348bb756577ee4cd9b5c899e554c9a756b0939ac1b0c7ce66cf2139db",\n   "contenido": "e7102e79075f911177eb670d5daf24b0ae8b19436ea27a9573d5c1cc0e6b4d3a",\n   "origen": "/home/claude/work/FSAP2_buildings/prediccion_2025.parquet",\n   "nodo": "FSAP2_buildings"\n  },\n  "FSAP_contents/v23_theft.npz": {\n   "bytes": 752028,\n   "sha256": "bbd6650303c3b940e075927cee9803413d87a9e9cffef5a2b37a2f79da0e3c4f",\n   "contenido": "5d3a97a2a8a843dd3227040e887ce13890b74705163658ce68f37d02762fa1de",\n   "origen": "/home/claude/work/FSAP_contents/v23_theft.npz",\n   "nodo": "FSAP_contents"\n  },\n  "FSAP_contents/v23_escape_of_water.npz": {\n   "bytes": 752028,\n   "sha256": "96f944cf147439c979d9195a4064c7645ed9ee0ecd700dc0507f76155ef7140c",\n   "contenido": "835011fbc9b139b7bb596313e81e268dfafa1ec55caf3be17d2ca48a10f2613b",\n   "origen": "/home/claude/work/FSAP_contents/v23_escape_of_water.npz",\n   "nodo": "FSAP_contents"\n  },\n  "FSAP_contents/v23_accidental_damage.npz": {\n   "bytes": 752028,\n   "sha256": "d9f5e0b63937d106d5dfc8c5cd9ca8c6e7811eed16ca5af8d86f983e8bfafb1d",\n   "contenido": "7196137a11c967c1b46013a7f9c8999025a9217e84e4a1ee019f3f9c3378421a",\n   "origen": "/home/claude/work/FSAP_contents/v23_accidental_damage.npz",\n   "nodo": "FSAP_contents"\n  },\n  "FSAP_contents/v23_flood.npz": {\n   "bytes": 752028,\n   "sha256": "25a2adf3ca65e4dabda00284e627724e269b73bf6e634a82010ac0ec76c8bb63",\n   "contenido": "4677f9dd8df9a0ba503e3f3dd5c26426051e838a14e3b89dc7f74d342eac6405",\n   "origen": "/home/claude/work/FSAP_contents/v23_flood.npz",\n   "nodo": "FSAP_contents"\n  },\n  "FSAP_contents/v24_theft.npy": {\n   "bytes": 1500056,\n   "sha256": "cfe68a13afbbbda3335b46d7cfb01805c1bdaeec7f25938a8e194544a23cca29",\n   "contenido": "ea18aa845ea0529618fa4ff33927d3b5d1cbe8143bf5a9ca4d868599e7383105",\n   "origen": "/home/claude/work/FSAP_contents/v24_theft.npy",\n   "nodo": "FSAP_contents"\n  },\n  "FSAP_contents/v24_escape_of_water.npy": {\n   "bytes": 1500056,\n   "sha256": "8d9c3341a4c85ac780078b1ce295e0150c34348bc92f4e8eb43a05e0981db399",\n   "contenido": "a94ad4cc693a579dcf8e47c8d79f80796eeccf9cdc61f0f091ba67028ce96931",\n   "origen": "/home/claude/work/FSAP_contents/v24_escape_of_water.npy",\n   "nodo": "FSAP_contents"\n  },\n  "FSAP_contents/v24_accidental_damage.npy": {\n   "bytes": 1500056,\n   "sha256": "e57933947da62042f91b3d4156ffcba74131d13ea7abce90a0581fc04c4f9e54",\n   "contenido": "02e339ff1a89c3f29e9fbb3c1ce7745c26ad79304b17bf8305b78f211f872f6e",\n   "origen": "/home/claude/work/FSAP_contents/v24_accidental_damage.npy",\n   "nodo": "FSAP_contents"\n  },\n  "FSAP_contents/v24_flood.npy": {\n   "bytes": 1500056,\n   "sha256": "4aea53a01b546eab33b72268bd2c42c4f2eccf1037d1eb009fffde7c291afdda",\n   "contenido": "b62fcbbeae71f7114d61f92c467c75e12b481d35b0d9608bb4fa5e65c4e3514d",\n   "origen": "/home/claude/work/FSAP_contents/v24_flood.npy",\n   "nodo": "FSAP_contents"\n  },\n  "FSAP_contents/prod_theft.npy": {\n   "bytes": 1500112,\n   "sha256": "22035d69d31c08bebfe539f8ee09296f50c36b40f9f8038de13d7963a80eb8c4",\n   "contenido": "fa471b8c9d0a3be4da31c3d01f621ffac76532ae95fb0fc45c6f6e2f1cf6bf26",\n   "origen": "/home/claude/work/FSAP_contents/prod_theft.npy",\n   "nodo": "FSAP_contents"\n  },\n  "FSAP_contents/prod_escape_of_water.npy": {\n   "bytes": 1500112,\n   "sha256": "abafd8da5f6fac80251ff1a8baff4722847dacd2dfd1773007d0ef6d6a4cebec",\n   "contenido": "f6b43f005c642ed73589661ca5362a7c5c4701ac6a0dfcf42af5234c7089e7fe",\n   "origen": "/home/claude/work/FSAP_contents/prod_escape_of_water.npy",\n   "nodo": "FSAP_contents"\n  },\n  "FSAP_contents/prod_accidental_damage.npy": {\n   "bytes": 1500112,\n   "sha256": "26641f6003ed76e7835d5145e3b231ffb560c05c58a838304552eea38b745ca7",\n   "contenido": "c5005e29e122cd98ad8e1445b4ff022e881bf4402efb4b176c04ef4553e6cb56",\n   "origen": "/home/claude/work/FSAP_contents/prod_accidental_damage.npy",\n   "nodo": "FSAP_contents"\n  },\n  "FSAP_contents/prod_flood.npy": {\n   "bytes": 1500112,\n   "sha256": "ccdbfe8c871e39f9267eab378e8d92c545dab44bee281ef2bc9d2374b4f0e61b",\n   "contenido": "87988ea67c8bc26a02b02e45fb634158adf1b9c4c7404409adcfdedd9c7eb0e3",\n   "origen": "/home/claude/work/FSAP_contents/prod_flood.npy",\n   "nodo": "FSAP_contents"\n  },\n  "sel_FSAP_contents.parquet": {\n   "bytes": 1583200,\n   "sha256": "5eb0386a57ee6ef715989e06a3b5f3fc4f767069fa2e65582c7ff87f0d91060a",\n   "contenido": "3d908254711203ca9da04c4113973934a7587415b84f0bc99d6daf99dd2eadc9",\n   "origen": "/home/claude/work/sel_FSAP_contents.parquet",\n   "nodo": "FSAP_contents"\n  },\n  "FSAP_contents/v24.parquet": {\n   "bytes": 2907654,\n   "sha256": "b451b67b5a22020bd3ac2ba43fee68d1230a54087520433b6656e8dcfc08c161",\n   "contenido": "0293229b0a70a4d2d42df5e1f8bb0dc9b165d4be888db03694573bbdec786b29",\n   "origen": "/home/claude/work/FSAP_contents/v24.parquet",\n   "nodo": "FSAP_contents"\n  },\n  "FSAP_contents/prediccion_2025.parquet": {\n   "bytes": 2846950,\n   "sha256": "4e2721f75b11090cc3ebad37e97ba9060c0a5f6cbaf77346afcf27f56a90eb8a",\n   "contenido": "1ab6563930d58c91eff5e41a50032f1c2f12b22c7c44a740a5525eb099e5295b",\n   "origen": "/home/claude/work/FSAP_contents/prediccion_2025.parquet",\n   "nodo": "FSAP_contents"\n  },\n  "FSAP2_contents/v24_theft.npy": {\n   "bytes": 1500056,\n   "sha256": "d6b330d8f55508cd5e02ff459c851cfc523af68ec9f4c0e3dbf570299dffd1e0",\n   "contenido": "7107917ab6d0d5d51e547332db9fe78942be2e937cc3facbcff61c71687fa279",\n   "origen": "/home/claude/work/FSAP2_contents/v24_theft.npy",\n   "nodo": "FSAP2_contents"\n  },\n  "FSAP2_contents/v24_escape_of_water.npy": {\n   "bytes": 1500056,\n   "sha256": "6ee11df4a9fbb6f6747f7cb6c2a5579c9944285ac17b0dddb9689053df041ab3",\n   "contenido": "92fae148d1d97b8c1eb90c71fa2a1d298b49f13e04757a31467d094bf01791cc",\n   "origen": "/home/claude/work/FSAP2_contents/v24_escape_of_water.npy",\n   "nodo": "FSAP2_contents"\n  },\n  "FSAP2_contents/v24_accidental_damage.npy": {\n   "bytes": 1500056,\n   "sha256": "f276e677263db38f971491c24b246149671008da91c867f369abdd3983040ebf",\n   "contenido": "e960de5b9987702323fbf6d1e34b6ab219bcd85f44d10d5a188c934d893b76ab",\n   "origen": "/home/claude/work/FSAP2_contents/v24_accidental_damage.npy",\n   "nodo": "FSAP2_contents"\n  },\n  "FSAP2_contents/v24_flood.npy": {\n   "bytes": 1500056,\n   "sha256": "690e7b5b3c497c2df855d6f6923f96b64e5689a529d9485e46523e211b2e6205",\n   "contenido": "ead8ec1ee0c314469be019b1252e921a9298fa66db856bc1771b41cbb7624ac0",\n   "origen": "/home/claude/work/FSAP2_contents/v24_flood.npy",\n   "nodo": "FSAP2_contents"\n  },\n  "FSAP2_contents/prod_theft.npy": {\n   "bytes": 1500112,\n   "sha256": "9d80e12df99f83f36c854a80dacc4c6740e072bb7b7d65dc329234bb13421af3",\n   "contenido": "1a927dc5ef777c95e949079d24f5fc4262cf2415096071644805896878b34906",\n   "origen": "/home/claude/work/FSAP2_contents/prod_theft.npy",\n   "nodo": "FSAP2_contents"\n  },\n  "FSAP2_contents/prod_escape_of_water.npy": {\n   "bytes": 1500112,\n   "sha256": "7eadcd7a3e4a07457d4504c283513bbae835e79778005a97a9557d91fe8bc898",\n   "contenido": "48c21af0c89bd88414f418c9ce2d937ff2bb3c113118794381c343f602847cbb",\n   "origen": "/home/claude/work/FSAP2_contents/prod_escape_of_water.npy",\n   "nodo": "FSAP2_contents"\n  },\n  "FSAP2_contents/prod_accidental_damage.npy": {\n   "bytes": 1500112,\n   "sha256": "46bd228c7352a8e8f2a157ebceb28c3c1ab00446094982d9d3b2fe7273e18188",\n   "contenido": "1665c7d122790a4acabbd09aeab95146ae5797da0a4a07bb155f30538a872ac8",\n   "origen": "/home/claude/work/FSAP2_contents/prod_accidental_damage.npy",\n   "nodo": "FSAP2_contents"\n  },\n  "FSAP2_contents/prod_flood.npy": {\n   "bytes": 1500112,\n   "sha256": "88cf7525daf6c80c83d7f34c84f987be492c2566d05ded5be9a3b85cb904bc87",\n   "contenido": "79a9daa3ce6174dff505c36fb5f93badf37387bbdc93f7fc67ecb21ca6d02b9e",\n   "origen": "/home/claude/work/FSAP2_contents/prod_flood.npy",\n   "nodo": "FSAP2_contents"\n  },\n  "FSAP2_contents/v24.parquet": {\n   "bytes": 2907654,\n   "sha256": "c170d5bb8d9f3fce5ad61a2b8d220b8a3589a423ab702f94f2a469b064538fa2",\n   "contenido": "73364651f8348112037de7604cdefdce01c8da21b5101ed62708efcb189e2ca9",\n   "origen": "/home/claude/work/FSAP2_contents/v24.parquet",\n   "nodo": "FSAP2_contents"\n  },\n  "FSAP2_contents/prediccion_2025.parquet": {\n   "bytes": 2846950,\n   "sha256": "1bfdd92e10f76ded32d5e3b1b20e1e482105a99cfbc1f6624330064289a2d4c8",\n   "contenido": "3893b4c9fa4772ec7222ae8e0a7a4e05dea79773beeba143a5d7df9e178fd560",\n   "origen": "/home/claude/work/FSAP2_contents/prediccion_2025.parquet",\n   "nodo": "FSAP2_contents"\n  },\n  "sel_FSAXc4_contents.parquet": {\n   "bytes": 1583200,\n   "sha256": "8d1bc6092d030392a8cccb718c6977a9fe5d961a66830699f3f68b6277bf6e91",\n   "contenido": "ead7277d3490bef603db7eca4ae65918116cb91115418c0dc55118a29a747857",\n   "origen": "/home/claude/work/sel_FSAXc4_contents.parquet",\n   "nodo": "FSAXc4_contents"\n  },\n  "FSAXc4_contents/v24.parquet": {\n   "bytes": 2907654,\n   "sha256": "148c0fc478d5ecf6bfba4b7e49be6e092c305da96c74812e3007c1cff47e071b",\n   "contenido": "e55f3fa521d2d54e87fdf45777ca156709b8dd59eb04995ff327a8d287f44ec0",\n   "origen": "/home/claude/work/FSAXc4_contents/v24.parquet",\n   "nodo": "FSAXc4_contents"\n  },\n  "FSAXc4_contents/prediccion_2025.parquet": {\n   "bytes": 2846950,\n   "sha256": "7c1314c30b7dc66012acba3c31bdaf8614ba4003dfceb3c32c6d4947d5528693",\n   "contenido": "8e3d2185f0c53b77d7d0056fb503423cadd62ed2037b83cbd5b41ac47b373bcf",\n   "origen": "/home/claude/work/FSAXc4_contents/prediccion_2025.parquet",\n   "nodo": "FSAXc4_contents"\n  },\n  "CFS_motor_4s/v24.parquet": {\n   "bytes": 3958636,\n   "sha256": "1056f044b2b4ad4f44e00b707a5add3c8f270eb9f9d8eba50adc876b7f993d3b",\n   "contenido": "51da611d6fe392eacaa0a4cef21ec05d918b2b4aadb0d4e176a625d2bbd4d037",\n   "origen": "/home/claude/work/CFS_motor_4s/v24.parquet",\n   "nodo": "CFS_motor_4s"\n  },\n  "CFS_motor_4s/prediccion_2025.parquet": {\n   "bytes": 3739277,\n   "sha256": "1a561d3974c5ada13d39cbb495f5d9fe8103899800d1c787369c83d7157ad270",\n   "contenido": "6a89e9b402ad16fc5558457556de8cd7baf3b826e20d0582d786e36d1e837c32",\n   "origen": "/home/claude/work/CFS_motor_4s/prediccion_2025.parquet",\n   "nodo": "CFS_motor_4s"\n  },\n  "sel_CFSS_motor.parquet": {\n   "bytes": 2184468,\n   "sha256": "3999126bf6a41a40e143a698c962734342576b243d86d14a100323638b27cd30",\n   "contenido": "451a411d6a0766749ae965a0820de4b424aaaf87f110bb83a2d2001d9a13681d",\n   "origen": "/home/claude/work/sel_CFSS_motor.parquet",\n   "nodo": "CFSS_motor"\n  },\n  "CFSS_motor/v24.parquet": {\n   "bytes": 3958638,\n   "sha256": "7e7dc1ae042af79eff417300700d29ffc7ebf1470285d0b46a1cade635aa5c64",\n   "contenido": "ddf65387e08b4ca4b810a8bb5990e03dc8a01ec1831191c07940407cbee0cd98",\n   "origen": "/home/claude/work/CFSS_motor/v24.parquet",\n   "nodo": "CFSS_motor"\n  },\n  "CFSS_motor/prediccion_2025.parquet": {\n   "bytes": 3739277,\n   "sha256": "01879b6503846c2bcc787dffc568e2b4e54702e832a6188f8e28954a93636b0f",\n   "contenido": "f2c7f58463492168b95abf1288ce321dd5787f905d1d1bb78b26c6b6db03feda",\n   "origen": "/home/claude/work/CFSS_motor/prediccion_2025.parquet",\n   "nodo": "CFSS_motor"\n  },\n  "sel_CFSS4_buildings.parquet": {\n   "bytes": 1574274,\n   "sha256": "24720d1605dc96e864cc854e5d4b15140f810e45f824dd7b006181a7c77234dc",\n   "contenido": "f0fe47fda321f32a6886447d26bf0f7318d45f3bd2fb3b372dbb71a1f85609d6",\n   "origen": "/home/claude/work/sel_CFSS4_buildings.parquet",\n   "nodo": "CFSS4_buildings"\n  },\n  "CFSS4_buildings/v24.parquet": {\n   "bytes": 2867461,\n   "sha256": "5d62ca730fe58c37b2eba947cf3dd55bdbfe2f0c0d6664887aaf5e3aae985992",\n   "contenido": "a6a5c40bf934e4e0e77673c0e2d5199154b1c368676a5068a56d7d3a4431d48b",\n   "origen": "/home/claude/work/CFSS4_buildings/v24.parquet",\n   "nodo": "CFSS4_buildings"\n  },\n  "CFSS4_buildings/prediccion_2025.parquet": {\n   "bytes": 2820627,\n   "sha256": "a0e9b7f325f203659ddc5d28b01b2a6d99025a4b3600a3188f6c2135f6e97287",\n   "contenido": "fe236611aa1b5f4bf67f521e2cf0d8361375f08d04f2dd47bcecdce0f34417e6",\n   "origen": "/home/claude/work/CFSS4_buildings/prediccion_2025.parquet",\n   "nodo": "CFSS4_buildings"\n  },\n  "sel_FSAS4_buildings.parquet": {\n   "bytes": 1574274,\n   "sha256": "30fd205adbccd000ad5778e586ba50e9bfe878c7c27c5537c6050662d7d7fa0b",\n   "contenido": "eea1f6aea6bebe86dd5b365c93be4ac6d61702b0b41d74a9c1529d0e520f0224",\n   "origen": "/home/claude/work/sel_FSAS4_buildings.parquet",\n   "nodo": "FSAS4_buildings"\n  },\n  "FSAS4_buildings/v24.parquet": {\n   "bytes": 2867461,\n   "sha256": "efbc789e12135c4cb664adc8a75f483f50b5b26c41382c7a3cd480a7044555d7",\n   "contenido": "9bbd5ac062a378429f68cdb3622482968d6c309afed2951a8df731e39a5ed707",\n   "origen": "/home/claude/work/FSAS4_buildings/v24.parquet",\n   "nodo": "FSAS4_buildings"\n  },\n  "FSAS4_buildings/prediccion_2025.parquet": {\n   "bytes": 2820627,\n   "sha256": "e4c66502dafbd11706cede41cef0bf9157aa97d8fedc3186b90206b0fc51f513",\n   "contenido": "4f95dfb24f5f7168295054cbdc7168340c19ddea285dddf365ce25946fe599c4",\n   "origen": "/home/claude/work/FSAS4_buildings/prediccion_2025.parquet",\n   "nodo": "FSAS4_buildings"\n  },\n  "sel_MLP3_contents.parquet": {\n   "bytes": 1583200,\n   "sha256": "b0d6c5d6bd232a08fe68307a36882d2384f24100370a455d07a0e9c4999292bc",\n   "contenido": "0d43dc219dd678adf113bf7a8262d744d1a5bc74ed04109aa80b13e4496d099e",\n   "origen": "/home/claude/work/sel_MLP3_contents.parquet",\n   "nodo": "MLP3_contents"\n  },\n  "MLP3_contents/v24.parquet": {\n   "bytes": 2907654,\n   "sha256": "a32a06f39dc47a92bc4637fdf6cf7bba051f046d3395aab377d6f4b8abb60f09",\n   "contenido": "c5882d87bf4cd7d6f9a1986ee19f815b299d1a434e9980801b634c58ae6c49b1",\n   "origen": "/home/claude/work/MLP3_contents/v24.parquet",\n   "nodo": "MLP3_contents"\n  },\n  "MLP3_contents/prediccion_2025.parquet": {\n   "bytes": 2846950,\n   "sha256": "0dc7c784cbde615726f71c0f7fce473177a583de03dca0f06e69790227041d0d",\n   "contenido": "ec3a333653ace9fcc07e505a350ae2199272df0ec8f7e5273d87888b2c8248b8",\n   "origen": "/home/claude/work/MLP3_contents/prediccion_2025.parquet",\n   "nodo": "MLP3_contents"\n  },\n  "sel_B_CFST_motor_tendencia.parquet": {\n   "bytes": 2184468,\n   "sha256": "bab64358c15bc2937aebfbb52bf8614bb47030519bc895733796652ad6fa801e",\n   "contenido": "4c555e7e642ccb6716f2f2897b052cb6c08a41cef4ea0f134bd6098211a0fcf5",\n   "origen": "/mnt/user-data/uploads/Hakaton/Hackaton/Semana 1/Modelos_claude_b/runs/B_CFST_motor_tendencia/sel_B_CFST_motor_tendencia.parquet",\n   "nodo": "B_CFST_motor_tendencia"\n  },\n  "B_CFST_motor_tendencia/v24.parquet": {\n   "bytes": 3958636,\n   "sha256": "7c964a8afbff304b89badc17fd359f64bbd2937f3605573280cf794c719c883a",\n   "contenido": "b947fcda025e9854f2927db224abc5270cf716a38a7ae91c596f6272ba128dbc",\n   "origen": "/mnt/user-data/uploads/Hakaton/Hackaton/Semana 1/Modelos_claude_b/runs/B_CFST_motor_tendencia/v24.parquet",\n   "nodo": "B_CFST_motor_tendencia"\n  },\n  "B_CFST_motor_tendencia/prediccion_2025.parquet": {\n   "bytes": 3739277,\n   "sha256": "ce175b1f8bcf03869df82cbdc72bcb7576fbd0946b5ecd7a37eca3690c7cbd14",\n   "contenido": "ec1912f0241c5a0df24f89b1bf4de0c8ab641e4faba8c53f73a773b49518d1c2",\n   "origen": "/mnt/user-data/uploads/Hakaton/Hackaton/Semana 1/Modelos_claude_b/runs/B_CFST_motor_tendencia/prediccion_2025.parquet",\n   "nodo": "B_CFST_motor_tendencia"\n  },\n  "sel_B_CFST_buildings_tendencia.parquet": {\n   "bytes": 1574274,\n   "sha256": "b4ca50868d4f646fd99a2723e52cbc907f392e7d39985e9ded229a0aa48e6903",\n   "contenido": "c3d1abfbea05f10c764fd520865693f0b9c9f38966c69f6536fb7c5be8cffa40",\n   "origen": "/mnt/user-data/uploads/Hakaton/Hackaton/Semana 1/Modelos_claude_b/runs/B_CFST_buildings_tendencia/sel_B_CFST_buildings_tendencia.parquet",\n   "nodo": "B_CFST_buildings_tendencia"\n  },\n  "B_CFST_buildings_tendencia/v24.parquet": {\n   "bytes": 2867461,\n   "sha256": "548b3870f8e77e57d7134703845a59d53d83cd7c4142451e12f1494bb207f88d",\n   "contenido": "d84db3d67f3bdb02d19b07c215c6c96b1e99d9c4d81dc4d7cc9ec1af5f731845",\n   "origen": "/mnt/user-data/uploads/Hakaton/Hackaton/Semana 1/Modelos_claude_b/runs/B_CFST_buildings_tendencia/v24.parquet",\n   "nodo": "B_CFST_buildings_tendencia"\n  },\n  "B_CFST_buildings_tendencia/prediccion_2025.parquet": {\n   "bytes": 2820627,\n   "sha256": "d07de2fdf5237cd391ba7304d36d3b1b343bd42a562e9ea048b3dc73194cc724",\n   "contenido": "e21facaf57951f15763253ca1dcd360779c330225a25764fcef5b98e33dbd626",\n   "origen": "/mnt/user-data/uploads/Hakaton/Hackaton/Semana 1/Modelos_claude_b/runs/B_CFST_buildings_tendencia/prediccion_2025.parquet",\n   "nodo": "B_CFST_buildings_tendencia"\n  },\n  "sel_B_FSAST_buildings_tendencia.parquet": {\n   "bytes": 1574274,\n   "sha256": "849c9710b1370ce397096291cc7816283dce9ab06cd2027b3336aa81ffa270b3",\n   "contenido": "e584bb6a5cb622f93c28576cbc90705cf9d1d128cacc726fd3bb6d416c0e4ded",\n   "origen": "/mnt/user-data/uploads/Hakaton/Hackaton/Semana 1/Modelos_claude_b/runs/B_FSAST_buildings_tendencia/sel_B_FSAST_buildings_tendencia.parquet",\n   "nodo": "B_FSAST_buildings_tendencia"\n  },\n  "B_FSAST_buildings_tendencia/v24.parquet": {\n   "bytes": 2867461,\n   "sha256": "b5eedbdfacf43bb4867863bc8bb2f69009dd9204d51abec087034ffd295013c4",\n   "contenido": "d5213e778557c64815a8df1f26905156a26ebdb7d9f003da9bf659a6319c5c20",\n   "origen": "/mnt/user-data/uploads/Hakaton/Hackaton/Semana 1/Modelos_claude_b/runs/B_FSAST_buildings_tendencia/v24.parquet",\n   "nodo": "B_FSAST_buildings_tendencia"\n  },\n  "B_FSAST_buildings_tendencia/prediccion_2025.parquet": {\n   "bytes": 2820627,\n   "sha256": "38e947a6be4ae6856a9d54608ec033e3bf036e472b5427c4930d94fb6430e9dc",\n   "contenido": "15a21294e404af324988cdd9b0b6548925c4eca8d2a354f91ff08e1a32a043c6",\n   "origen": "/mnt/user-data/uploads/Hakaton/Hackaton/Semana 1/Modelos_claude_b/runs/B_FSAST_buildings_tendencia/prediccion_2025.parquet",\n   "nodo": "B_FSAST_buildings_tendencia"\n  },\n  "S22_claude_propio/S22_claude_propio.csv": {\n   "bytes": 13012725,\n   "sha256": "c21a02e14de4a89f892604239b7cfd9ac4e95ea7d175460a25ba806a5a3541b0",\n   "contenido": "c21a02e14de4a89f892604239b7cfd9ac4e95ea7d175460a25ba806a5a3541b0",\n   "origen": "/home/claude/work/S22_claude_propio/S22_claude_propio.csv",\n   "nodo": "S22_claude_propio"\n  },\n  "S22_claude_propio/prediccion_2025.parquet": {\n   "bytes": 8557978,\n   "sha256": "35ffd557f43a32ad9a379a1df22386fc671e1f02f799c24f24e39c59da4719cf",\n   "contenido": "29ae79020644eebe6c5da25df5a99dfba0c48831cb5d52798c03d8ac98a1ee81",\n   "origen": "/home/claude/work/S22_claude_propio/prediccion_2025.parquet",\n   "nodo": "S22_claude_propio"\n  },\n  "S30_claude_b/S30_claude_b.csv": {\n   "bytes": 13012442,\n   "sha256": "fd9528e0b1ee4b674dea63834a97523fd4ddc743d1495a422ce31a2eb76e3bc5",\n   "contenido": "fd9528e0b1ee4b674dea63834a97523fd4ddc743d1495a422ce31a2eb76e3bc5",\n   "origen": "/mnt/user-data/uploads/Hakaton/Hackaton/Semana 1/Modelos_claude_b/runs/S30_claude_b/S30_claude_b.csv",\n   "nodo": "S30_claude_b"\n  },\n  "S30_claude_b/prediccion_2025.parquet": {\n   "bytes": 8557976,\n   "sha256": "231c6d174fa28fb73da432d4bae1dfb2287e7ec8bf9f07fa37f7bad772543e13",\n   "contenido": "d0c6cf5f0f3da2f01a519138648c74ccc91f64b2c2d9386a52aab199abb74e94",\n   "origen": "/mnt/user-data/uploads/Hakaton/Hackaton/Semana 1/Modelos_claude_b/runs/S30_claude_b/prediccion_2025.parquet",\n   "nodo": "S30_claude_b"\n  }\n },\n "faltan": [\n  "CFS_motor/v23_own_damage.npz",\n  "CFS_motor/v23_third_party_prop.npz",\n  "CFS_motor/v23_bodily_injury.npz",\n  "CFS_motor/v23_theft.npz",\n  "CFS_motor/v23_glass_hail.npz",\n  "CFS_motor/v24_own_damage.npy",\n  "CFS_motor/v24_third_party_prop.npy",\n  "CFS_motor/v24_bodily_injury.npy",\n  "CFS_motor/v24_theft.npy",\n  "CFS_motor/v24_glass_hail.npy"\n ],\n "datos": {\n  "originales": {\n   "addresses.parquet": "2f1a9a157649a73ed1b0ee9b2caa3660b4c2a11766af3c64ab5131389079c0b1",\n   "claims.parquet": "323adf2e0f6c5975a026479d725d4cd04f51fce3aa069f897a99f664c6e392c0",\n   "customers.parquet": "065458a0b3aa9d3499e80b3b25b76baf0ef6245146477b6861f6ea35edf581eb",\n   "policies_buildings.parquet": "eb84372c23c16ed0619d89c8eb1396ab6916e5ec46a5907ce42c3dfce4a91cbe",\n   "policies_contents.parquet": "8c067cc80baed583100df999e1c3d5e389295702f7bac21d5331bbd639b83fcd",\n   "policies_motor.parquet": "4d2d7ae03971d5d6df3a71bbaf0ebc168250a6807b27d1561d38b638434dc95e",\n   "sample_submission.parquet": "9529f3b6a679d3410350baa7c2ef3fb085d1d8843a838360c5852dfe820d4d95",\n   "test_buildings.parquet": "7b2b96507de29241c70617c17df66146ed30b51f826ce5219bbda5da536d4957",\n   "test_contents.parquet": "c2212f62f3b7a21431c0073277549c3db5e63857565a8b88855e79a65fa1b8b6",\n   "test_motor.parquet": "8d28fb9fd091caf2a044913a013592226a4c0917d5c6fcc9c5e8cdcec2ab4fc5",\n   "weather.parquet": "b8407f0f3c7279a49297c969a4ab2905641b232910e316fb806730065987cd49"\n  },\n  "capa": {\n   "policies_motor": "d6b5e28f2b291c6ce285bc75c2426f63d78b2556b40273197f4a42a0652c9ef8",\n   "policies_buildings": "0378b2801849ba257cc06fc671e7d3ae4f7eade41a20efc6d08fb1eb6cbaf0f9",\n   "policies_contents": "09cdc7e56daf6765cf93c338351f84bf7d474d4fe148e88b35caa5f74f890dd4",\n   "test_motor": "27ddb9a3e6f9b56af223c9d50d01586a81d732dcb571a23bb793810e0655f43b",\n   "test_buildings": "99e983e3203fee57ee32f6e47166cdd8d0bb033455c236e1967839dac88276ae",\n   "test_contents": "55de22d263c113646568a5fe778b71635ff51a475bc78f747e69baad5e3ff39a",\n   "customers": "065458a0b3aa9d3499e80b3b25b76baf0ef6245146477b6861f6ea35edf581eb",\n   "addresses": "2f1a9a157649a73ed1b0ee9b2caa3660b4c2a11766af3c64ab5131389079c0b1",\n   "weather": "b8407f0f3c7279a49297c969a4ab2905641b232910e316fb806730065987cd49",\n   "claims": "31b1d8007ca9ee3a6dd00e73f99e7a7fc0d1fb8b13087b687342c999bd7b8336",\n   "sample_submission": "9529f3b6a679d3410350baa7c2ef3fb085d1d8843a838360c5852dfe820d4d95"\n  },\n  "decisiones": {\n   "Limpieza/runs/capendorse_20261017T010844Z_fbf8cc20/ledger/cumulative_retrospective_changes.parquet": {\n    "uso": "marca audit_capital_scale_confirmed (rama CAP02R a CAP06R)",\n    "sha256": "19682d0792858cf4b4df0172ef8658cfb1ff9a28f4431c5ba8e5bb5d25a704bd"\n   },\n   "Limpieza/runs/capfix_20261016T184904Z_099a22b1/ledger/all_capital_changes.parquet": {\n    "uso": "marca audit_capital_cap01",\n    "sha256": "4724971a9e788854cfa6e229b9acfd6b29cdb4012f3b2d87fd93b0cf662a921c"\n   },\n   "Decision_criterios_capital_claude/runs/dec_20261017T024905Z_c6e024cb/tables/autorizadas.csv": {\n    "uso": "marca audit_capital_scale_confirmed (lista D66)",\n    "sha256": "bf307355c2170b5d843751f0c543f6ffed25f318cb2e102a0fe07476b366d7c0"\n   }\n  },\n  "bases": {\n   "tables/t0_por_row_id.parquet": {\n    "bytes": 20018041,\n    "sha256": "caa8294e1d89dca5dfdc8b3de4db3f9a6fc2cba930506ec96a6abf5c31d3b340",\n    "contenido": "3bb22d23e7b5c10111c323733a808793acabd407918d0ab8ccb59ee6dfe0f70a"\n   },\n   "tables/asignacion_por_siniestro.parquet": {\n    "bytes": 4805740,\n    "sha256": "535327e1bbba95f171820d27e345988b3c2cbdeec6f86eefefaedefc52992ae5",\n    "contenido": "f760aed27500d172209434c89967949ded6e324b7ecd8a414c34dceb065f2ed3"\n   },\n   "tables/factores_desarrollo_conteos.csv": {\n    "bytes": 23362,\n    "sha256": "e02c7aa48b476ad7626cd7b2581745149830eabab19c4d2e95894a4e94d4e4b7",\n    "contenido": "e02c7aa48b476ad7626cd7b2581745149830eabab19c4d2e95894a4e94d4e4b7"\n   },\n   "tables/ibnr_por_peligro_anio.csv": {\n    "bytes": 4243,\n    "sha256": "4e1a723c0aa5691c64384fad32d6062b074a9794d72a4138a6f020f5ea572d71",\n    "contenido": "4e1a723c0aa5691c64384fad32d6062b074a9794d72a4138a6f020f5ea572d71"\n   },\n   "tables/completitud_2024_por_mes.csv": {\n    "bytes": 8572,\n    "sha256": "34ebf9ae430f8949f54d4e1d05cc1fd855e1bb40a86c0099a482ad44a70a9ff2",\n    "contenido": "34ebf9ae430f8949f54d4e1d05cc1fd855e1bb40a86c0099a482ad44a70a9ff2"\n   },\n   "tables/chain_ladder_conteos.csv": {\n    "bytes": 2366,\n    "sha256": "d643025262cbc9781febbb94217ffeb657ceadc0f750a17ea6c64d95ec0a88ef",\n    "contenido": "d643025262cbc9781febbb94217ffeb657ceadc0f750a17ea6c64d95ec0a88ef"\n   }\n  },\n  "f0": {\n   "tables/f0_motor.parquet": {\n    "bytes": 98141422,\n    "sha256": "c503d0142e83818b2c202747ddbce53882ed9add6f8309ced714655caf25b392",\n    "contenido": "20f1a2ecfb2c012244ad8c4222ec8503e98f2efcc021c40cf1089fcc70afbbe6"\n   },\n   "tables/f0_buildings.parquet": {\n    "bytes": 66919547,\n    "sha256": "2def2e2433062704e3762eae63f5b000e73d8284a62a6cc1a689aa63c2842454",\n    "contenido": "1506da562e28d3f28a9338b93c25424bfbdbe68c0efa1010867743977b0a4362"\n   },\n   "tables/f0_contents.parquet": {\n    "bytes": 65668739,\n    "sha256": "42328ddba499a77f9cc3c814506dc97fde7c3e2adf88cfc491058c7de92c76ff",\n    "contenido": "7b06b565833323090adcff185f2b5ab869b61936df75850acb69b80fe420c190"\n   },\n   "tables/particiones.parquet": {\n    "bytes": 15848536,\n    "sha256": "d365981d07feec50e487ff0d8fa4b9e5b8a00d5af8f8e521e03f2a158931035b",\n    "contenido": "0987960a3498983daa25bbd27bd4258e3e4dbfbf372bedb33ccaa02bba6c4105"\n   }\n  },\n  "f0_contrato_bloques": "5ddb45d8e2e07d2432f77cc7abe19226322b24391a813ae9e8e7efd071602972"\n },\n "mezcla": {\n  "insumos/exposicion_motor.parquet": {\n   "bytes": 8727967,\n   "sha256": "5555d11ca01e0afa16d6872295f24030f38be01e26e278a624d5fe53c47d54b1",\n   "contenido": "7a05a4165ca5029f508e188ff5dec1139a56fdd24f0b861655f61cefc99d70d5",\n   "origen": "Hackaton/Semana 1/Modelos_claude/insumos/exposicion_motor.parquet",\n   "uso": "exposición que lee el modo mezcla cuando no hay feat_<ramo>.parquet; columnas row_id, e_ef_coste y exposure"\n  },\n  "insumos/exposicion_buildings.parquet": {\n   "bytes": 6057353,\n   "sha256": "5d6a099be0d72858ef749257ed374dde80f62fe4ef07bc657df48b710f8e2673",\n   "contenido": "940ccd79edf673e1cbd7209cd8437f0fa8947cdca3f72156383833740fb124dc",\n   "origen": "Hackaton/Semana 1/Modelos_claude/insumos/exposicion_buildings.parquet",\n   "uso": "exposición que lee el modo mezcla cuando no hay feat_<ramo>.parquet; columnas row_id, e_ef_coste y exposure"\n  },\n  "insumos/exposicion_contents.parquet": {\n   "bytes": 6166024,\n   "sha256": "e115a4b0565cd134f47568b5cf0c35a9a931c72bd254967c17bf7bedd586ab5c",\n   "contenido": "d1b4ed9b35bc488254e1386b71ca3ec85699339bb1411d108a5186cdd9a0f881",\n   "origen": "Hackaton/Semana 1/Modelos_claude/insumos/exposicion_contents.parquet",\n   "uso": "exposición que lee el modo mezcla cuando no hay feat_<ramo>.parquet; columnas row_id, e_ef_coste y exposure"\n  }\n }\n}', 'receta/receta.json': '{\n "nombre": "Receta de S30_claude_b (selección final de la parte 1) y S22_claude_propio (respaldo)",\n "version": "2026-10-19",\n "entorno_de_referencia": "entorno/entorno_referencia.json",\n "entradas": {\n  "descripcion": "Rutas relativas a la carpeta «Hackaton/Semana 1» del proyecto (variable IAA_SEMANA1). Las etapas de capa base, bases actuariales y F0 que las producen desde los once Parquet originales están en la receta de datos (etapas 00 a 02).",\n  "archivos": [\n   "Datos/weather.parquet",\n   "Datos/sample_submission.parquet",\n   "Matriz_F0_claude/runs/f0_20261017T141732Z_65df7fca/tables/f0_motor.parquet",\n   "Matriz_F0_claude/runs/f0_20261017T141732Z_65df7fca/tables/f0_buildings.parquet",\n   "Matriz_F0_claude/runs/f0_20261017T141732Z_65df7fca/tables/f0_contents.parquet",\n   "Matriz_F0_claude/runs/f0_20261017T141732Z_65df7fca/tables/particiones.parquet",\n   "Matriz_F0_claude/runs/f0_20261017T141732Z_65df7fca/contrato_disponibilidad.json",\n   "Bases_actuariales_claude/runs/bases_20261017T120251Z_45443ff0/tables/asignacion_por_siniestro.parquet"\n  ]\n },\n "objetivos": {\n  "S30": {\n   "nodo": "S30_claude_b",\n   "csv": "S30_claude_b/S30_claude_b.csv",\n   "sha256": "fd9528e0b1ee4b674dea63834a97523fd4ddc743d1495a422ce31a2eb76e3bc5"\n  },\n  "S22": {\n   "nodo": "S22_claude_propio",\n   "csv": "S22_claude_propio/S22_claude_propio.csv",\n   "sha256": "c21a02e14de4a89f892604239b7cfd9ac4e95ea7d175460a25ba806a5a3541b0"\n  }\n },\n "nodos": [\n  {\n   "id": "variables",\n   "etapa": "variables",\n   "linea": "principal",\n   "script": "src/modelos/construir_variables.py",\n   "args": [],\n   "requiere": [],\n   "salidas": [\n    "feat_motor.parquet",\n    "feat_buildings.parquet",\n    "feat_contents.parquet"\n   ],\n   "verificar": [\n    "feat_motor.parquet",\n    "feat_buildings.parquet",\n    "feat_contents.parquet"\n   ],\n   "objetivos": [\n    "S22",\n    "S30"\n   ],\n   "evidencia": "construido una vez el 17-10 a las 13:14 (nube); comun.py con SHA 7b475c1b… igual al de los manifiestos de S03 y S04b",\n   "nota": "F0 (D68) + asignación de siniestros (D67) + clima original; variables F1, F2 y F4 con corte al 1 de enero del año del periodo",\n   "minutos_ref": 1\n  },\n  {\n   "id": "sel_total_F0124",\n   "etapa": "seleccion_v23",\n   "linea": "principal",\n   "script": "src/modelos/evaluar.py",\n   "args": [\n    "total_F0124",\n    "F0124",\n    "receta/especificaciones/cfg_total.json",\n    "total"\n   ],\n   "requiere": [\n    "variables"\n   ],\n   "salidas": [\n    "sel_total_F0124.parquet"\n   ],\n   "verificar": [\n    "sel_total_F0124.parquet"\n   ],\n   "objetivos": [\n    "S22",\n    "S30"\n   ],\n   "evidencia": "cola1.sh, log_cola1.txt",\n   "nota": "sus etiquetas y son las de V23 que usa mezcla3.labels",\n   "minutos_ref": 2\n  },\n  {\n   "id": "sel_total_F01234",\n   "etapa": "seleccion_v23",\n   "linea": "principal",\n   "script": "src/modelos/evaluar.py",\n   "args": [\n    "total_F01234",\n    "F01234",\n    "receta/especificaciones/cfg_total.json",\n    "total"\n   ],\n   "requiere": [\n    "variables"\n   ],\n   "salidas": [\n    "sel_total_F01234.parquet"\n   ],\n   "verificar": [\n    "sel_total_F01234.parquet"\n   ],\n   "objetivos": [\n    "S22",\n    "S30"\n   ],\n   "evidencia": "cola1.sh, log_cola1.txt",\n   "nota": "",\n   "minutos_ref": 2\n  },\n  {\n   "id": "sel_total_motor_red",\n   "etapa": "seleccion_v23",\n   "linea": "principal",\n   "script": "src/modelos/evaluar.py",\n   "args": [\n    "total_motor_red",\n    "F01234",\n    "receta/especificaciones/cfg_total_motor_red.json",\n    "total"\n   ],\n   "requiere": [\n    "variables"\n   ],\n   "salidas": [\n    "sel_total_motor_red.parquet"\n   ],\n   "verificar": [\n    "sel_total_motor_red.parquet"\n   ],\n   "objetivos": [\n    "S22",\n    "S30"\n   ],\n   "evidencia": "cola9.sh, log_cola9.txt",\n   "nota": "",\n   "minutos_ref": 2\n  },\n  {\n   "id": "sel_hS2",\n   "etapa": "seleccion_v23",\n   "linea": "principal",\n   "script": "src/modelos/evaluar.py",\n   "args": [\n    "hS2",\n    "F0124",\n    "receta/especificaciones/cfg_hogar_s2.json",\n    "total",\n    "buildings,contents"\n   ],\n   "requiere": [\n    "variables"\n   ],\n   "salidas": [\n    "sel_hS2.parquet"\n   ],\n   "verificar": [\n    "sel_hS2.parquet"\n   ],\n   "objetivos": [\n    "S22",\n    "S30"\n   ],\n   "evidencia": "cola19.sh, log_hS2.txt",\n   "nota": "",\n   "minutos_ref": 1\n  },\n  {\n   "id": "sel_hS3",\n   "etapa": "seleccion_v23",\n   "linea": "principal",\n   "script": "src/modelos/evaluar.py",\n   "args": [\n    "hS3",\n    "F0124",\n    "receta/especificaciones/cfg_hogar_s3.json",\n    "total",\n    "buildings"\n   ],\n   "requiere": [\n    "variables"\n   ],\n   "salidas": [\n    "sel_hS3.parquet"\n   ],\n   "verificar": [\n    "sel_hS3.parquet"\n   ],\n   "objetivos": [\n    "S22",\n    "S30"\n   ],\n   "evidencia": "cola29.sh, log_hS3.txt",\n   "nota": "",\n   "minutos_ref": 1\n  },\n  {\n   "id": "sel_cat1a",\n   "etapa": "seleccion_v23",\n   "linea": "principal",\n   "script": "src/modelos/cat_experimento.py",\n   "args": [\n    "cat1a",\n    "motor"\n   ],\n   "requiere": [\n    "variables"\n   ],\n   "salidas": [\n    "sel_cat1a.parquet"\n   ],\n   "verificar": [\n    "sel_cat1a.parquet"\n   ],\n   "objetivos": [\n    "S22",\n    "S30"\n   ],\n   "evidencia": "cola7.sh, log_cola7.txt (mejor iteración 393)",\n   "nota": "",\n   "minutos_ref": 4\n  },\n  {\n   "id": "S03_lgbm_F0124",\n   "etapa": "produccion",\n   "linea": "principal",\n   "script": "src/modelos/produccion.py",\n   "args": [\n    "S03_lgbm_F0124",\n    "F0124",\n    "receta/especificaciones/cfg_s03.json",\n    "--semillas=3"\n   ],\n   "requiere": [\n    "variables"\n   ],\n   "salidas": [\n    "S03_lgbm_F0124/v24.parquet",\n    "S03_lgbm_F0124/prediccion_2025.parquet",\n    "S03_lgbm_F0124/S03_lgbm_F0124.csv"\n   ],\n   "verificar": [\n    "S03_lgbm_F0124/v24.parquet",\n    "S03_lgbm_F0124/prediccion_2025.parquet",\n    "S03_lgbm_F0124/S03_lgbm_F0124.csv"\n   ],\n   "objetivos": [\n    "S22",\n    "S30"\n   ],\n   "evidencia": "README de Modelos_claude, «Cómo reproducir»; manifest.json de la corrida",\n   "nota": "",\n   "minutos_ref": 12\n  },\n  {\n   "id": "S04b_lgbm_F0124_F3autos",\n   "etapa": "produccion",\n   "linea": "principal",\n   "script": "src/modelos/produccion.py",\n   "args": [\n    "S04b_lgbm_F0124_F3autos",\n    "F01234",\n    "receta/especificaciones/cfg_m595_f3.json",\n    "--semillas=3",\n    "--solo=motor",\n    "--reusar=S03_lgbm_F0124"\n   ],\n   "requiere": [\n    "S03_lgbm_F0124"\n   ],\n   "salidas": [\n    "S04b_lgbm_F0124_F3autos/v24.parquet",\n    "S04b_lgbm_F0124_F3autos/prediccion_2025.parquet",\n    "S04b_lgbm_F0124_F3autos/S04b_lgbm_F0124_F3autos.csv"\n   ],\n   "verificar": [\n    "S04b_lgbm_F0124_F3autos/v24.parquet",\n    "S04b_lgbm_F0124_F3autos/prediccion_2025.parquet",\n    "S04b_lgbm_F0124_F3autos/S04b_lgbm_F0124_F3autos.csv"\n   ],\n   "objetivos": [\n    "S22",\n    "S30"\n   ],\n   "evidencia": "cola2.sh; manifest.json de la corrida",\n   "nota": "",\n   "minutos_ref": 8\n  },\n  {\n   "id": "S06c_lgbm_motor_red",\n   "etapa": "produccion",\n   "linea": "principal",\n   "script": "src/modelos/produccion.py",\n   "args": [\n    "S06c_lgbm_motor_red",\n    "F01234",\n    "receta/especificaciones/cfg_motor_red.json",\n    "--semillas=3",\n    "--solo=motor",\n    "--reusar=S04b_lgbm_F0124_F3autos"\n   ],\n   "requiere": [\n    "S04b_lgbm_F0124_F3autos"\n   ],\n   "salidas": [\n    "S06c_lgbm_motor_red/v24.parquet",\n    "S06c_lgbm_motor_red/prediccion_2025.parquet",\n    "S06c_lgbm_motor_red/S06c_lgbm_motor_red.csv"\n   ],\n   "verificar": [\n    "S06c_lgbm_motor_red/v24.parquet",\n    "S06c_lgbm_motor_red/prediccion_2025.parquet",\n    "S06c_lgbm_motor_red/S06c_lgbm_motor_red.csv"\n   ],\n   "objetivos": [\n    "S22",\n    "S30"\n   ],\n   "evidencia": "cola9.sh, log_cola9.txt",\n   "nota": "",\n   "minutos_ref": 8\n  },\n  {\n   "id": "S08h_hogar_F3s2",\n   "etapa": "produccion",\n   "linea": "principal",\n   "script": "src/modelos/produccion.py",\n   "args": [\n    "S08h_hogar_F3s2",\n    "F0124",\n    "receta/especificaciones/cfg_hogar_s2.json",\n    "--solo=buildings,contents",\n    "--reusar=S04b_lgbm_F0124_F3autos",\n    "--semillas=3"\n   ],\n   "requiere": [\n    "S04b_lgbm_F0124_F3autos"\n   ],\n   "salidas": [\n    "S08h_hogar_F3s2/v24.parquet",\n    "S08h_hogar_F3s2/prediccion_2025.parquet",\n    "S08h_hogar_F3s2/S08h_hogar_F3s2.csv"\n   ],\n   "verificar": [\n    "S08h_hogar_F3s2/v24.parquet",\n    "S08h_hogar_F3s2/prediccion_2025.parquet",\n    "S08h_hogar_F3s2/S08h_hogar_F3s2.csv"\n   ],\n   "objetivos": [\n    "S22",\n    "S30"\n   ],\n   "evidencia": "cola19.sh, log_S08h.txt",\n   "nota": "",\n   "minutos_ref": 4\n  },\n  {\n   "id": "S12h_edif_md1000",\n   "etapa": "produccion",\n   "linea": "principal",\n   "script": "src/modelos/produccion.py",\n   "args": [\n    "S12h_edif_md1000",\n    "F0124",\n    "receta/especificaciones/cfg_hogar_s3.json",\n    "--solo=buildings",\n    "--reusar=S08h_hogar_F3s2",\n    "--semillas=3"\n   ],\n   "requiere": [\n    "S08h_hogar_F3s2"\n   ],\n   "salidas": [\n    "S12h_edif_md1000/v24.parquet",\n    "S12h_edif_md1000/prediccion_2025.parquet",\n    "S12h_edif_md1000/S12h_edif_md1000.csv"\n   ],\n   "verificar": [\n    "S12h_edif_md1000/v24.parquet",\n    "S12h_edif_md1000/prediccion_2025.parquet",\n    "S12h_edif_md1000/S12h_edif_md1000.csv"\n   ],\n   "objetivos": [\n    "S22",\n    "S30"\n   ],\n   "evidencia": "cola29.sh, log_S12h.txt",\n   "nota": "",\n   "minutos_ref": 3\n  },\n  {\n   "id": "CAT_motor",\n   "etapa": "produccion",\n   "linea": "principal",\n   "script": "src/modelos/cat_produccion.py",\n   "args": [\n    "CAT_motor",\n    "motor",\n    "393",\n    "--semillas=2"\n   ],\n   "requiere": [\n    "variables"\n   ],\n   "salidas": [\n    "CAT_motor/v24.parquet",\n    "CAT_motor/prediccion_2025.parquet"\n   ],\n   "verificar": [\n    "CAT_motor/v24.parquet",\n    "CAT_motor/prediccion_2025.parquet"\n   ],\n   "objetivos": [\n    "S22",\n    "S30"\n   ],\n   "evidencia": "cola10.sh",\n   "nota": "",\n   "minutos_ref": 40\n  },\n  {\n   "id": "CAT_motor_s23",\n   "etapa": "produccion",\n   "linea": "principal",\n   "script": "src/modelos/cat_produccion.py",\n   "args": [\n    "CAT_motor_s23",\n    "motor",\n    "393",\n    "--semillas=2",\n    "--semilla_inicial=2"\n   ],\n   "requiere": [\n    "variables"\n   ],\n   "salidas": [\n    "CAT_motor_s23/v24.parquet",\n    "CAT_motor_s23/prediccion_2025.parquet"\n   ],\n   "verificar": [\n    "CAT_motor_s23/v24.parquet",\n    "CAT_motor_s23/prediccion_2025.parquet"\n   ],\n   "objetivos": [\n    "S22",\n    "S30"\n   ],\n   "evidencia": "cola30.sh y cola31.sh, log_cat_s23.txt",\n   "nota": "",\n   "minutos_ref": 40\n  },\n  {\n   "id": "CAT_motor_4s",\n   "etapa": "produccion",\n   "linea": "principal",\n   "script": "src/modelos/combinar_semillas.py",\n   "args": [\n    "CAT_motor_4s",\n    "CAT_motor",\n    "CAT_motor_s23"\n   ],\n   "requiere": [\n    "CAT_motor",\n    "CAT_motor_s23"\n   ],\n   "salidas": [\n    "CAT_motor_4s/v24.parquet",\n    "CAT_motor_4s/prediccion_2025.parquet"\n   ],\n   "verificar": [\n    "CAT_motor_4s/v24.parquet",\n    "CAT_motor_4s/prediccion_2025.parquet"\n   ],\n   "objetivos": [\n    "S22",\n    "S30"\n   ],\n   "evidencia": "resumen.json de CAT_motor_4s (corridas CAT_motor y CAT_motor_s23, semillas [2, 2])",\n   "nota": "",\n   "minutos_ref": 1\n  },\n  {\n   "id": "CFS_motor_v23",\n   "etapa": "partes",\n   "linea": "principal",\n   "script": "src/modelos/cat_frecsev.py",\n   "args": [\n    "CFS_motor_v23",\n    "motor"\n   ],\n   "requiere": [\n    "variables"\n   ],\n   "salidas": [\n    "CFS_motor_v23/v23_own_damage.npz",\n    "CFS_motor_v23/v23_third_party_prop.npz",\n    "CFS_motor_v23/v23_bodily_injury.npz",\n    "CFS_motor_v23/v23_theft.npz",\n    "CFS_motor_v23/v23_glass_hail.npz",\n    "sel_CFS_motor_v23.parquet"\n   ],\n   "verificar": [\n    "CFS_motor_v23/v23_own_damage.npz",\n    "CFS_motor_v23/v23_third_party_prop.npz",\n    "CFS_motor_v23/v23_bodily_injury.npz",\n    "CFS_motor_v23/v23_theft.npz",\n    "CFS_motor_v23/v23_glass_hail.npz",\n    "sel_CFS_motor_v23.parquet"\n   ],\n   "objetivos": [\n    "S22",\n    "S30"\n   ],\n   "evidencia": "cola37.sh, log_cfs_motor_v23.txt",\n   "nota": "",\n   "minutos_ref": 60\n  },\n  {\n   "id": "CFS_motor",\n   "etapa": "partes",\n   "linea": "principal",\n   "script": "src/modelos/cat_frecsev.py",\n   "args": [\n    "CFS_motor",\n    "motor",\n    "--prod",\n    "--semillas=2"\n   ],\n   "requiere": [\n    "variables"\n   ],\n   "salidas": [\n    "CFS_motor/v23_own_damage.npz",\n    "CFS_motor/v23_third_party_prop.npz",\n    "CFS_motor/v23_bodily_injury.npz",\n    "CFS_motor/v23_theft.npz",\n    "CFS_motor/v23_glass_hail.npz",\n    "CFS_motor/v24_own_damage.npy",\n    "CFS_motor/v24_third_party_prop.npy",\n    "CFS_motor/v24_bodily_injury.npy",\n    "CFS_motor/v24_theft.npy",\n    "CFS_motor/v24_glass_hail.npy",\n    "CFS_motor/prod_own_damage.npy",\n    "CFS_motor/prod_third_party_prop.npy",\n    "CFS_motor/prod_bodily_injury.npy",\n    "CFS_motor/prod_theft.npy",\n    "CFS_motor/prod_glass_hail.npy",\n    "sel_CFS_motor.parquet",\n    "CFS_motor/v24.parquet",\n    "CFS_motor/prediccion_2025.parquet"\n   ],\n   "verificar": [\n    "CFS_motor/prod_own_damage.npy",\n    "CFS_motor/prod_third_party_prop.npy",\n    "CFS_motor/prod_bodily_injury.npy",\n    "CFS_motor/prod_theft.npy",\n    "CFS_motor/prod_glass_hail.npy",\n    "CFS_motor/prediccion_2025.parquet"\n   ],\n   "objetivos": [\n    "S22",\n    "S30"\n   ],\n   "evidencia": "cola23.sh y cola25.sh (V23), cola29.sh (--prod), log_cfs_motor.txt y log_cfs_motor_prod.txt",\n   "nota": "la V23 y la V24 originales de esta corrida se calcularon en float64, antes del paso a float32 que documenta cat_frecsev.py; aguas abajo solo se usan sus partes de 2025 (hechas ya en float32) y sus mejores iteraciones, iguales a las de CFS_motor_v23",\n   "minutos_ref": 150\n  },\n  {\n   "id": "CFS_motor_s23",\n   "etapa": "partes",\n   "linea": "principal",\n   "script": "src/modelos/cat_frecsev.py",\n   "args": [\n    "CFS_motor_s23",\n    "motor",\n    "--prod",\n    "--semillas=2",\n    "--semilla_inicial=2",\n    "--desde=CFS_motor"\n   ],\n   "requiere": [\n    "CFS_motor"\n   ],\n   "salidas": [\n    "CFS_motor_s23/v24_own_damage.npy",\n    "CFS_motor_s23/v24_third_party_prop.npy",\n    "CFS_motor_s23/v24_bodily_injury.npy",\n    "CFS_motor_s23/v24_theft.npy",\n    "CFS_motor_s23/v24_glass_hail.npy",\n    "CFS_motor_s23/prod_own_damage.npy",\n    "CFS_motor_s23/prod_third_party_prop.npy",\n    "CFS_motor_s23/prod_bodily_injury.npy",\n    "CFS_motor_s23/prod_theft.npy",\n    "CFS_motor_s23/prod_glass_hail.npy",\n    "CFS_motor_s23/v24.parquet",\n    "CFS_motor_s23/prediccion_2025.parquet"\n   ],\n   "verificar": [\n    "CFS_motor_s23/v24_own_damage.npy",\n    "CFS_motor_s23/v24_third_party_prop.npy",\n    "CFS_motor_s23/v24_bodily_injury.npy",\n    "CFS_motor_s23/v24_theft.npy",\n    "CFS_motor_s23/v24_glass_hail.npy",\n    "CFS_motor_s23/prod_own_damage.npy",\n    "CFS_motor_s23/prod_third_party_prop.npy",\n    "CFS_motor_s23/prod_bodily_injury.npy",\n    "CFS_motor_s23/prod_theft.npy",\n    "CFS_motor_s23/prod_glass_hail.npy",\n    "CFS_motor_s23/v24.parquet",\n    "CFS_motor_s23/prediccion_2025.parquet"\n   ],\n   "objetivos": [\n    "S22",\n    "S30"\n   ],\n   "evidencia": "cola32.sh, log_cfs_motor_s23.txt (5.736 s)",\n   "nota": "",\n   "minutos_ref": 96\n  },\n  {\n   "id": "CFS_buildings",\n   "etapa": "partes",\n   "linea": "principal",\n   "script": "src/modelos/cat_frecsev.py",\n   "args": [\n    "CFS_buildings",\n    "buildings",\n    "--prod",\n    "--semillas=2",\n    "--f3s2"\n   ],\n   "requiere": [\n    "variables"\n   ],\n   "salidas": [\n    "CFS_buildings/v23_storm.npz",\n    "CFS_buildings/v23_flood.npz",\n    "CFS_buildings/v23_escape_of_water.npz",\n    "CFS_buildings/v23_subsidence.npz",\n    "CFS_buildings/v23_fire.npz",\n    "CFS_buildings/v23_accidental_damage.npz",\n    "CFS_buildings/v24_storm.npy",\n    "CFS_buildings/v24_flood.npy",\n    "CFS_buildings/v24_escape_of_water.npy",\n    "CFS_buildings/v24_subsidence.npy",\n    "CFS_buildings/v24_fire.npy",\n    "CFS_buildings/v24_accidental_damage.npy",\n    "CFS_buildings/prod_storm.npy",\n    "CFS_buildings/prod_flood.npy",\n    "CFS_buildings/prod_escape_of_water.npy",\n    "CFS_buildings/prod_subsidence.npy",\n    "CFS_buildings/prod_fire.npy",\n    "CFS_buildings/prod_accidental_damage.npy",\n    "sel_CFS_buildings.parquet",\n    "CFS_buildings/v24.parquet",\n    "CFS_buildings/prediccion_2025.parquet"\n   ],\n   "verificar": [\n    "CFS_buildings/v23_storm.npz",\n    "CFS_buildings/v23_flood.npz",\n    "CFS_buildings/v23_escape_of_water.npz",\n    "CFS_buildings/v23_subsidence.npz",\n    "CFS_buildings/v23_fire.npz",\n    "CFS_buildings/v23_accidental_damage.npz",\n    "CFS_buildings/v24_storm.npy",\n    "CFS_buildings/v24_flood.npy",\n    "CFS_buildings/v24_escape_of_water.npy",\n    "CFS_buildings/v24_subsidence.npy",\n    "CFS_buildings/v24_fire.npy",\n    "CFS_buildings/v24_accidental_damage.npy",\n    "CFS_buildings/prod_storm.npy",\n    "CFS_buildings/prod_flood.npy",\n    "CFS_buildings/prod_escape_of_water.npy",\n    "CFS_buildings/prod_subsidence.npy",\n    "CFS_buildings/prod_fire.npy",\n    "CFS_buildings/prod_accidental_damage.npy",\n    "sel_CFS_buildings.parquet",\n    "CFS_buildings/v24.parquet",\n    "CFS_buildings/prediccion_2025.parquet"\n   ],\n   "objetivos": [\n    "S22",\n    "S30"\n   ],\n   "evidencia": "cola31.sh, log_cfs_buildings.txt (2.606 s)",\n   "nota": "",\n   "minutos_ref": 44\n  },\n  {\n   "id": "CFS_buildings_s23",\n   "etapa": "partes",\n   "linea": "principal",\n   "script": "src/modelos/cat_frecsev.py",\n   "args": [\n    "CFS_buildings_s23",\n    "buildings",\n    "--prod",\n    "--semillas=2",\n    "--f3s2",\n    "--desde=CFS_buildings",\n    "--semilla_inicial=2"\n   ],\n   "requiere": [\n    "CFS_buildings"\n   ],\n   "salidas": [\n    "CFS_buildings_s23/v24_storm.npy",\n    "CFS_buildings_s23/v24_flood.npy",\n    "CFS_buildings_s23/v24_escape_of_water.npy",\n    "CFS_buildings_s23/v24_subsidence.npy",\n    "CFS_buildings_s23/v24_fire.npy",\n    "CFS_buildings_s23/v24_accidental_damage.npy",\n    "CFS_buildings_s23/prod_storm.npy",\n    "CFS_buildings_s23/prod_flood.npy",\n    "CFS_buildings_s23/prod_escape_of_water.npy",\n    "CFS_buildings_s23/prod_subsidence.npy",\n    "CFS_buildings_s23/prod_fire.npy",\n    "CFS_buildings_s23/prod_accidental_damage.npy",\n    "CFS_buildings_s23/v24.parquet",\n    "CFS_buildings_s23/prediccion_2025.parquet"\n   ],\n   "verificar": [\n    "CFS_buildings_s23/v24_storm.npy",\n    "CFS_buildings_s23/v24_flood.npy",\n    "CFS_buildings_s23/v24_escape_of_water.npy",\n    "CFS_buildings_s23/v24_subsidence.npy",\n    "CFS_buildings_s23/v24_fire.npy",\n    "CFS_buildings_s23/v24_accidental_damage.npy",\n    "CFS_buildings_s23/prod_storm.npy",\n    "CFS_buildings_s23/prod_flood.npy",\n    "CFS_buildings_s23/prod_escape_of_water.npy",\n    "CFS_buildings_s23/prod_subsidence.npy",\n    "CFS_buildings_s23/prod_fire.npy",\n    "CFS_buildings_s23/prod_accidental_damage.npy",\n    "CFS_buildings_s23/v24.parquet",\n    "CFS_buildings_s23/prediccion_2025.parquet"\n   ],\n   "objetivos": [\n    "S22",\n    "S30"\n   ],\n   "evidencia": "cola42.sh, log_cfs_buildings_s23.txt (2.205 s)",\n   "nota": "",\n   "minutos_ref": 37\n  },\n  {\n   "id": "FSA_contents",\n   "etapa": "produccion",\n   "linea": "principal",\n   "script": "src/modelos/frecsev2.py",\n   "args": [\n    "FSA_contents",\n    "contents",\n    "--prod",\n    "--semillas=2",\n    "--aditivo"\n   ],\n   "requiere": [\n    "variables"\n   ],\n   "salidas": [\n    "sel_FSA_contents.parquet",\n    "FSA_contents/v24.parquet",\n    "FSA_contents/prediccion_2025.parquet"\n   ],\n   "verificar": [\n    "sel_FSA_contents.parquet",\n    "FSA_contents/v24.parquet",\n    "FSA_contents/prediccion_2025.parquet"\n   ],\n   "objetivos": [\n    "S22",\n    "S30"\n   ],\n   "evidencia": "cola33.sh, log_fsa_contents.txt",\n   "nota": "",\n   "minutos_ref": 20\n  },\n  {\n   "id": "FSAP_buildings",\n   "etapa": "partes",\n   "linea": "principal",\n   "script": "src/modelos/frecsev2.py",\n   "args": [\n    "FSAP_buildings",\n    "buildings",\n    "--prod",\n    "--semillas=2",\n    "--aditivo",\n    "--partes"\n   ],\n   "requiere": [\n    "variables"\n   ],\n   "salidas": [\n    "FSAP_buildings/v23_storm.npz",\n    "FSAP_buildings/v23_flood.npz",\n    "FSAP_buildings/v23_escape_of_water.npz",\n    "FSAP_buildings/v23_subsidence.npz",\n    "FSAP_buildings/v23_fire.npz",\n    "FSAP_buildings/v23_accidental_damage.npz",\n    "FSAP_buildings/v24_storm.npy",\n    "FSAP_buildings/v24_flood.npy",\n    "FSAP_buildings/v24_escape_of_water.npy",\n    "FSAP_buildings/v24_subsidence.npy",\n    "FSAP_buildings/v24_fire.npy",\n    "FSAP_buildings/v24_accidental_damage.npy",\n    "FSAP_buildings/prod_storm.npy",\n    "FSAP_buildings/prod_flood.npy",\n    "FSAP_buildings/prod_escape_of_water.npy",\n    "FSAP_buildings/prod_subsidence.npy",\n    "FSAP_buildings/prod_fire.npy",\n    "FSAP_buildings/prod_accidental_damage.npy",\n    "sel_FSAP_buildings.parquet",\n    "FSAP_buildings/v24.parquet",\n    "FSAP_buildings/prediccion_2025.parquet"\n   ],\n   "verificar": [\n    "FSAP_buildings/v23_storm.npz",\n    "FSAP_buildings/v23_flood.npz",\n    "FSAP_buildings/v23_escape_of_water.npz",\n    "FSAP_buildings/v23_subsidence.npz",\n    "FSAP_buildings/v23_fire.npz",\n    "FSAP_buildings/v23_accidental_damage.npz",\n    "FSAP_buildings/v24_storm.npy",\n    "FSAP_buildings/v24_flood.npy",\n    "FSAP_buildings/v24_escape_of_water.npy",\n    "FSAP_buildings/v24_subsidence.npy",\n    "FSAP_buildings/v24_fire.npy",\n    "FSAP_buildings/v24_accidental_damage.npy",\n    "FSAP_buildings/prod_storm.npy",\n    "FSAP_buildings/prod_flood.npy",\n    "FSAP_buildings/prod_escape_of_water.npy",\n    "FSAP_buildings/prod_subsidence.npy",\n    "FSAP_buildings/prod_fire.npy",\n    "FSAP_buildings/prod_accidental_damage.npy",\n    "sel_FSAP_buildings.parquet",\n    "FSAP_buildings/v24.parquet",\n    "FSAP_buildings/prediccion_2025.parquet"\n   ],\n   "objetivos": [\n    "S22",\n    "S30"\n   ],\n   "evidencia": "cola39.sh, log_fsap.txt",\n   "nota": "",\n   "minutos_ref": 25\n  },\n  {\n   "id": "FSAP2_buildings",\n   "etapa": "partes",\n   "linea": "principal",\n   "script": "src/modelos/frecsev2.py",\n   "args": [\n    "FSAP2_buildings",\n    "buildings",\n    "--prod",\n    "--semillas=2",\n    "--aditivo",\n    "--partes",\n    "--desde=FSAP_buildings",\n    "--semilla_inicial=2"\n   ],\n   "requiere": [\n    "FSAP_buildings"\n   ],\n   "salidas": [\n    "FSAP2_buildings/v24_storm.npy",\n    "FSAP2_buildings/v24_flood.npy",\n    "FSAP2_buildings/v24_escape_of_water.npy",\n    "FSAP2_buildings/v24_subsidence.npy",\n    "FSAP2_buildings/v24_fire.npy",\n    "FSAP2_buildings/v24_accidental_damage.npy",\n    "FSAP2_buildings/prod_storm.npy",\n    "FSAP2_buildings/prod_flood.npy",\n    "FSAP2_buildings/prod_escape_of_water.npy",\n    "FSAP2_buildings/prod_subsidence.npy",\n    "FSAP2_buildings/prod_fire.npy",\n    "FSAP2_buildings/prod_accidental_damage.npy",\n    "FSAP2_buildings/v24.parquet",\n    "FSAP2_buildings/prediccion_2025.parquet"\n   ],\n   "verificar": [\n    "FSAP2_buildings/v24_storm.npy",\n    "FSAP2_buildings/v24_flood.npy",\n    "FSAP2_buildings/v24_escape_of_water.npy",\n    "FSAP2_buildings/v24_subsidence.npy",\n    "FSAP2_buildings/v24_fire.npy",\n    "FSAP2_buildings/v24_accidental_damage.npy",\n    "FSAP2_buildings/prod_storm.npy",\n    "FSAP2_buildings/prod_flood.npy",\n    "FSAP2_buildings/prod_escape_of_water.npy",\n    "FSAP2_buildings/prod_subsidence.npy",\n    "FSAP2_buildings/prod_fire.npy",\n    "FSAP2_buildings/prod_accidental_damage.npy",\n    "FSAP2_buildings/v24.parquet",\n    "FSAP2_buildings/prediccion_2025.parquet"\n   ],\n   "objetivos": [\n    "S22",\n    "S30"\n   ],\n   "evidencia": "cola40.sh y cola40b.sh, log_fsap2.txt",\n   "nota": "",\n   "minutos_ref": 15\n  },\n  {\n   "id": "FSAP_contents",\n   "etapa": "partes",\n   "linea": "principal",\n   "script": "src/modelos/frecsev2.py",\n   "args": [\n    "FSAP_contents",\n    "contents",\n    "--prod",\n    "--semillas=2",\n    "--aditivo",\n    "--partes"\n   ],\n   "requiere": [\n    "variables"\n   ],\n   "salidas": [\n    "FSAP_contents/v23_theft.npz",\n    "FSAP_contents/v23_escape_of_water.npz",\n    "FSAP_contents/v23_accidental_damage.npz",\n    "FSAP_contents/v23_flood.npz",\n    "FSAP_contents/v24_theft.npy",\n    "FSAP_contents/v24_escape_of_water.npy",\n    "FSAP_contents/v24_accidental_damage.npy",\n    "FSAP_contents/v24_flood.npy",\n    "FSAP_contents/prod_theft.npy",\n    "FSAP_contents/prod_escape_of_water.npy",\n    "FSAP_contents/prod_accidental_damage.npy",\n    "FSAP_contents/prod_flood.npy",\n    "sel_FSAP_contents.parquet",\n    "FSAP_contents/v24.parquet",\n    "FSAP_contents/prediccion_2025.parquet"\n   ],\n   "verificar": [\n    "FSAP_contents/v23_theft.npz",\n    "FSAP_contents/v23_escape_of_water.npz",\n    "FSAP_contents/v23_accidental_damage.npz",\n    "FSAP_contents/v23_flood.npz",\n    "FSAP_contents/v24_theft.npy",\n    "FSAP_contents/v24_escape_of_water.npy",\n    "FSAP_contents/v24_accidental_damage.npy",\n    "FSAP_contents/v24_flood.npy",\n    "FSAP_contents/prod_theft.npy",\n    "FSAP_contents/prod_escape_of_water.npy",\n    "FSAP_contents/prod_accidental_damage.npy",\n    "FSAP_contents/prod_flood.npy",\n    "sel_FSAP_contents.parquet",\n    "FSAP_contents/v24.parquet",\n    "FSAP_contents/prediccion_2025.parquet"\n   ],\n   "objetivos": [\n    "S22",\n    "S30"\n   ],\n   "evidencia": "cola39.sh, log_fsap.txt",\n   "nota": "",\n   "minutos_ref": 25\n  },\n  {\n   "id": "FSAP2_contents",\n   "etapa": "partes",\n   "linea": "principal",\n   "script": "src/modelos/frecsev2.py",\n   "args": [\n    "FSAP2_contents",\n    "contents",\n    "--prod",\n    "--semillas=2",\n    "--aditivo",\n    "--partes",\n    "--desde=FSAP_contents",\n    "--semilla_inicial=2"\n   ],\n   "requiere": [\n    "FSAP_contents"\n   ],\n   "salidas": [\n    "FSAP2_contents/v24_theft.npy",\n    "FSAP2_contents/v24_escape_of_water.npy",\n    "FSAP2_contents/v24_accidental_damage.npy",\n    "FSAP2_contents/v24_flood.npy",\n    "FSAP2_contents/prod_theft.npy",\n    "FSAP2_contents/prod_escape_of_water.npy",\n    "FSAP2_contents/prod_accidental_damage.npy",\n    "FSAP2_contents/prod_flood.npy",\n    "FSAP2_contents/v24.parquet",\n    "FSAP2_contents/prediccion_2025.parquet"\n   ],\n   "verificar": [\n    "FSAP2_contents/v24_theft.npy",\n    "FSAP2_contents/v24_escape_of_water.npy",\n    "FSAP2_contents/v24_accidental_damage.npy",\n    "FSAP2_contents/v24_flood.npy",\n    "FSAP2_contents/prod_theft.npy",\n    "FSAP2_contents/prod_escape_of_water.npy",\n    "FSAP2_contents/prod_accidental_damage.npy",\n    "FSAP2_contents/prod_flood.npy",\n    "FSAP2_contents/v24.parquet",\n    "FSAP2_contents/prediccion_2025.parquet"\n   ],\n   "objetivos": [\n    "S22",\n    "S30"\n   ],\n   "evidencia": "cola40.sh y cola40b.sh, log_fsap2.txt",\n   "nota": "",\n   "minutos_ref": 15\n  },\n  {\n   "id": "FSAXc4_contents",\n   "etapa": "severidad",\n   "linea": "principal",\n   "script": "src/modelos/severidad_cfs3.py",\n   "args": [\n    "FSAP_contents",\n    "contents",\n    "FSAXc4_contents",\n    "--v24=FSAP_contents,FSAP2_contents",\n    "--prod=FSAP_contents,FSAP2_contents"\n   ],\n   "requiere": [\n    "FSAP_contents",\n    "FSAP2_contents"\n   ],\n   "salidas": [\n    "sel_FSAXc4_contents.parquet",\n    "FSAXc4_contents/v24.parquet",\n    "FSAXc4_contents/prediccion_2025.parquet"\n   ],\n   "verificar": [\n    "sel_FSAXc4_contents.parquet",\n    "FSAXc4_contents/v24.parquet",\n    "FSAXc4_contents/prediccion_2025.parquet"\n   ],\n   "objetivos": [\n    "S22",\n    "S30"\n   ],\n   "evidencia": "resumen.json de FSAXc4_contents (origen v23 FSAP_contents; v24 y prod FSAP_contents y FSAP2_contents; lambda 20)",\n   "nota": "",\n   "minutos_ref": 1\n  },\n  {\n   "id": "CFS_motor_4s",\n   "etapa": "severidad",\n   "linea": "principal",\n   "script": "src/modelos/combinar_semillas.py",\n   "args": [\n    "CFS_motor_4s",\n    "CFS_motor",\n    "CFS_motor_s23"\n   ],\n   "requiere": [\n    "CFS_motor",\n    "CFS_motor_s23"\n   ],\n   "salidas": [\n    "CFS_motor_4s/v24.parquet",\n    "CFS_motor_4s/prediccion_2025.parquet"\n   ],\n   "verificar": [\n    "CFS_motor_4s/prediccion_2025.parquet"\n   ],\n   "objetivos": [\n    "S22"\n   ],\n   "evidencia": "finalizar_s15.sh",\n   "nota": "solo hace falta su resumen.json, que severidad_cfs.py lee como referencia; su V24 hereda la V24 en float64 de CFS_motor y no se compara",\n   "minutos_ref": 1\n  },\n  {\n   "id": "CFSS_motor",\n   "etapa": "severidad",\n   "linea": "principal",\n   "script": "src/modelos/severidad_cfs.py",\n   "args": [\n    "CFS_motor_4s",\n    "motor",\n    "CFSS_motor",\n    "--v23=CFS_motor_v23",\n    "--v24=CFS_motor_s23",\n    "--prod=CFS_motor,CFS_motor_s23"\n   ],\n   "requiere": [\n    "CFS_motor_4s",\n    "CFS_motor_v23"\n   ],\n   "salidas": [\n    "sel_CFSS_motor.parquet",\n    "CFSS_motor/v24.parquet",\n    "CFSS_motor/prediccion_2025.parquet"\n   ],\n   "verificar": [\n    "sel_CFSS_motor.parquet",\n    "CFSS_motor/v24.parquet",\n    "CFSS_motor/prediccion_2025.parquet"\n   ],\n   "objetivos": [\n    "S22"\n   ],\n   "evidencia": "resumen.json de CFSS_motor (origen CFS_motor_4s); misma orden que CFSSr_motor de cola45.sh sin --tope",\n   "nota": "",\n   "minutos_ref": 1\n  },\n  {\n   "id": "CFSS4_buildings",\n   "etapa": "severidad",\n   "linea": "principal",\n   "script": "src/modelos/severidad_cfs.py",\n   "args": [\n    "CFS_buildings",\n    "buildings",\n    "CFSS4_buildings",\n    "--v24=CFS_buildings,CFS_buildings_s23",\n    "--prod=CFS_buildings,CFS_buildings_s23"\n   ],\n   "requiere": [\n    "CFS_buildings_s23"\n   ],\n   "salidas": [\n    "sel_CFSS4_buildings.parquet",\n    "CFSS4_buildings/v24.parquet",\n    "CFSS4_buildings/prediccion_2025.parquet"\n   ],\n   "verificar": [\n    "sel_CFSS4_buildings.parquet",\n    "CFSS4_buildings/v24.parquet",\n    "CFSS4_buildings/prediccion_2025.parquet"\n   ],\n   "objetivos": [\n    "S22"\n   ],\n   "evidencia": "cola42.sh",\n   "nota": "",\n   "minutos_ref": 1\n  },\n  {\n   "id": "FSAS4_buildings",\n   "etapa": "severidad",\n   "linea": "principal",\n   "script": "src/modelos/severidad_cfs.py",\n   "args": [\n    "FSAP_buildings",\n    "buildings",\n    "FSAS4_buildings",\n    "--v24=FSAP_buildings,FSAP2_buildings",\n    "--prod=FSAP_buildings,FSAP2_buildings"\n   ],\n   "requiere": [\n    "FSAP_buildings",\n    "FSAP2_buildings"\n   ],\n   "salidas": [\n    "sel_FSAS4_buildings.parquet",\n    "FSAS4_buildings/v24.parquet",\n    "FSAS4_buildings/prediccion_2025.parquet"\n   ],\n   "verificar": [\n    "sel_FSAS4_buildings.parquet",\n    "FSAS4_buildings/v24.parquet",\n    "FSAS4_buildings/prediccion_2025.parquet"\n   ],\n   "objetivos": [\n    "S22"\n   ],\n   "evidencia": "cola40.sh",\n   "nota": "",\n   "minutos_ref": 1\n  },\n  {\n   "id": "MLP3_contents",\n   "etapa": "produccion",\n   "linea": "principal",\n   "script": "src/modelos/mlp_tweedie.py",\n   "args": [\n    "MLP3_contents",\n    "contents",\n    "--semillas=3",\n    "--v24",\n    "--prod",\n    "--wd=0.0001",\n    "--f3s2"\n   ],\n   "requiere": [\n    "variables"\n   ],\n   "salidas": [\n    "sel_MLP3_contents.parquet",\n    "MLP3_contents/v24.parquet",\n    "MLP3_contents/prediccion_2025.parquet"\n   ],\n   "verificar": [\n    "sel_MLP3_contents.parquet",\n    "MLP3_contents/v24.parquet",\n    "MLP3_contents/prediccion_2025.parquet"\n   ],\n   "objetivos": [\n    "S22"\n   ],\n   "evidencia": "cola21.sh, log_mlp3_contents.txt",\n   "nota": "red neuronal con torch en CPU; solo entra en S22 (peso 0,1)",\n   "minutos_ref": 3\n  },\n  {\n   "id": "B_CFST_motor_tendencia",\n   "etapa": "linea_b",\n   "linea": "B",\n   "script": "src/linea_b/severidad_tendencia.py",\n   "args": [\n    "motor",\n    "B_CFST_motor_tendencia",\n    "--regla=tendencia",\n    "--v23=CFS_motor_v23",\n    "--v24=CFS_motor_s23",\n    "--prod=CFS_motor,CFS_motor_s23"\n   ],\n   "requiere": [\n    "CFS_motor",\n    "CFS_motor_s23",\n    "CFS_motor_v23"\n   ],\n   "salidas": [\n    "sel_B_CFST_motor_tendencia.parquet",\n    "B_CFST_motor_tendencia/v24.parquet",\n    "B_CFST_motor_tendencia/prediccion_2025.parquet"\n   ],\n   "verificar": [\n    "sel_B_CFST_motor_tendencia.parquet",\n    "B_CFST_motor_tendencia/v24.parquet",\n    "B_CFST_motor_tendencia/prediccion_2025.parquet"\n   ],\n   "objetivos": [\n    "S30"\n   ],\n   "evidencia": "resumen.json de B_CFST_motor_tendencia (regla tendencia; v23 CFS_motor_v23; v24 CFS_motor_s23; prod CFS_motor,CFS_motor_s23); Modelos_claude_b/README.md §3",\n   "nota": "",\n   "minutos_ref": 1\n  },\n  {\n   "id": "B_CFST_buildings_tendencia",\n   "etapa": "linea_b",\n   "linea": "B",\n   "script": "src/linea_b/severidad_tendencia.py",\n   "args": [\n    "buildings",\n    "B_CFST_buildings_tendencia",\n    "--regla=tendencia",\n    "--v23=CFS_buildings",\n    "--v24=CFS_buildings,CFS_buildings_s23",\n    "--prod=CFS_buildings,CFS_buildings_s23"\n   ],\n   "requiere": [\n    "CFS_buildings",\n    "CFS_buildings_s23"\n   ],\n   "salidas": [\n    "sel_B_CFST_buildings_tendencia.parquet",\n    "B_CFST_buildings_tendencia/v24.parquet",\n    "B_CFST_buildings_tendencia/prediccion_2025.parquet"\n   ],\n   "verificar": [\n    "sel_B_CFST_buildings_tendencia.parquet",\n    "B_CFST_buildings_tendencia/v24.parquet",\n    "B_CFST_buildings_tendencia/prediccion_2025.parquet"\n   ],\n   "objetivos": [\n    "S30"\n   ],\n   "evidencia": "resumen.json de B_CFST_buildings_tendencia (regla tendencia; v23 CFS_buildings; v24 CFS_buildings,CFS_buildings_s23; prod CFS_buildings,CFS_buildings_s23); Modelos_claude_b/README.md §3",\n   "nota": "",\n   "minutos_ref": 1\n  },\n  {\n   "id": "B_FSAST_buildings_tendencia",\n   "etapa": "linea_b",\n   "linea": "B",\n   "script": "src/linea_b/severidad_tendencia.py",\n   "args": [\n    "buildings",\n    "B_FSAST_buildings_tendencia",\n    "--regla=tendencia",\n    "--v23=FSAP_buildings",\n    "--v24=FSAP_buildings,FSAP2_buildings",\n    "--prod=FSAP_buildings,FSAP2_buildings"\n   ],\n   "requiere": [\n    "FSAP2_buildings",\n    "FSAP_buildings"\n   ],\n   "salidas": [\n    "sel_B_FSAST_buildings_tendencia.parquet",\n    "B_FSAST_buildings_tendencia/v24.parquet",\n    "B_FSAST_buildings_tendencia/prediccion_2025.parquet"\n   ],\n   "verificar": [\n    "sel_B_FSAST_buildings_tendencia.parquet",\n    "B_FSAST_buildings_tendencia/v24.parquet",\n    "B_FSAST_buildings_tendencia/prediccion_2025.parquet"\n   ],\n   "objetivos": [\n    "S30"\n   ],\n   "evidencia": "resumen.json de B_FSAST_buildings_tendencia (regla tendencia; v23 FSAP_buildings; v24 FSAP_buildings,FSAP2_buildings; prod FSAP_buildings,FSAP2_buildings); Modelos_claude_b/README.md §3",\n   "nota": "",\n   "minutos_ref": 1\n  },\n  {\n   "id": "S22_claude_propio",\n   "etapa": "mezcla",\n   "linea": "principal",\n   "script": "src/modelos/mezcla3.py",\n   "args": [\n    "S22_claude_propio",\n    "receta/especificaciones/spec_s22_propio.json",\n    "--forma=motor:1.09:0.95",\n    "--nivel=s17",\n    "--escribir"\n   ],\n   "requiere": [\n    "sel_total_F0124",\n    "S04b_lgbm_F0124_F3autos",\n    "CAT_motor_4s",\n    "sel_cat1a",\n    "CFSS_motor",\n    "sel_total_F01234",\n    "S06c_lgbm_motor_red",\n    "sel_total_motor_red",\n    "S12h_edif_md1000",\n    "sel_hS3",\n    "CFSS4_buildings",\n    "FSAS4_buildings",\n    "S08h_hogar_F3s2",\n    "sel_hS2",\n    "MLP3_contents",\n    "FSA_contents",\n    "FSAXc4_contents"\n   ],\n   "salidas": [\n    "S22_claude_propio/S22_claude_propio.csv",\n    "S22_claude_propio/prediccion_2025.parquet"\n   ],\n   "verificar": [\n    "S22_claude_propio/S22_claude_propio.csv",\n    "S22_claude_propio/prediccion_2025.parquet"\n   ],\n   "objetivos": [\n    "S22"\n   ],\n   "evidencia": "finalizar_s22.sh; D86",\n   "nota": "SHA-256 entregado c21a02e14de4a89f892604239b7cfd9ac4e95ea7d175460a25ba806a5a3541b0 (Kaggle 57370117, público 0,12988)",\n   "minutos_ref": 1\n  },\n  {\n   "id": "S30_claude_b",\n   "etapa": "mezcla",\n   "linea": "B",\n   "script": "src/linea_b/mezcla3_b.py",\n   "args": [\n    "S30_claude_b",\n    "receta/especificaciones/spec_s30.json",\n    "--forma=motor:1.09:0.95,contents:1:0.95",\n    "--nivel=s17",\n    "--escribir"\n   ],\n   "requiere": [\n    "sel_total_F0124",\n    "S04b_lgbm_F0124_F3autos",\n    "CAT_motor_4s",\n    "sel_cat1a",\n    "B_CFST_motor_tendencia",\n    "sel_total_F01234",\n    "S06c_lgbm_motor_red",\n    "sel_total_motor_red",\n    "S12h_edif_md1000",\n    "sel_hS3",\n    "B_CFST_buildings_tendencia",\n    "B_FSAST_buildings_tendencia",\n    "S08h_hogar_F3s2",\n    "sel_hS2",\n    "FSA_contents",\n    "FSAXc4_contents"\n   ],\n   "salidas": [\n    "S30_claude_b/S30_claude_b.csv",\n    "S30_claude_b/prediccion_2025.parquet"\n   ],\n   "verificar": [\n    "S30_claude_b/S30_claude_b.csv",\n    "S30_claude_b/prediccion_2025.parquet"\n   ],\n   "objetivos": [\n    "S30"\n   ],\n   "evidencia": "Modelos_claude_b/README.md («Candidato S30_claude_b»); reproducir_s30_b.py; D88 y D89",\n   "nota": "SHA-256 entregado fd9528e0b1ee4b674dea63834a97523fd4ddc743d1495a422ce31a2eb76e3bc5 (Kaggle 57371361, público 0,12996); selección final de la parte 1",\n   "minutos_ref": 1\n  }\n ],\n "etapas_datos": {\n  "descripcion": "Etapas 00 a 03, de los once Parquet originales a las variables. reproducir.py datos las ejecuta en una raíz de proyecto mínima dentro de --trabajo, porque bases_actuariales.py y construir_f0.py localizan sus entradas subiendo hasta RUTAS_PROYECTO.json y escriben su corrida junto al script; los scripts se copian allí sin cambios y se comprueba su SHA-256. Las corridas nuevas se enlazan con los nombres que espera comun.py (f0_20261017T141732Z_65df7fca y bases_20261017T120251Z_45443ff0).",\n  "pasos": [\n   {\n    "id": "capa",\n    "script": "src/datos/n62_codex/rebuild_base.py",\n    "args": [\n     "--raw",\n     "<originales>",\n     "--rules",\n     "src/datos/n62_codex/rules.json",\n     "--out",\n     "<trabajo>/proyecto/Hackaton/Semana 1/Limpieza/runs/preaudit_20261016T211816Z_817285a5"\n    ],\n    "autor": "Codex (N62, COD-20261018-104)",\n    "verifica": "SHA-256 de las once tablas igual al de la capa preaudit"\n   },\n   {\n    "id": "bases",\n    "script": "src/datos/bases_actuariales.py",\n    "args": [],\n    "autor": "Claude (D67)",\n    "verifica": "huella de contenido de t0_por_row_id y asignacion_por_siniestro; bytes de las tablas de desarrollo"\n   },\n   {\n    "id": "f0",\n    "script": "src/datos/construir_f0.py",\n    "args": [\n     "--ramo motor",\n     "--run <id> --ramo buildings",\n     "--run <id> --ramo contents",\n     "--run <id> --finalizar"\n    ],\n    "autor": "Claude (D68)",\n    "verifica": "huella de contenido de las tres matrices y de las particiones; bloques del contrato de disponibilidad",\n    "decisiones_congeladas": "tres listas de row_id del proyecto que solo alimentan marcas audit_* (excluidas de los predictores), leídas de --decisiones y comprobadas por SHA-256"\n   },\n   {\n    "id": "variables",\n    "script": "src/modelos/construir_variables.py",\n    "args": [],\n    "autor": "Claude",\n    "verifica": "huella de contenido de feat_<ramo>.parquet"\n   }\n  ],\n  "prueba_19_oct": "En el entorno de referencia: once tablas con el SHA de preaudit; bases y F0 iguales en contenido; variables iguales en contenido y marcos de modelo (matriz X con tipos y categorías, etiquetas, offsets y particiones) idénticos a los de las variables originales. Los bytes de F0 y de las variables cambian porque pandas 3 guarda las fechas de periodo en segundos y la VM las guardó en nanosegundos."\n }\n}\n', 'src/datos/n62_codex/rules.json': '{\n  "version": "N62_frozen_preaudit_v1",\n  "originals_sha256": {\n    "addresses.parquet": "2f1a9a157649a73ed1b0ee9b2caa3660b4c2a11766af3c64ab5131389079c0b1",\n    "claims.parquet": "323adf2e0f6c5975a026479d725d4cd04f51fce3aa069f897a99f664c6e392c0",\n    "customers.parquet": "065458a0b3aa9d3499e80b3b25b76baf0ef6245146477b6861f6ea35edf581eb",\n    "policies_buildings.parquet": "eb84372c23c16ed0619d89c8eb1396ab6916e5ec46a5907ce42c3dfce4a91cbe",\n    "policies_contents.parquet": "8c067cc80baed583100df999e1c3d5e389295702f7bac21d5331bbd639b83fcd",\n    "policies_motor.parquet": "4d2d7ae03971d5d6df3a71bbaf0ebc168250a6807b27d1561d38b638434dc95e",\n    "sample_submission.parquet": "9529f3b6a679d3410350baa7c2ef3fb085d1d8843a838360c5852dfe820d4d95",\n    "test_buildings.parquet": "7b2b96507de29241c70617c17df66146ed30b51f826ce5219bbda5da536d4957",\n    "test_contents.parquet": "c2212f62f3b7a21431c0073277549c3db5e63857565a8b88855e79a65fa1b8b6",\n    "test_motor.parquet": "8d28fb9fd091caf2a044913a013592226a4c0917d5c6fcc9c5e8cdcec2ab4fc5",\n    "weather.parquet": "b8407f0f3c7279a49297c969a4ab2905641b232910e316fb806730065987cd49"\n  },\n  "ID01": {\n    "pattern": "[0-9]{1,8}",\n    "width": 8\n  },\n  "EXP01": {\n    "inclusive_days": true,\n    "denominator": 365,\n    "decimals": 4\n  },\n  "CAP01": {\n    "thresholds": {\n      "buildings": 10000000,\n      "contents": 1000000\n    },\n    "scale": 1000,\n    "context_fields": {\n      "buildings": [\n        "customer_id",\n        "address_id",\n        "construction_type",\n        "occupancy",\n        "accidental_damage_cover",\n        "voluntary_excess",\n        "compulsory_excess",\n        "smoke_alarms",\n        "mortgage"\n      ],\n      "contents": [\n        "customer_id",\n        "address_id",\n        "tenure",\n        "security_level",\n        "high_value_items",\n        "accidental_damage_cover",\n        "works_from_home",\n        "lodgers",\n        "bicycle_cover",\n        "voluntary_excess",\n        "compulsory_excess"\n      ]\n    },\n    "approved_sets": {\n      "policies_buildings": {\n        "count": 802,\n        "row_ids_sha256": "b262deab1c0cb371635c1b042032708b2ba42324e9c706e6c43e4fdc7cb95c20"\n      },\n      "policies_contents": {\n        "count": 731,\n        "row_ids_sha256": "af19838c5321bc453d20c733ca9c60a91076f2cef8def69bd54bf2039014e8f0"\n      },\n      "test_buildings": {\n        "count": 272,\n        "row_ids_sha256": "9f8490d0fb51aeb9b311e3357d11b00d5086ff3fe28a14b410198af545144ce6"\n      },\n      "test_contents": {\n        "count": 268,\n        "row_ids_sha256": "0bed421cb45f521b29b90be0d2850051e9b2b414416e3d97924c95ca71596b15"\n      }\n    },\n    "selection": "Most recent strictly earlier non-extreme original anchor; complete same context; start gap1..400; divided capital ratio0.9..1.1. Never reselect after correcting."\n  },\n  "CAT01": {\n    "map": {\n      "Brick": "brick",\n      "BRICK": "brick",\n      "NON_STANDARD": "non_standard",\n      "STONE": "stone",\n      "T": "t",\n      "W": "w"\n    }\n  },\n  "AGE01": {\n    "calendar_completed": true,\n    "feb29_on_feb28_nonleap": "abort",\n    "maximum_source_difference": 1\n  },\n  "DUP01": {\n    "count": 604,\n    "pairs_sha256": "ad61daaa2e217bbf3259932a72cfae5ec8c9931dc53c30294f414e5178396689"\n  },\n  "REN00": {\n    "cohort_source_file_sha256": "4e68ecadb8a00976a1601b5067ce73c87a43c74069cb542c1cafbdf8d9af764f",\n    "approved_sets": {\n      "buildings": {\n        "count": 1010,\n        "policy_ids_sha256": "1b68021fc8111739d1bece6eafe3f6acc8bc1dd54098c83bf5a9f5ec6f9efcdf"\n      },\n      "contents": {\n        "count": 219,\n        "policy_ids_sha256": "64da56e61f813b75b58eb3e08451dee91b0d7c9359d853bb90372eef7b604a94"\n      },\n      "motor": {\n        "count": 1513,\n        "policy_ids_sha256": "049e7b3480b1f35612007d3e580bfe3544991d164334fcebe6c5ab9bc12dfc84"\n      }\n    }\n  },\n  "PRE00": {\n    "cohort_source_file_sha256": "90d1cba5de82320aae81966aa1d248446cb2c758d6880378bbde855a4d1226bf",\n    "count": 413,\n    "claim_ids_sha256": "f31e2e94e7bb9d9bca6f50929be016275be72848eb6a2afa33976ef7f14d471c"\n  }\n}\n'}


### Inspect the dependency graph

Each row is a stage needed for S30. The `requires` column shows which earlier stages supply its inputs. This graph is frozen; inspecting it does not train a model. The data-building stages precede the first row.


In [ ]:
recipe = json.loads(DATAFILES["receta/receta.json"])
nodes = {node["id"]: node for node in recipe["nodos"]}
needed, stack = set(), [recipe["objetivos"]["S30"]["nodo"]]
while stack:
    current = stack.pop()
    if current not in needed:
        needed.add(current)
        stack.extend(nodes[current]["requiere"])
kinds = {"variables": "features", "seleccion_v23": "2023 selection", "produccion": "production", "partes": "peril components", "severidad": "severity", "linea_b": "trend-adjusted severity", "mezcla": "blend"}
graph = [{"stage": node["id"], "kind": kinds[node["etapa"]], "requires": ", ".join(node["requiere"]), "source": node["script"]} for node in recipe["nodos"] if node["id"] in needed]
print(f"Stages in the S30 graph: {len(graph)}")
display(pd.DataFrame(graph))


In [ ]:
for rel, text in {**SOURCES, **DATAFILES}.items():
    dest = PACKAGE / rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    if rel.endswith(".py"): compile(text, str(dest), "exec")
    dest.write_text(text, encoding="utf-8", newline="\n")
work = RUN / "run"
work_dir = work / "work"
work_dir.mkdir(parents=True, exist_ok=True)
config = PACKAGE / "receta/especificaciones/cfg_total.json"
work_config = work_dir / "cfg_total.json"
if work_config.exists() and work_config.read_bytes() != config.read_bytes():
    raise RuntimeError(f"Existing configuration differs: {work_config}")
if not work_config.exists():
    shutil.copyfile(config, work_config)
cmd = [sys.executable, "-X", "utf8", str(PACKAGE / "reproducir.py"), "todo", "--objetivo=S30", f"--semana1={developed.parent}", f"--originales={developed}", f"--trabajo={work}", "--continuar-si-difiere"]
print("Starting S30. Logs and receipts are in", work)
result = subprocess.run(cmd, cwd=RUN, env=dict(os.environ, PYTHONUTF8="1", PYTHONHASHSEED="0", OMP_NUM_THREADS="2", OPENBLAS_NUM_THREADS="2", MKL_NUM_THREADS="2"))
if result.returncode not in (0, 3): raise RuntimeError(f"S30 run failed with code {result.returncode}; inspect {work / 'registros'}")
print("Runner return code:", result.returncode)


## Apply the S41 level factors and check the output

The three exact factors below are frozen from the historical credibility rule. I apply them to the newly generated six-decimal S30 CSV, preserving the template order. A hash comparison checks the final file against the historical S41 receipt.


In [ ]:
factors = {'motor': 0.9962490564279887, 'buildings': 1.0259995860825672, 'contents': 1.0126005668944555}
s30 = work / "work/S30_claude_b/S30_claude_b.csv"
pred = pd.read_csv(s30, dtype={"row_id": str})
line_lookup = pd.concat([pd.read_parquet(developed / f"test_{line}.parquet", columns=["row_id"]).assign(line=line) for line in LINES]).set_index("row_id").line
labels = line_lookup.reindex(pred.row_id).to_numpy()
assert pd.notna(labels).all()
for line, factor in factors.items():
    pred.loc[labels == line, "prediction"] *= factor
expected_ids = pd.read_parquet(developed / "sample_submission.parquet", columns=["row_id"]).row_id.astype(str)
assert list(pred.columns) == ["row_id", "prediction"] and pred.row_id.is_unique and pred.row_id.equals(expected_ids)
assert np.isfinite(pred.prediction).all() and (pred.prediction > 0).all()
out = RUN / "submission_week1_S41_candidate.csv"
pred.to_csv(out, index=False, float_format="%.6f", lineterminator="\n")
h = hashlib.sha256(out.read_bytes()).hexdigest()
expected_sha = "d8661c2874e9112bd3dd62ea4470a465ee309832dee53fd9dcaa4144dac643a5"
print("S41 SHA-256:", h, "matches historical:", h == expected_sha)
if h != expected_sha:
    raise RuntimeError(f"S41 differs from the historical CSV. Preserve and investigate {out} before treating this as an exact reproduction.")
final_output = EXPORT / "submission_week1_S41.csv"
shutil.copyfile(out, final_output)
print("Output:", final_output)


## Outputs and troubleshooting

After the final checks pass, I find `submission_week1_S41.csv` in the `EXPORT` directory printed near the beginning. Without an override, Kaggle exports to a run-specific subdirectory of `/kaggle/working`; local execution exports beneath the new run directory. I can set `IAA_NOTEBOOK_EXPORT` before starting to choose a persistent private destination. I keep row-level outputs private.

| If execution stops | What I check |
| --- | --- |
| Missing or ambiguous input directory | The explicit market path, required filenames and attached competition data |
| Missing library or version mismatch | The optional installation cell, kernel restart and reference environment |
| Failed model stage | The stage's log and receipt under the printed run directory |
| Final fingerprint mismatch | Environment and generated artifacts; I retain the file and investigate the difference |

For Week 1, return code 3 means that the historical final CSV matched while intermediate artifacts differed. I preserve and inspect the difference report. Within the same session the runner can reuse completed nodes after its hash checks; a new notebook session creates a new run folder.

I do not bypass a failed fingerprint check or interpret a resumed run as an uninterrupted execution. A schema-valid CSV and an exact historical file match answer different questions.


## License and materials

My project code and documentation are covered by the MIT License below. Competition data and third-party packages retain their own terms and are not included in that license.

MIT License

Copyright (c) 2026 Manuel Rodríguez

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.
